# Border Surveillance Research — Consolidated Source Code

This notebook consolidates the **source code cells** from the five uploaded original notebooks.

## Important provenance and safety notes

- Each source notebook is kept in its own section, and its code-cell order is preserved.
- Code is copied as source; this is **not yet a refactor or a validated runnable pipeline**.
- Cell outputs, execution counts, and saved visualizations are intentionally omitted to keep this notebook source-focused.
- Common embedded credential patterns were redacted. Use Kaggle Secrets or environment variables for credentials.
- Original uploaded notebooks should remain unchanged as the historical record.
- Some cells may depend on Kaggle paths, datasets, checkpoints, installed packages, or earlier cells. Run sections only after reviewing those dependencies.
- Duplicate, obsolete, exploratory, and debugging code has **not** been removed yet. We will identify those during the repository-cleaning pass.



# Source notebook: `Framework_SeniorProject.ipynb`

Original notebook cells: 98. Non-empty source-code cells included: 98. The original code-cell order is preserved below.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q kaggle

from google.colab import files
uploaded = files.upload()

In [ ]:
import os
import json

# Create Kaggle configuration directory
os.makedirs('/root/.kaggle', exist_ok=True)

# Save API credentials
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])

# Secure credentials
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("Kaggle API configured successfully!")

In [ ]:
!pip install -q --upgrade kaggle
!kaggle --version

In [ ]:
!kaggle datasets list -s "LLVIP"

In [ ]:
!kaggle datasets files afradhossain/llvip-dataset

In [ ]:
!kaggle datasets files --help

In [ ]:
!kaggle datasets files afradhossain/llvip-dataset \
    --page-token "CfDJ8CXkJvnxKG5Ktgac76JjV5Hw7ov2B4d_0dx-8z_ayiwmaC82CvRwsfWSPdFukEa8MMRVvxv5iapLyO8Bnx0yVirNM9tELmllT4Sz24lLs-KcKOLEXSbRKi3IlDCLXc1Ev2aPFmS3tW4CXaEQie6ntGzZMKem" \
    --page-size 200

In [ ]:
!kaggle datasets files afradhossain/llvip-dataset \
    --page-token "CfDJ8CXkJvnxKG5Ktgac76JjV5FQC6A-TOd4XWDM2DjSvJCdXWDlJKNJljs9EfQtExP-_168JgZV7CvXA2Eu3S5O9ACc-lTQHq6p78fFlj25PR3lWGg0E5etnwxh1JeMg3zg26iVPYNfrH7QQbmNYDIqLY3iEGxw" \
    --page-size 200

In [ ]:
!kaggle datasets files afradhossain/llvip-dataset \
    --page-token "CfDJ8CXkJvnxKG5Ktgac76JjV5FnIvdacUQBj3IZ-9Ln8jSGXTuBC5pu6nvD0M4LUb2pRrPew1pi6RRRKUuUdYCsxwPbUWXpsj4LK8EwI1F5g6nmnS1ZhkSWdHmXtTb_8wdEMQ5qwQF_thTxv0FGbYIkyC7E31lB" \
    --page-size 200

In [ ]:
!kaggle datasets download -d afradhossain/llvip-dataset \
    -p /content --unzip

In [ ]:
!ls -lah /content/LLVIP

In [ ]:
from pathlib import Path

ROOT = Path("/content/LLVIP")

for folder in ["visible", "infrared", "Annotations"]:
    path = ROOT / folder
    print(f"\n{folder}:")
    if path.exists():
        for item in sorted(path.iterdir()):
            if item.is_dir():
                print(f"  {item.name}: {len(list(item.iterdir()))} files")
            else:
                print(f"  {item.name}")
    else:
        print("  NOT FOUND")


In [ ]:
import subprocess
import json
import re

dataset = "afradhossain/llvip-dataset"
page_token = None
image_files = []
annotation_count = 0
page_count = 0

while True:
    cmd = ["kaggle", "datasets", "files", dataset, "--page-size", "200"]
    if page_token:
        cmd += ["--page-token", page_token]

    result = subprocess.run(cmd, capture_output=True, text=True)
    output = result.stdout

    if result.returncode != 0:
        print("Error:", result.stderr)
        break

    page_count += 1

    for line in output.splitlines():
        filename = line.strip().split()[0] if line.strip() else ""
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            image_files.append(filename)
        elif filename.lower().endswith(".xml"):
            annotation_count += 1

    match = re.search(r"Next Page Token\s*=\s*(\S+)", output)
    if not match:
        break

    page_token = match.group(1)

print("Pages checked:", page_count)
print("XML annotations found:", annotation_count)
print("Image files found:", len(image_files))
print("\nSample image paths:")
for path in image_files[:20]:
    print(path)

In [ ]:
!kaggle datasets download -d afradhossain/llvip-dataset -p /content

In [ ]:
from pathlib import Path

ROOT = Path("/content/LLVIP")

for folder in [
    "visible/train",
    "visible/test",
    "infrared/train",
    "infrared/test",
    "Annotations"
]:
    path = ROOT / folder
    print(folder, "EXISTS" if path.exists() else "MISSING")
    if path.exists():
        print("  Files:", len(list(path.iterdir())))

In [ ]:
from pathlib import Path

ROOT = Path("/content/LLVIP")

def get_stems(folder, extension=None):
    files = Path(folder).iterdir()
    return {
        f.stem for f in files
        if f.is_file() and (
            extension is None or f.suffix.lower() == extension
        )
    }

visible_train = get_stems(ROOT / "visible/train", ".jpg")
infrared_train = get_stems(ROOT / "infrared/train", ".jpg")
visible_test = get_stems(ROOT / "visible/test", ".jpg")
infrared_test = get_stems(ROOT / "infrared/test", ".jpg")
annotations = get_stems(ROOT / "Annotations", ".xml")

print("TRAIN")
print("Visible:", len(visible_train))
print("Infrared:", len(infrared_train))
print("Matching pairs:", len(visible_train & infrared_train))
print("Missing infrared:", len(visible_train - infrared_train))

print("\nTEST")
print("Visible:", len(visible_test))
print("Infrared:", len(infrared_test))
print("Matching pairs:", len(visible_test & infrared_test))
print("Missing infrared:", len(visible_test - infrared_test))

all_images = visible_train | visible_test
print("\nANNOTATIONS")
print("XML files:", len(annotations))
print("Images with annotations:", len(all_images & annotations))
print("Images without annotations:", len(all_images - annotations))

In [ ]:
import random

SEED = 0
N_TRAIN = 4000
N_VAL = 500

random.seed(SEED)

all_train = sorted(visible_train)

assert len(all_train) >= N_TRAIN + N_VAL

random.shuffle(all_train)

train_names = all_train[:N_TRAIN]
val_names = all_train[N_TRAIN:N_TRAIN + N_VAL]
test_names = sorted(visible_test)

print("Training:", len(train_names))
print("Validation:", len(val_names))
print("Testing:", len(test_names))

print("Train/validation overlap:",
      len(set(train_names) & set(val_names)))

print("Train/test overlap:",
      len(set(train_names) & set(test_names)))

print("Validation/test overlap:",
      len(set(val_names) & set(test_names)))

In [ ]:
!pip -q install ultralytics ensemble-boxes pycocotools

In [ ]:
# Experiment 1 — LLVIP RGB / thermal / early-fusion / late-fusion comparison
# Corrected Colab script.
#
# Dataset expected at:
#   /content/LLVIP/
#       visible/{train,test}/
#       infrared/{train,test}/
#       Annotations/*.xml
#
# Method:
#   - 4,000 training images sampled from the official LLVIP train split
#   - 500 mutually exclusive validation images sampled from the remaining
#     official LLVIP train split
#   - all 3,463 official test images
#   - same YOLO model, seed, image size, epochs, and batch size per modality
#
# Important correction:
#   Validation images are sourced from ROOT/visible/train and
#   ROOT/infrared/train, but written to the separate WORK/.../val folders.
#   The original script incorrectly tried to read ROOT/.../val, which does
#   not exist in this LLVIP layout, and could also overlap train/val names.

# %% 0. Install dependencies (run once in a Colab cell if needed)
# !pip -q install ultralytics ensemble-boxes pycocotools

# %% 1. Configuration and imports
import os
import json
import random
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

ROOT = Path("/content/LLVIP")
WORK = Path("/content/work_exp1_corrected")

N_TRAIN = 4000
N_VAL = 500
EPOCHS = 20
IMGSZ = 640
BATCH = 16
SEED = 0
MODEL = "yolo11n.pt"

# Ground-truth box area bins at original-image pixel scale.
SIZE_BINS = [
    (0, 16**2),
    (16**2, 32**2),
    (32**2, 64**2),
    (64**2, 1e10),
]
SIZE_BIN_LABELS = ["<16^2", "16-32^2", "32-64^2", ">64^2"]

VARIANTS = ["rgb", "thermal", "blend", "vt"]

random.seed(SEED)
np.random.seed(SEED)

assert ROOT.exists(), f"LLVIP root not found: {ROOT}"

# %% 2. Verify dataset structure, filenames, and annotation coverage
def jpg_stems(folder):
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"Missing folder: {folder}")
    return {p.stem for p in folder.glob("*.jpg") if p.is_file()}

visible_train = jpg_stems(ROOT / "visible" / "train")
infrared_train = jpg_stems(ROOT / "infrared" / "train")
visible_test = jpg_stems(ROOT / "visible" / "test")
infrared_test = jpg_stems(ROOT / "infrared" / "test")
annotation_stems = {
    p.stem for p in (ROOT / "Annotations").glob("*.xml") if p.is_file()
}

assert visible_train == infrared_train, (
    "Training visible/infrared filenames do not match. "
    f"Visible-only: {len(visible_train - infrared_train)}, "
    f"IR-only: {len(infrared_train - visible_train)}"
)
assert visible_test == infrared_test, (
    "Test visible/infrared filenames do not match. "
    f"Visible-only: {len(visible_test - infrared_test)}, "
    f"IR-only: {len(infrared_test - visible_test)}"
)

all_image_stems = visible_train | visible_test
missing_xml = all_image_stems - annotation_stems
assert not missing_xml, (
    f"{len(missing_xml)} images have no XML annotation. "
    f"Examples: {sorted(missing_xml)[:10]}"
)

print("Dataset verification passed.")
print(f"Train pairs: {len(visible_train)}")
print(f"Test pairs: {len(visible_test)}")
print(f"Images with XML annotations: {len(all_image_stems)}")

# %% 3. Reproducible, mutually exclusive train/validation/test split
# This reproduces the split already verified in the Colab session:
# sorted filenames -> shuffle with seed 0 -> first 4,000 train, next 500 val.
all_train_names = sorted(visible_train)
all_test_names = sorted(visible_test)

assert len(all_train_names) >= N_TRAIN + N_VAL, (
    f"Need at least {N_TRAIN + N_VAL} train images; "
    f"found {len(all_train_names)}"
)

split_rng = random.Random(SEED)
shuffled_train_names = all_train_names.copy()
split_rng.shuffle(shuffled_train_names)

train_names = shuffled_train_names[:N_TRAIN]
val_names = shuffled_train_names[N_TRAIN:N_TRAIN + N_VAL]
test_names = all_test_names

assert len(train_names) == N_TRAIN
assert len(val_names) == N_VAL
assert len(set(train_names) & set(val_names)) == 0
assert len(set(train_names) & set(test_names)) == 0
assert len(set(val_names) & set(test_names)) == 0

print("\nSplit verification:")
print("Training:", len(train_names))
print("Validation:", len(val_names))
print("Official test:", len(test_names))
print("Train/validation overlap:", len(set(train_names) & set(val_names)))
print("Train/test overlap:", len(set(train_names) & set(test_names)))
print("Validation/test overlap:", len(set(val_names) & set(test_names)))

# Save exact split membership for reproducibility.
WORK.mkdir(parents=True, exist_ok=True)
with open(WORK / "split_manifest.json", "w") as f:
    json.dump(
        {
            "seed": SEED,
            "n_train": len(train_names),
            "n_val": len(val_names),
            "n_test": len(test_names),
            "train": train_names,
            "val": val_names,
            "test": test_names,
        },
        f,
        indent=2,
    )

# %% 4. Read VOC annotations and convert to YOLO labels
def read_boxes(xml_path):
    root = ET.parse(str(xml_path)).getroot()
    size_node = root.find("size")
    if size_node is None:
        raise ValueError(f"XML has no <size> element: {xml_path}")

    width_node = size_node.find("width")
    height_node = size_node.find("height")
    if width_node is None or height_node is None:
        raise ValueError(f"XML has no width/height: {xml_path}")

    width = int(float(width_node.text))
    height = int(float(height_node.text))
    if width <= 0 or height <= 0:
        raise ValueError(f"Invalid XML image dimensions in {xml_path}")

    boxes = []
    for obj in root.findall("object"):
        box_node = obj.find("bndbox")
        if box_node is None:
            continue
        coords = []
        for key in ("xmin", "ymin", "xmax", "ymax"):
            node = box_node.find(key)
            if node is None or node.text is None:
                raise ValueError(f"Malformed bounding box in {xml_path}")
            coords.append(float(node.text))

        x1, y1, x2, y2 = coords
        # Ignore malformed/degenerate boxes; keep valid annotations unchanged.
        if x2 > x1 and y2 > y1:
            boxes.append((x1, y1, x2, y2))

    return width, height, boxes


def make_variant(source_split, output_split, names, variant):
    """Read from official source split and write into model-specific split."""
    image_dir = WORK / variant / "images" / output_split
    label_dir = WORK / variant / "labels" / output_split
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    for name in names:
        visible_path = ROOT / "visible" / source_split / f"{name}.jpg"
        infrared_path = ROOT / "infrared" / source_split / f"{name}.jpg"
        xml_path = ROOT / "Annotations" / f"{name}.xml"

        rgb = cv2.imread(str(visible_path), cv2.IMREAD_COLOR)
        ir = cv2.imread(str(infrared_path), cv2.IMREAD_GRAYSCALE)
        if rgb is None:
            raise FileNotFoundError(f"Could not read visible image: {visible_path}")
        if ir is None:
            raise FileNotFoundError(f"Could not read infrared image: {infrared_path}")

        height, width = rgb.shape[:2]
        if ir.shape[:2] != (height, width):
            # LLVIP pairs are intended to be registered. Resize only to ensure
            # equal tensor dimensions; this does not independently prove alignment.
            ir = cv2.resize(ir, (width, height), interpolation=cv2.INTER_LINEAR)

        ir_bgr = cv2.merge([ir, ir, ir])

        if variant == "rgb":
            output = rgb
        elif variant == "thermal":
            output = ir_bgr
        elif variant == "blend":
            # Static pixel-level early fusion: equal-weight visible/IR blend.
            output = cv2.addWeighted(rgb, 0.5, ir_bgr, 0.5, 0)
        elif variant == "vt":
            # OpenCV images are BGR: thermal replaces the red channel.
            output = cv2.merge([rgb[:, :, 0], rgb[:, :, 1], ir])
        else:
            raise ValueError(f"Unknown variant: {variant}")

        if not cv2.imwrite(str(image_dir / f"{name}.jpg"), output):
            raise IOError(f"Could not write image: {image_dir / f'{name}.jpg'}")

        xml_width, xml_height, boxes = read_boxes(xml_path)
        if (xml_width, xml_height) != (width, height):
            raise ValueError(
                f"Image/XML dimension mismatch for {name}: "
                f"image={width}x{height}, XML={xml_width}x{xml_height}"
            )

        label_path = label_dir / f"{name}.txt"
        with open(label_path, "w") as label_file:
            for x1, y1, x2, y2 in boxes:
                # VOC coordinates are converted to normalized YOLO xywh.
                cx = ((x1 + x2) / 2.0) / width
                cy = ((y1 + y2) / 2.0) / height
                bw = (x2 - x1) / width
                bh = (y2 - y1) / height
                label_file.write(
                    f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n"
                )


# %% 5. Build model-specific datasets
# Rebuild only the prepared data folders; model outputs are stored separately.
for variant in VARIANTS:
    variant_dir = WORK / variant
    if variant_dir.exists():
        shutil.rmtree(variant_dir)

for variant in VARIANTS:
    print(f"\nPreparing {variant} dataset...")
    make_variant("train", "train", train_names, variant)
    make_variant("train", "val", val_names, variant)
    make_variant("test", "test", test_names, variant)

    yaml_path = WORK / f"{variant}.yaml"
    yaml_path.write_text(
        f"path: {WORK / variant}\n"
        "train: images/train\n"
        "val: images/val\n"
        "test: images/test\n"
        f"names: {{0: person}}\n"
    )

print("\nAll four datasets prepared.")

# %% 6. Train identical YOLO detectors
from ultralytics import YOLO

RUNS = WORK / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

for variant in VARIANTS:
    print(f"\nTraining variant: {variant}")
    model = YOLO(MODEL)
    model.train(
        data=str(WORK / f"{variant}.yaml"),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        seed=SEED,
        batch=BATCH,
        project=str(RUNS),
        name=variant,
        exist_ok=True,
        deterministic=True,
        verbose=False,
    )

print("\nTraining finished for all four variants.")

# %% 7. Predict on the official test split
from ensemble_boxes import weighted_boxes_fusion

def predict(variant, names, split="test"):
    weights_path = RUNS / variant / "weights" / "best.pt"
    if not weights_path.exists():
        raise FileNotFoundError(f"Missing trained weights: {weights_path}")

    model = YOLO(str(weights_path))
    predictions = {}

    for index, name in enumerate(names, start=1):
        image_path = WORK / variant / "images" / split / f"{name}.jpg"
        result = model.predict(
            str(image_path),
            imgsz=IMGSZ,
            conf=0.001,
            iou=0.6,
            verbose=False,
        )[0]

        height, width = result.orig_shape
        boxes = result.boxes.xyxyn.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        predictions[name] = (boxes, scores, (width, height))

        if index % 500 == 0:
            print(f"{variant}: predicted {index}/{len(names)} test images")

    return predictions


# %% 8. Ground truth and size-stratified COCO AP50 evaluation
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def gt_for(name):
    return read_boxes(ROOT / "Annotations" / f"{name}.xml")


def eval_size_bins(predictions, names, tag):
    images = []
    annotations = []
    detections = []
    annotation_id = 1

    for image_id, name in enumerate(names):
        width, height, gt_boxes = gt_for(name)
        images.append(
            {"id": image_id, "width": width, "height": height}
        )

        for x1, y1, x2, y2 in gt_boxes:
            box_width = x2 - x1
            box_height = y2 - y1
            annotations.append(
                {
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": 1,
                    "bbox": [x1, y1, box_width, box_height],
                    "area": box_width * box_height,
                    "iscrowd": 0,
                }
            )
            annotation_id += 1

        boxes, scores, _ = predictions[name]
        for (x1, y1, x2, y2), score in zip(boxes, scores):
            detections.append(
                {
                    "image_id": image_id,
                    "category_id": 1,
                    "score": float(score),
                    "bbox": [
                        float(x1 * width),
                        float(y1 * height),
                        float((x2 - x1) * width),
                        float((y2 - y1) * height),
                    ],
                }
            )

    ground_truth = COCO()
    ground_truth.dataset = {
        "images": images,
        "annotations": annotations,
        "categories": [{"id": 1, "name": "person"}],
    }
    ground_truth.createIndex()

    if detections:
        detection_results = ground_truth.loadRes(detections)
    else:
        # COCO.loadRes([]) is not supported consistently; construct empty result.
        detection_results = COCO()
        detection_results.dataset = {
            "images": images,
            "annotations": [],
            "categories": [{"id": 1, "name": "person"}],
        }
        detection_results.createIndex()

    evaluator = COCOeval(ground_truth, detection_results, "bbox")
    evaluator.params.iouThrs = np.array([0.50] + list(np.arange(0.55, 0.96, 0.05)))
    evaluator.params.areaRng = [[0, 1e10]] + [list(b) for b in SIZE_BINS]
    evaluator.params.areaRngLbl = ["all"] + SIZE_BIN_LABELS
    evaluator.params.maxDets = [1, 10, 100]
    evaluator.evaluate()
    evaluator.accumulate()

    precision = evaluator.eval["precision"]
    row = {"config": tag}

    # First IoU threshold is 0.50; last maxDets index is 100.
    for area_index, label in enumerate(evaluator.params.areaRngLbl):
        values = precision[0, :, 0, area_index, 2]
        valid = values[values > -1]
        row[f"AP50 {label}"] = (
            float(np.mean(valid)) if len(valid) else float("nan")
        )

    # Include the number of ground-truth boxes in each original-pixel-area bin.
    areas = [ann["area"] for ann in annotations]
    row["GT boxes all"] = len(areas)
    for (low, high), label in zip(SIZE_BINS, SIZE_BIN_LABELS):
        row[f"GT boxes {label}"] = sum(low <= area < high for area in areas)

    return row


# %% 9. Late fusion (weighted boxes fusion) and final results
predictions = {
    variant: predict(variant, test_names, split="test")
    for variant in VARIANTS
}


def late_fuse(pred_a, pred_b, weight_a=1.0, weight_b=1.0, iou_threshold=0.55):
    fused = {}

    for name in pred_a:
        boxes_a, scores_a, image_size = pred_a[name]
        boxes_b, scores_b, _ = pred_b[name]

        if len(boxes_a) + len(boxes_b) == 0:
            fused[name] = (np.zeros((0, 4)), np.zeros(0), image_size)
            continue

        boxes, scores, _ = weighted_boxes_fusion(
            [
                np.clip(boxes_a, 0, 1).tolist(),
                np.clip(boxes_b, 0, 1).tolist(),
            ],
            [scores_a.tolist(), scores_b.tolist()],
            [
                [0] * len(scores_a),
                [0] * len(scores_b),
            ],
            weights=[weight_a, weight_b],
            iou_thr=iou_threshold,
            skip_box_thr=0.001,
        )

        fused[name] = (
            np.asarray(boxes, dtype=float).reshape(-1, 4),
            np.asarray(scores, dtype=float),
            image_size,
        )

    return fused


predictions["late_WBF(rgb+thermal)"] = late_fuse(
    predictions["rgb"], predictions["thermal"]
)

result_rows = [
    eval_size_bins(pred, test_names, config_name)
    for config_name, pred in predictions.items()
]

results = pd.DataFrame(result_rows).round(4)
results_path = WORK / "exp1_results.csv"
results.to_csv(results_path, index=False)

print("\nFinal Experiment 1 results:")
print(results.to_string(index=False))
print(f"\nSaved results to: {results_path}")
print(f"Saved split manifest to: {WORK / 'split_manifest.json'}")

# %% 10. Optional: package the key outputs for download
# from google.colab import files
# files.download(str(results_path))
# files.download(str(WORK / "split_manifest.json"))

# Reporting notes:
# - The test metrics are computed on the official LLVIP test split.
# - Validation images are held out from the official training split.
# - The "blend" and "vt" variants are fixed input-channel/pixel fusion baselines.
# - The late-fusion result combines RGB and thermal detector predictions using WBF.
# - SIZE_BINS are based on original-image bounding-box pixel area, not width alone.
# - Report the exact split manifest, model checkpoint, and package versions for
#   reproducibility. Do not describe resizing as proof of RGB/IR registration.


In [ ]:
from google.colab import files

files.download(
    "/content/work_exp1_corrected/exp1_results.csv"
)

In [ ]:
import os
import zipfile

base = "/content/work_exp1_corrected/runs"

with zipfile.ZipFile("/content/exp1_training_logs.zip", "w") as z:
    for variant in ["rgb", "thermal", "blend", "vt"]:
        path = os.path.join(base, variant, "results.csv")
        if os.path.exists(path):
            z.write(path, arcname=f"{variant}/results.csv")

files.download("/content/exp1_training_logs.zip")

In [ ]:

from pathlib import Path

base = Path("/content/work_exp1_corrected/runs")

variants = ["rgb", "thermal", "blend", "vt"]

for name in variants:
    weights = base / name / "weights" / "best.pt"
    if weights.exists():
        print(f"FOUND: {name} — {weights.stat().st_size / (1024**2):.1f} MB")
    else:
        print(f"MISSING: {name}")


In [ ]:

print("Prediction dictionary available:", "predictions" in globals())

if "predictions" in globals():
    for name, pred in predictions.items():
        print(f"{name}: {len(pred)} images")

print("\nTest image list available:", "test_names" in globals())

if "test_names" in globals():
    print("Test images:", len(test_names))

print("\nPrepared test datasets:")
from pathlib import Path

for name in ["rgb", "thermal", "blend", "vt"]:
    folder = Path("/content/work_exp1_corrected") / name / "images" / "test"
    print(f"{name}: {len(list(folder.glob('*.jpg')))} images")


In [ ]:

import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pathlib import Path

# Mutually exclusive integer-pixel-area bins.
# COCO includes both endpoints, so use non-overlapping ranges.
SIZE_BINS = [
    (0, 255),
    (256, 1023),
    (1024, 4095),
    (4096, 10**10),
]

SIZE_BIN_LABELS = [
    "<16²",
    "16²–<32²",
    "32²–<64²",
    "≥64²",
]

def corrected_eval_size_bins(pred, names, tag):
    images = []
    annotations = []
    detections = []
    annotation_id = 1

    assert set(pred.keys()) == set(names), (
        f"Prediction image mismatch for {tag}"
    )

    for image_id, name in enumerate(names):
        width, height, gt_boxes = read_boxes(
            ROOT / "Annotations" / f"{name}.xml"
        )

        boxes, scores, pred_size = pred[name]

        assert tuple(pred_size) == (width, height), (
            f"Image dimension mismatch: {name}, "
            f"prediction={pred_size}, XML={(width, height)}"
        )

        images.append({
            "id": image_id,
            "width": width,
            "height": height
        })

        for x1, y1, x2, y2 in gt_boxes:
            box_width = x2 - x1
            box_height = y2 - y1
            area = box_width * box_height

            annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 1,
                "bbox": [x1, y1, box_width, box_height],
                "area": area,
                "iscrowd": 0
            })
            annotation_id += 1

        for (x1, y1, x2, y2), score in zip(boxes, scores):
            detections.append({
                "image_id": image_id,
                "category_id": 1,
                "score": float(score),
                "bbox": [
                    float(x1 * width),
                    float(y1 * height),
                    float((x2 - x1) * width),
                    float((y2 - y1) * height)
                ]
            })

    gt = COCO()
    gt.dataset = {
        "images": images,
        "annotations": annotations,
        "categories": [{"id": 1, "name": "person"}]
    }
    gt.createIndex()

    if detections:
        dt = gt.loadRes(detections)
    else:
        dt = COCO()
        dt.dataset = {
            "images": images,
            "annotations": [],
            "categories": [{"id": 1, "name": "person"}]
        }
        dt.createIndex()

    evaluator = COCOeval(gt, dt, "bbox")
    evaluator.params.iouThrs = np.array(
        [0.50] + list(np.arange(0.55, 0.96, 0.05))
    )
    evaluator.params.areaRng = (
        [[0, 10**10]] + [list(b) for b in SIZE_BINS]
    )
    evaluator.params.areaRngLbl = ["all"] + SIZE_BIN_LABELS
    evaluator.params.maxDets = [1, 10, 100]

    evaluator.evaluate()
    evaluator.accumulate()

    precision = evaluator.eval["precision"]
    row = {"config": tag}

    for area_index, label in enumerate(
        evaluator.params.areaRngLbl
    ):
        values = precision[0, :, 0, area_index, 2]
        valid = values[values > -1]

        row[f"AP50 {label}"] = (
            float(np.mean(valid)) if len(valid) else float("nan")
        )

    # Count GT boxes using the same inclusive boundaries as COCO.
    areas = [ann["area"] for ann in annotations]
    row["GT boxes all"] = len(areas)

    for (low, high), label in zip(SIZE_BINS, SIZE_BIN_LABELS):
        row[f"GT boxes {label}"] = sum(
            low <= area <= high for area in areas
        )

    return row


# Re-evaluate all four individual models and late fusion.
result_rows = []

for config_name, pred in predictions.items():
    print("Evaluating:", config_name)

    result_rows.append(
        corrected_eval_size_bins(
            pred, test_names, config_name
        )
    )

corrected_results = pd.DataFrame(result_rows).round(4)

output_path = (
    Path("/content/work_exp1_corrected")
    / "exp1_results_corrected_bins.csv"
)

corrected_results.to_csv(output_path, index=False)

print("\nCORRECTED EXPERIMENT 1 RESULTS")
print(corrected_results.to_string(index=False))
print("\nSaved to:", output_path)


In [ ]:

import shutil
import pickle
from pathlib import Path

WORK = Path("/content/work_exp1_corrected")
BACKUP = Path("/content/experiment1_backup")
BACKUP.mkdir(exist_ok=True)

# 1. Copy trained models and training logs
shutil.copytree(
    WORK / "runs",
    BACKUP / "runs",
    dirs_exist_ok=True
)

# 2. Copy evaluation results
for filename in [
    "exp1_results.csv",
    "exp1_results_corrected_bins.csv",
    "split_manifest.json",
    "rgb.yaml",
    "thermal.yaml",
    "blend.yaml",
    "vt.yaml",
]:
    source = WORK / filename
    if source.exists():
        shutil.copy2(source, BACKUP / filename)

# 3. Save all predictions for future evaluation
with open(BACKUP / "test_predictions.pkl", "wb") as f:
    pickle.dump(predictions, f, protocol=pickle.HIGHEST_PROTOCOL)

# 4. Create ZIP archive
archive = shutil.make_archive(
    "/content/experiment1_backup",
    "zip",
    root_dir="/content",
    base_dir="experiment1_backup"
)

size_mb = Path(archive).stat().st_size / (1024 ** 2)

print("Backup created successfully!")
print("Location:", archive)
print(f"Size: {size_mb:.2f} MB")
print("\nContents:")
for item in sorted(BACKUP.rglob("*")):
    if item.is_file():
        print(item.relative_to(BACKUP))


In [ ]:
from google.colab import files
files.download("/content/experiment1_backup.zip")

In [ ]:

from pathlib import Path

ROOT = Path("/content/Anti-UAV410")

print("Dataset exists:", ROOT.exists())

if ROOT.exists():
    for split in ["train", "val", "test"]:
        folder = ROOT / split
        print(f"\n{split}:")

        if folder.exists():
            sequences = [p for p in folder.iterdir() if p.is_dir()]
            print("Sequences:", len(sequences))

            for seq in sequences[:5]:
                print(" ", seq.name)
        else:
            print("Folder not found")



In [ ]:
from pathlib import Path

for path in [
    Path("/content/Anti-UAV410"),
    Path("/content/drive/MyDrive/Anti-UAV410"),
    Path("/content/AntiUAV410"),
    Path("/content/drive/MyDrive/AntiUAV410"),
]:
    print(path, "→", path.exists())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive")

for item in root.iterdir():
    print(item.name)

In [ ]:
!kaggle datasets list -s "Anti-UAV"

In [ ]:
!kaggle datasets files farhanzzzzz/anti-uav410

In [ ]:
!kaggle datasets files huseyincavus/anti-uav410


In [ ]:

from pathlib import Path
import subprocess

ROOT = Path("/content/drive/MyDrive/Anti-UAV410")
ROOT.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    [
        "kaggle", "datasets", "download",
        "-d", "farhanzzzzz/anti-uav410",
        "-p", str(ROOT),
        "--unzip"
    ],
    text=True
)

print("Download process exit code:", result.returncode)
print("Download folder:", ROOT)


In [ ]:
!df -h /content

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/content/Anti-UAV410")
ROOT.mkdir(parents=True, exist_ok=True)

result = subprocess.run([
    "kaggle", "datasets", "download",
    "-d", "farhanzzzzz/anti-uav410",
    "-p", str(ROOT),
    "--unzip"
])

print("Exit code:", result.returncode)
print("Download location:", ROOT)

In [ ]:
from pathlib import Path

ROOT = Path("/content/Anti-UAV410")

print("Root exists:", ROOT.exists())
print("\nTop-level contents:")

for item in sorted(ROOT.iterdir()):
    if item.is_dir():
        print(f"[DIR]  {item.name}")
    else:
        print(f"[FILE] {item.name} ({item.stat().st_size / 1e6:.1f} MB)")

In [ ]:
!pwd
!ls -lah /content
!df -h /content

In [ ]:
!kaggle datasets list -s "Anti-UAV410" -p 1

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("/content/kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets list -s "Anti-UAV410"

In [ ]:
!kaggle datasets files huseyincavus/anti-uav410 --page-size 100

In [ ]:
!kaggle datasets files huseyincavus/anti-uav410 --page-token "PASTE_NEXT_PAGE_TOKEN_HERE" --page-size 100

In [ ]:
!kaggle datasets files --help

In [ ]:
import subprocess

token = input("Paste the Next Page Token: ").strip()

result = subprocess.run(
    [
        "kaggle", "datasets", "files",
        "huseyincavus/anti-uav410",
        "--page-token", token,
        "--page-size", "200",
        "-v"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

In [ ]:

import subprocess
import csv
import re
import time
from collections import Counter
from tqdm.auto import tqdm
from IPython.display import display, clear_output

DATASET = "huseyincavus/anti-uav410"
PAGE_SIZE = 200
MAX_PAGES = 500

all_files = {}
page_token = None
page = 0
start_time = time.time()

progress = tqdm(
    total=None,
    desc="Scanning Kaggle dataset",
    unit="page",
    dynamic_ncols=True
)

while page < MAX_PAGES:
    cmd = [
        "kaggle", "datasets", "files", DATASET,
        "--page-size", str(PAGE_SIZE), "-v"
    ]

    if page_token:
        cmd += ["--page-token", page_token]

    progress.set_postfix(
        files=len(all_files),
        status="Fetching page"
    )

    result = subprocess.run(
        cmd, capture_output=True, text=True
    )

    output = result.stdout + "\n" + result.stderr

    if result.returncode != 0:
        progress.close()
        print("\nKaggle error:")
        print(output[-2000:])
        break

    previous_count = len(all_files)

    for line in result.stdout.splitlines():
        line = line.strip()
        if not line:
            continue

        try:
            row = next(csv.reader([line]))
            if len(row) >= 2:
                name = row[0].strip().strip('"')
                size = row[1].strip().strip('"')

                if name.lower() not in ("name", "filename"):
                    if name and size.isdigit():
                        all_files[name] = int(size)
        except Exception:
            pass

    page += 1
    match = re.search(
        r"Next Page Token\s*=\s*(\S+)",
        output,
        re.IGNORECASE
    )

    progress.update(1)
    progress.set_postfix(
        files=len(all_files),
        new=len(all_files) - previous_count,
        elapsed=f"{time.time()-start_time:.0f}s"
    )

    if not match:
        progress.close()
        print("\nAll available pages have been scanned.")
        break

    page_token = match.group(1)
    time.sleep(1)

else:
    progress.close()
    print("Stopped at MAX_PAGES. Listing may be incomplete.")

# Final report
print("\n========== DATASET SUMMARY ==========")
print("Pages checked:", page)
print("Unique files:", len(all_files))
print("Total size (GB):", round(sum(all_files.values()) / 1e9, 2))
print("Elapsed time (minutes):",
      round((time.time()-start_time)/60, 2))

folders = Counter()
extensions = Counter()
annotation_files = []

for name in all_files:
    parts = name.split("/")
    folders["/".join(parts[:-1]) or "(root)"] += 1
    extensions[name.rsplit(".", 1)[-1].lower()] += 1

    if any(x in name.lower() for x in
           ["label", "annotation", "json", "xml", "txt"]):
        annotation_files.append(name)

print("\nFile extensions:")
for ext, count in extensions.most_common():
    print(f"{ext}: {count}")

print("\nTop folders:")
for folder, count in folders.most_common(40):
    print(f"{count:6}  {folder}")

print("\nPossible annotation files:")
for name in annotation_files[:100]:
    print(name)

if len(annotation_files) > 100:
    print("... and", len(annotation_files)-100, "more")


In [ ]:

import subprocess, csv, re, time
from tqdm.auto import tqdm

assert "page_token" in globals() and page_token, (
    "The previous pagination token is unavailable. "
    "The runtime may have reset."
)

extra_pages = 100
start = time.time()
found_rgb_labels = []
found_splits = set()

bar = tqdm(total=extra_pages, desc="Checking remaining dataset", unit="page")

for i in range(extra_pages):
    cmd = [
        "kaggle", "datasets", "files", DATASET,
        "--page-size", str(PAGE_SIZE), "-v",
        "--page-token", page_token
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    output = result.stdout + "\n" + result.stderr

    if result.returncode != 0:
        print("\nKaggle error:", output[-1500:])
        break

    for line in result.stdout.splitlines():
        try:
            row = next(csv.reader([line.strip()]))
            if len(row) >= 2:
                name = row[0].strip().strip('"')
                size = row[1].strip().strip('"')
                if name and size.isdigit():
                    all_files[name] = int(size)

                    if name.lower().endswith("rgb_label.json"):
                        found_rgb_labels.append(name)

                    first_folder = name.split("/")[0].lower()
                    if first_folder in ("train", "val", "test"):
                        found_splits.add(first_folder)
        except Exception:
            pass

    bar.update(1)
    bar.set_postfix(
        files=len(all_files),
        splits=",".join(sorted(found_splits)) or "test so far",
        rgb_labels=len(found_rgb_labels)
    )

    match = re.search(
        r"Next Page Token\s*=\s*(\S+)",
        output,
        re.IGNORECASE
    )

    if not match:
        print("\nReached the final page.")
        page_token = None
        break

    page_token = match.group(1)

    # Stop as soon as we find evidence of another split
    # or RGB annotations.
    if found_rgb_labels or "train" in found_splits or "val" in found_splits:
        print("\nFound another split or RGB annotation. Stopping for inspection.")
        break

    time.sleep(1)

bar.close()

print("\nNew scan duration:", round(time.time()-start, 1), "seconds")
print("Total unique files collected:", len(all_files))
print("Splits encountered:", sorted(found_splits))
print("RGB annotation files:", found_rgb_labels[:20])
print("Latest page token available:", bool(page_token))


In [ ]:

import subprocess

result = subprocess.run(
    ["kaggle", "datasets", "metadata", DATASET],
    capture_output=True,
    text=True
)

print(result.stdout[:12000])
if result.stderr:
    print(result.stderr[:2000])


In [ ]:

import json

with open("/content/dataset-metadata.json", "r") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=2))


In [ ]:

import subprocess, csv, re, time
from tqdm.auto import tqdm

if not page_token:
    print("The previous scan already reached the final page.")
else:
    MAX_EXTRA_PAGES = 200
    start = time.time()
    new_paths = set()

    bar = tqdm(total=MAX_EXTRA_PAGES, desc="Verifying RGB and splits", unit="page")

    for i in range(MAX_EXTRA_PAGES):
        cmd = [
            "kaggle", "datasets", "files", DATASET,
            "--page-size", str(PAGE_SIZE), "-v",
            "--page-token", page_token
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)
        output = result.stdout + "\n" + result.stderr

        if result.returncode != 0:
            print("\nKaggle error:", output[-1500:])
            break

        for line in result.stdout.splitlines():
            try:
                row = next(csv.reader([line.strip()]))
                if len(row) >= 2:
                    name = row[0].strip().strip('"')
                    size = row[1].strip().strip('"')

                    if name and size.isdigit():
                        all_files[name] = int(size)
                        new_paths.add(name)

            except Exception:
                pass

        bar.update(1)

        names_lower = [n.lower() for n in new_paths]
        rgb_files = [
            n for n in names_lower
            if any(x in n for x in
                   ["rgb", "visible", "color", "colour"])
        ]
        other_splits = [
            n for n in names_lower
            if n.startswith(("train/", "val/", "validation/"))
        ]
        rgb_labels = [
            n for n in names_lower
            if "rgb_label" in n
        ]

        bar.set_postfix(
            total_files=len(all_files),
            rgb_names=len(rgb_files),
            other_splits=len(other_splits)
        )

        match = re.search(
            r"Next Page Token\s*=\s*(\S+)",
            output,
            re.IGNORECASE
        )

        if not match:
            page_token = None
            print("\nReached the final page.")
            break

        page_token = match.group(1)

        if rgb_files or other_splits or rgb_labels:
            print("\nFound potentially relevant files. Stopping.")
            break

        time.sleep(1)

    bar.close()

    print("\n========== VERIFICATION ==========")
    print("Total unique files:", len(all_files))
    print("New paths scanned:", len(new_paths))
    print("RGB/visible-like paths:", len(rgb_files))
    print("RGB annotation files:", len(rgb_labels))
    print("Train/validation paths:", len(other_splits))

    print("\nSample RGB/visible paths:")
    for n in rgb_files[:20]:
        print(n)

    print("\nSample train/validation paths:")
    for n in other_splits[:20]:
        print(n)

    print("\nElapsed minutes:", round((time.time()-start)/60, 1))


In [ ]:

import subprocess, json

sequence = "01_1667_0001-1500"

# Download only the annotation file
result = subprocess.run([
    "kaggle", "datasets", "download",
    "huseyincavus/anti-uav410",
    "-f", f"train/{sequence}/IR_label.json",
    "-p", "/content/anti_uav_inspect"
], capture_output=True, text=True)

print(result.stdout)
print(result.stderr[-1000:] if result.stderr else "")

# Unzip the downloaded annotation
import glob, zipfile, os

zips = glob.glob("/content/anti_uav_inspect/*.zip")

if zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/content/anti_uav_inspect")

    files = glob.glob("/content/anti_uav_inspect/**/*", recursive=True)
    json_files = [f for f in files if f.endswith(".json")]

    for path in json_files:
        with open(path, "r") as f:
            data = json.load(f)

        print("\nAnnotation file:", path)
        print("Data type:", type(data).__name__)
        print("Top-level keys:",
              list(data.keys()) if isinstance(data, dict) else "Not a dictionary")
        print("Sample:")
        print(str(data)[:3000])
else:
    print("No ZIP downloaded. Check the Kaggle error above.")


In [ ]:

import os
import json
import glob

folder = "/content/anti_uav_inspect"

print("Downloaded files:")
for f in glob.glob(folder + "/**/*", recursive=True):
    if os.path.isfile(f):
        print(os.path.basename(f), "-", os.path.getsize(f), "bytes")

# Locate and read the JSON file
json_files = glob.glob(folder + "/**/IR_label.json", recursive=True)

if json_files:
    path = json_files[0]

    with open(path, "r") as f:
        data = json.load(f)

    print("\nAnnotation file:", path)
    print("Data type:", type(data).__name__)

    if isinstance(data, dict):
        print("Top-level keys:", list(data.keys()))

    print("\nSample annotation:")
    print(json.dumps(data, indent=2)[:4000])

else:
    print("\nIR_label.json not found.")


In [ ]:

import json
import os

path = "/content/anti_uav_inspect/IR_label.json"

with open(path, "r") as f:
    data = json.load(f)

print("ANNOTATION STRUCTURE")
print("=" * 50)

for key, value in data.items():
    print(f"\n{key}:")
    print("  Type:", type(value).__name__)

    if isinstance(value, list):
        print("  Number of entries:", len(value))
        print("  First 5 entries:", value[:5])
    else:
        print("  Value:", str(value)[:300])

print("\nBOUNDING BOX INSPECTION")
print("=" * 50)

rects = data.get("gt_rect", [])

for i, rect in enumerate(rects[:10]):
    print(f"Frame {i}: {rect}")

print("\nEXISTENCE LABEL COUNTS")
print("=" * 50)

from collections import Counter

print(Counter(data.get("exist", [])))

print("\nFRAME COUNT:", len(data.get("exist", [])))
print("BOX COUNT:", len(rects))


In [ ]:

import subprocess
import os
from PIL import Image
from IPython.display import display

sequence = "01_1667_0001-1500"
frame = "000001.jpg"

folder = "/content/anti_uav_inspect/frame"
os.makedirs(folder, exist_ok=True)

result = subprocess.run([
    "kaggle", "datasets", "download",
    "huseyincavus/anti-uav410",
    "-f", f"train/{sequence}/{frame}",
    "-p", folder
], capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print(result.stderr[-1000:])

# Locate and extract the downloaded file if necessary
import glob
import zipfile

for z in glob.glob(folder + "/*.zip"):
    with zipfile.ZipFile(z) as archive:
        archive.extractall(folder)

images = glob.glob(folder + "/**/*.jpg", recursive=True)

if images:
    img = Image.open(images[0])
    print("Image:", images[0])
    print("Dimensions:", img.size)
    print("Image mode:", img.mode)
    display(img)
else:
    print("No JPG found. Check the download output.")


In [ ]:

import numpy as np
from PIL import Image
import glob

image_path = glob.glob(
    "/content/anti_uav_inspect/frame/**/*.jpg",
    recursive=True
)[0]

img = np.array(Image.open(image_path).convert("RGB"))

r, g, b = img[:,:,0], img[:,:,1], img[:,:,2]

print("Image shape:", img.shape)
print("R-G identical:", np.array_equal(r, g))
print("G-B identical:", np.array_equal(g, b))
print("R-B identical:", np.array_equal(r, b))

print("\nChannel statistics:")
for name, channel in zip(["Red", "Green", "Blue"], [r, g, b]):
    print(
        name,
        "min:", channel.min(),
        "max:", channel.max(),
        "mean:", round(channel.mean(), 2),
        "std:", round(channel.std(), 2)
    )

print("\nUnique RGB colors:", len(np.unique(img.reshape(-1, 3), axis=0)))


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import glob

path = glob.glob(
    "/content/anti_uav_inspect/frame/**/*.jpg",
    recursive=True
)[0]

img = np.array(Image.open(path).convert("RGB"))

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

axes[0].imshow(img)
axes[0].set_title("Original image")

axes[1].imshow(img[:,:,0], cmap="gray")
axes[1].set_title("Red channel")

axes[2].imshow(img[:,:,1], cmap="gray")
axes[2].set_title("Green channel")

axes[3].imshow(img.mean(axis=2), cmap="gray")
axes[3].set_title("Grayscale")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
!kaggle datasets list -s "Anti-UAV300"

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("/content/kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle authentication configured.")

In [ ]:
!kaggle datasets list -s "Anti-UAV300"

In [ ]:
!kaggle datasets files mm1991/anti-uav-dataset-300

In [ ]:
!mkdir -p /content/anti_uav_inspect

!kaggle datasets download mm1991/anti-uav-dataset-300 \
  -f label_new/train.json \
  -p /content/anti_uav_inspect

!kaggle datasets download mm1991/anti-uav-dataset-300 \
  -f test/20190925_111757_1_1/visible.json \
  -p /content/anti_uav_inspect

!kaggle datasets download mm1991/anti-uav-dataset-300 \
  -f test/20190925_111757_1_1/infrared.json \
  -p /content/anti_uav_inspect

In [ ]:
import os
import json

for root, dirs, files in os.walk("/content/anti_uav_inspect"):
    for file in files:
        path = os.path.join(root, file)
        print("\nFILE:", file, "| SIZE:", os.path.getsize(path))

        if file.endswith(".json"):
            with open(path, "r") as f:
                data = json.load(f)

            print("Type:", type(data).__name__)
            print("Keys:", list(data.keys()) if isinstance(data, dict) else "Not a dictionary")
            print("Sample:", str(data)[:700])

In [ ]:
import json
import os
import numpy as np

folder = "/content/anti_uav_inspect"

with open(os.path.join(folder, "visible.json")) as f:
    rgb = json.load(f)

with open(os.path.join(folder, "infrared.json")) as f:
    ir = json.load(f)

# Number of annotated frames
print("VISIBLE frames:", len(rgb["exist"]))
print("INFRARED frames:", len(ir["exist"]))

# Check whether frame-existence labels match
rgb_exist = np.array(rgb["exist"])
ir_exist = np.array(ir["exist"])

print("\nExistence labels identical:",
      np.array_equal(rgb_exist, ir_exist))

print("Visible positive frames:", int(rgb_exist.sum()))
print("Infrared positive frames:", int(ir_exist.sum()))

# Check bounding-box availability and equality
rgb_boxes = np.array(rgb["gt_rect"], dtype=float)
ir_boxes = np.array(ir["gt_rect"], dtype=float)

print("\nRGB bbox shape:", rgb_boxes.shape)
print("IR bbox shape:", ir_boxes.shape)

if rgb_boxes.shape == ir_boxes.shape:
    differences = np.abs(rgb_boxes - ir_boxes)
    print("Bounding boxes exactly identical:",
          np.allclose(rgb_boxes, ir_boxes))
    print("Mean absolute bbox difference:",
          round(float(differences.mean()), 3))

print("\nFirst 5 RGB boxes:")
print(rgb_boxes[:5])

print("\nFirst 5 IR boxes:")
print(ir_boxes[:5])

In [ ]:
!kaggle datasets download mm1991/anti-uav-dataset-300 \
  -f test/20190925_111757_1_1/visible.mp4 \
  -p /content/anti_uav_inspect

!kaggle datasets download mm1991/anti-uav-dataset-300 \
  -f test/20190925_111757_1_1/infrared.mp4 \
  -p /content/anti_uav_inspect

In [ ]:
import cv2
import os
import json

folder = "/content/anti_uav_inspect"

videos = {
    "Visible": os.path.join(folder, "visible.mp4"),
    "Infrared": os.path.join(folder, "infrared.mp4")
}

for name, path in videos.items():
    cap = cv2.VideoCapture(path)

    print(f"\n{name} video")
    print("Width:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))
    print("Height:", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    print("Frame count:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    print("FPS:", cap.get(cv2.CAP_PROP_FPS))

    cap.release()

In [ ]:
import cv2
import json
import matplotlib.pyplot as plt

folder = "/content/anti_uav_inspect"

# Load annotations
with open(folder + "/visible.json") as f:
    rgb_ann = json.load(f)

with open(folder + "/infrared.json") as f:
    ir_ann = json.load(f)

# Read the first frame from each video
rgb_cap = cv2.VideoCapture(folder + "/visible.mp4")
ir_cap = cv2.VideoCapture(folder + "/infrared.mp4")

_, rgb_frame = rgb_cap.read()
_, ir_frame = ir_cap.read()

rgb_cap.release()
ir_cap.release()

# Draw modality-specific ground-truth boxes
def draw_box(frame, box):
    x, y, w, h = map(int, box)
    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 3)
    return frame

rgb_frame = draw_box(rgb_frame, rgb_ann["gt_rect"][0])
ir_frame = draw_box(ir_frame, ir_ann["gt_rect"][0])

# Convert BGR to RGB for matplotlib
rgb_frame = cv2.cvtColor(rgb_frame, cv2.COLOR_BGR2RGB)
ir_frame = cv2.cvtColor(ir_frame, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(rgb_frame)
axes[0].set_title("Visible frame 1 — Ground truth")
axes[0].axis("off")

axes[1].imshow(ir_frame, cmap="gray")
axes[1].set_title("Infrared frame 1 — Ground truth")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import json
import matplotlib.pyplot as plt

folder = "/content/anti_uav_inspect"

with open(folder + "/visible.json") as f:
    rgb_ann = json.load(f)

with open(folder + "/infrared.json") as f:
    ir_ann = json.load(f)

rgb_cap = cv2.VideoCapture(folder + "/visible.mp4")
ir_cap = cv2.VideoCapture(folder + "/infrared.mp4")

frame_indices = [0, 249, 499, 749, 999]

fig, axes = plt.subplots(5, 2, figsize=(12, 20))

for row, idx in enumerate(frame_indices):
    rgb_cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ir_cap.set(cv2.CAP_PROP_POS_FRAMES, idx)

    _, rgb = rgb_cap.read()
    _, ir = ir_cap.read()

    for frame, box in [
        (rgb, rgb_ann["gt_rect"][idx]),
        (ir, ir_ann["gt_rect"][idx])
    ]:
        x, y, w, h = map(int, box)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 3)

    axes[row, 0].imshow(cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f"Visible — frame {idx+1}")

    axes[row, 1].imshow(cv2.cvtColor(ir, cv2.COLOR_BGR2RGB))
    axes[row, 1].set_title(f"Infrared — frame {idx+1}")

    axes[row, 0].axis("off")
    axes[row, 1].axis("off")

rgb_cap.release()
ir_cap.release()

plt.tight_layout()
plt.show()

In [ ]:
!kaggle datasets download mm1991/anti-uav-dataset-300 \
  -f label_new/val.json \
  -p /content/anti_uav_inspect

!kaggle datasets download mm1991/anti-uav-dataset-300 \
  -f label_new/test.json \
  -p /content/anti_uav_inspect

In [ ]:
import json
import os

folder = "/content/anti_uav_inspect"

splits = {}

for split in ["train", "val", "test"]:
    path = os.path.join(folder, split + ".json")

    with open(path) as f:
        data = json.load(f)

    splits[split] = set(data.keys())
    print(f"{split}: {len(splits[split])} sequences")

print("\nSplit overlap:")
print("Train ∩ Validation:", len(splits["train"] & splits["val"]))
print("Train ∩ Test:", len(splits["train"] & splits["test"]))
print("Validation ∩ Test:", len(splits["val"] & splits["test"]))

print("\nTotal unique sequences:",
      len(set.union(*splits.values())))

In [ ]:
!df -h /content


In [ ]:
!mkdir -p /content/Anti-UAV300

!kaggle datasets download mm1991/anti-uav-dataset-300 \
    -p /content/Anti-UAV300 \
    --unzip

In [ ]:
import os

root = "/content/Anti-UAV300"

print("Archive exists:",
      os.path.exists(root + "/anti-uav-dataset-300.zip"))

for split in ["train", "val", "test"]:
    path = os.path.join(root, split)
    if os.path.exists(path):
        print(split, "folders:", len(os.listdir(path)))

In [ ]:
import os

root = "/content/Anti-UAV300"
required = [
    "visible.mp4",
    "infrared.mp4",
    "visible.json",
    "infrared.json"
]

missing = []

for split in ["train", "val", "test"]:
    split_path = os.path.join(root, split)

    for seq in os.listdir(split_path):
        seq_path = os.path.join(split_path, seq)

        if not os.path.isdir(seq_path):
            continue

        absent = [
            f for f in required
            if not os.path.isfile(os.path.join(seq_path, f))
        ]

        if absent:
            missing.append((split, seq, absent))

print("Missing files:", len(missing))
for item in missing[:20]:
    print(item)

print("Dataset structure check completed.")

In [ ]:
import os, cv2, json
import numpy as np

root = "/content/Anti-UAV300"
report = []

for split in ["train", "val", "test"]:
    split_path = os.path.join(root, split)

    for seq in os.listdir(split_path):
        p = os.path.join(split_path, seq)
        if not os.path.isdir(p):
            continue

        row = {"split": split, "sequence": seq}

        for modality in ["visible", "infrared"]:
            video = cv2.VideoCapture(os.path.join(p, modality + ".mp4"))
            frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
            video.release()

            with open(os.path.join(p, modality + ".json")) as f:
                data = json.load(f)

            boxes = data["gt_rect"]
            exists = data["exist"]

            row[modality + "_frames"] = frames
            row[modality + "_annotations"] = len(boxes)
            row[modality + "_exist"] = len(exists)

        report.append(row)

print("Sequences checked:", len(report))
print("First 5 results:")
for r in report[:5]:
    print(r)

mismatch = [
    r for r in report
    if not (
        r["visible_frames"] == r["visible_annotations"] == r["visible_exist"]
        and r["infrared_frames"] == r["infrared_annotations"] == r["infrared_exist"]
    )
]

print("\nSequences with frame/annotation count mismatch:", len(mismatch))
for r in mismatch[:10]:
    print(r)

In [ ]:
import os, json
import numpy as np

root = "/content/Anti-UAV300"

results = []

for split in ["train", "val", "test"]:
    split_path = os.path.join(root, split)

    for seq in os.listdir(split_path):
        p = os.path.join(split_path, seq)

        if not os.path.isdir(p):
            continue

        with open(os.path.join(p, "visible.json")) as f:
            vis = json.load(f)

        with open(os.path.join(p, "infrared.json")) as f:
            ir = json.load(f)

        vis_exist = np.array(vis["exist"])
        ir_exist = np.array(ir["exist"])

        results.append({
            "split": split,
            "sequence": seq,
            "same_length": len(vis_exist) == len(ir_exist),
            "same_presence": np.array_equal(vis_exist, ir_exist),
            "visible_positive": int(vis_exist.sum()),
            "infrared_positive": int(ir_exist.sum())
        })

print("Sequences checked:", len(results))
print("Different frame-presence arrays:",
      sum(not r["same_presence"] for r in results))

print("Different lengths:",
      sum(not r["same_length"] for r in results))

print("\nFirst 5 results:")
for r in results[:5]:
    print(r)

In [ ]:
import os, json
import numpy as np

root = "/content/Anti-UAV300"

total_both = 0
total_visible_only = 0
total_ir_only = 0
total_neither = 0
total_frames = 0

for split in ["train", "val", "test"]:
    split_path = os.path.join(root, split)

    for seq in os.listdir(split_path):
        p = os.path.join(split_path, seq)

        if not os.path.isdir(p):
            continue

        with open(os.path.join(p, "visible.json")) as f:
            vis = np.array(json.load(f)["exist"]).astype(bool)

        with open(os.path.join(p, "infrared.json")) as f:
            ir = np.array(json.load(f)["exist"]).astype(bool)

        total_both += np.sum(vis & ir)
        total_visible_only += np.sum(vis & ~ir)
        total_ir_only += np.sum(~vis & ir)
        total_neither += np.sum(~vis & ~ir)
        total_frames += len(vis)

print("Total paired frames:", total_frames)
print("Drone present in both:", total_both)
print("Visible only:", total_visible_only)
print("Infrared only:", total_ir_only)
print("Absent in both:", total_neither)

print("\nCheck:", total_both + total_visible_only +
      total_ir_only + total_neither == total_frames)

In [ ]:
import os, json
import cv2
import numpy as np
import matplotlib.pyplot as plt

root = "/content/Anti-UAV300"
found = False

for split in ["train", "val", "test"]:
    split_path = os.path.join(root, split)

    for seq in os.listdir(split_path):
        p = os.path.join(split_path, seq)

        if not os.path.isdir(p):
            continue

        with open(os.path.join(p, "visible.json")) as f:
            vis = json.load(f)

        with open(os.path.join(p, "infrared.json")) as f:
            ir = json.load(f)

        v = np.array(vis["exist"]).astype(bool)
        t = np.array(ir["exist"]).astype(bool)

        indices = np.where(~v & t)[0]

        if len(indices) > 0:
            idx = int(indices[0])

            cap_v = cv2.VideoCapture(
                os.path.join(p, "visible.mp4")
            )
            cap_t = cv2.VideoCapture(
                os.path.join(p, "infrared.mp4")
            )

            cap_v.set(cv2.CAP_PROP_POS_FRAMES, idx)
            cap_t.set(cv2.CAP_PROP_POS_FRAMES, idx)

            _, frame_v = cap_v.read()
            _, frame_t = cap_t.read()

            cap_v.release()
            cap_t.release()

            if frame_v is not None and frame_t is not None:
                print("Split:", split)
                print("Sequence:", seq)
                print("Frame:", idx)
                print("Visible bbox:", vis["gt_rect"][idx])
                print("Infrared bbox:", ir["gt_rect"][idx])

                plt.figure(figsize=(12, 5))

                plt.subplot(1, 2, 1)
                plt.imshow(cv2.cvtColor(
                    frame_v, cv2.COLOR_BGR2RGB
                ))
                plt.title("Visible — annotation absent")
                plt.axis("off")

                plt.subplot(1, 2, 2)
                plt.imshow(cv2.cvtColor(
                    frame_t, cv2.COLOR_BGR2RGB
                ))
                plt.title("Infrared — annotation present")
                plt.axis("off")

                plt.show()
                found = True
                break

        if found:
            break

    if found:
        break

In [ ]:
import os, cv2, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

root = "/content/Anti-UAV300/train/20190925_152412_1_4"

with open(os.path.join(root, "visible.json")) as f:
    vis = json.load(f)

with open(os.path.join(root, "infrared.json")) as f:
    ir = json.load(f)

cap_v = cv2.VideoCapture(os.path.join(root, "visible.mp4"))
cap_i = cv2.VideoCapture(os.path.join(root, "infrared.mp4"))

for idx in [470, 471, 472, 473, 474]:
    cap_v.set(cv2.CAP_PROP_POS_FRAMES, idx)
    cap_i.set(cv2.CAP_PROP_POS_FRAMES, idx)

    _, frame_v = cap_v.read()
    _, frame_i = cap_i.read()

    print(
        f"Frame {idx}: RGB={vis['exist'][idx]}, "
        f"IR={ir['exist'][idx]}"
    )

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].imshow(cv2.cvtColor(frame_v, cv2.COLOR_BGR2RGB))
    ax[0].set_title(f"Visible — frame {idx}")
    ax[0].axis("off")

    ax[1].imshow(cv2.cvtColor(frame_i, cv2.COLOR_BGR2RGB))
    ax[1].set_title(f"Infrared — frame {idx}")
    ax[1].axis("off")

    bbox = ir["gt_rect"][idx]
    if len(bbox) == 4:
        x, y, w, h = bbox
        ax[1].add_patch(Rectangle(
            (x, y), w, h,
            fill=False, edgecolor="red", linewidth=2
        ))

    plt.show()

cap_v.release()
cap_i.release()

In [ ]:
import os, json, cv2

root = "/content/Anti-UAV300"

stats = {
    "visible": {"valid": 0, "invalid": 0, "empty": 0},
    "infrared": {"valid": 0, "invalid": 0, "empty": 0}
}

examples = []

for split in ["train", "val", "test"]:
    split_path = os.path.join(root, split)

    for seq in os.listdir(split_path):
        p = os.path.join(split_path, seq)

        if not os.path.isdir(p):
            continue

        for modality in ["visible", "infrared"]:

            video = cv2.VideoCapture(
                os.path.join(p, modality + ".mp4")
            )

            width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
            video.release()

            with open(os.path.join(p, modality + ".json")) as f:
                data = json.load(f)

            for i, (exists, bbox) in enumerate(
                zip(data["exist"], data["gt_rect"])
            ):

                if not exists:
                    if not bbox:
                        stats[modality]["empty"] += 1
                    else:
                        stats[modality]["invalid"] += 1
                    continue

                if len(bbox) != 4:
                    stats[modality]["invalid"] += 1
                    continue

                x, y, w, h = map(float, bbox)

                valid = (
                    x >= 0 and y >= 0 and
                    w > 0 and h > 0 and
                    x + w <= width and
                    y + h <= height
                )

                if valid:
                    stats[modality]["valid"] += 1
                else:
                    stats[modality]["invalid"] += 1
                    if len(examples) < 10:
                        examples.append(
                            (split, seq, modality, i, bbox, width, height)
                        )

print("ANNOTATION VALIDATION")
print(stats)

print("\nInvalid examples:")
for item in examples:
    print(item)

In [ ]:
import os, json
from collections import Counter

root = "/content/Anti-UAV300/train"

for seq in [
    "20190925_152412_1_4",
    "20190926_141816_1_1"
]:
    print("\nSequence:", seq)

    for modality in ["visible", "infrared"]:
        path = os.path.join(root, seq, modality + ".json")

        with open(path) as f:
            data = json.load(f)

        print("\n", modality)
        print("Exist value types and values:",
              Counter((type(x).__name__, repr(x))
                      for x in data["exist"]))

        print("First zero-box examples:")
        shown = 0

        for i, bbox in enumerate(data["gt_rect"]):
            if bbox == [0, 0, 0, 0]:
                print(
                    "Frame:", i,
                    "| exist:", repr(data["exist"][i]),
                    "| bbox:", bbox
                )
                shown += 1

                if shown == 5:
                    break

In [ ]:
import os, json
from collections import Counter

root = "/content/Anti-UAV300"

stats = {
    "visible": Counter(),
    "infrared": Counter()
}

examples = []

for split in ["train", "val", "test"]:
    split_path = os.path.join(root, split)

    for seq in os.listdir(split_path):
        p = os.path.join(split_path, seq)

        if not os.path.isdir(p):
            continue

        for modality in ["visible", "infrared"]:
            with open(os.path.join(p, modality + ".json")) as f:
                data = json.load(f)

            for i, (exist, bbox) in enumerate(
                zip(data["exist"], data["gt_rect"])
            ):
                zero_box = bbox == [0, 0, 0, 0]

                if exist == 1 and zero_box:
                    stats[modality]["positive_with_zero_box"] += 1
                    if len(examples) < 10:
                        examples.append(
                            (split, seq, modality, i, exist, bbox)
                        )

                elif exist == 0 and zero_box:
                    stats[modality]["negative_with_zero_box"] += 1

                elif exist == 0 and not zero_box:
                    stats[modality]["negative_with_nonzero_box"] += 1

                elif exist == 1 and not zero_box:
                    stats[modality]["positive_with_nonzero_box"] += 1

print("ANNOTATION CONSISTENCY AUDIT")
for modality, counts in stats.items():
    print("\n", modality.upper())
    for key, value in counts.items():
        print(key, ":", value)

print("\nFirst inconsistent examples:")
for x in examples:
    print(x)

In [ ]:
import os, json, cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

root = "/content/Anti-UAV300/train"

examples = [
    ("20190925_152412_1_4", "visible"),
    ("20190925_152412_1_4", "infrared")
]

for seq, modality in examples:
    folder = os.path.join(root, seq)

    with open(os.path.join(folder, modality + ".json")) as f:
        data = json.load(f)

    idx = next(
        i for i, (e, b) in enumerate(
            zip(data["exist"], data["gt_rect"])
        )
        if e == 0 and b != [0, 0, 0, 0]
    )

    cap = cv2.VideoCapture(
        os.path.join(folder, modality + ".mp4")
    )
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    cap.release()

    bbox = data["gt_rect"][idx]

    print(
        "\nModality:", modality,
        "| Frame:", idx,
        "| exist:", data["exist"][idx],
        "| bbox:", bbox
    )

    if ok:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(9, 5))
        plt.imshow(frame)

        x, y, w, h = bbox
        plt.gca().add_patch(
            Rectangle(
                (x, y), w, h,
                fill=False, edgecolor="red", linewidth=2
            )
        )

        plt.title(f"{modality.upper()} — frame {idx}")
        plt.axis("off")
        plt.show()

In [ ]:
import os, json, cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

root = "/content/Anti-UAV300/train"

examples = [
    ("20190925_152412_1_4", "visible"),
    ("20190925_152412_1_4", "infrared")
]

for seq, modality in examples:
    folder = os.path.join(root, seq)

    with open(os.path.join(folder, modality + ".json")) as f:
        data = json.load(f)

    candidates = [
        i for i, (e, b) in enumerate(
            zip(data["exist"], data["gt_rect"])
        )
        if e == 0
        and isinstance(b, list)
        and len(b) == 4
        and any(float(v) != 0 for v in b)
    ]

    if not candidates:
        print(f"No suitable example found: {modality}")
        continue

    idx = candidates[0]

    cap = cv2.VideoCapture(
        os.path.join(folder, modality + ".mp4")
    )
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    cap.release()

    bbox = data["gt_rect"][idx]

    print(
        f"\nModality: {modality}",
        f"| Frame: {idx}",
        f"| exist: {data['exist'][idx]}",
        f"| bbox: {bbox}"
    )

    if ok:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        x, y, w, h = map(float, bbox)

        plt.figure(figsize=(9, 5))
        plt.imshow(frame)
        plt.gca().add_patch(
            Rectangle(
                (x, y), w, h,
                fill=False, edgecolor="red", linewidth=2
            )
        )
        plt.title(f"{modality.upper()} — frame {idx}")
        plt.axis("off")
        plt.show()

In [ ]:
import os, json
from collections import Counter

root = "/content/Anti-UAV300"

stats = {
    "visible": Counter(),
    "infrared": Counter()
}

for split in ["train", "val", "test"]:
    split_path = os.path.join(root, split)

    for seq in os.listdir(split_path):
        p = os.path.join(split_path, seq)

        if not os.path.isdir(p):
            continue

        for modality in ["visible", "infrared"]:
            with open(os.path.join(p, modality + ".json")) as f:
                data = json.load(f)

            for exist, bbox in zip(data["exist"], data["gt_rect"]):

                if not bbox:
                    category = "empty_bbox"

                elif len(bbox) != 4:
                    category = "malformed_bbox"

                elif all(float(v) == 0 for v in bbox):
                    category = "zero_bbox"

                else:
                    category = "nonzero_bbox"

                stats[modality][f"exist_{exist}_{category}"] += 1

print("CORRECTED ANNOTATION AUDIT")

for modality, counts in stats.items():
    print("\n", modality.upper())
    for category, count in sorted(counts.items()):
        print(f"{category}: {count}")

In [ ]:
import os, json
from collections import defaultdict

root = "/content/Anti-UAV300"

results = defaultdict(lambda: {
    "valid_positive": 0,
    "valid_negative": 0,
    "excluded_positive": 0,
    "excluded_conflict": 0
})

for split in ["train", "val", "test"]:
    for seq in os.listdir(os.path.join(root, split)):
        p = os.path.join(root, split, seq)

        if not os.path.isdir(p):
            continue

        for modality in ["visible", "infrared"]:
            with open(os.path.join(p, modality + ".json")) as f:
                data = json.load(f)

            for exist, bbox in zip(data["exist"], data["gt_rect"]):
                valid_box = (
                    isinstance(bbox, list)
                    and len(bbox) == 4
                    and float(bbox[2]) > 0
                    and float(bbox[3]) > 0
                    and float(bbox[0]) >= 0
                    and float(bbox[1]) >= 0
                )

                key = (split, modality)

                if exist == 1 and valid_box:
                    results[key]["valid_positive"] += 1

                elif exist == 1:
                    results[key]["excluded_positive"] += 1

                elif exist == 0 and valid_box:
                    results[key]["excluded_conflict"] += 1

                else:
                    results[key]["valid_negative"] += 1

for key, counts in sorted(results.items()):
    print("\n", key)
    for name, value in counts.items():
        print(name, ":", value)

In [ ]:
import os, json, cv2, math
from collections import Counter

root = "/content/Anti-UAV300"

results = {
    (split, modality): Counter()
    for split in ["train", "val", "test"]
    for modality in ["visible", "infrared"]
}

for split in ["train", "val", "test"]:
    for seq in os.listdir(os.path.join(root, split)):
        p = os.path.join(root, split, seq)

        if not os.path.isdir(p):
            continue

        for modality in ["visible", "infrared"]:
            video = cv2.VideoCapture(
                os.path.join(p, modality + ".mp4")
            )
            W = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
            H = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
            video.release()

            with open(os.path.join(p, modality + ".json")) as f:
                data = json.load(f)

            for exist, bbox in zip(data["exist"], data["gt_rect"]):
                key = (split, modality)

                valid = False

                if isinstance(bbox, list) and len(bbox) == 4:
                    try:
                        x, y, w, h = map(float, bbox)

                        valid = (
                            all(map(math.isfinite, [x, y, w, h]))
                            and x >= 0 and y >= 0
                            and w > 0 and h > 0
                            and x + w <= W
                            and y + h <= H
                        )
                    except (ValueError, TypeError):
                        valid = False

                if exist == 1 and valid:
                    results[key]["positive_valid"] += 1
                elif exist == 1:
                    results[key]["positive_excluded"] += 1
                elif exist == 0 and valid:
                    results[key]["negative_conflict"] += 1
                else:
                    results[key]["negative_clean"] += 1

print("STRICT BOUNDING-BOX VALIDATION")

for key, counts in sorted(results.items()):
    print("\n", key)
    for name, value in counts.items():
        print(f"{name}: {value}")

In [ ]:
import os

BASE = "/content/anti_uav_yolo"

for modality in ["visible", "infrared"]:
    for split in ["train", "val", "test"]:
        os.makedirs(
            f"{BASE}/{modality}/images/{split}",
            exist_ok=True
        )
        os.makedirs(
            f"{BASE}/{modality}/labels/{split}",
            exist_ok=True
        )

print("YOLO dataset directories created.")

for modality in ["visible", "infrared"]:
    print(modality, "ready")

In [ ]:
import os
import cv2
import json
import csv
import math
from collections import defaultdict

SOURCE = "/content/Anti-UAV300"
OUTPUT = "/content/anti_uav_yolo"
STEP = 10

modalities = ["visible", "infrared"]
splits = ["train", "val", "test"]

summary = defaultdict(lambda: defaultdict(int))
manifest = []

for split in splits:
    split_path = os.path.join(SOURCE, split)

    for seq in sorted(os.listdir(split_path)):
        seq_path = os.path.join(split_path, seq)

        if not os.path.isdir(seq_path):
            continue

        for modality in modalities:

            video_path = os.path.join(
                seq_path, modality + ".mp4"
            )
            json_path = os.path.join(
                seq_path, modality + ".json"
            )

            with open(json_path) as f:
                annotations = json.load(f)

            exists = annotations["exist"]
            boxes = annotations["gt_rect"]

            cap = cv2.VideoCapture(video_path)

            W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

            frame_idx = 0

            while True:
                success, frame = cap.read()

                if not success:
                    break

                if frame_idx % STEP == 0:

                    flag = exists[frame_idx]
                    bbox = boxes[frame_idx]

                    valid_box = False

                    if isinstance(bbox, list) and len(bbox) == 4:
                        try:
                            x, y, w, h = map(float, bbox)

                            valid_box = (
                                all(map(math.isfinite, [x, y, w, h]))
                                and x >= 0 and y >= 0
                                and w > 0 and h > 0
                                and x + w <= W
                                and y + h <= H
                            )
                        except (ValueError, TypeError):
                            valid_box = False

                    # Apply consistent annotation filtering
                    if flag == 1 and valid_box:
                        status = "positive"

                    elif flag == 0 and not valid_box:
                        status = "negative"

                    else:
                        status = "excluded"

                    summary[modality][split + "_" + status] += 1

                    if status == "excluded":
                        reason = (
                            "positive_without_valid_box"
                            if flag == 1
                            else "negative_with_nonzero_or_invalid_box"
                        )

                        manifest.append([
                            split, seq, modality,
                            frame_idx, flag, bbox, reason
                        ])

                    else:
                        image_name = f"{seq}_{frame_idx:06d}.jpg"

                        image_path = os.path.join(
                            OUTPUT, modality, "images",
                            split, image_name
                        )

                        label_path = os.path.join(
                            OUTPUT, modality, "labels",
                            split, image_name.replace(".jpg", ".txt")
                        )

                        cv2.imwrite(image_path, frame)

                        if status == "positive":
                            xc = (x + w / 2) / W
                            yc = (y + h / 2) / H
                            nw = w / W
                            nh = h / H

                            with open(label_path, "w") as f:
                                f.write(
                                    f"0 {xc:.8f} {yc:.8f} "
                                    f"{nw:.8f} {nh:.8f}\n"
                                )
                        else:
                            # Empty label file = background image
                            open(label_path, "w").close()

                frame_idx += 1

            cap.release()

            if frame_idx != len(exists) or frame_idx != len(boxes):
                raise RuntimeError(
                    f"Frame mismatch: {split}/{seq}/{modality}: "
                    f"video={frame_idx}, "
                    f"exist={len(exists)}, boxes={len(boxes)}"
                )

            print(f"Completed: {split}/{seq}/{modality}")

# Save exclusion manifest
manifest_path = os.path.join(OUTPUT, "exclusion_manifest.csv")

with open(manifest_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "split", "sequence", "modality",
        "frame_index", "exist", "bbox", "reason"
    ])
    writer.writerows(manifest)

# Print summary
print("\n========== EXTRACTION SUMMARY ==========")

for modality in modalities:
    print("\n", modality.upper())

    for split in splits:
        for status in ["positive", "negative", "excluded"]:
            key = split + "_" + status
            print(f"{split} {status}: {summary[modality][key]}")

print("\nExclusion manifest:", manifest_path)
print("Dataset preparation completed.")

In [ ]:
import os
import cv2
import json
import csv
import math
import shutil
from collections import defaultdict

SOURCE = "/content/Anti-UAV300"
OUTPUT = "/content/anti_uav_yolo"
STEP = 10

modalities = ["visible", "infrared"]
splits = ["train", "val", "test"]

# Rebuild clean output directories
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)

for modality in modalities:
    for split in splits:
        os.makedirs(f"{OUTPUT}/{modality}/images/{split}", exist_ok=True)
        os.makedirs(f"{OUTPUT}/{modality}/labels/{split}", exist_ok=True)

summary = defaultdict(lambda: defaultdict(int))
manifest = []

for split in splits:
    split_path = os.path.join(SOURCE, split)

    for seq in sorted(os.listdir(split_path)):
        seq_path = os.path.join(split_path, seq)

        if not os.path.isdir(seq_path):
            continue

        for modality in modalities:
            video_path = os.path.join(seq_path, modality + ".mp4")
            json_path = os.path.join(seq_path, modality + ".json")

            with open(json_path) as f:
                annotations = json.load(f)

            exists = annotations["exist"]
            boxes = annotations["gt_rect"]

            cap = cv2.VideoCapture(video_path)
            W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

            frame_idx = 0

            while True:
                success, frame = cap.read()
                if not success:
                    break

                if frame_idx % STEP == 0:
                    flag = int(exists[frame_idx])
                    bbox = boxes[frame_idx]

                    parsed = None

                    if isinstance(bbox, (list, tuple)) and len(bbox) == 4:
                        try:
                            parsed = list(map(float, bbox))
                        except (ValueError, TypeError):
                            parsed = None

                    is_zero = (
                        parsed is not None
                        and all(math.isfinite(v) for v in parsed)
                        and all(v == 0 for v in parsed)
                    )

                    valid_box = (
                        parsed is not None
                        and all(math.isfinite(v) for v in parsed)
                        and parsed[0] >= 0
                        and parsed[1] >= 0
                        and parsed[2] > 0
                        and parsed[3] > 0
                        and parsed[0] + parsed[2] <= W
                        and parsed[1] + parsed[3] <= H
                    )

                    if flag == 1 and valid_box:
                        status = "positive"
                    elif flag == 0 and (bbox == [] or is_zero):
                        status = "negative"
                    else:
                        status = "excluded"

                    summary[modality][f"{split}_{status}"] += 1

                    if status == "excluded":
                        reason = (
                            "positive_invalid_bbox"
                            if flag == 1
                            else "negative_nonzero_or_malformed_bbox"
                        )

                        manifest.append([
                            split, seq, modality, frame_idx,
                            flag, bbox, reason
                        ])

                    else:
                        image_name = f"{seq}_{frame_idx:06d}.jpg"

                        image_path = (
                            f"{OUTPUT}/{modality}/images/{split}/{image_name}"
                        )
                        label_path = (
                            f"{OUTPUT}/{modality}/labels/{split}/"
                            f"{image_name.replace('.jpg', '.txt')}"
                        )

                        if not cv2.imwrite(image_path, frame):
                            raise RuntimeError(f"Failed to save {image_path}")

                        if status == "positive":
                            x, y, w, h = parsed

                            xc = (x + w / 2) / W
                            yc = (y + h / 2) / H
                            nw = w / W
                            nh = h / H

                            with open(label_path, "w") as f:
                                f.write(
                                    f"0 {xc:.8f} {yc:.8f} "
                                    f"{nw:.8f} {nh:.8f}\n"
                                )
                        else:
                            open(label_path, "w").close()

                frame_idx += 1

            cap.release()

            if frame_idx != len(exists) or frame_idx != len(boxes):
                raise RuntimeError(
                    f"Frame mismatch: {split}/{seq}/{modality}"
                )

        print(f"Completed sequence: {split}/{seq}")

manifest_path = f"{OUTPUT}/exclusion_manifest.csv"

with open(manifest_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "split", "sequence", "modality",
        "frame_index", "exist", "bbox", "reason"
    ])
    writer.writerows(manifest)

print("\n========== STRICT EXTRACTION SUMMARY ==========")

for modality in modalities:
    print(f"\n{modality.upper()}")
    for split in splits:
        for status in ["positive", "negative", "excluded"]:
            print(
                f"{split} {status}: "
                f"{summary[modality][f'{split}_{status}']}"
            )

print("\nCorrected datasets saved:", OUTPUT)
print("Exclusion manifest:", manifest_path)

In [ ]:
import os
import yaml

BASE = "/content/anti_uav_yolo"

for modality in ["visible", "infrared"]:

    config = {
        "path": f"{BASE}/{modality}",
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": 1,
        "names": {
            0: "drone"
        }
    }

    yaml_path = f"{BASE}/{modality}/data.yaml"

    with open(yaml_path, "w") as f:
        yaml.safe_dump(config, f, sort_keys=False)

    print("Created:", yaml_path)

print("\nBoth YOLO configurations created.")


In [ ]:
import os
import cv2
from collections import defaultdict

BASE = "/content/anti_uav_yolo"
modalities = ["visible", "infrared"]
splits = ["train", "val", "test"]

all_passed = True

for modality in modalities:
    print(f"\n========== {modality.upper()} ==========")

    for split in splits:
        image_dir = f"{BASE}/{modality}/images/{split}"
        label_dir = f"{BASE}/{modality}/labels/{split}"

        images = [
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        labels = [
            f for f in os.listdir(label_dir)
            if f.endswith(".txt")
        ]

        image_stems = {os.path.splitext(f)[0] for f in images}
        label_stems = {os.path.splitext(f)[0] for f in labels}

        missing_labels = image_stems - label_stems
        orphan_labels = label_stems - image_stems

        positive = 0
        negative = 0
        invalid = 0
        corrupt = 0

        for img_name in images:
            stem = os.path.splitext(img_name)[0]
            img_path = os.path.join(image_dir, img_name)
            label_path = os.path.join(label_dir, stem + ".txt")

            img = cv2.imread(img_path)

            if img is None:
                corrupt += 1
                continue

            if not os.path.exists(label_path):
                continue

            with open(label_path) as f:
                lines = [line.strip() for line in f if line.strip()]

            if not lines:
                negative += 1
                continue

            positive += 1

            for line in lines:
                parts = line.split()

                try:
                    values = list(map(float, parts))
                except ValueError:
                    invalid += 1
                    continue

                if (
                    len(values) != 5
                    or values[0] != 0
                    or not all(0 <= v <= 1 for v in values[1:])
                    or values[3] <= 0
                    or values[4] <= 0
                ):
                    invalid += 1

        passed = (
            not missing_labels
            and not orphan_labels
            and invalid == 0
            and corrupt == 0
        )

        all_passed &= passed

        print(f"\n{split.upper()}")
        print("Images:", len(images))
        print("Label files:", len(labels))
        print("Positive:", positive)
        print("Negative:", negative)
        print("Missing labels:", len(missing_labels))
        print("Orphan labels:", len(orphan_labels))
        print("Invalid annotations:", invalid)
        print("Corrupt images:", corrupt)
        print("Status:", "PASS" if passed else "FAIL")

print("\n================================")
print("FINAL RESULT:", "ALL CHECKS PASSED" if all_passed else "CHECKS FAILED")

In [ ]:
import os

for path in [
    "/content/Anti-UAV300",
    "/content/anti_uav_yolo",
    "/content/anti_uav_yolo/visible/data.yaml",
    "/content/anti_uav_yolo/infrared/data.yaml"
]:
    print(path, "EXISTS" if os.path.exists(path) else "MISSING")

In [ ]:
import os
import shutil

paths = [
    "/content/Anti-UAV300",
    "/content/anti_uav_yolo",
    "/content/anti_uav_inspect"
]

print("========== BACKUP SIZE CHECK ==========")

total = 0

for path in paths:
    if os.path.exists(path):
        size = shutil.disk_usage(path).used if False else 0

        for root, dirs, files in os.walk(path):
            for f in files:
                fp = os.path.join(root, f)
                try:
                    size += os.path.getsize(fp)
                except OSError:
                    pass

        total += size
        print(f"{path}: {size / (1024**3):.2f} GB")
    else:
        print(f"{path}: Not found")

print(f"\nTotal: {total / (1024**3):.2f} GB")

disk = shutil.disk_usage("/content")
print(f"Available Colab storage: {disk.free / (1024**3):.2f} GB")

In [ ]:
import os
import tarfile

BACKUP = "/content/research_backup"
os.makedirs(BACKUP, exist_ok=True)

items = [
    (
        "/content/Anti-UAV300",
        BACKUP + "/anti_uav_original.tar"
    ),
    (
        "/content/anti_uav_yolo",
        BACKUP + "/anti_uav_experiment.tar"
    ),
    (
        "/content/anti_uav_inspect",
        BACKUP + "/anti_uav_inspection.tar"
    )
]

for source, destination in items:
    print(f"\nCreating: {os.path.basename(destination)}")

    with tarfile.open(destination, "w") as archive:
        archive.add(source, arcname=os.path.basename(source))

    size = os.path.getsize(destination) / (1024**3)
    print(f"Completed: {size:.2f} GB")

print("\nAll three archives created.")

In [ ]:
import os
import shutil

paths = [
    "/content/Anti-UAV300",
    "/content/anti_uav_yolo",
    "/content/anti_uav_inspect",
    "/content/research_backup"
]

for path in paths:
    print(
        f"{path}:",
        "EXISTS" if os.path.exists(path) else "MISSING"
    )

print("\nStorage:")
print(shutil.disk_usage("/content"))

In [ ]:
import os

BACKUP = "/content/research_backup"

print("Backup directory contents:")

for item in os.listdir(BACKUP):
    path = os.path.join(BACKUP, item)
    print(item, round(os.path.getsize(path) / (1024**3), 3), "GB")

# Source notebook: `exp-2(1).ipynb`

Original notebook cells: 40. Non-empty source-code cells included: 40. The original code-cell order is preserved below.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch
import os

print("=" * 50)
print("KAGGLE ENVIRONMENT CHECK")
print("=" * 50)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
        print(
            "VRAM:",
            round(
                torch.cuda.get_device_properties(i).total_memory / 1024**3,
                2
            ),
            "GB"
        )

print("Working directory:", os.getcwd())
print("Environment check complete.")

In [ ]:

import os
import hashlib
from pathlib import Path

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

REPO_ID = "SakibM19/border-surveillance-research-backups"
REPO_TYPE = "dataset"

TEST_PATH = "Experiment2/Kaggle_Access_Test/kaggle_access_test.txt"

print("=" * 60)
print("KAGGLE - HUGGING FACE ACCESS VERIFICATION")
print("=" * 60)

# --------------------------------------------------
# 1. LOAD TOKEN SECURELY
# --------------------------------------------------

print("\n[1] Loading Kaggle Secret...")

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN secret is empty."

print("Secret loaded successfully.")

# --------------------------------------------------
# 2. VERIFY ACCOUNT AND REPOSITORY
# --------------------------------------------------

print("\n[2] Checking Hugging Face account...")

api = HfApi(token=HF_TOKEN)

account = api.whoami()
print("Authenticated account:", account.get("name", "Verified"))

info = api.repo_info(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE
)

assert info.private, "Repository is not private."

print("Repository:", REPO_ID)
print("Repository type:", REPO_TYPE)
print("Private repository: VERIFIED")

# --------------------------------------------------
# 3. CREATE TEST FILE
# --------------------------------------------------

print("\n[3] Creating test file...")

local_test = Path("/kaggle/working/kaggle_hf_test.txt")

test_content = (
    "Kaggle Hugging Face backup verification\n"
    "Repository: SakibM19/border-surveillance-research-backups\n"
    "Purpose: WRITE, DOWNLOAD, SHA256 integrity test\n"
)

local_test.write_text(test_content, encoding="utf-8")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

local_hash = sha256_file(local_test)
print("Local SHA256:", local_hash)

# --------------------------------------------------
# 4. TEST WRITE ACCESS
# --------------------------------------------------

print("\n[4] Uploading test file...")

api.upload_file(
    path_or_fileobj=str(local_test),
    path_in_repo=TEST_PATH,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    commit_message="Verify Kaggle Hugging Face backup access"
)

print("UPLOAD: SUCCESS")

# --------------------------------------------------
# 5. VERIFY REMOTE LISTING
# --------------------------------------------------

print("\n[5] Checking remote file listing...")

remote_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE
)

assert TEST_PATH in remote_files, "Uploaded file missing remotely."

print("REMOTE LISTING: PASS")

# --------------------------------------------------
# 6. DOWNLOAD REMOTE FILE
# --------------------------------------------------

print("\n[6] Downloading remote test file...")

downloaded = hf_hub_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    filename=TEST_PATH,
    token=HF_TOKEN,
    force_download=True
)

print("DOWNLOAD: SUCCESS")

# --------------------------------------------------
# 7. VERIFY SHA256
# --------------------------------------------------

print("\n[7] Comparing file integrity...")

remote_hash = sha256_file(downloaded)

print("Local SHA256: ", local_hash)
print("Remote SHA256:", remote_hash)

assert local_hash == remote_hash, "SHA256 mismatch!"

print("SHA256 MATCH: PASS")

# --------------------------------------------------
# FINAL RESULT
# --------------------------------------------------

print("\n" + "=" * 60)
print("HUGGING FACE BACKUP VERIFICATION: PASSED")
print("=" * 60)
print("Account authentication: PASS")
print("Private repository access: PASS")
print("WRITE permission: PASS")
print("Remote listing: PASS")
print("Remote download: PASS")
print("SHA256 integrity: PASS")
print("\nDATASET RESTORATION: NOT STARTED")
print("MODEL TRAINING: NOT STARTED")
print("=" * 60)


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

print("HF_TOKEN loaded:", bool(HF_TOKEN))
print("Token length:", len(HF_TOKEN))

In [ ]:
import hashlib
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = "SakibM19/border-surveillance-research-backups"
REPO_TYPE = "dataset"
TEST_PATH = "Experiment2/Kaggle_Access_Test/kaggle_access_test.txt"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("1. AUTHENTICATING...")

api = HfApi(token=HF_TOKEN)
account = api.whoami()
print("Authenticated account:", account["name"])

info = api.repo_info(REPO_ID, repo_type=REPO_TYPE)
assert info.private
print("Private repository: VERIFIED")

print("\n2. CREATING TEST FILE...")

local_file = Path("/kaggle/working/hf_test.txt")
local_file.write_text(
    "Kaggle Hugging Face backup integrity test.\n",
    encoding="utf-8"
)
local_hash = sha256(local_file)

print("\n3. UPLOADING...")

api.upload_file(
    path_or_fileobj=str(local_file),
    path_in_repo=TEST_PATH,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    commit_message="Kaggle backup access test"
)
print("UPLOAD: SUCCESS")

print("\n4. VERIFYING REMOTE LISTING...")

remote_files = api.list_repo_files(
    REPO_ID,
    repo_type=REPO_TYPE
)
assert TEST_PATH in remote_files
print("REMOTE LISTING: PASS")

print("\n5. DOWNLOADING...")

downloaded = hf_hub_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    filename=TEST_PATH,
    token=HF_TOKEN,
    force_download=True
)
print("DOWNLOAD: SUCCESS")

print("\n6. VERIFYING SHA256...")

remote_hash = sha256(downloaded)
assert local_hash == remote_hash

print("SHA256 MATCH: PASS")

print("\n" + "=" * 55)
print("HUGGING FACE CONNECTION TEST: PASSED")
print("Authentication: PASS")
print("Private repository: PASS")
print("Write access: PASS")
print("Remote listing: PASS")
print("Download: PASS")
print("Integrity: PASS")
print("DATASET RESTORATION: NOT STARTED")
print("TRAINING: NOT STARTED")
print("=" * 55)

In [ ]:

import os
import json
import hashlib
import shutil
import tarfile
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
import yaml

# ============================================================
# EXPERIMENT 2 — KAGGLE RESTORATION AND CONFIGURATION
# ============================================================

REPO_ID = "SakibM19/border-surveillance-research-backups"
REPO_TYPE = "dataset"

BASE = Path("/kaggle/working")
PART_DIR = BASE / "EXP2_DOWNLOAD_PARTS"
ARCHIVE = BASE / "Experiment2_Restored.tar"
RESTORE_DIR = BASE / "EXP2_RESTORED"
DATA_ROOT = RESTORE_DIR / "AntiUAV_Experiment2"

EXPECTED_ARCHIVE_SHA = (
    "c644f40169369fb843373ae2e7ac54a0d44bffe6128b6198fa3d398bbca73ec1"
)

PARTS = [
    ("experiment2_part_01.tar.part", 1073741824,
     "0ea56e507fa7a3a4cb845a5cc47aabfc584542b47a341bfddef829464e1c49f4"),
    ("experiment2_part_02.tar.part", 1073741824,
     "0d136123b8871f5ae16610f0908ad84b493ac61a45653bb2138c5e55b02e9743"),
    ("experiment2_part_03.tar.part", 1073741824,
     "ff5e5c04fd1fc702b93d1aa329ebd7e1b89790aaab9cba71fbca9ad90ef8020d"),
    ("experiment2_part_04.tar.part", 1073741824,
     "5970dbb37aa7b1772241fca0efe8875d5f207b1a5eecf9295b7a762b9b31d241"),
    ("experiment2_part_05.tar.part", 619188224,
     "357d71ff0999f2820f5333b263d90527222b53e632710a7d87032a760bd2661e"),
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert HF_TOKEN, "HF_TOKEN is not loaded."
api = HfApi(token=HF_TOKEN)

print("=" * 65)
print("EXPERIMENT 2 — KAGGLE RESTORATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. DOWNLOAD AND VERIFY ORIGINAL FIVE PARTS
# ------------------------------------------------------------

PART_DIR.mkdir(parents=True, exist_ok=True)
verified_parts = []

for i, (filename, expected_size, expected_hash) in enumerate(PARTS, 1):
    remote_path = f"Experiment2/{filename}"
    local_path = PART_DIR / filename

    print(f"\nDownloading part {i}/5: {filename}")

    downloaded = hf_hub_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        filename=remote_path,
        token=HF_TOKEN,
        local_dir=str(PART_DIR)
    )

    downloaded = Path(downloaded)

    assert downloaded.stat().st_size == expected_size, (
        f"Part {i} size mismatch."
    )

    actual_hash = sha256_file(downloaded)

    assert actual_hash == expected_hash, (
        f"Part {i} SHA256 mismatch."
    )

    verified_parts.append(downloaded)
    print(f"PART {i}: VERIFIED | {expected_size:,} bytes")

print("\nALL FIVE ORIGINAL PARTS VERIFIED.")

# ------------------------------------------------------------
# 2. RECONSTRUCT ORIGINAL ARCHIVE
# ------------------------------------------------------------

print("\nReconstructing archive...")

with open(ARCHIVE, "wb") as output:
    for part in verified_parts:
        with open(part, "rb") as source:
            shutil.copyfileobj(source, output, length=8 * 1024 * 1024)

archive_hash = sha256_file(ARCHIVE)

assert archive_hash == EXPECTED_ARCHIVE_SHA, (
    "Reconstructed archive SHA256 mismatch."
)

print("ARCHIVE SHA256:", archive_hash)
print("ARCHIVE INTEGRITY: PASS")

# Free space by deleting the verified individual parts.
shutil.rmtree(PART_DIR)

# ------------------------------------------------------------
# 3. CHECK ARCHIVE PATH SAFETY AND EXTRACT
# ------------------------------------------------------------

print("\nChecking archive contents...")

with tarfile.open(ARCHIVE, "r:") as tar:
    members = tar.getmembers()

    assert len(members) == 118766, (
        f"Unexpected archive entry count: {len(members)}"
    )

    for member in members:
        member_path = Path(member.name)

        assert not member_path.is_absolute(), (
            f"Absolute path found: {member.name}"
        )
        assert ".." not in member_path.parts, (
            f"Unsafe path found: {member.name}"
        )
        assert not (member.issym() or member.islnk()), (
            f"Unexpected link found: {member.name}"
        )

    print("Archive entries:", len(members))
    print("Path safety: PASS")

    # Extract only after all members pass validation.
    RESTORE_DIR.mkdir(parents=True, exist_ok=True)
    tar.extractall(path=RESTORE_DIR, members=members)

assert DATA_ROOT.is_dir(), f"Dataset root not found: {DATA_ROOT}"

print("EXTRACTION: COMPLETE")

# Archive no longer needed after verified extraction.
ARCHIVE.unlink()

# ------------------------------------------------------------
# 4. VERIFY DATASET SPLITS
# ------------------------------------------------------------

print("\nDATASET FILE VERIFICATION")

expected_counts = {
    "visible": {"train": 14949, "val": 6186, "test": 8539},
    "infrared": {"train": 14955, "val": 6198, "test": 8541},
}

dataset_summary = {}

for modality, splits in expected_counts.items():
    dataset_summary[modality] = {}

    for split, expected in splits.items():
        image_dir = DATA_ROOT / "datasets" / modality / "images" / split
        label_dir = DATA_ROOT / "datasets" / modality / "labels" / split

        assert image_dir.is_dir(), f"Missing: {image_dir}"
        assert label_dir.is_dir(), f"Missing: {label_dir}"

        image_count = sum(1 for _ in image_dir.glob("*.jpg"))
        label_count = sum(1 for _ in label_dir.glob("*.txt"))

        assert image_count == expected, (
            f"{modality} {split}: expected {expected}, got {image_count}"
        )
        assert label_count == expected, (
            f"{modality} {split}: label count mismatch"
        )

        dataset_summary[modality][split] = {
            "images": image_count,
            "labels": label_count
        }

        print(
            f"{modality.upper()} {split}: "
            f"{image_count:,} images | {label_count:,} labels | PASS"
        )

# ------------------------------------------------------------
# 5. CREATE KAGGLE-SPECIFIC YAML CONFIGURATIONS
# ------------------------------------------------------------

print("\nCREATING KAGGLE CONFIGURATIONS")

CONFIG_DIR = DATA_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

config_records = []

for modality in ["visible", "infrared"]:
    dataset_path = DATA_ROOT / "datasets" / modality

    config = {
        "path": str(dataset_path),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": 1,
        "names": {0: "drone"}
    }

    config_path = CONFIG_DIR / f"{modality}_data_kaggle.yaml"

    with open(config_path, "w") as f:
        yaml.safe_dump(config, f, sort_keys=False)

    config_hash = sha256_file(config_path)

    config_records.append({
        "name": modality,
        "local_path": str(config_path),
        "remote_path": (
            f"Experiment2/Kaggle_Training_Configs/"
            f"{modality}_data_kaggle.yaml"
        ),
        "sha256": config_hash
    })

    print(f"{modality.upper()} CONFIG: CREATED")
    print("Path:", config_path)
    print("SHA256:", config_hash)

# ------------------------------------------------------------
# 6. BACK UP CONFIGURATIONS AND RESTORATION MANIFEST
# ------------------------------------------------------------

print("\nUPLOADING KAGGLE CONFIGURATIONS")

for item in config_records:
    api.upload_file(
        path_or_fileobj=item["local_path"],
        path_in_repo=item["remote_path"],
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        commit_message=f"Back up Kaggle {item['name']} training config"
    )

    remote_download = hf_hub_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        filename=item["remote_path"],
        token=HF_TOKEN,
        force_download=True
    )

    assert sha256_file(remote_download) == item["sha256"], (
        f"Remote config verification failed: {item['name']}"
    )

    print(item["name"].upper(), "REMOTE BACKUP: VERIFIED")

manifest = {
    "experiment": "Experiment 2 - Anti-UAV300",
    "platform": "Kaggle",
    "source_repository": REPO_ID,
    "archive_sha256": EXPECTED_ARCHIVE_SHA,
    "dataset_summary": dataset_summary,
    "configurations": config_records,
    "restoration_status": "PASS",
    "training_status": "NOT_STARTED"
}

manifest_path = BASE / "Experiment2_Kaggle_Restoration_Manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

manifest_hash = sha256_file(manifest_path)

manifest_remote = (
    "Experiment2/Kaggle_Restoration/"
    "Experiment2_Kaggle_Restoration_Manifest.json"
)

api.upload_file(
    path_or_fileobj=str(manifest_path),
    path_in_repo=manifest_remote,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    commit_message="Back up Kaggle Experiment 2 restoration manifest"
)

remote_manifest = hf_hub_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    filename=manifest_remote,
    token=HF_TOKEN,
    force_download=True
)

assert sha256_file(remote_manifest) == manifest_hash

print("\n" + "=" * 65)
print("EXPERIMENT 2 — KAGGLE RESTORATION: PASSED")
print("=" * 65)
print("Original archive: SHA256 VERIFIED")
print("Dataset extraction: COMPLETE")
print("Dataset image and label counts: VERIFIED")
print("Visible configuration: VERIFIED")
print("Infrared configuration: VERIFIED")
print("Hugging Face configuration backups: VERIFIED")
print("Restoration manifest: REMOTELY VERIFIED")
print("TRAINING: NOT STARTED")
print("=" * 65)


In [ ]:
# ============================================================
# EXPERIMENT 2 — FINAL KAGGLE TRAINING CELL
# Anti-UAV300 | YOLO11n | Visible + Infrared
# Hugging Face verified checkpoint backups
# ============================================================

import os
import sys
import json
import time
import hashlib
import zipfile
import traceback
from pathlib import Path

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

# ------------------------------------------------------------
# 1. HUGGING FACE AUTHENTICATION
# ------------------------------------------------------------

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

REPO_ID = "SakibM19/border-surveillance-research-backups"
REPO_TYPE = "dataset"

api = HfApi(token=HF_TOKEN)

account = api.whoami()
assert account["name"] == "SakibM19", "Unexpected Hugging Face account"

repo_info = api.repo_info(REPO_ID, repo_type=REPO_TYPE)
assert repo_info.private, "Repository must be private"

print("Hugging Face authentication: PASS")
print("Repository:", REPO_ID)

# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

ROOT = Path("/kaggle/working/EXP2_RESTORED/AntiUAV_Experiment2")
DATA_ROOT = ROOT / "datasets"

OUTPUT = Path("/kaggle/working/EXP2_TRAINING")
OUTPUT.mkdir(parents=True, exist_ok=True)

assert DATA_ROOT.exists(), f"Dataset directory missing: {DATA_ROOT}"

# ------------------------------------------------------------
# 3. DOWNLOAD CONFIGURATIONS FROM VERIFIED HF BACKUPS
# ------------------------------------------------------------

import yaml

CONFIGS = {}

for modality in ["visible", "infrared"]:

    remote_config = (
        f"Experiment2/Training_Configs/"
        f"{modality}_data_restored.yaml"
    )

    print(f"\nDownloading {modality} configuration...")

    source = Path(hf_hub_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        filename=remote_config,
        token=HF_TOKEN,
        force_download=True
    ))

    assert source.is_file(), f"Configuration unavailable: {remote_config}"

    with open(source, "r") as f:
        cfg = yaml.safe_load(f)

    modality_root = DATA_ROOT / modality

    assert modality_root.exists(), f"Missing dataset: {modality_root}"

    # Use the actual Kaggle restoration directory.
    cfg["path"] = str(modality_root)

    local_yaml = OUTPUT / f"{modality}_kaggle.yaml"

    with open(local_yaml, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    for split in ["train", "val", "test"]:
        split_path = modality_root / cfg[split]
        assert split_path.exists(), (
            f"{modality} {split} directory missing: {split_path}"
        )

    CONFIGS[modality] = local_yaml

    print(f"{modality.upper()} configuration: PASS")
    print("Train:", cfg["train"])
    print("Validation:", cfg["val"])
    print("Test:", cfg["test"])

print("\nBoth configurations verified.")

# ------------------------------------------------------------
# 4. GPU AND ULTRALYTICS
# ------------------------------------------------------------

import torch

assert torch.cuda.is_available(), "CUDA GPU unavailable"

print("\nGPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

try:
    from ultralytics import YOLO
except ImportError:
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "ultralytics"
    ])
    from ultralytics import YOLO

# ------------------------------------------------------------
# 5. SHA256 AND VERIFIED BACKUP FUNCTION
# ------------------------------------------------------------

VERIFY_DIR = OUTPUT / "remote_verification"
VERIFY_DIR.mkdir(exist_ok=True)

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def upload_and_verify(local_path, remote_path):

    local_path = Path(local_path)

    assert local_path.is_file(), f"Missing backup: {local_path}"

    local_hash = sha256(local_path)

    api.upload_file(
        path_or_fileobj=str(local_path),
        path_in_repo=remote_path,
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        token=HF_TOKEN
    )

    downloaded = hf_hub_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        filename=remote_path,
        token=HF_TOKEN,
        force_download=True,
        local_dir=str(VERIFY_DIR)
    )

    remote_hash = sha256(downloaded)

    assert local_hash == remote_hash, (
        f"SHA256 verification failed: {remote_path}"
    )

    print("BACKUP VERIFIED:", remote_path)

    return local_hash

# ------------------------------------------------------------
# 6. TRAINING CONFIGURATION
# ------------------------------------------------------------

EPOCHS = 50
IMGSZ = 640
BATCH = 16
DEVICE = 0

RUN_SUMMARY = {
    "experiment": "Experiment2_AntiUAV300",
    "repository": REPO_ID,
    "model": "YOLO11n",
    "epochs_requested": EPOCHS,
    "image_size": IMGSZ,
    "batch_size": BATCH,
    "gpu": torch.cuda.get_device_name(0),
    "runs": {}
}

# ------------------------------------------------------------
# 7. TRAIN ONE MODALITY
# ------------------------------------------------------------

def train_modality(modality):

    print("\n" + "=" * 65)
    print("STARTING", modality.upper(), "TRAINING")
    print("=" * 65)

    remote_base = f"Experiment2/Training/{modality.capitalize()}"

    model = YOLO("yolo11n.pt")

    state = {"last_epoch_verified": 0}

    # Backup callback: checkpoint is uploaded and verified
    # before training proceeds to the next epoch.
    def backup_checkpoint(trainer):

        epoch = int(trainer.epoch) + 1

        weights_dir = Path(trainer.save_dir) / "weights"
        last_checkpoint = weights_dir / "last.pt"

        if not last_checkpoint.exists():
            raise RuntimeError(
                f"Epoch {epoch}: last.pt was not found. "
                "Stopping to protect experiment integrity."
            )

        remote_checkpoint = (
            f"{remote_base}/checkpoints/"
            f"epoch_{epoch:03d}_last.pt"
        )

        digest = upload_and_verify(
            last_checkpoint,
            remote_checkpoint
        )

        upload_and_verify(
            last_checkpoint,
            f"{remote_base}/checkpoints/latest_last.pt"
        )

        manifest = {
            "modality": modality,
            "epoch": epoch,
            "checkpoint": remote_checkpoint,
            "sha256": digest,
            "verified": True,
            "timestamp_utc": time.strftime(
                "%Y-%m-%d %H:%M:%S",
                time.gmtime()
            )
        }

        manifest_path = (
            OUTPUT / f"{modality}_latest_backup_manifest.json"
        )

        manifest_path.write_text(json.dumps(manifest, indent=2))

        upload_and_verify(
            manifest_path,
            f"{remote_base}/manifests/latest_backup_manifest.json"
        )

        state["last_epoch_verified"] = epoch

        print(
            f"\nCHECKPOINT VERIFIED: "
            f"{modality.upper()} — EPOCH {epoch}/{EPOCHS}\n"
        )

    # Upload updated metrics after each epoch.
    def backup_metrics(trainer):

        csv_path = Path(trainer.save_dir) / "results.csv"

        if csv_path.exists():
            upload_and_verify(
                csv_path,
                f"{remote_base}/logs/results.csv"
            )

    model.add_callback("on_model_save", backup_checkpoint)
    model.add_callback("on_fit_epoch_end", backup_metrics)

    results = model.train(
        data=str(CONFIGS[modality]),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=4,
        project=str(OUTPUT),
        name=modality,
        exist_ok=True,
        pretrained=True,
        optimizer="auto",
        seed=0,
        deterministic=True,
        patience=15,
        save=True,
        save_period=1,
        val=True,
        plots=True,
        verbose=True,
        amp=True
    )

    # --------------------------------------------------------
    # FINAL OUTPUT BACKUP
    # --------------------------------------------------------

    save_dir = Path(results.save_dir)
    weights_dir = save_dir / "weights"

    for filename in ["best.pt", "last.pt"]:

        file_path = weights_dir / filename

        if file_path.exists():
            upload_and_verify(
                file_path,
                f"{remote_base}/final/{filename}"
            )

    for filename in ["results.csv", "args.yaml"]:

        file_path = save_dir / filename

        if file_path.exists():
            upload_and_verify(
                file_path,
                f"{remote_base}/final/{filename}"
            )

    # Archive the complete run directory.
    archive_path = OUTPUT / f"{modality}_complete_results.zip"

    with zipfile.ZipFile(
        archive_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as archive:

        for file_path in save_dir.rglob("*"):

            if file_path.is_file():
                archive.write(
                    file_path,
                    file_path.relative_to(save_dir)
                )

    upload_and_verify(
        archive_path,
        f"{remote_base}/final/{modality}_complete_results.zip"
    )

    RUN_SUMMARY["runs"][modality] = {
        "status": "COMPLETED",
        "epochs_verified": state["last_epoch_verified"],
        "results_directory": str(save_dir)
    }

    print("\n", modality.upper(), "TRAINING COMPLETED")

# ------------------------------------------------------------
# 8. RUN VISIBLE THEN INFRARED
# ------------------------------------------------------------

try:

    train_modality("visible")

    train_modality("infrared")

    RUN_SUMMARY["overall_status"] = "BOTH_RUNS_COMPLETED"

except Exception as error:

    RUN_SUMMARY["overall_status"] = "STOPPED_WITH_ERROR"
    RUN_SUMMARY["error"] = repr(error)
    RUN_SUMMARY["traceback"] = traceback.format_exc()

    print("\nTRAINING STOPPED:")
    print(repr(error))

    print(
        "\nPreviously verified Hugging Face checkpoints "
        "remain in the repository."
    )

finally:

    summary_path = OUTPUT / "Experiment2_Training_Summary.json"

    summary_path.write_text(
        json.dumps(RUN_SUMMARY, indent=2)
    )

    try:
        upload_and_verify(
            summary_path,
            "Experiment2/Training/Experiment2_Training_Summary.json"
        )

    except Exception as backup_error:
        print("SUMMARY BACKUP FAILED:", repr(backup_error))

    print("\n" + "=" * 65)
    print("EXPERIMENT 2 FINAL STATUS")
    print(json.dumps(RUN_SUMMARY, indent=2))
    print("=" * 65)

In [ ]:
# ============================================================
# EXPERIMENT 2 — INDEPENDENT HUGGING FACE BACKUP AUDIT
# READ-ONLY: Does not modify or delete remote files.
# ============================================================

import hashlib
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "SakibM19/border-surveillance-research-backups"
TYPE = "dataset"

api = HfApi(token=TOKEN)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("CONNECTING TO HUGGING FACE...")
api.whoami()
print("Authentication: PASS")

remote_files = {
    f.path: f for f in api.list_repo_tree(
        repo_id=REPO,
        repo_type=TYPE,
        path_in_repo="Experiment2/Training",
        recursive=True
    )
    if hasattr(f, "path")
}

print("Remote training files found:", len(remote_files))

required = []

for modality in ["Visible", "Infrared"]:
    base = f"Experiment2/Training/{modality}"

    required.extend([
        f"{base}/checkpoints/epoch_050_last.pt",
        f"{base}/checkpoints/latest_last.pt",
        f"{base}/manifests/latest_backup_manifest.json",
        f"{base}/final/best.pt",
        f"{base}/final/last.pt",
        f"{base}/final/results.csv",
        f"{base}/final/args.yaml",
        f"{base}/final/{modality.lower()}_complete_results.zip",
    ])

required.append(
    "Experiment2/Training/Experiment2_Training_Summary.json"
)

missing = [p for p in required if p not in remote_files]

print("\nREMOTE FILE EXISTENCE AUDIT")
print("--------------------------------")

for path in required:
    status = "FOUND" if path in remote_files else "MISSING"
    print(status, path)

print("\nDOWNLOADING AND VERIFYING IMPORTANT FILES")
print("--------------------------------")

audit_dir = Path("/kaggle/working/EXP2_REMOTE_AUDIT")
audit_dir.mkdir(exist_ok=True)

verify_paths = [
    p for p in required
    if p.endswith((
        "epoch_050_last.pt",
        "best.pt",
        "last.pt",
        ".zip",
        "latest_backup_manifest.json",
        "Training_Summary.json"
    ))
]

verified = []

for remote_path in verify_paths:
    if remote_path not in remote_files:
        continue

    local = hf_hub_download(
        repo_id=REPO,
        repo_type=TYPE,
        filename=remote_path,
        token=TOKEN,
        force_download=True,
        local_dir=str(audit_dir)
    )

    digest = sha256(local)
    size = Path(local).stat().st_size

    verified.append(remote_path)

    print(
        "SHA256 PASS |",
        f"{size / (1024**2):.2f} MB |",
        digest[:16],
        "|",
        remote_path
    )

print("\n" + "=" * 60)

if not missing and len(verified) == len(verify_paths):
    print("EXPERIMENT 2 REMOTE BACKUP AUDIT: PASS")
    print("All required files found and selected files re-downloaded.")
else:
    print("EXPERIMENT 2 REMOTE BACKUP AUDIT: INCOMPLETE")
    print("Missing files:", missing)
    print("Verified files:", len(verified), "/", len(verify_paths))

print("=" * 60)

In [ ]:

# ============================================================
# EXPERIMENT 2 — ANTI-UAV300 TEST-SET EVALUATION
# Visible + Infrared | Evaluation only | HF backup included
# ============================================================

import os
import json
import yaml
import hashlib
import zipfile
import shutil
from pathlib import Path
from datetime import datetime, timezone

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download
from ultralytics import YOLO

# -------------------- CONFIGURATION --------------------

REPO = "SakibM19/border-surveillance-research-backups"
REPO_TYPE = "dataset"

BASE = Path("/kaggle/working/EXP2_RESTORED/AntiUAV_Experiment2")
DATASETS = BASE / "datasets"

EVAL_ROOT = Path("/kaggle/working/EXP2_TEST_EVALUATION")
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=TOKEN)

modalities = {
    "Visible": {
        "dataset": DATASETS / "visible",
        "config": "Experiment2/Training_Configs/visible_data_restored.yaml",
        "model": "Experiment2/Training/Visible/final/best.pt",
    },
    "Infrared": {
        "dataset": DATASETS / "infrared",
        "config": "Experiment2/Training_Configs/infrared_data_restored.yaml",
        "model": "Experiment2/Training/Infrared/final/best.pt",
    },
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def normalize_split(value):
    """Keep dataset-relative paths when adapting restored YAML files."""
    if isinstance(value, list):
        return [normalize_split(v) for v in value]

    value = str(value).replace("\\", "/")
    if "/images/" in value:
        return "images/" + value.split("/images/", 1)[1]

    if value.startswith("/"):
        raise ValueError(f"Could not safely normalize dataset path: {value}")

    return value

# -------------------- PRE-FLIGHT CHECKS --------------------

print("=" * 65)
print("EXPERIMENT 2 — TEST EVALUATION PRE-FLIGHT")
print("=" * 65)

assert DATASETS.exists(), f"Restored dataset not found: {DATASETS}"

api.whoami()
print("Hugging Face authentication: PASS")

import torch
assert torch.cuda.is_available(), "CUDA GPU is not available."
print("GPU:", torch.cuda.get_device_name(0))

for name, info in modalities.items():
    dataset_root = info["dataset"]

    assert dataset_root.exists(), f"{name} dataset missing"
    assert (dataset_root / "images" / "test").exists(), (
        f"{name} test images missing"
    )
    assert (dataset_root / "labels" / "test").exists(), (
        f"{name} test labels missing"
    )

    print(
        f"{name}: test images = "
        f"{len(list((dataset_root / 'images' / 'test').glob('*')))}"
    )

print("\nPre-flight checks: PASS")

# -------------------- EVALUATION --------------------

summary = {
    "experiment": "Experiment2_AntiUAV300",
    "stage": "Independent test-set evaluation",
    "repository": REPO,
    "evaluation_time_utc": datetime.now(timezone.utc).isoformat(),
    "model": "YOLO11n",
    "image_size": 640,
    "split": "test",
    "device": torch.cuda.get_device_name(0),
    "results": {},
}

for name, info in modalities.items():

    print("\n" + "=" * 65)
    print(f"STARTING {name.upper()} TEST EVALUATION")
    print("=" * 65)

    # Download the verified best model from Hugging Face.
    model_path = hf_hub_download(
        repo_id=REPO,
        repo_type=REPO_TYPE,
        filename=info["model"],
        token=TOKEN,
        force_download=True,
    )

    print("Model downloaded:", info["model"])
    print("Model size:", round(Path(model_path).stat().st_size / (1024**2), 2), "MB")

    # Download the corresponding restored dataset configuration.
    config_path = hf_hub_download(
        repo_id=REPO,
        repo_type=REPO_TYPE,
        filename=info["config"],
        token=TOKEN,
        force_download=True,
    )

    with open(config_path, "r") as f:
        data_config = yaml.safe_load(f)

    # Point the YAML to the restored Kaggle dataset.
    data_config["path"] = str(info["dataset"])

    for split in ["train", "val", "test"]:
        if split in data_config:
            data_config[split] = normalize_split(data_config[split])

    local_config = EVAL_ROOT / f"{name.lower()}_test_data.yaml"

    with open(local_config, "w") as f:
        yaml.safe_dump(data_config, f, sort_keys=False)

    # Confirm that the test image path resolves.
    test_path = info["dataset"] / "images" / "test"
    assert test_path.exists(), f"Test directory missing: {test_path}"

    # Load model and evaluate on the untouched test split.
    model = YOLO(model_path)

    run_dir = EVAL_ROOT / name

    metrics = model.val(
        data=str(local_config),
        split="test",
        imgsz=640,
        batch=16,
        device=0,
        workers=4,
        plots=True,
        save_json=True,
        save_txt=True,
        save_conf=True,
        project=str(EVAL_ROOT),
        name=name,
        exist_ok=True,
        verbose=True,
    )

    # Extract evaluation metrics.
    box = metrics.box

    result = {
        "status": "COMPLETED",
        "modality": name,
        "model_repository_path": info["model"],
        "test_split": "test",
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
        "per_class_mAP50_95": [float(x) for x in box.maps],
        "results_directory": str(run_dir),
    }

    # Preserve Ultralytics' complete metrics dictionary as well.
    if hasattr(metrics, "results_dict"):
        result["ultralytics_results"] = {
            str(k): float(v)
            for k, v in metrics.results_dict.items()
        }

    summary["results"][name] = result

    with open(run_dir / "evaluation_metrics.json", "w") as f:
        json.dump(result, f, indent=2)

    print(f"\n{name.upper()} TEST EVALUATION COMPLETE")
    print("Precision:", round(result["precision"], 5))
    print("Recall:", round(result["recall"], 5))
    print("mAP50:", round(result["mAP50"], 5))
    print("mAP50-95:", round(result["mAP50_95"], 5))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# -------------------- SAVE MASTER SUMMARY --------------------

summary_path = EVAL_ROOT / "Experiment2_Test_Evaluation_Summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

# Include the exact evaluation configurations in the evidence.
# The model weights are already backed up separately on HF.

# -------------------- CREATE COMPLETE EVIDENCE ARCHIVE --------------------

archive_path = EVAL_ROOT / "Experiment2_Test_Evaluation_Evidence.zip"

with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in EVAL_ROOT.rglob("*"):
        if file.is_file() and file != archive_path:
            z.write(file, file.relative_to(EVAL_ROOT))

print("\nEvidence archive created:", archive_path)
print("Archive size:", round(archive_path.stat().st_size / (1024**2), 2), "MB")

# -------------------- UPLOAD TO HUGGING FACE --------------------

remote_archive = "Experiment2/Evaluation/Experiment2_Test_Evaluation_Evidence.zip"
remote_summary = "Experiment2/Evaluation/Experiment2_Test_Evaluation_Summary.json"

print("\nUPLOADING TEST EVALUATION EVIDENCE...")

api.upload_file(
    path_or_fileobj=str(archive_path),
    path_in_repo=remote_archive,
    repo_id=REPO,
    repo_type=REPO_TYPE,
    token=TOKEN,
)

api.upload_file(
    path_or_fileobj=str(summary_path),
    path_in_repo=remote_summary,
    repo_id=REPO,
    repo_type=REPO_TYPE,
    token=TOKEN,
)

# -------------------- REMOTE DOWNLOAD + SHA256 VERIFICATION --------------------

print("\nVERIFYING REMOTE BACKUP...")

verify_dir = EVAL_ROOT / "remote_verification"
verify_dir.mkdir(exist_ok=True)

for remote_path, local_path in [
    (remote_archive, archive_path),
    (remote_summary, summary_path),
]:
    downloaded = hf_hub_download(
        repo_id=REPO,
        repo_type=REPO_TYPE,
        filename=remote_path,
        token=TOKEN,
        force_download=True,
        local_dir=str(verify_dir),
    )

    local_hash = sha256(local_path)
    remote_hash = sha256(downloaded)

    assert local_hash == remote_hash, (
        f"SHA256 MISMATCH: {remote_path}"
    )

    print(
        "SHA256 MATCH: PASS |",
        round(Path(downloaded).stat().st_size / (1024**2), 2),
        "MB |",
        local_hash[:16],
        "|",
        remote_path,
    )

# -------------------- FINAL STATUS --------------------

print("\n" + "=" * 65)
print("EXPERIMENT 2 TEST EVALUATION STATUS")
print("=" * 65)

for name in modalities:
    print(name, ":", summary["results"][name]["status"])

print("REMOTE BACKUP: VERIFIED")
print("TEST EVALUATION: COMPLETED")
print("TRAINING: NOT PERFORMED")
print("=" * 65)


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "SakibM19/border-surveillance-research-backups"

api = HfApi(token=TOKEN)

files = api.list_repo_files(
    repo_id=REPO,
    repo_type="dataset"
)

print("EXPERIMENT 3 — REMOTE FILE INVENTORY")
print("=" * 55)

exp3_files = [
    f for f in files
    if "experiment3" in f.lower()
    or "range" in f.lower()
    or "distance" in f.lower()
]

if exp3_files:
    for f in exp3_files:
        print(f)
else:
    print("No Experiment 3 / range / distance files found.")

print("=" * 55)
print("TOTAL MATCHING FILES:", len(exp3_files))

In [ ]:

from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
import zipfile
import shutil

REPO = "SakibM19/border-surveillance-research-backups"
TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

ROOT = Path("/kaggle/working/EXP2_PAPER_EVIDENCE")
ROOT.mkdir(parents=True, exist_ok=True)

archives = {
    "Visible_Training": "Experiment2/Training/Visible/final/visible_complete_results.zip",
    "Infrared_Training": "Experiment2/Training/Infrared/final/infrared_complete_results.zip",
    "Test_Evaluation": "Experiment2/Evaluation/Experiment2_Test_Evaluation_Evidence.zip",
}

figure_root = ROOT / "Figures"
figure_root.mkdir(exist_ok=True)

print("DOWNLOADING VERIFIED EXPERIMENT 2 ARCHIVES")

for name, remote_path in archives.items():
    print("\nProcessing:", name)

    downloaded = hf_hub_download(
        repo_id=REPO,
        repo_type="dataset",
        filename=remote_path,
        token=TOKEN,
        force_download=True,
    )

    extract_dir = ROOT / name
    extract_dir.mkdir(exist_ok=True)

    with zipfile.ZipFile(downloaded, "r") as z:
        z.extractall(extract_dir)

    print("Extracted:", extract_dir)

    destination = figure_root / name
    destination.mkdir(exist_ok=True)

    images = [
        p for p in extract_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ]

    for image in images:
        target = destination / image.name

        # Avoid overwriting files with the same name.
        if target.exists():
            target = destination / f"{image.parent.name}_{image.name}"

        shutil.copy2(image, target)

    print("Figures found:", len(images))

print("\n" + "=" * 60)
print("EXPERIMENT 2 PAPER FIGURES")
print("=" * 60)

all_figures = list(figure_root.rglob("*"))
all_figures = [p for p in all_figures if p.is_file()]

for p in all_figures:
    print(p.relative_to(ROOT))

print("\nTOTAL FIGURE FILES:", len(all_figures))
print("FIGURE DIRECTORY:", figure_root)
print("=" * 60)


In [ ]:

from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

ROOT = Path("/kaggle/working/EXP2_PAPER_EVIDENCE/Figures")

categories = [
    "Visible_Training",
    "Infrared_Training",
    "Test_Evaluation",
]

for category in categories:
    folder = ROOT / category

    files = sorted([
        p for p in folder.iterdir()
        if p.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ])

    if not files:
        continue

    thumb_w, thumb_h = 360, 260
    label_h = 42
    columns = 3
    rows = (len(files) + columns - 1) // columns

    sheet = Image.new(
        "RGB",
        (columns * thumb_w, rows * (thumb_h + label_h)),
        "white"
    )

    draw = ImageDraw.Draw(sheet)

    for i, path in enumerate(files):
        try:
            img = Image.open(path).convert("RGB")
            img.thumbnail((thumb_w - 12, thumb_h - 12))

            x = (i % columns) * thumb_w
            y = (i // columns) * (thumb_h + label_h)

            sheet.paste(img, (x + 6, y + 6))

            draw.text(
                (x + 6, y + thumb_h),
                path.name[:48],
                fill="black"
            )

        except Exception as e:
            print("Could not load:", path.name, e)

    output = Path("/kaggle/working/EXP2_PAPER_EVIDENCE") / (
        category + "_CONTACT_SHEET.jpg"
    )

    sheet.save(output, quality=90)

    print("\n", category, "|", len(files), "figures")
    display(sheet)

    print("Saved:", output)


In [ ]:

import shutil
from IPython.display import FileLink, display

source = "/kaggle/working/EXP2_PAPER_EVIDENCE/Figures"
archive = "/kaggle/working/Experiment2_All_64_Paper_Figures"

shutil.make_archive(archive, "zip", source)

print("Download all Experiment 2 figures:")
display(FileLink(archive + ".zip"))


In [ ]:
from IPython.display import FileLink, display

display(FileLink(
    "/kaggle/working/EXP2_PAPER_EVIDENCE/Test_Evaluation_CONTACT_SHEET.jpg"
))

In [ ]:

import zipfile
from pathlib import Path
from IPython.display import display, FileLink

folder = Path("/kaggle/working/EXP2_PAPER_EVIDENCE")

files = [
    folder / "Visible_Training_CONTACT_SHEET.jpg",
    folder / "Infrared_Training_CONTACT_SHEET.jpg",
    folder / "Test_Evaluation_CONTACT_SHEET.jpg",
]

zip_path = Path("/kaggle/working/Experiment2_Contact_Sheets.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files:
        if file.exists():
            z.write(file, arcname=file.name)
            print("Added:", file.name)
        else:
            print("MISSING:", file.name)

print("\nZIP created:", zip_path)
print("Size:", round(zip_path.stat().st_size / 1024, 2), "KB")


In [ ]:

from pathlib import Path
from PIL import Image

ROOT = Path("/kaggle/working/EXP2_PAPER_EVIDENCE/Figures")

print("=" * 75)
print("EXPERIMENT 2 — ORIGINAL FIGURE RESOLUTION AUDIT")
print("=" * 75)

for category in ["Visible_Training", "Infrared_Training", "Test_Evaluation"]:
    folder = ROOT / category

    print(f"\n--- {category} ---")

    for p in sorted(folder.iterdir()):
        if p.suffix.lower() not in [".png", ".jpg", ".jpeg"]:
            continue

        try:
            with Image.open(p) as img:
                width, height = img.size
                dpi = img.info.get("dpi", "Not specified")

                print(
                    f"{p.name:42s} "
                    f"{width:5d} x {height:5d} | DPI: {dpi}"
                )

        except Exception as e:
            print("ERROR:", p.name, e)

print("\n" + "=" * 75)
print("AUDIT COMPLETE")
print("=" * 75)


In [ ]:

from pathlib import Path

ROOT = Path("/kaggle/working/EXP2_PAPER_EVIDENCE")

print("=" * 75)
print("EXPERIMENT 2 — LOCATING SOURCE DATA FOR IEEE FIGURES")
print("=" * 75)

targets = {
    "results.csv",
    "Experiment2_Test_Evaluation_Summary.json",
    "Experiment2_Training_Summary.json",
}

for target in sorted(targets):
    print(f"\n--- {target} ---")
    matches = list(ROOT.rglob(target))

    if not matches:
        print("NOT FOUND")
    else:
        for p in matches:
            print(f"FOUND: {p}")
            print(f"SIZE: {p.stat().st_size:,} bytes")

print("\n--- FIGURE SOURCE DIRECTORIES ---")
for p in sorted(ROOT.iterdir()):
    if p.is_dir():
        print(p)

print("\nSOURCE FILE SEARCH COMPLETE")


In [ ]:

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/kaggle/working/EXP2_PAPER_EVIDENCE")

print("=" * 75)
print("EXPERIMENT 2 — SOURCE DATA INSPECTION")
print("=" * 75)

# 1. Inspect Visible training results
for modality in ["Visible_Training", "Infrared_Training"]:
    csv_path = ROOT / modality / "results.csv"

    print(f"\n{'=' * 65}")
    print(f"{modality.upper()} — RESULTS.CSV")
    print("=" * 65)

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    print("Rows (epochs):", len(df))
    print("\nAvailable columns:")
    for col in df.columns:
        print(" -", col)

    print("\nFirst 3 rows:")
    print(df.head(3).to_string(index=False))

    print("\nLast 3 rows:")
    print(df.tail(3).to_string(index=False))

    print("\nMissing values:", int(df.isna().sum().sum()))

# 2. Inspect test evaluation summary
summary_path = (
    ROOT / "Test_Evaluation" /
    "Experiment2_Test_Evaluation_Summary.json"
)

print("\n" + "=" * 65)
print("TEST EVALUATION SUMMARY")
print("=" * 65)

with open(summary_path, "r") as f:
    summary = json.load(f)

print(json.dumps(summary, indent=2))

print("\n" + "=" * 75)
print("SOURCE DATA INSPECTION COMPLETE")
print("=" * 75)


In [ ]:

from pathlib import Path
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import shutil

# ============================================================
# PATHS
# ============================================================

ROOT = Path("/kaggle/working/EXP2_PAPER_EVIDENCE")
OUT = ROOT / "IEEE_300DPI"

OUT.mkdir(parents=True, exist_ok=True)

DPI = 600

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 150,
    "savefig.dpi": DPI,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# ============================================================
# HELPER: SAVE PNG + VECTOR PDF
# ============================================================

manifest = []

def save_figure(fig, filename, description):
    png_path = OUT / f"{filename}.png"
    pdf_path = OUT / f"{filename}.pdf"

    fig.savefig(
        png_path,
        dpi=DPI,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        format="pdf",
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    manifest.append({
        "filename": filename,
        "description": description,
        "png": str(png_path),
        "pdf": str(pdf_path),
        "png_dpi": DPI
    })

    print("CREATED:", filename)

# ============================================================
# LOAD AND VALIDATE TRAINING DATA
# ============================================================

data = {}

for modality in ["Visible", "Infrared"]:

    csv_path = ROOT / f"{modality}_Training" / "results.csv"

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    required = [
        "epoch",
        "train/box_loss",
        "train/cls_loss",
        "train/dfl_loss",
        "val/box_loss",
        "val/cls_loss",
        "val/dfl_loss",
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)"
    ]

    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"{modality}: Missing columns: {missing}")

    if len(df) != 50:
        raise ValueError(
            f"{modality}: Expected 50 epochs, found {len(df)}"
        )

    if df[required].isna().any().any():
        raise ValueError(f"{modality}: Missing values detected")

    data[modality] = df

    print(
        f"{modality}: {len(df)} epochs loaded and validated"
    )

# ============================================================
# FIGURES 1–3: VISIBLE
# FIGURES 4–6: INFRARED
# ============================================================

for modality in ["Visible", "Infrared"]:

    df = data[modality]
    epochs = df["epoch"]

    # --------------------------------------------------------
    # FIGURE A: TRAINING AND VALIDATION LOSSES
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        1, 3,
        figsize=(10, 3.3),
        constrained_layout=True
    )

    loss_types = [
        ("box", "Box Loss"),
        ("cls", "Classification Loss"),
        ("dfl", "Distribution Focal Loss")
    ]

    for ax, (key, title) in zip(axes, loss_types):

        ax.plot(
            epochs,
            df[f"train/{key}_loss"],
            label="Training",
            linewidth=1.8
        )

        ax.plot(
            epochs,
            df[f"val/{key}_loss"],
            label="Validation",
            linewidth=1.8,
            linestyle="--"
        )

        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False)

    fig.suptitle(
        f"{modality} Modality: Training and Validation Losses",
        fontsize=12
    )

    save_figure(
        fig,
        f"Experiment2_{modality}_Training_Validation_Loss",
        f"{modality} training and validation loss across 50 epochs"
    )

    # --------------------------------------------------------
    # FIGURE B: PRECISION AND RECALL
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(6.5, 4.2))

    ax.plot(
        epochs,
        df["metrics/precision(B)"],
        label="Precision",
        linewidth=2
    )

    ax.plot(
        epochs,
        df["metrics/recall(B)"],
        label="Recall",
        linewidth=2,
        linestyle="--"
    )

    ax.set_title(f"{modality} Modality: Precision and Recall")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Metric")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)

    save_figure(
        fig,
        f"Experiment2_{modality}_Precision_Recall",
        f"{modality} precision and recall across training epochs"
    )

    # --------------------------------------------------------
    # FIGURE C: mAP
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(6.5, 4.2))

    ax.plot(
        epochs,
        df["metrics/mAP50(B)"],
        label="mAP@0.50",
        linewidth=2
    )

    ax.plot(
        epochs,
        df["metrics/mAP50-95(B)"],
        label="mAP@0.50:0.95",
        linewidth=2,
        linestyle="--"
    )

    ax.set_title(f"{modality} Modality: Validation mAP")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("mAP")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)

    save_figure(
        fig,
        f"Experiment2_{modality}_Validation_mAP",
        f"{modality} validation mAP across training epochs"
    )

# ============================================================
# FIGURE 7: INDEPENDENT TEST-SET COMPARISON
# ============================================================

summary_path = (
    ROOT / "Test_Evaluation" /
    "Experiment2_Test_Evaluation_Summary.json"
)

with open(summary_path, "r") as f:
    summary = json.load(f)

metrics = ["precision", "recall", "mAP50", "mAP50_95"]
metric_labels = [
    "Precision",
    "Recall",
    "mAP@0.50",
    "mAP@0.50:0.95"
]

visible_values = [
    summary["results"]["Visible"][m] for m in metrics
]

infrared_values = [
    summary["results"]["Infrared"][m] for m in metrics
]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.bar(
    x - width / 2,
    visible_values,
    width,
    label="Visible"
)

ax.bar(
    x + width / 2,
    infrared_values,
    width,
    label="Infrared"
)

ax.set_title("Anti-UAV300 Independent Test-Set Performance")
ax.set_ylabel("Metric")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, rotation=10, ha="right")
ax.set_ylim(0, 1.08)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)

save_figure(
    fig,
    "Experiment2_Visible_vs_Infrared_Test_Comparison",
    "Independent test-set comparison of Visible and Infrared models"
)

# ============================================================
# SAVE SOURCE DATA AND MANIFEST
# ============================================================

source_dir = OUT / "Source_Data"
source_dir.mkdir(exist_ok=True)

for modality in ["Visible", "Infrared"]:
    shutil.copy2(
        ROOT / f"{modality}_Training" / "results.csv",
        source_dir / f"{modality}_results.csv"
    )

shutil.copy2(
    summary_path,
    source_dir / "Experiment2_Test_Evaluation_Summary.json"
)

with open(OUT / "Figure_Manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# ============================================================
# FINAL AUDIT
# ============================================================

print("\n" + "=" * 70)
print("IEEE FIGURE GENERATION COMPLETE")
print("=" * 70)

print("Output directory:", OUT)
print("Figures generated:", len(manifest))
print("PNG resolution:", DPI, "DPI")
print("PDF versions: Vector export")
print("Source CSV and JSON files: Preserved")

for item in manifest:
    png = Path(item["png"])
    pdf = Path(item["pdf"])

    print(
        f"{item['filename']}: "
        f"PNG {png.stat().st_size / 1024:.1f} KB | "
        f"PDF {pdf.stat().size if hasattr(pdf.stat(), 'size') else pdf.stat().st_size / 1024:.1f} KB"
    )

print("\nOUTPUT FILES:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(OUT)} | {p.stat().st_size:,} bytes")

print("\nAUDIT COMPLETE")


In [ ]:

from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
import math

ROOT = Path(
    "/kaggle/working/EXP2_PAPER_EVIDENCE/IEEE_300DPI"
)

files = sorted(ROOT.glob("*.png"))

thumb_w = 900
thumb_h = 560
label_h = 45
columns = 2
rows = math.ceil(len(files) / columns)

sheet = Image.new(
    "RGB",
    (columns * thumb_w, rows * (thumb_h + label_h)),
    "white"
)

draw = ImageDraw.Draw(sheet)

for i, path in enumerate(files):
    img = Image.open(path).convert("RGB")
    img.thumbnail((thumb_w - 30, thumb_h - 30))

    x = (i % columns) * thumb_w
    y = (i // columns) * (thumb_h + label_h)

    px = x + (thumb_w - img.width) // 2
    py = y + (thumb_h - img.height) // 2

    sheet.paste(img, (px, py))

    draw.text(
        (x + 12, y + thumb_h + 8),
        path.stem,
        fill="black"
    )

output = ROOT / "IEEE_Figures_Review_Contact_Sheet.jpg"
sheet.save(output, quality=92)

print("CONTACT SHEET CREATED:", output)
print("Figures included:", len(files))
print("Use this sheet only for visual inspection.")


In [ ]:

from pathlib import Path
import zipfile
import hashlib
import json
import tempfile
import shutil
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

# ============================================================
# CONFIGURATION
# ============================================================

ROOT = Path("/kaggle/working/EXP2_PAPER_EVIDENCE")
FIG_DIR = ROOT / "IEEE_300DPI"

REPO_ID = "SakibM19/border-surveillance-research-backups"
REPO_TYPE = "dataset"

ARCHIVE = Path(
    "/kaggle/working/Experiment2_IEEE_600DPI_Paper_Evidence.zip"
)

REMOTE_PATH = (
    "Experiment2/Paper_Evidence/IEEE_600DPI/"
    "Experiment2_IEEE_600DPI_Paper_Evidence.zip"
)

# ============================================================
# LOAD TOKEN SECURELY
# ============================================================

TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

if not TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Kaggle Secrets.")

api = HfApi(token=TOKEN)

# ============================================================
# CHECK REQUIRED FILES
# ============================================================

required = [
    FIG_DIR / "Experiment2_Visible_Training_Validation_Loss.png",
    FIG_DIR / "Experiment2_Visible_Training_Validation_Loss.pdf",
    FIG_DIR / "Experiment2_Visible_Precision_Recall.png",
    FIG_DIR / "Experiment2_Visible_Precision_Recall.pdf",
    FIG_DIR / "Experiment2_Visible_Validation_mAP.png",
    FIG_DIR / "Experiment2_Visible_Validation_mAP.pdf",

    FIG_DIR / "Experiment2_Infrared_Training_Validation_Loss.png",
    FIG_DIR / "Experiment2_Infrared_Training_Validation_Loss.pdf",
    FIG_DIR / "Experiment2_Infrared_Precision_Recall.png",
    FIG_DIR / "Experiment2_Infrared_Precision_Recall.pdf",
    FIG_DIR / "Experiment2_Infrared_Validation_mAP.png",
    FIG_DIR / "Experiment2_Infrared_Validation_mAP.pdf",

    FIG_DIR / "Experiment2_Visible_vs_Infrared_Test_Comparison.png",
    FIG_DIR / "Experiment2_Visible_vs_Infrared_Test_Comparison.pdf",

    FIG_DIR / "Figure_Manifest.json",
    FIG_DIR / "Source_Data/Visible_results.csv",
    FIG_DIR / "Source_Data/Infrared_results.csv",
    FIG_DIR / "Source_Data/Experiment2_Test_Evaluation_Summary.json"
]

missing = [str(p) for p in required if not p.is_file()]

if missing:
    raise FileNotFoundError(
        "Required evidence files missing:\n" + "\n".join(missing)
    )

print("REQUIRED FILE AUDIT: PASS")
print("Files to package:", len(required))

# ============================================================
# CREATE ARCHIVE
# ============================================================

if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(
    ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in required:
        zf.write(path, path.relative_to(FIG_DIR))

# Verify ZIP integrity
with zipfile.ZipFile(ARCHIVE, "r") as zf:
    bad_file = zf.testzip()
    if bad_file:
        raise RuntimeError(f"ZIP integrity failure: {bad_file}")

    archived_files = zf.namelist()

if len(archived_files) != len(required):
    raise RuntimeError("Archive file count mismatch.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

local_sha = sha256_file(ARCHIVE)

print("\nARCHIVE CREATED")
print("Path:", ARCHIVE)
print("Size:", round(ARCHIVE.stat().st_size / (1024 * 1024), 3), "MB")
print("Files:", len(archived_files))
print("Local SHA256:", local_sha)

# ============================================================
# AUTHENTICATION AND REPOSITORY ACCESS
# ============================================================

api.repo_info(repo_id=REPO_ID, repo_type=REPO_TYPE)
print("\nPRIVATE REPOSITORY ACCESS: PASS")

# ============================================================
# UPLOAD
# ============================================================

api.upload_file(
    path_or_fileobj=str(ARCHIVE),
    path_in_repo=REMOTE_PATH,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    commit_message="Add verified Experiment 2 IEEE-ready figures and source evidence"
)

print("UPLOAD: SUCCESS")

# ============================================================
# REMOTE LISTING VERIFICATION
# ============================================================

remote_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE
)

if REMOTE_PATH not in remote_files:
    raise RuntimeError("Uploaded archive not found in remote repository.")

print("REMOTE LISTING: PASS")

# ============================================================
# DOWNLOAD AND SHA256 VERIFICATION
# ============================================================

with tempfile.TemporaryDirectory() as temp_dir:

    downloaded = hf_hub_download(
        repo_id=REPO_ID,
        filename=REMOTE_PATH,
        repo_type=REPO_TYPE,
        token=TOKEN,
        force_download=True,
        local_dir=temp_dir
    )

    downloaded = Path(downloaded)
    remote_sha = sha256_file(downloaded)

    if local_sha != remote_sha:
        raise RuntimeError("SHA256 mismatch after remote download.")

    with zipfile.ZipFile(downloaded, "r") as zf:
        if zf.testzip() is not None:
            raise RuntimeError("Downloaded archive failed ZIP integrity.")

        remote_contents = zf.namelist()

    if set(remote_contents) != set(archived_files):
        raise RuntimeError("Remote archive contents do not match local archive.")

print("\n" + "=" * 65)
print("EXPERIMENT 2 — IEEE FIGURE BACKUP AUDIT")
print("=" * 65)
print("Archive creation       : PASS")
print("ZIP integrity          : PASS")
print("Private repository     : PASS")
print("Upload                 : PASS")
print("Remote listing         : PASS")
print("Remote download        : PASS")
print("SHA256 integrity       : PASS")
print("Archive contents       : MATCH")
print("SHA256                 :", local_sha)
print("=" * 65)
print("FINAL STATUS: VERIFIED REMOTE BACKUP")


In [ ]:

# ============================================================
# EXPERIMENT 3 — HUGGING FACE BACKUP PREFLIGHT
# ============================================================

import os
import json
import hashlib
import tempfile
from datetime import datetime, timezone

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = "SakibM19/border-surveillance-research-backups"
REPO_TYPE = "dataset"

print("=" * 65)
print("EXPERIMENT 3 — BACKUP PREFLIGHT")
print("=" * 65)

# 1. Load secret securely
TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert TOKEN, "HF_TOKEN was not found in Kaggle Secrets."

api = HfApi(token=TOKEN)

# 2. Verify authentication and private repository access
who = api.whoami()
print("Authentication: PASS")
print("Account:", who.get("name", "authenticated"))

info = api.repo_info(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=TOKEN
)

assert info.private is True, "Repository is not private."
print("Private repository: PASS")

# 3. Create a small preflight test artifact
test_content = {
    "experiment": "Experiment3",
    "purpose": "Hugging Face backup preflight",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PREFLIGHT_TEST"
}

local_path = "/kaggle/working/Experiment3_HF_Preflight.json"

with open(local_path, "w") as f:
    json.dump(test_content, f, indent=2)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

local_hash = sha256_file(local_path)

# 4. Upload test artifact
remote_path = "Experiment3/Preflight/Experiment3_HF_Preflight.json"

api.upload_file(
    path_or_fileobj=local_path,
    path_in_repo=remote_path,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=TOKEN,
    commit_message="Experiment 3: verify backup connectivity"
)

print("Upload: PASS")

# 5. Verify remote listing
remote_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=TOKEN
)

assert remote_path in remote_files, "Uploaded file missing from remote listing."
print("Remote listing: PASS")

# 6. Download remote copy
downloaded_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=remote_path,
    repo_type=REPO_TYPE,
    token=TOKEN,
    force_download=True
)

remote_hash = sha256_file(downloaded_path)

assert local_hash == remote_hash, "SHA-256 mismatch!"
print("Remote download: PASS")
print("SHA-256 integrity: PASS")

print("\n" + "=" * 65)
print("EXPERIMENT 3 PREFLIGHT: PASSED")
print("Authentication       : PASS")
print("Private repository   : PASS")
print("Write access         : PASS")
print("Remote listing       : PASS")
print("Download             : PASS")
print("SHA-256 integrity    : PASS")
print("=" * 65)


In [ ]:

# ============================================================
# EXPERIMENT 3
# RANGE-TO-PIXEL AND DISTANCE-ERROR ANALYSIS
#
# CPU ONLY — NO MODEL TRAINING
# All camera/target specifications are explicit assumptions.
# ============================================================

import os
import json
import math
import hashlib
import zipfile
import shutil
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime, timezone
from huggingface_hub import HfApi, hf_hub_download

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

REPO_ID = "SakibM19/border-surveillance-research-backups"
REPO_TYPE = "dataset"

OUT = "/kaggle/working/Experiment3_Range_Distance_Analysis"
FIG = os.path.join(OUT, "Figures")
DATA = os.path.join(OUT, "Data")
REPORT = os.path.join(OUT, "Reports")

for folder in [OUT, FIG, DATA, REPORT]:
    os.makedirs(folder, exist_ok=True)

CONFIG = {
    "experiment": "Experiment3_Range_to_Pixel_and_Distance_Error",
    "status": "COMPUTATIONAL_GEOMETRIC_ANALYSIS",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "method": "Pinhole camera projection approximation",
    "target": {
        "name": "DJI Mini 2 (reference target)",
        "width_m": 0.289,
        "width_status": "ASSUMED_NOT_MANUFACTURER_VERIFIED"
    },
    "RGB": {
        "width_px": 1920,
        "height_px": 1080,
        "horizontal_fov_deg": 20.0,
        "specification_status": "ASSUMED_NOT_VERIFIED"
    },
    "Thermal": {
        "width_px": 640,
        "height_px": 512,
        "horizontal_fov_deg": 18.0,
        "specification_status": "ASSUMED_NOT_VERIFIED"
    },
    "distances_m": [50, 100, 150, 200, 250, 300],
    "pixel_perturbations": [-3, -2, -1, 0, 1, 2, 3],
    "limitations": [
        "Idealized pinhole-camera geometry",
        "Assumed target width and camera field of view",
        "No real camera calibration",
        "No lens distortion or perspective variation",
        "No real-world detection or ranging measurements",
        "Pixel perturbations are deterministic sensitivity tests"
    ]
}

with open(os.path.join(REPORT, "Experiment3_Configuration.json"), "w") as f:
    json.dump(CONFIG, f, indent=4)

# ------------------------------------------------------------
# 2. CAMERA GEOMETRY
# ------------------------------------------------------------

TARGET_WIDTH_M = CONFIG["target"]["width_m"]
DISTANCES = CONFIG["distances_m"]
PERTURBATIONS = CONFIG["pixel_perturbations"]

cameras = {
    "RGB": CONFIG["RGB"],
    "Thermal": CONFIG["Thermal"]
}

camera_results = {}

for name, cam in cameras.items():
    hfov_rad = math.radians(cam["horizontal_fov_deg"])
    fx = cam["width_px"] / (2.0 * math.tan(hfov_rad / 2.0))

    camera_results[name] = {
        "fx_pixels": fx,
        "horizontal_fov_deg": cam["horizontal_fov_deg"],
        "image_width_px": cam["width_px"],
        "image_height_px": cam["height_px"]
    }

# ------------------------------------------------------------
# 3. RANGE-TO-PIXEL CALCULATIONS
# ------------------------------------------------------------

projection_rows = []
error_rows = []

for camera_name, cam in cameras.items():

    fx = camera_results[camera_name]["fx_pixels"]

    for distance in DISTANCES:

        true_pixels = (fx * TARGET_WIDTH_M) / distance

        projection_rows.append({
            "camera": camera_name,
            "distance_m": distance,
            "target_width_m": TARGET_WIDTH_M,
            "focal_length_pixels": fx,
            "projected_width_pixels": true_pixels
        })

        for delta in PERTURBATIONS:

            measured_pixels = true_pixels + delta

            if measured_pixels <= 0:
                estimated_distance = np.nan
                absolute_error = np.nan
                relative_error = np.nan
                valid = False
            else:
                estimated_distance = (
                    fx * TARGET_WIDTH_M / measured_pixels
                )

                absolute_error = abs(
                    estimated_distance - distance
                )

                relative_error = (
                    absolute_error / distance
                ) * 100.0

                valid = True

            error_rows.append({
                "camera": camera_name,
                "true_distance_m": distance,
                "true_width_pixels": true_pixels,
                "pixel_perturbation": delta,
                "measured_width_pixels": measured_pixels,
                "estimated_distance_m": estimated_distance,
                "absolute_error_m": absolute_error,
                "relative_error_percent": relative_error,
                "valid_estimate": valid
            })

projection_df = pd.DataFrame(projection_rows)
error_df = pd.DataFrame(error_rows)

projection_df.to_csv(
    os.path.join(DATA, "Range_to_Pixel_Projection.csv"),
    index=False
)

error_df.to_csv(
    os.path.join(DATA, "Pixel_Perturbation_Distance_Errors.csv"),
    index=False
)

# ------------------------------------------------------------
# 4. SUMMARY METRICS
# ------------------------------------------------------------

summary_rows = []

for camera_name in cameras:

    subset = error_df[
        (error_df["camera"] == camera_name) &
        (error_df["pixel_perturbation"] != 0) &
        (error_df["valid_estimate"])
    ]

    errors = subset["estimated_distance_m"].values - \
             subset["true_distance_m"].values

    mae = float(np.mean(np.abs(errors)))
    rmse = float(np.sqrt(np.mean(errors ** 2)))
    mean_relative_error = float(
        subset["relative_error_percent"].mean()
    )

    total_nonzero = len(
        error_df[
            (error_df["camera"] == camera_name) &
            (error_df["pixel_perturbation"] != 0)
        ]
    )

    valid_count = len(subset)

    summary_rows.append({
        "camera": camera_name,
        "valid_perturbed_estimates": valid_count,
        "invalid_perturbed_estimates": total_nonzero - valid_count,
        "MAE_m": mae,
        "RMSE_m": rmse,
        "mean_relative_error_percent": mean_relative_error
    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    os.path.join(DATA, "Experiment3_Error_Summary.csv"),
    index=False
)

# Per-distance sensitivity metrics
distance_summary = (
    error_df[
        (error_df["pixel_perturbation"] != 0) &
        (error_df["valid_estimate"])
    ]
    .groupby(["camera", "true_distance_m"])
    .agg(
        valid_cases=("absolute_error_m", "count"),
        MAE_m=("absolute_error_m", "mean"),
        RMSE_m=("absolute_error_m",
                lambda x: np.sqrt(np.mean(np.square(x)))),
        mean_relative_error_percent=(
            "relative_error_percent", "mean"
        )
    )
    .reset_index()
)

distance_summary.to_csv(
    os.path.join(DATA, "Distance_Level_Error_Summary.csv"),
    index=False
)

# ------------------------------------------------------------
# 5. FIGURE 1 — PROJECTED TARGET WIDTH
# ------------------------------------------------------------

plt.figure(figsize=(9, 5.5))

for camera_name in cameras:
    subset = projection_df[
        projection_df["camera"] == camera_name
    ]

    plt.plot(
        subset["distance_m"],
        subset["projected_width_pixels"],
        marker="o",
        linewidth=2,
        label=camera_name
    )

plt.xlabel("Target Distance (m)")
plt.ylabel("Projected Target Width (pixels)")
plt.title("Projected Target Width vs. Distance")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(FIG, "Figure3_1_Projected_Width.png"),
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(
    os.path.join(FIG, "Figure3_1_Projected_Width.pdf"),
    bbox_inches="tight"
)
plt.close()

# ------------------------------------------------------------
# 6. FIGURE 2 — ABSOLUTE DISTANCE ERROR
# ------------------------------------------------------------

plt.figure(figsize=(9, 5.5))

for camera_name in cameras:
    subset = distance_summary[
        distance_summary["camera"] == camera_name
    ]

    plt.plot(
        subset["true_distance_m"],
        subset["MAE_m"],
        marker="o",
        linewidth=2,
        label=camera_name
    )

plt.xlabel("True Target Distance (m)")
plt.ylabel("Mean Absolute Distance Error (m)")
plt.title("Distance Estimation Error vs. Target Distance")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(FIG, "Figure3_2_Distance_Error.png"),
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(
    os.path.join(FIG, "Figure3_2_Distance_Error.pdf"),
    bbox_inches="tight"
)
plt.close()

# ------------------------------------------------------------
# 7. FIGURE 3 — RELATIVE DISTANCE ERROR
# ------------------------------------------------------------

plt.figure(figsize=(9, 5.5))

for camera_name in cameras:
    subset = distance_summary[
        distance_summary["camera"] == camera_name
    ]

    plt.plot(
        subset["true_distance_m"],
        subset["mean_relative_error_percent"],
        marker="o",
        linewidth=2,
        label=camera_name
    )

plt.xlabel("True Target Distance (m)")
plt.ylabel("Mean Relative Error (%)")
plt.title("Relative Distance Error vs. Target Distance")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(FIG, "Figure3_3_Relative_Error.png"),
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(
    os.path.join(FIG, "Figure3_3_Relative_Error.pdf"),
    bbox_inches="tight"
)
plt.close()

# ------------------------------------------------------------
# 8. FIGURE 4 — PIXEL ERROR SENSITIVITY
# ------------------------------------------------------------

plt.figure(figsize=(9, 5.5))

for camera_name in cameras:

    subset = error_df[
        (error_df["camera"] == camera_name) &
        (error_df["valid_estimate"])
    ]

    grouped = subset.groupby(
        "pixel_perturbation"
    )["relative_error_percent"].mean()

    plt.plot(
        grouped.index,
        grouped.values,
        marker="o",
        linewidth=2,
        label=camera_name
    )

plt.xlabel("Pixel Measurement Perturbation (pixels)")
plt.ylabel("Mean Relative Error (%)")
plt.title("Sensitivity to Pixel Measurement Error")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(FIG, "Figure3_4_Pixel_Sensitivity.png"),
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(
    os.path.join(FIG, "Figure3_4_Pixel_Sensitivity.pdf"),
    bbox_inches="tight"
)
plt.close()

# ------------------------------------------------------------
# 9. FIGURE MANIFEST AND TECHNICAL REPORT
# ------------------------------------------------------------

figure_manifest = {
    "figure_count": 4,
    "raster_format": "PNG",
    "raster_dpi": 600,
    "vector_format": "PDF",
    "figures": [
        "Figure3_1_Projected_Width",
        "Figure3_2_Distance_Error",
        "Figure3_3_Relative_Error",
        "Figure3_4_Pixel_Sensitivity"
    ],
    "source_data": [
        "Range_to_Pixel_Projection.csv",
        "Pixel_Perturbation_Distance_Errors.csv",
        "Experiment3_Error_Summary.csv",
        "Distance_Level_Error_Summary.csv"
    ]
}

with open(os.path.join(REPORT, "Figure_Manifest.json"), "w") as f:
    json.dump(figure_manifest, f, indent=4)

technical_report = {
    "experiment": CONFIG["experiment"],
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "camera_geometry": camera_results,
    "overall_error_summary": summary_df.to_dict(orient="records"),
    "interpretation_warning": (
        "These are idealized simulated geometric sensitivity results. "
        "They do not establish real-world ranging accuracy."
    )
}

with open(os.path.join(REPORT, "Experiment3_Technical_Report.json"), "w") as f:
    json.dump(technical_report, f, indent=4)

# ------------------------------------------------------------
# 10. CREATE COMPLETE EVIDENCE ARCHIVE
# ------------------------------------------------------------

ARCHIVE = "/kaggle/working/Experiment3_Range_Distance_Evidence.zip"

if os.path.exists(ARCHIVE):
    os.remove(ARCHIVE)

with zipfile.ZipFile(
    ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED
) as zf:

    for root, dirs, files in os.walk(OUT):
        for filename in files:
            full_path = os.path.join(root, filename)
            relative_path = os.path.relpath(full_path, OUT)
            zf.write(full_path, relative_path)

# Local SHA-256
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

local_hash = sha256_file(ARCHIVE)

# ------------------------------------------------------------
# 11. UPLOAD TO PRIVATE HUGGING FACE REPOSITORY
# ------------------------------------------------------------

from kaggle_secrets import UserSecretsClient

TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert TOKEN, "HF_TOKEN missing."

api = HfApi(token=TOKEN)

remote_archive = (
    "Experiment3/Paper_Evidence/"
    "Experiment3_Range_Distance_Evidence.zip"
)

remote_report = (
    "Experiment3/Paper_Evidence/"
    "Experiment3_Technical_Report.json"
)

api.upload_file(
    path_or_fileobj=ARCHIVE,
    path_in_repo=remote_archive,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=TOKEN,
    commit_message="Experiment 3: upload range-distance evidence"
)

api.upload_file(
    path_or_fileobj=os.path.join(
        REPORT, "Experiment3_Technical_Report.json"
    ),
    path_in_repo=remote_report,
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=TOKEN,
    commit_message="Experiment 3: upload technical report"
)

# ------------------------------------------------------------
# 12. REMOTE DOWNLOAD AND INTEGRITY VERIFICATION
# ------------------------------------------------------------

remote_download = hf_hub_download(
    repo_id=REPO_ID,
    filename=remote_archive,
    repo_type=REPO_TYPE,
    token=TOKEN,
    force_download=True
)

remote_hash = sha256_file(remote_download)

assert local_hash == remote_hash, "REMOTE SHA-256 MISMATCH"

with zipfile.ZipFile(remote_download, "r") as zf:
    assert zf.testzip() is None, "Remote ZIP integrity failed."

remote_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=TOKEN
)

assert remote_archive in remote_files
assert remote_report in remote_files

# ------------------------------------------------------------
# 13. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("EXPERIMENT 3 — RESULTS")
print("=" * 68)

print("\nCAMERA GEOMETRY")
for camera_name, values in camera_results.items():
    print(
        f"{camera_name}: fx={values['fx_pixels']:.3f} pixels"
    )

print("\nPROJECTED TARGET WIDTH")
print(projection_df.round(4).to_string(index=False))

print("\nERROR SUMMARY — NONZERO PIXEL PERTURBATIONS")
print(summary_df.round(4).to_string(index=False))

print("\nINVALID ESTIMATES ARE REPORTED SEPARATELY.")
print("\nARCHIVE:", ARCHIVE)
print("Archive size (MB):", round(os.path.getsize(ARCHIVE)/1e6, 3))
print("SHA-256:", local_hash)

print("\n" + "=" * 68)
print("EXPERIMENT 3 FINAL STATUS")
print("=" * 68)
print("Geometric analysis       : PASS")
print("CSV evidence             : PASS")
print("Figures (600 DPI PNG)    : 4 GENERATED")
print("Vector PDF figures       : 4 GENERATED")
print("Technical report         : PASS")
print("Archive creation         : PASS")
print("HF upload                : PASS")
print("Remote listing           : PASS")
print("Remote download          : PASS")
print("SHA-256 integrity        : PASS")
print("ZIP integrity            : PASS")
print("GPU training             : NOT REQUIRED")
print("=" * 68)


In [ ]:

# ============================================================
# RESEARCH MASTER AUDIT — EXPERIMENTS 1, 2 AND 3
# CPU ONLY | READ-ONLY REMOTE INVENTORY
# ============================================================

import os, json, hashlib, zipfile, tempfile
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

REPO = "SakibM19/border-surveillance-research-backups"
TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert TOKEN, "HF_TOKEN missing from Kaggle Secrets."

api = HfApi(token=TOKEN)

print("=" * 70)
print("RESEARCH MASTER AUDIT")
print("=" * 70)

# 1. Remote repository inventory
files = api.list_repo_files(
    repo_id=REPO,
    repo_type="dataset",
    token=TOKEN
)

print("\nREMOTE REPOSITORY ACCESS: PASS")
print("Total remote files:", len(files))

# 2. Locate Experiment 1 and 2 evidence
keywords = [
    "prediction", "predictions", "labels", "ground_truth",
    "ground-truth", "test_evaluation", "validation",
    "results.json", "detection", "inference",
    "experiment1", "experiment2"
]

matches = [
    p for p in files
    if any(k in p.lower() for k in keywords)
    and ("experiment1/" in p.lower() or
         "experiment2/" in p.lower())
]

print("\nRELEVANT EXPERIMENT 1/2 FILES")
print("-" * 70)

for p in matches:
    print(p)

# 3. Verify Experiment 3 remote archive
exp3_archive = (
    "Experiment3/Paper_Evidence/"
    "Experiment3_Range_Distance_Evidence.zip"
)

assert exp3_archive in files, "Experiment 3 archive missing remotely."

downloaded = hf_hub_download(
    repo_id=REPO,
    filename=exp3_archive,
    repo_type="dataset",
    token=TOKEN,
    force_download=True
)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

with zipfile.ZipFile(downloaded) as z:
    assert z.testzip() is None
    members = z.namelist()

print("\nEXPERIMENT 3 REMOTE ARCHIVE")
print("ZIP integrity: PASS")
print("SHA-256:", sha256(downloaded))
print("Files inside archive:", len(members))

# 4. Independent mathematical consistency check
# Recalculate expected geometry from the declared assumptions.

import math
import numpy as np
import pandas as pd

target_width = 0.289
distances = [50, 100, 150, 200, 250, 300]

camera_config = {
    "RGB": {"width": 1920, "hfov": 20.0},
    "Thermal": {"width": 640, "hfov": 18.0}
}

independent_rows = []

for name, c in camera_config.items():
    fx = c["width"] / (
        2 * math.tan(math.radians(c["hfov"]) / 2)
    )

    for distance in distances:
        pixels = fx * target_width / distance
        independent_rows.append({
            "camera": name,
            "distance_m": distance,
            "focal_length_px": fx,
            "projected_width_px": pixels
        })

independent = pd.DataFrame(independent_rows)

print("\nINDEPENDENT GEOMETRY RECOMPUTATION")
print(independent.round(4).to_string(index=False))

# 5. Inspect archive structure for source files
print("\nEXPERIMENT 3 ARCHIVE CONTENTS")
for member in members:
    print(member)

print("\n" + "=" * 70)
print("MASTER AUDIT COMPLETE")
print("=" * 70)
print("Experiment 3 remote archive: VERIFIED")
print("Independent geometry: RECOMPUTED")
print("Experiment 1/2 inventory: GENERATED")
print("Experiment 4 evaluation: NOT STARTED")
print("No training performed.")
print("No existing remote files modified.")
print("=" * 70)


In [ ]:

print("EXPERIMENT 1 AND 2 — PREDICTION EVIDENCE INVENTORY")
print("=" * 70)

for path in matches:
    print(path)

print("\nTOTAL MATCHING FILES:", len(matches))


In [ ]:

# ============================================================
# EXPERIMENT 4 — PREDICTION EVIDENCE DISCOVERY
# READ-ONLY | NO TRAINING | CPU ONLY
# ============================================================

import os, json, zipfile
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

REPO = "SakibM19/border-surveillance-research-backups"
TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

assert TOKEN, "HF_TOKEN missing"

archives = {
    "EXP1_FINAL": 
        "Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip",

    "EXP1_PAPER":
        "Experiment1/Paper_Evidence/Experiment1_Paper_Evidence.zip",

    "EXP2_TEST":
        "Experiment2/Evaluation/Experiment2_Test_Evaluation_Evidence.zip"
}

out = "/kaggle/working/EXP4_Evidence_Discovery"
os.makedirs(out, exist_ok=True)

keywords = [
    "predict", "prediction", "ground", "label", "conf",
    "detection", "result", "metric", "json", "csv",
    "txt", "manifest"
]

all_inventory = {}

print("=" * 70)
print("EXPERIMENT 4 — EVIDENCE DISCOVERY")
print("=" * 70)

for name, remote_path in archives.items():

    print(f"\n--- {name} ---")

    local = hf_hub_download(
        repo_id=REPO,
        filename=remote_path,
        repo_type="dataset",
        token=TOKEN,
        force_download=True
    )

    with zipfile.ZipFile(local, "r") as z:

        assert z.testzip() is None, f"{name}: ZIP integrity failure"

        members = z.namelist()
        relevant = [
            m for m in members
            if any(k in m.lower() for k in keywords)
        ]

        all_inventory[name] = {
            "archive": remote_path,
            "total_files": len(members),
            "relevant_files": relevant,
            "all_files": members
        }

        print("ZIP integrity: PASS")
        print("Total files:", len(members))
        print("Potentially relevant files:")

        for item in relevant:
            print(" ", item)

# Also inspect the Experiment 1 CSV and Experiment 2 summary
extra_files = [
    "Experiment1/exp1_results.csv",
    "Experiment2/Evaluation/Experiment2_Test_Evaluation_Summary.json"
]

for remote_path in extra_files:

    print("\n---", remote_path, "---")

    local = hf_hub_download(
        repo_id=REPO,
        filename=remote_path,
        repo_type="dataset",
        token=TOKEN,
        force_download=True
    )

    if remote_path.endswith(".json"):
        with open(local, "r") as f:
            data = json.load(f)
        print(json.dumps(data, indent=2)[:12000])

    elif remote_path.endswith(".csv"):
        import pandas as pd
        df = pd.read_csv(local)
        print("Rows:", len(df))
        print("Columns:", list(df.columns))
        print(df.head(10).to_string(index=False))

report_path = os.path.join(out, "Evidence_Discovery_Report.json")

with open(report_path, "w") as f:
    json.dump(all_inventory, f, indent=2)

print("\n" + "=" * 70)
print("EVIDENCE DISCOVERY COMPLETE")
print("Report:", report_path)
print("No remote files modified.")
print("No model training performed.")
print("=" * 70)


In [ ]:
# EXPERIMENT 4 — CHECK FOR PER-IMAGE PREDICTION EVIDENCE

import json
import os

report_path = "/kaggle/working/EXP4_Evidence_Discovery/Evidence_Discovery_Report.json"

with open(report_path, "r") as f:
    report = json.load(f)

prediction_terms = [
    "predict", "prediction", "detections", "detection",
    "predictions", "results", "labels"
]

for archive_name, info in report.items():
    if not isinstance(info, dict) or "all_files" not in info:
        continue

    files = info["all_files"]

    print("\n" + "=" * 65)
    print(archive_name)
    print("Total files:", len(files))

    # Exclude ground-truth label directories when searching for predictions
    prediction_candidates = [
        p for p in files
        if any(term in p.lower() for term in prediction_terms)
        and not (
            "/labels/" in p.lower()
            or p.lower().startswith("labels/")
            or "ground_truth" in p.lower()
            or "ground-truth" in p.lower()
        )
    ]

    ground_truth_files = [
        p for p in files
        if "/labels/" in p.lower()
        or p.lower().startswith("labels/")
        or "ground_truth" in p.lower()
        or "ground-truth" in p.lower()
    ]

    print("Ground-truth candidates:", len(ground_truth_files))
    print("Prediction/result candidates:", len(prediction_candidates))

    for p in prediction_candidates[:50]:
        print("  ", p)

    if len(prediction_candidates) > 50:
        print("  ...", len(prediction_candidates) - 50, "more")

print("\n" + "=" * 65)
print("PREDICTION EVIDENCE CHECK COMPLETE")

In [ ]:

# EXPERIMENT 4 — PREDICTION FORMAT AND LABEL SCHEMA AUDIT
# Read-only inspection. No training and no remote uploads.

import os
import json
import zipfile
import pickle
import numpy as np
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

REPO = "SakibM19/border-surveillance-research-backups"
TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

ARCHIVES = {
    "EXP1_FINAL": "Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip",
    "EXP2_TEST": "Experiment2/Evaluation/Experiment2_Test_Evaluation_Evidence.zip",
}

OUT = "/kaggle/working/EXP4_Schema_Audit"
os.makedirs(OUT, exist_ok=True)

def describe(obj, depth=0):
    """Print a compact description of nested prediction structures."""
    prefix = "  " * depth
    if isinstance(obj, dict):
        print(prefix + f"dict with {len(obj)} keys:", list(obj.keys())[:20])
        for key in list(obj.keys())[:5]:
            print(prefix + f"  [{key!r}]")
            describe(obj[key], depth + 2)
    elif isinstance(obj, (list, tuple)):
        print(prefix + f"{type(obj).__name__}, length={len(obj)}")
        if obj:
            describe(obj[0], depth + 1)
    elif isinstance(obj, np.ndarray):
        print(prefix + f"ndarray shape={obj.shape}, dtype={obj.dtype}")
        if obj.size:
            print(prefix + "sample:", repr(obj.reshape(-1)[:8].tolist()))
    else:
        print(prefix + f"{type(obj).__name__}: {repr(obj)[:250]}")

for name, remote_path in ARCHIVES.items():
    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    archive_path = hf_hub_download(
        repo_id=REPO,
        filename=remote_path,
        repo_type="dataset",
        token=TOKEN,
        force_download=False
    )

    with zipfile.ZipFile(archive_path, "r") as z:
        assert z.testzip() is None, f"{name}: ZIP integrity failed"
        members = z.namelist()

        if name == "EXP1_FINAL":
            targets = [
                m for m in members
                if m.endswith(".pkl") or "threshold_results" in m.lower()
            ]
        else:
            targets = [
                m for m in members
                if m.endswith("predictions.json")
                or (
                    "/labels/" in m.replace("\\", "/").lower()
                    and m.lower().endswith(".txt")
                )
            ]

        print("Selected files:", len(targets))

        for member in targets:
            if name == "EXP2_TEST" and "/labels/" in member.replace("\\", "/").lower():
                continue

            print("\n---", member, "---")
            raw = z.read(member)

            if member.endswith(".json"):
                try:
                    data = json.loads(raw.decode("utf-8"))
                    describe(data)
                    print("JSON sample:", json.dumps(data[:2] if isinstance(data, list) else data, indent=2)[:1800])
                except Exception as e:
                    print("JSON inspection error:", repr(e))

            elif member.endswith(".pkl"):
                try:
                    data = pickle.loads(raw)
                    describe(data)
                except Exception as e:
                    print("PICKLE inspection error:", repr(e))

            elif member.endswith(".csv"):
                print(raw.decode("utf-8", errors="replace")[:2500])

        if name == "EXP2_TEST":
            label_files = [
                m for m in members
                if "/labels/" in m.replace("\\", "/").lower()
                and m.lower().endswith(".txt")
            ]
            print("\nGround-truth label TXT files:", len(label_files))
            for member in label_files[:3]:
                print("\nLABEL SAMPLE:", member)
                print(z.read(member).decode("utf-8", errors="replace")[:700])

print("\n" + "=" * 65)
print("SCHEMA AUDIT COMPLETE")
print("No model training performed.")
print("No remote files modified.")


In [ ]:

# EXPERIMENT 4 — TEST SPLIT ALIGNMENT AND GROUND-TRUTH FORMAT AUDIT
# Read-only. No training, no remote modifications.

import os, json, zipfile, collections
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

REPO = "SakibM19/border-surveillance-research-backups"
TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

REMOTE = "Experiment2/Evaluation/Experiment2_Test_Evaluation_Evidence.zip"

archive = hf_hub_download(
    repo_id=REPO,
    filename=REMOTE,
    repo_type="dataset",
    token=TOKEN
)

def split_from_path(path):
    p = path.lower().replace("\\", "/").split("/")
    for s in ("train", "val", "test"):
        if s in p:
            return s
    return "unknown"

def stem(path):
    return os.path.splitext(os.path.basename(path))[0]

with zipfile.ZipFile(archive, "r") as z:
    assert z.testzip() is None
    members = z.namelist()

    print("=" * 70)
    print("ARCHIVE STRUCTURE")
    print("=" * 70)

    for modality in ("Visible", "Infrared"):
        image_paths = [
            p for p in members
            if p.lower().endswith((".jpg", ".jpeg", ".png"))
            and p.split("/")[0].lower() == modality.lower()
            and "/images/" in p.lower()
        ]

        label_paths = [
            p for p in members
            if p.lower().endswith(".txt")
            and p.split("/")[0].lower() == modality.lower()
            and "/labels/" in p.lower()
        ]

        image_sets = collections.defaultdict(set)
        label_sets = collections.defaultdict(dict)

        for p in image_paths:
            image_sets[split_from_path(p)].add(stem(p))

        for p in label_paths:
            label_sets[split_from_path(p)][stem(p)] = p

        prediction_path = f"{modality}/predictions.json"
        predictions = json.loads(z.read(prediction_path))

        pred_ids = [str(x["image_id"]) for x in predictions]
        pred_set = set(pred_ids)

        print(f"\n--- {modality.upper()} ---")
        print("Images by split:", {k: len(v) for k, v in image_sets.items()})
        print("Labels by split:", {k: len(v) for k, v in label_sets.items()})
        print("Prediction records:", len(predictions))
        print("Unique prediction image IDs:", len(pred_set))
        print("Duplicate prediction IDs:", len(pred_ids) - len(pred_set))

        for split in ("train", "val", "test", "unknown"):
            imgs = image_sets.get(split, set())
            labs = label_sets.get(split, {})
            if not imgs and not labs:
                continue

            matched = pred_set & imgs
            print(
                f"{split:7s} | images={len(imgs):6d} "
                f"| prediction IDs matched={len(matched):6d} "
                f"| prediction coverage={len(matched)/max(1,len(imgs))*100:6.2f}% "
                f"| labels={len(labs):6d}"
            )

        # Match predictions against labels from the split with the
        # highest overlap, but report all split overlaps above.
        best_split = max(
            image_sets,
            key=lambda s: len(pred_set & image_sets[s]),
            default="unknown"
        )

        matched_ids = pred_set & image_sets.get(best_split, set())
        labels = label_sets.get(best_split, {})

        print("\nBest-overlap split:", best_split)
        print("Prediction IDs in that split:", len(matched_ids))
        print("Matched IDs with label files:", len(matched_ids & set(labels)))
        print("Matched IDs without label files:", len(matched_ids - set(labels)))

        # Inspect label column counts and numeric ranges.
        column_counts = collections.Counter()
        sixth_values = []
        sample_rows = []
        nonnumeric = 0

        for image_id in list(matched_ids)[:]:
            label_path = labels.get(image_id)
            if not label_path:
                continue

            content = z.read(label_path).decode("utf-8", errors="replace")
            for line in content.splitlines():
                parts = line.strip().split()
                if not parts:
                    continue
                column_counts[len(parts)] += 1
                if len(sample_rows) < 12:
                    sample_rows.append((label_path, line.strip()))
                try:
                    vals = [float(x) for x in parts]
                    if len(vals) >= 6:
                        sixth_values.append(vals[5])
                except Exception:
                    nonnumeric += 1

        print("\nLabel row field-count distribution:", dict(column_counts))
        print("Nonnumeric label rows:", nonnumeric)
        if sixth_values:
            print(
                "Sixth-column range:",
                min(sixth_values), "to", max(sixth_values)
            )
            print(
                "Sixth-column examples:",
                sixth_values[:20]
            )

        print("\nLabel row examples:")
        for path, line in sample_rows:
            print(path, "=>", line)

    print("\n" + "=" * 70)
    print("ALIGNMENT AUDIT COMPLETE")
    print("No model training performed.")
    print("No remote files modified.")


In [ ]:

# EXPERIMENT 4 — DIRECT ID MATCH + ARCHIVE METADATA AUDIT
# Read-only. No training or uploads.

import os, json, zipfile, collections
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

REPO = "SakibM19/border-surveillance-research-backups"
TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REMOTE = "Experiment2/Evaluation/Experiment2_Test_Evaluation_Evidence.zip"

archive = hf_hub_download(
    repo_id=REPO, filename=REMOTE,
    repo_type="dataset", token=TOKEN
)

with zipfile.ZipFile(archive, "r") as z:
    assert z.testzip() is None
    members = z.namelist()

    print("TOTAL ARCHIVE FILES:", len(members))
    print("\n--- TOP-LEVEL PATHS ---")
    roots = sorted(set(p.split("/")[0] for p in members if p))
    print(roots)

    print("\n--- METADATA / REPORT FILES ---")
    metadata = [
        p for p in members
        if any(k in p.lower() for k in
               ["summary", "manifest", "config", "args", "evaluation", "report"])
        and p.lower().endswith((".json", ".yaml", ".yml", ".txt", ".csv"))
    ]
    for p in metadata[:60]:
        print("\nFILE:", p)
        raw = z.read(p).decode("utf-8", errors="replace")
        print(raw[:1800])

    for modality in ("Visible", "Infrared"):
        print("\n" + "=" * 65)
        print(modality.upper())
        print("=" * 65)

        pred_path = f"{modality}/predictions.json"
        label_prefix = f"{modality}/labels/"

        predictions = json.loads(z.read(pred_path))
        prediction_ids = {str(x["image_id"]) for x in predictions}

        label_paths = [
            p for p in members
            if p.startswith(label_prefix) and p.lower().endswith(".txt")
        ]
        label_map = {
            os.path.splitext(os.path.basename(p))[0]: p
            for p in label_paths
        }
        label_ids = set(label_map)

        matched = prediction_ids & label_ids
        pred_only = prediction_ids - label_ids
        label_only = label_ids - prediction_ids

        print("Prediction records:", len(predictions))
        print("Unique prediction IDs:", len(prediction_ids))
        print("Ground-truth label files:", len(label_ids))
        print("Matched IDs:", len(matched))
        print("Prediction IDs without labels:", len(pred_only))
        print("Labels without predictions:", len(label_only))
        print("ID coverage:", round(100 * len(matched) / max(1, len(label_ids)), 3), "%")

        print("\nPrediction category IDs:",
              dict(collections.Counter(str(x["category_id"]) for x in predictions)))

        # Label schema across the complete set of matched files.
        field_counts = collections.Counter()
        class_counts = collections.Counter()
        sixth_values = []
        empty_files = 0
        malformed = 0
        sample_lines = []

        for image_id in matched:
            content = z.read(label_map[image_id]).decode(
                "utf-8", errors="replace"
            )
            rows = [line.strip() for line in content.splitlines() if line.strip()]

            if not rows:
                empty_files += 1

            for line in rows:
                parts = line.split()
                field_counts[len(parts)] += 1
                try:
                    vals = [float(v) for v in parts]
                    class_counts[parts[0]] += 1
                    if len(vals) >= 6:
                        sixth_values.append(vals[5])
                    if len(sample_lines) < 10:
                        sample_lines.append(line)
                except Exception:
                    malformed += 1

        print("\nLabel row field counts:", dict(field_counts))
        print("Class IDs:", dict(class_counts))
        print("Empty label files among matched IDs:", empty_files)
        print("Malformed rows:", malformed)
        if sixth_values:
            print("Sixth-column min/max:",
                  min(sixth_values), max(sixth_values))
        print("Sample label rows:")
        for line in sample_lines:
            print(" ", line)

        print("\nExample unmatched prediction IDs:", list(pred_only)[:8])
        print("Example unmatched label IDs:", list(label_only)[:8])

print("\nDIRECT MATCH AND METADATA AUDIT COMPLETE.")
print("No remote files modified.")


In [ ]:

# EXPERIMENT 4 — LOCATE ORIGINAL ANT-UAV300 TEST ANNOTATIONS
# Read-only inspection of restored dataset and evaluation YAML files.

import os
from pathlib import Path

ROOT = Path("/kaggle/working/EXP2_RESTORED/AntiUAV_Experiment2")

print("RESTORED DATASET EXISTS:", ROOT.exists())
print("ROOT:", ROOT)

for modality in ("visible", "infrared"):
    print("\n" + "=" * 60)
    print(modality.upper())
    print("=" * 60)

    modality_root = ROOT / "datasets" / modality

    if not modality_root.exists():
        print("Dataset directory not found:", modality_root)
        continue

    for split in ("train", "val", "test"):
        image_dir = modality_root / split / "images"
        label_dir = modality_root / split / "labels"

        image_count = (
            sum(1 for p in image_dir.iterdir() if p.is_file())
            if image_dir.exists() else 0
        )
        label_files = (
            list(label_dir.glob("*.txt"))
            if label_dir.exists() else []
        )

        print(
            f"{split:5s} | images={image_count:6d} "
            f"| label files={len(label_files):6d}"
        )

        if split == "test" and label_files:
            print("Test label examples:")
            for p in label_files[:3]:
                print(" ", p.name, "=>", repr(p.read_text()[:300]))

print("\n" + "=" * 60)
print("EVALUATION YAML FILES")
print("=" * 60)

for name in ("visible_test_data.yaml", "infrared_test_data.yaml"):
    candidates = [
        Path("/kaggle/working/EXP2_TEST_EVALUATION") / name,
        Path("/kaggle/working") / name,
    ]

    for path in candidates:
        if path.exists():
            print("\n---", path, "---")
            print(path.read_text()[:4000])
            break
    else:
        print(name, "not found in expected locations")

print("\nANNOTATION DISCOVERY COMPLETE.")
print("No files modified.")


In [ ]:

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

REPO_ID = "SakibM19/border-surveillance-research-backups"
TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

api = HfApi(token=TOKEN)

print("=" * 65)
print("HUGGING FACE BACKUP PREFLIGHT")
print("=" * 65)

try:
    info = api.repo_info(
        repo_id=REPO_ID,
        repo_type="dataset"
    )

    print("Authentication       : PASS")
    print("Repository           :", info.id)
    print("Private repository   :", info.private)

    files = api.list_repo_files(
        repo_id=REPO_ID,
        repo_type="dataset"
    )

    exp2_files = [
        f for f in files
        if f.startswith("Experiment2/")
    ]

    print("\nExperiment 2 files found:", len(exp2_files))
    print("\nRELEVANT BACKUP FILES:")

    for path in exp2_files:
        if any(term in path.lower() for term in (
            "archive", "dataset", "anti-uav",
            "visible", "infrared", "test",
            "split", "yaml", "zip"
        )):
            print(" ", path)

    print("\nPREFLIGHT: PASS")

except Exception as e:
    print("\nPREFLIGHT: FAILED")
    print(type(e).__name__, str(e))


In [ ]:

from huggingface_hub import HfApi

REPO_ID = "SakibM19/border-surveillance-research-backups"

api = HfApi(token=TOKEN)

print("=" * 75)
print("COMPLETE EXPERIMENT 2 REMOTE FILE INVENTORY")
print("=" * 75)

entries = list(
    api.list_repo_tree(
        repo_id=REPO_ID,
        repo_type="dataset",
        path_in_repo="Experiment2",
        recursive=True,
        expand=True
    )
)

files = [x for x in entries if hasattr(x, "size")]

print("Files found:", len(files))
print()

total_bytes = 0

for item in sorted(files, key=lambda x: x.path):
    size = item.size
    total_bytes += size
    print(f"{size / (1024**2):10.2f} MB | {item.path}")

print("\n" + "=" * 75)
print(f"TOTAL SIZE: {total_bytes / (1024**3):.3f} GB")
print("=" * 75)

print("\nDATASET / ARCHIVE CANDIDATES:")
candidates = [
    x for x in files
    if any(term in x.path.lower() for term in (
        "dataset", "archive", "part_", "part-", ".tar", ".zip"
    ))
]

for item in candidates:
    print(f"{item.size / (1024**2):10.2f} MB | {item.path}")

if not candidates:
    print("No obvious dataset archive files found.")


In [ ]:

import json
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO_ID = "SakibM19/border-surveillance-research-backups"

manifest_files = [
    "Experiment2/Dataset_Audits/Experiment2_Dataset_Integrity_Audit.json",
    "Experiment2/Kaggle_Restoration/Experiment2_Kaggle_Restoration_Manifest.json",
    "Experiment2/Restoration_Audit/Experiment2_Restoration_Manifest.json",
]

print("=" * 70)
print("EXPERIMENT 2 — RESTORATION MANIFEST INSPECTION")
print("=" * 70)

for remote_path in manifest_files:
    print("\n" + "-" * 70)
    print(remote_path)
    print("-" * 70)

    try:
        local_path = hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=remote_path,
            token=TOKEN,
            local_dir="/kaggle/working/EXP2_MANIFEST_INSPECTION"
        )

        with open(local_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print(json.dumps(data, indent=2))

    except Exception as e:
        print("INSPECTION ERROR:", type(e).__name__, str(e))

print("\nMANIFEST INSPECTION COMPLETE.")


In [ ]:

import shutil
from pathlib import Path

print("=" * 65)
print("KAGGLE STORAGE PREFLIGHT")
print("=" * 65)

for path in ["/kaggle/working", "/kaggle/temp"]:
    p = Path(path)
    if p.exists():
        usage = shutil.disk_usage(path)
        print(f"\nLocation: {path}")
        print(f"Total    : {usage.total / (1024**3):.2f} GB")
        print(f"Used     : {usage.used / (1024**3):.2f} GB")
        print(f"Free     : {usage.free / (1024**3):.2f} GB")

required_gb = 11.0
free_gb = shutil.disk_usage("/kaggle/working").free / (1024**3)

print("\nEstimated minimum free space:", required_gb, "GB")
print("Available free space         :", round(free_gb, 2), "GB")

if free_gb >= required_gb:
    print("\nSTORAGE PREFLIGHT: PASS")
    print("Ready to prepare verified restoration.")
else:
    print("\nSTORAGE PREFLIGHT: INSUFFICIENT")
    print("Do not download the archive yet.")


In [ ]:

import os
import hashlib
import tarfile
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO_ID = "SakibM19/border-surveillance-research-backups"
BASE = Path("/kaggle/working/EXP2_RECOVERY")
PARTS_DIR = BASE / "parts"
ARCHIVE = BASE / "Experiment2_Restored.tar"
EXTRACT_DIR = Path("/kaggle/working/EXP2_RESTORED")

BASE.mkdir(parents=True, exist_ok=True)
PARTS_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_ARCHIVE_SHA256 = (
    "c644f40169369fb843373ae2e7ac54a0d44bffe6128b6198fa3d398bbca73ec1"
)

PARTS = [
    ("Experiment2/experiment2_part_01.tar.part", 1073741824,
     "0ea56e507fa7a3a4cb845a5cc47aabfc584542b47a341bfddef829464e1c49f4"),
    ("Experiment2/experiment2_part_02.tar.part", 1073741824,
     "0d136123b8871f5ae16610f0908ad84b493ac61a45653bb2138c5e55b02e9743"),
    ("Experiment2/experiment2_part_03.tar.part", 1073741824,
     "ff5e5c04fd1fc702b93d1aa329ebd7e1b89790aaab9cba71fbca9ad90ef8020d"),
    ("Experiment2/experiment2_part_04.tar.part", 1073741824,
     "5970dbb37aa7b1772241fca0efe8875d5f207b1a5eecf9295b7a762b9b31d241"),
    ("Experiment2/experiment2_part_05.tar.part", 619188224,
     "357d71ff0999f2820f5333b263d90527222b53e632710a7d87032a760bd2661e"),
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=" * 70)
print("EXPERIMENT 2 — VERIFIED DATASET RESTORATION")
print("=" * 70)

# Step 1: Download and verify every archive part.
verified_parts = []

for i, (remote_path, expected_size, expected_sha) in enumerate(PARTS, 1):
    print(f"\n[{i}/5] Downloading {remote_path}")

    local_path = Path(hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=remote_path,
        token=TOKEN,
        local_dir=str(PARTS_DIR)
    ))

    actual_size = local_path.stat().st_size
    actual_sha = sha256_file(local_path)

    size_ok = actual_size == expected_size
    sha_ok = actual_sha == expected_sha

    print(f"  Size: {actual_size:,} bytes | {'PASS' if size_ok else 'FAIL'}")
    print(f"  SHA256: {actual_sha}")
    print(f"  Hash check: {'PASS' if sha_ok else 'FAIL'}")

    if not size_ok or not sha_ok:
        raise RuntimeError(f"Part {i} failed integrity verification.")

    verified_parts.append(local_path)

print("\nALL FIVE PARTS VERIFIED.")

# Step 2: Reassemble in the original order.
print("\nReassembling archive...")

with open(ARCHIVE, "wb") as output:
    for i, part in enumerate(verified_parts, 1):
        print(f"Appending part {i}/5...")
        with open(part, "rb") as source:
            shutil.copyfileobj(source, output, length=8 * 1024 * 1024)

print("Reassembly complete.")
print("Archive size:", f"{ARCHIVE.stat().st_size:,}", "bytes")

# Step 3: Verify the combined archive.
print("\nVerifying combined archive SHA256...")
archive_sha = sha256_file(ARCHIVE)
print("Actual  :", archive_sha)
print("Expected:", EXPECTED_ARCHIVE_SHA256)

if archive_sha != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError("Combined archive SHA256 mismatch. Extraction stopped.")

print("COMBINED ARCHIVE SHA256: PASS")

# Step 4: Inspect archive before extraction.
print("\nChecking TAR archive...")
with tarfile.open(ARCHIVE, "r:") as tar:
    members = tar.getmembers()
    print("Archive entries:", len(members))

    # Reject unsafe paths before extraction.
    root = EXTRACT_DIR.resolve()
    for member in members:
        target = (EXTRACT_DIR / member.name).resolve()
        if target != root and root not in target.parents:
            raise RuntimeError(f"Unsafe archive path detected: {member.name}")

print("TAR inspection: PASS")

# Step 5: Extract.
print("\nExtracting dataset...")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with tarfile.open(ARCHIVE, "r:") as tar:
    tar.extractall(path=EXTRACT_DIR, filter="data")

print("\nEXTRACTION COMPLETE.")
print("Destination:", EXTRACT_DIR)
print("Archive SHA256 verified:", archive_sha)
print("All source parts individually verified.")


In [ ]:

from pathlib import Path
import yaml

ROOT = Path("/kaggle/working/EXP2_RESTORED")

print("=" * 70)
print("EXPERIMENT 2 — ORIGINAL TEST ANNOTATION DISCOVERY")
print("=" * 70)

# Locate dataset directories
dataset_dirs = list(ROOT.rglob("visible")) + list(ROOT.rglob("infrared"))

for modality in ("visible", "infrared"):
    candidates = [
        p for p in dataset_dirs
        if p.is_dir() and p.name.lower() == modality
        and (p / "test").exists()
    ]

    print(f"\n{'=' * 55}")
    print(modality.upper())
    print("=" * 55)

    if not candidates:
        print("Dataset directory not found.")
        continue

    modality_root = candidates[0]
    print("Dataset root:", modality_root)

    for split in ("train", "val", "test"):
        split_root = modality_root / split
        image_dir = split_root / "images"
        label_dir = split_root / "labels"

        if not split_root.exists():
            print(f"{split}: NOT FOUND")
            continue

        images = list(image_dir.glob("*")) if image_dir.exists() else []
        labels = list(label_dir.glob("*.txt")) if label_dir.exists() else []

        print(f"\n{split.upper()}")
        print("Images:", len(images))
        print("Label files:", len(labels))

        if split == "test" and labels:
            print("Sample original annotation files:")
            for label_file in labels[:3]:
                print(f"\n  {label_file.name}")
                lines = label_file.read_text(
                    encoding="utf-8"
                ).strip().splitlines()

                for line in lines[:3]:
                    print("   ", line)

# Locate YAML files
print("\n" + "=" * 70)
print("DATASET YAML FILES")
print("=" * 70)

for yaml_file in ROOT.rglob("*.yaml"):
    print("\nFile:", yaml_file)
    try:
        data = yaml.safe_load(yaml_file.read_text(encoding="utf-8"))
        print(data)
    except Exception as e:
        print("Could not parse:", e)

print("\nANNOTATION DISCOVERY COMPLETE.")


In [ ]:

from pathlib import Path

ROOT = Path("/kaggle/working/EXP2_RESTORED/AntiUAV_Experiment2")

print("=" * 70)
print("RESTORED DATASET STRUCTURE AUDIT")
print("=" * 70)

print("\nRoot exists:", ROOT.exists())
print("Root:", ROOT)

print("\nTOP-LEVEL DIRECTORIES:")
for p in sorted(ROOT.iterdir()):
    if p.is_dir():
        print("DIR :", p.name)
    else:
        print("FILE:", p.name)

print("\nALL IMAGE/LABEL DIRECTORIES:")
for p in sorted(ROOT.rglob("*")):
    if p.is_dir() and p.name.lower() in ("images", "labels"):
        print("\n", p.relative_to(ROOT))
        try:
            children = list(p.iterdir())
            print("  Immediate entries:", len(children))
            for child in children[:8]:
                print("  ", child.name)
        except Exception as e:
            print("  Error:", e)

print("\nPOTENTIAL TEST DIRECTORIES:")
for p in sorted(ROOT.rglob("*")):
    if p.is_dir() and "test" in p.name.lower():
        print(" ", p.relative_to(ROOT))

print("\nSTRUCTURE AUDIT COMPLETE.")


In [ ]:

from pathlib import Path
from collections import Counter

ROOT = Path("/kaggle/working/EXP2_RESTORED/AntiUAV_Experiment2")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

print("=" * 70)
print("EXPERIMENT 2 — ORIGINAL TEST ANNOTATION AUDIT")
print("=" * 70)

for modality in ("visible", "infrared"):
    image_dir = ROOT / "datasets" / modality / "images" / "test"
    label_dir = ROOT / "datasets" / modality / "labels" / "test"

    images = [
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]
    labels = list(label_dir.glob("*.txt"))

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}

    matched = image_stems & label_stems
    images_without_labels = image_stems - label_stems
    labels_without_images = label_stems - image_stems

    empty_labels = 0
    row_counts = Counter()
    field_counts = Counter()
    class_counts = Counter()
    malformed = 0
    samples = []

    for label_file in labels:
        lines = [
            line.strip()
            for line in label_file.read_text(
                encoding="utf-8"
            ).splitlines()
            if line.strip()
        ]

        if not lines:
            empty_labels += 1
            continue

        row_counts[len(lines)] += 1

        for line in lines:
            fields = line.split()
            field_counts[len(fields)] += 1

            if len(fields) not in (5, 6):
                malformed += 1
                continue

            try:
                class_counts[fields[0]] += 1
                values = [float(x) for x in fields[1:]]
                if not all(0 <= x <= 1 for x in values[:4]):
                    malformed += 1
            except ValueError:
                malformed += 1

            if len(samples) < 3:
                samples.append((label_file.name, line))

    print(f"\n{'-' * 60}")
    print(modality.upper())
    print(f"{'-' * 60}")
    print("Test images             :", len(images))
    print("Original label files    :", len(labels))
    print("Matched image-label IDs :", len(matched))
    print("Images without labels   :", len(images_without_labels))
    print("Labels without images   :", len(labels_without_images))
    print("Empty label files       :", empty_labels)
    print("Annotation field counts :", dict(field_counts))
    print("Annotation class counts:", dict(class_counts))
    print("Malformed annotation rows:", malformed)

    print("\nSample original annotations:")
    for filename, line in samples:
        print(f"  {filename}: {line}")

print("\nORIGINAL TEST ANNOTATION AUDIT COMPLETE.")


In [ ]:
from pathlib import Path

ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

keywords = [
    "exp4", "experiment4", "operator",
    "alert", "verification", "evidence"
]

print("=" * 75)
print("EXPERIMENT 4 — INITIAL FILE INVENTORY")
print("=" * 75)

for root in ROOTS:
    print(f"\nROOT: {root}")
    print("Exists:", root.exists())

    if not root.exists():
        continue

    print("\nTop-level contents:")
    for item in sorted(root.iterdir()):
        kind = "DIR " if item.is_dir() else "FILE"
        print(f"  {kind}: {item.name}")

    print("\nRelevant files:")
    matches = []

    for path in root.rglob("*"):
        if path.is_file() and any(
            word in path.name.lower()
            or word in str(path.parent).lower()
            for word in keywords
        ):
            matches.append(path)

    if matches:
        for path in sorted(matches):
            try:
                size_mb = path.stat().st_size / (1024 * 1024)
                print(f"  {path}  ({size_mb:.2f} MB)")
            except OSError:
                print(f"  {path}")
    else:
        print("  No matching files found.")

print("\nINVENTORY COMPLETE")

# Source notebook: `framework_2(1).ipynb`

Original notebook cells: 96. Non-empty source-code cells included: 94. The original code-cell order is preserved below.

In [ ]:
import os
import torch
import shutil

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU currently available")

print("\nFree storage:",
      round(shutil.disk_usage("/content").free / (1024**3), 2),
      "GB")

In [ ]:
!pip install -q kaggle ultralytics opencv-python pycocotools ensemble-boxes

In [ ]:
import torch
import ultralytics
import cv2

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)

shutil.copy(
    "/content/kaggle.json",
    "/root/.kaggle/kaggle.json"
)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle authentication file configured.")

In [ ]:
!kaggle datasets list -s anti-uav-dataset-300

In [ ]:
import os
import shutil

DATA_DIR = "/content/Anti-UAV300"
os.makedirs(DATA_DIR, exist_ok=True)

# Check available storage
free_gb = shutil.disk_usage("/content").free / (1024**3)
print(f"Available storage: {free_gb:.2f} GB")

if free_gb < 20:
    raise RuntimeError("Insufficient storage. Stop before downloading.")

print("\nDownloading complete Anti-UAV300 dataset...")

!kaggle datasets download \
    mm1991/anti-uav-dataset-300 \
    -p /content/Anti-UAV300 \
    --unzip

print("\nDownload and extraction command finished.")

In [ ]:
import os

ROOT = "/content/Anti-UAV300"

print("Root contents:")
for item in os.listdir(ROOT):
    path = os.path.join(ROOT, item)
    print(item, "—", "Folder" if os.path.isdir(path) else "File")

print("\nChecking sequence folders...")

sequence_count = 0
missing = []

for split in ["train", "val", "test"]:
    split_path = os.path.join(ROOT, split)

    if not os.path.isdir(split_path):
        print(f"{split}: folder not found")
        continue

    folders = [
        f for f in os.listdir(split_path)
        if os.path.isdir(os.path.join(split_path, f))
    ]

    sequence_count += len(folders)
    print(f"{split}: {len(folders)} sequences")

    for folder in folders:
        p = os.path.join(split_path, folder)

        required = [
            "visible.mp4",
            "infrared.mp4",
            "visible.json",
            "infrared.json"
        ]

        for filename in required:
            if not os.path.isfile(os.path.join(p, filename)):
                missing.append(os.path.join(p, filename))

print("\nTotal sequences:", sequence_count)
print("Missing required files:", len(missing))

if missing:
    print(*missing[:20], sep="\n")

In [ ]:
import os
import shutil

for path in [
    "/content/Anti-UAV300",
]:
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))

    print(path, f"{total / (1024**3):.2f} GB")

print("Free space:",
      f"{shutil.disk_usage('/content').free / (1024**3):.2f} GB")

In [ ]:
import os

PROJECT = "/content/AntiUAV_Experiment2"

folders = [
    "datasets",
    "datasets/visible",
    "datasets/infrared",
    "manifests",
    "configs",
    "runs",
    "results",
    "logs",
    "backups"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT, folder), exist_ok=True)

print("Experiment 2 workspace created:")
for folder in folders:
    print("✓", folder)

print("\nOriginal dataset:", "/content/Anti-UAV300")
print("Project workspace:", PROJECT)

In [ ]:
import os
import json

ROOT = "/content/Anti-UAV300"

sequence = sorted(
    os.listdir(os.path.join(ROOT, "train"))
)[0]

folder = os.path.join(ROOT, "train", sequence)

for modality in ["visible", "infrared"]:
    path = os.path.join(folder, f"{modality}.json")

    with open(path, "r") as f:
        data = json.load(f)

    print(f"\n{modality.upper()}")
    print("Keys:", list(data.keys()))
    print("Frames:", len(data["gt_rect"]))
    print("Exist entries:", len(data["exist"]))
    print("First bbox:", data["gt_rect"][0])
    print("First existence value:", data["exist"][0])

In [ ]:

import os
import cv2
import json
import math
import csv
import yaml
from collections import defaultdict

SOURCE = "/content/Anti-UAV300"
PROJECT = "/content/AntiUAV_Experiment2"

MODALITIES = ["visible", "infrared"]
SPLITS = ["train", "val", "test"]
STEP = 10

stats = defaultdict(lambda: {
    "positive": 0,
    "negative": 0,
    "excluded": 0,
    "frames_decoded": 0,
    "write_errors": 0
})

exclusions = []
sequence_errors = []

def is_zero_or_empty(box):
    if box is None or box == []:
        return True
    try:
        return all(float(v) == 0 for v in box)
    except (TypeError, ValueError):
        return False

def valid_bbox(box, width, height):
    try:
        if box is None or len(box) != 4:
            return False

        x, y, w, h = map(float, box)

        if not all(math.isfinite(v) for v in [x, y, w, h]):
            return False

        return (
            x >= 0 and y >= 0 and
            w > 0 and h > 0 and
            x + w <= width and
            y + h <= height
        )
    except (TypeError, ValueError):
        return False

for modality in MODALITIES:
    print(f"\n{'='*55}")
    print("PROCESSING:", modality.upper())
    print(f"{'='*55}")

    for split in SPLITS:
        sequence_root = os.path.join(SOURCE, split)
        image_out = os.path.join(
            PROJECT, "datasets", modality, "images", split
        )
        label_out = os.path.join(
            PROJECT, "datasets", modality, "labels", split
        )

        os.makedirs(image_out, exist_ok=True)
        os.makedirs(label_out, exist_ok=True)

        sequences = sorted(
            x for x in os.listdir(sequence_root)
            if os.path.isdir(os.path.join(sequence_root, x))
        )

        for seq_idx, sequence in enumerate(sequences, 1):
            folder = os.path.join(sequence_root, sequence)
            video_path = os.path.join(folder, f"{modality}.mp4")
            json_path = os.path.join(folder, f"{modality}.json")

            try:
                with open(json_path, "r") as f:
                    annotation = json.load(f)

                boxes = annotation["gt_rect"]
                existence = annotation["exist"]

                if len(boxes) != len(existence):
                    raise ValueError("Annotation length mismatch")

                cap = cv2.VideoCapture(video_path)

                if not cap.isOpened():
                    raise ValueError("Cannot open video")

                width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

                if width <= 0 or height <= 0:
                    raise ValueError("Invalid video dimensions")

                frame_idx = 0

                while True:
                    success, frame = cap.read()

                    if not success:
                        break

                    if frame_idx >= len(boxes):
                        raise ValueError("More video frames than annotations")

                    stats[(split, modality)]["frames_decoded"] += 1

                    if frame_idx % STEP == 0:
                        box = boxes[frame_idx]
                        exists = existence[frame_idx]

                        key = f"{sequence}_frame_{frame_idx:06d}"
                        image_path = os.path.join(
                            image_out, key + ".jpg"
                        )
                        label_path = os.path.join(
                            label_out, key + ".txt"
                        )

                        if int(exists) == 1 and valid_bbox(
                            box, width, height
                        ):
                            x, y, w, h = map(float, box)

                            xc = (x + w / 2) / width
                            yc = (y + h / 2) / height
                            nw = w / width
                            nh = h / height

                            label = f"0 {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}\n"
                            category = "positive"

                        elif int(exists) == 0 and is_zero_or_empty(box):
                            label = ""
                            category = "negative"

                        else:
                            stats[(split, modality)]["excluded"] += 1

                            exclusions.append({
                                "split": split,
                                "sequence": sequence,
                                "modality": modality,
                                "frame": frame_idx,
                                "exist": exists,
                                "bbox": json.dumps(box),
                                "reason": (
                                    "invalid_positive_bbox"
                                    if int(exists) == 1
                                    else "negative_with_nonzero_bbox"
                                )
                            })

                            frame_idx += 1
                            continue

                        if not cv2.imwrite(image_path, frame):
                            stats[(split, modality)]["write_errors"] += 1
                            frame_idx += 1
                            continue

                        with open(label_path, "w") as f:
                            f.write(label)

                        stats[(split, modality)][category] += 1

                    frame_idx += 1

                cap.release()

                if frame_idx != len(boxes):
                    raise ValueError(
                        f"Video/annotation mismatch: "
                        f"{frame_idx} vs {len(boxes)}"
                    )

            except Exception as e:
                sequence_errors.append({
                    "split": split,
                    "sequence": sequence,
                    "modality": modality,
                    "error": str(e)
                })

            if seq_idx % 20 == 0 or seq_idx == len(sequences):
                print(
                    f"{split}: {seq_idx}/{len(sequences)} sequences processed"
                )

# Save exclusion manifest
manifest_path = os.path.join(
    PROJECT, "manifests", "exclusion_manifest.csv"
)

with open(manifest_path, "w", newline="") as f:
    fields = [
        "split", "sequence", "modality",
        "frame", "exist", "bbox", "reason"
    ]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(exclusions)

# Save sequence processing errors
error_path = os.path.join(
    PROJECT, "manifests", "sequence_errors.csv"
)

with open(error_path, "w", newline="") as f:
    fields = ["split", "sequence", "modality", "error"]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(sequence_errors)

# Generate YOLO YAML files
for modality in MODALITIES:
    data_root = os.path.join(PROJECT, "datasets", modality)

    config = {
        "path": data_root,
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": 1,
        "names": {0: "drone"}
    }

    config_path = os.path.join(
        PROJECT, "configs", f"{modality}_data.yaml"
    )

    with open(config_path, "w") as f:
        yaml.safe_dump(config, f, sort_keys=False)

# Print summary
print("\n" + "="*65)
print("DATASET PREPARATION SUMMARY")
print("="*65)

for modality in MODALITIES:
    print(f"\n{modality.upper()}")

    for split in SPLITS:
        s = stats[(split, modality)]

        print(
            f"{split:5} | "
            f"Positive: {s['positive']:6} | "
            f"Negative: {s['negative']:5} | "
            f"Excluded: {s['excluded']:4} | "
            f"Decoded: {s['frames_decoded']:6} | "
            f"Write errors: {s['write_errors']}"
        )

print("\nTotal exclusions:", len(exclusions))
print("Sequence errors:", len(sequence_errors))
print("Exclusion manifest:", manifest_path)
print("Error manifest:", error_path)

print("\nYAML configurations:")
print(os.path.join(PROJECT, "configs", "visible_data.yaml"))
print(os.path.join(PROJECT, "configs", "infrared_data.yaml"))

print("\nPREPARATION FINISHED")


In [ ]:
import os
import tarfile

PROJECT = "/content/AntiUAV_Experiment2"
BACKUP = "/content/experiment2_checkpoint.tar"

print("Creating Experiment 2 checkpoint...")

with tarfile.open(BACKUP, "w") as tar:
    tar.add(
        PROJECT,
        arcname="AntiUAV_Experiment2"
    )

size = os.path.getsize(BACKUP) / (1024**3)

print(f"\nBackup created: {size:.2f} GB")
print("Location:", BACKUP)

In [ ]:
import os

BACKUP = "/content/experiment2_checkpoint.tar"
CHUNK_SIZE = 1024**3  # 1 GB
OUTPUT_DIR = "/content/experiment2_backup_parts"

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(BACKUP, "rb") as src:
    part = 1
    while True:
        data = src.read(CHUNK_SIZE)
        if not data:
            break

        path = os.path.join(
            OUTPUT_DIR,
            f"experiment2_part_{part:02d}.tar.part"
        )

        with open(path, "wb") as dst:
            dst.write(data)

        print(f"Created: {path} ({len(data)/(1024**3):.2f} GB)")
        part += 1

print("\nAll backup parts created.")

In [ ]:
from google.colab import files
import os

folder = "/content/experiment2_backup_parts"

for filename in sorted(os.listdir(folder)):
    print("Downloading:", filename)
    files.download(os.path.join(folder, filename))

In [ ]:

import os
from pathlib import Path
from collections import Counter
from PIL import Image

PROJECT = Path("/content/AntiUAV_Experiment2")

print("=" * 65)
print("EXPERIMENT 2 — DATASET INTEGRITY AUDIT")
print("=" * 65)

total_images = 0
total_labels = 0
total_errors = 0

for modality in ["visible", "infrared"]:
    for split in ["train", "val", "test"]:

        image_dir = PROJECT / "datasets" / modality / split / "images"
        label_dir = PROJECT / "datasets" / modality / split / "labels"

        print(f"\n{modality.upper()} — {split.upper()}")

        if not image_dir.exists() or not label_dir.exists():
            print("ERROR: Dataset directory missing!")
            total_errors += 1
            continue

        images = sorted(
            p for p in image_dir.iterdir()
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        )

        labels = sorted(label_dir.glob("*.txt"))

        image_stems = {p.stem for p in images}
        label_stems = {p.stem for p in labels}

        missing_labels = image_stems - label_stems
        orphan_labels = label_stems - image_stems

        corrupt_images = 0
        invalid_labels = 0
        positive = 0
        negative = 0

        for image_path in images:
            try:
                with Image.open(image_path) as img:
                    img.verify()
            except Exception:
                corrupt_images += 1

        for label_path in labels:
            try:
                content = label_path.read_text().strip()

                if not content:
                    negative += 1
                    continue

                valid_file = True

                for line in content.splitlines():
                    values = line.split()

                    if len(values) != 5:
                        valid_file = False
                        break

                    cls, x, y, w, h = map(float, values)

                    if (
                        cls != 0
                        or not all(map(
                            lambda v: v == v and abs(v) != float("inf"),
                            [x, y, w, h]
                        ))
                        or not (0 <= x <= 1)
                        or not (0 <= y <= 1)
                        or not (0 < w <= 1)
                        or not (0 < h <= 1)
                        or x - w/2 < 0
                        or y - h/2 < 0
                        or x + w/2 > 1
                        or y + h/2 > 1
                    ):
                        valid_file = False
                        break

                if valid_file:
                    positive += 1
                else:
                    invalid_labels += 1

            except Exception:
                invalid_labels += 1

        errors = (
            len(missing_labels)
            + len(orphan_labels)
            + corrupt_images
            + invalid_labels
        )

        total_images += len(images)
        total_labels += len(labels)
        total_errors += errors

        print(f"Images:          {len(images):,}")
        print(f"Label files:     {len(labels):,}")
        print(f"Positive labels: {positive:,}")
        print(f"Negative labels: {negative:,}")
        print(f"Missing labels:  {len(missing_labels)}")
        print(f"Orphan labels:   {len(orphan_labels)}")
        print(f"Corrupt images:  {corrupt_images}")
        print(f"Invalid labels:  {invalid_labels}")
        print(f"Total errors:    {errors}")

print("\n" + "=" * 65)
print(f"TOTAL IMAGES: {total_images:,}")
print(f"TOTAL LABEL FILES: {total_labels:,}")
print(f"TOTAL ERRORS: {total_errors}")
print("=" * 65)

if total_errors == 0:
    print("AUDIT PASSED — No integrity errors detected.")
else:
    print("AUDIT NEEDS ATTENTION — Review the errors above.")


In [ ]:
import os

paths = [
    "/content",
    "/content/AntiUAV_Experiment2",
    "/content/Anti-UAV300"
]

for path in paths:
    print("\n" + "=" * 55)
    print("PATH:", path)
    print("EXISTS:", os.path.exists(path))

    if os.path.exists(path):
        print("CONTENTS:")
        for item in sorted(os.listdir(path))[:30]:
            print("  ", item)

In [ ]:
import os
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nExperiment 2 workspace exists:",
      os.path.exists("/content/AntiUAV_Experiment2"))

print("Original dataset exists:",
      os.path.exists("/content/Anti-UAV300"))

print("\nContent directory:")
print(os.listdir("/content"))

In [ ]:
import os
import shutil

print("GPU available:", os.path.exists("/dev/nvidia0"))
print("Free disk space:", round(shutil.disk_usage("/content").free / (1024**3), 2), "GB")
print("Kaggle credentials exist:", os.path.exists("/root/.kaggle/kaggle.json"))

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)

shutil.copy(
    "/content/kaggle.json",
    "/root/.kaggle/kaggle.json"
)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle authentication file configured.")

In [ ]:
!kaggle datasets list -s anti-uav

In [ ]:
import os

os.makedirs("/content/datasets", exist_ok=True)

!kaggle datasets download \
    -d mm1991/anti-uav-dataset-300 \
    -p /content/datasets


In [ ]:
import os

zip_path = "/content/datasets/anti-uav-dataset-300.zip"

print("Archive exists:", os.path.exists(zip_path))
print("Archive size (GB):", round(os.path.getsize(zip_path) / (1024**3), 2))


In [ ]:
from google.colab import files
import os
import zipfile

uploaded = files.upload()

backup_path = "/content/experiment1_backup.zip"

if not os.path.exists(backup_path):
    raise FileNotFoundError("Please upload experiment1_backup.zip")

with zipfile.ZipFile(backup_path, "r") as z:
    bad_file = z.testzip()

if bad_file:
    raise RuntimeError(f"Backup archive is corrupted: {bad_file}")

print("Experiment 1 backup verified.")
print("Size:", round(os.path.getsize(backup_path) / (1024**2), 2), "MB")

In [ ]:
from google.colab import files
import os

UPLOAD_DIR = "/content/PROJECT_BACKUPS"
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("Select ALL Experiment 1 and Experiment 2 backup files.")
print("You can select multiple files in the file picker.")

uploaded = files.upload()

for filename, content in uploaded.items():
    path = os.path.join(UPLOAD_DIR, filename)
    with open(path, "wb") as f:
        f.write(content)

print("\n========== UPLOAD SUMMARY ==========")
for filename in uploaded:
    size = os.path.getsize(os.path.join(UPLOAD_DIR, filename))
    print(f"{filename}: {size / (1024**2):.2f} MB")

print(f"\nTotal files uploaded: {len(uploaded)}")
print(f"Saved in: {UPLOAD_DIR}")
print("Upload complete. Do not start training yet.")

In [ ]:
from google.colab import files
import os

UPLOAD_DIR = "/content/PROJECT_BACKUPS/Experiment1"
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("Select your Experiment 1 backup ZIP.")
uploaded = files.upload()

for filename, content in uploaded.items():
    path = os.path.join(UPLOAD_DIR, filename)
    with open(path, "wb") as f:
        f.write(content)

    print(f"Saved: {filename}")
    print(f"Size: {os.path.getsize(path) / (1024**2):.2f} MB")

print("\nExperiment 1 upload complete.")
print("Location:", UPLOAD_DIR)

In [ ]:
import os, zipfile

folder = "/content/PROJECT_BACKUPS/Experiment1"
files = os.listdir(folder)

for filename in files:
    path = os.path.join(folder, filename)
    print(f"File: {filename}")
    print(f"Size: {os.path.getsize(path) / (1024**2):.2f} MB")

    if filename.endswith(".zip"):
        with zipfile.ZipFile(path, "r") as z:
            bad = z.testzip()
            print("ZIP integrity:", "PASSED" if bad is None else f"FAILED: {bad}")
            print("Files inside:", len(z.namelist()))

In [ ]:
import os
import zipfile
from google.colab import files

source = "/content/PROJECT_BACKUPS/Experiment1"
backup = "/content/Experiment1_PERMANENT_BACKUP.zip"

with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, filenames in os.walk(source):
        for filename in filenames:
            path = os.path.join(root, filename)
            z.write(path, os.path.relpath(path, source))

print("Backup created:", os.path.getsize(backup) / (1024**2), "MB")
files.download(backup)

In [ ]:

!pip -q install -U "huggingface_hub[hf_xet]"

from huggingface_hub import HfApi
from getpass import getpass
import os

# Enter your NEW token securely
token = getpass("Enter your new Hugging Face WRITE token: ")

api = HfApi()

# Get account username
username = api.whoami(token=token)["name"]

repo_id = f"{username}/border-surveillance-research-backups"

# Create private repository
api.create_repo(
    repo_id=repo_id,
    repo_type="dataset",
    private=True,
    exist_ok=True,
    token=token
)

# Verify local backup folder
local_folder = "/content/PROJECT_BACKUPS/Experiment1"

if not os.path.isdir(local_folder):
    raise FileNotFoundError(local_folder)

print("Account:", username)
print("Repository:", repo_id)
print("Starting upload...")

# Upload Experiment 1
api.upload_folder(
    folder_path=local_folder,
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo="Experiment1",
    token=token,
    commit_message="Upload Experiment 1 research backups"
)

print("\nEXPERIMENT 1 ONLINE BACKUP COMPLETE")
print(f"https://huggingface.co/datasets/{repo_id}")


In [ ]:
from huggingface_hub import HfApi
from getpass import getpass

token = getpass("Enter your Hugging Face token: ")
api = HfApi()

repo_id = "SakibM19/border-surveillance-research-backups"

files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset",
    token=token
)

exp1_files = [f for f in files if f.startswith("Experiment1/")]

print("EXPERIMENT 1 ONLINE VERIFICATION")
print("--------------------------------")
for f in exp1_files:
    print(f)

print("--------------------------------")
print("Files found:", len(exp1_files))

if exp1_files:
    print("VERIFICATION PASSED")
else:
    print("No Experiment 1 files found. Check the repository.")

In [ ]:

import os
import zipfile
import pandas as pd

folder = "/content/PROJECT_BACKUPS/Experiment1"

print("=" * 65)
print("EXPERIMENT 1 — BACKUP AUDIT")
print("=" * 65)

if not os.path.isdir(folder):
    raise FileNotFoundError(
        "Experiment 1 backup folder is missing. "
        "We will restore it from Hugging Face."
    )

for filename in sorted(os.listdir(folder)):
    path = os.path.join(folder, filename)

    if not os.path.isfile(path):
        continue

    size_mb = os.path.getsize(path) / (1024 ** 2)
    print(f"\nFILE: {filename}")
    print(f"SIZE: {size_mb:.2f} MB")

    if filename.endswith(".zip"):
        with zipfile.ZipFile(path, "r") as z:
            bad_file = z.testzip()
            print("ZIP integrity:", "PASSED" if bad_file is None else f"FAILED: {bad_file}")
            print("Files inside:", len(z.namelist()))

            if filename == "experiment1_backup.zip":
                print("\nMAIN BACKUP CONTENTS:")
                for item in z.namelist():
                    print(" -", item)

    elif filename.endswith(".csv"):
        try:
            df = pd.read_csv(path)
            print("CSV rows:", len(df))
            print("CSV columns:", list(df.columns))
            print("\nFirst 5 rows:")
            print(df.head().to_string(index=False))
        except Exception as e:
            print("CSV inspection error:", str(e))

print("\n" + "=" * 65)
print("BACKUP AUDIT INSPECTION COMPLETE")
print("=" * 65)


In [ ]:

import os
import json
import zipfile
import pickle
import pandas as pd

backup_dir = "/content/PROJECT_BACKUPS/Experiment1"
work_dir = "/content/EXP1_AUDIT"

os.makedirs(work_dir, exist_ok=True)

# 1. Extract the main backup
main_zip = os.path.join(backup_dir, "experiment1_backup.zip")

with zipfile.ZipFile(main_zip, "r") as z:
    z.extractall(work_dir)

root = os.path.join(work_dir, "experiment1_backup")

print("=" * 65)
print("1. CORRECTED RESULTS")
print("=" * 65)

corrected_csv = os.path.join(root, "exp1_results_corrected_bins.csv")
if os.path.exists(corrected_csv):
    print(pd.read_csv(corrected_csv).to_string(index=False))
else:
    print("Corrected results CSV not found.")

# 2. Dataset split manifest
print("\n" + "=" * 65)
print("2. DATASET SPLIT MANIFEST")
print("=" * 65)

manifest_path = os.path.join(root, "split_manifest.json")
if os.path.exists(manifest_path):
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    print(json.dumps(manifest, indent=2)[:10000])
else:
    print("Manifest not found.")

# 3. Training histories and best checkpoints
print("\n" + "=" * 65)
print("3. MODEL TRAINING EVIDENCE")
print("=" * 65)

for model in ["rgb", "thermal", "blend", "vt"]:
    model_dir = os.path.join(root, "runs", model)
    results_path = os.path.join(model_dir, "results.csv")
    best_path = os.path.join(model_dir, "weights", "best.pt")
    last_path = os.path.join(model_dir, "weights", "last.pt")

    print(f"\nMODEL: {model.upper()}")
    print("best.pt exists:", os.path.exists(best_path))
    print("last.pt exists:", os.path.exists(last_path))

    if os.path.exists(results_path):
        df = pd.read_csv(results_path)
        print("Training epochs recorded:", len(df))
        print("Training columns:", list(df.columns))

        metric_cols = [
            c for c in df.columns
            if "metrics/" in c or "fitness" in c
        ]
        if metric_cols:
            print("Final recorded metrics:")
            print(df[metric_cols].tail(1).to_string(index=False))

# 4. Training log archive
print("\n" + "=" * 65)
print("4. TRAINING LOG ARCHIVE")
print("=" * 65)

logs_zip = os.path.join(backup_dir, "exp1_training_logs.zip")
with zipfile.ZipFile(logs_zip, "r") as z:
    for item in z.namelist():
        print(item)

# 5. Prediction file structure
print("\n" + "=" * 65)
print("5. SAVED TEST PREDICTIONS")
print("=" * 65)

pred_path = os.path.join(root, "test_predictions.pkl")

if os.path.exists(pred_path):
    with open(pred_path, "rb") as f:
        predictions = pickle.load(f)

    print("Object type:", type(predictions).__name__)

    if isinstance(predictions, dict):
        print("Top-level keys:", list(predictions.keys()))
        for key, value in predictions.items():
            print(
                f"{key}: type={type(value).__name__}, "
                f"length={len(value) if hasattr(value, '__len__') else 'N/A'}"
            )
    elif hasattr(predictions, "__len__"):
        print("Number of entries:", len(predictions))
else:
    print("Prediction file not found.")

print("\n" + "=" * 65)
print("STEP 2 INSPECTION COMPLETE")
print("=" * 65)


In [ ]:

import os
import zipfile
import pickle
import yaml
import numpy as np

root = "/content/EXP1_AUDIT/experiment1_backup"

# Restore if the extracted folder is missing
if not os.path.isdir(root):
    os.makedirs("/content/EXP1_AUDIT", exist_ok=True)
    with zipfile.ZipFile(
        "/content/PROJECT_BACKUPS/Experiment1/experiment1_backup.zip"
    ) as z:
        z.extractall("/content/EXP1_AUDIT")

# Inspect prediction structure
with open(os.path.join(root, "test_predictions.pkl"), "rb") as f:
    predictions = pickle.load(f)

print("=" * 60)
print("PREDICTION STRUCTURE")
print("=" * 60)

for model, image_predictions in predictions.items():
    print(f"\nMODEL: {model}")
    print("Number of test images:", len(image_predictions))

    first_key = next(iter(image_predictions))
    sample = image_predictions[first_key]

    print("Sample image key:", first_key)
    print("Sample object type:", type(sample).__name__)

    if isinstance(sample, dict):
        print("Prediction fields:", list(sample.keys()))

        for key, value in sample.items():
            if isinstance(value, np.ndarray):
                print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
            elif hasattr(value, "shape"):
                print(f"  {key}: shape={value.shape}")
            elif isinstance(value, (list, tuple)):
                print(f"  {key}: {type(value).__name__}, length={len(value)}")
            else:
                print(f"  {key}: {type(value).__name__}, value={str(value)[:150]}")
    elif isinstance(sample, np.ndarray):
        print("Sample array shape:", sample.shape)
        print("Sample array dtype:", sample.dtype)
    else:
        print("Sample preview:", str(sample)[:500])

# Inspect model configuration files
print("\n" + "=" * 60)
print("MODEL CONFIGURATION FILES")
print("=" * 60)

for model in ["rgb", "thermal", "blend", "vt"]:
    path = os.path.join(root, f"{model}.yaml")
    print(f"\n{model.upper()}")

    if os.path.exists(path):
        with open(path, "r") as f:
            config = yaml.safe_load(f)
        print(config)
    else:
        print("Configuration file missing.")

print("\nPREDICTION STRUCTURE INSPECTION COMPLETE")


In [ ]:

import os
import numpy as np
import pickle

root = "/content/EXP1_AUDIT/experiment1_backup"

with open(os.path.join(root, "test_predictions.pkl"), "rb") as f:
    predictions = pickle.load(f)

print("=" * 60)
print("1. EXACT PREDICTION FORMAT")
print("=" * 60)

for model, image_data in predictions.items():
    key = next(iter(image_data))
    sample = image_data[key]

    print(f"\n{model}:")
    print("Tuple length:", len(sample))

    for i, item in enumerate(sample):
        arr = np.asarray(item)
        print(
            f"Element {i}: shape={arr.shape}, "
            f"dtype={arr.dtype}, "
            f"sample={str(arr.flatten()[:8])}"
        )

print("\n" + "=" * 60)
print("2. DATASET PATH AVAILABILITY")
print("=" * 60)

paths = [
    "/content/work_exp1_corrected",
    "/content/work_exp1_corrected/rgb",
    "/content/work_exp1_corrected/thermal",
    "/content/work_exp1_corrected/blend",
    "/content/work_exp1_corrected/vt",
    "/content/EXP1_AUDIT/experiment1_backup"
]

for path in paths:
    print(path, "->", "EXISTS" if os.path.exists(path) else "MISSING")

print("\n" + "=" * 60)
print("3. SAMPLE GROUND-TRUTH LABEL CHECK")
print("=" * 60)

sample_key = str(next(iter(predictions["rgb"])))
print("Sample prediction key:", sample_key)

for model in ["rgb", "thermal", "blend", "vt"]:
    label_path = (
        f"/content/work_exp1_corrected/{model}/labels/test/{sample_key}.txt"
    )
    print(model, "->", "FOUND" if os.path.exists(label_path) else "NOT FOUND")
    if os.path.exists(label_path):
        with open(label_path) as f:
            print("  Label contents:", f.read()[:500])

print("\nINSPECTION COMPLETE")


In [ ]:
import json
from pathlib import Path

backup = Path("/content/EXP1_AUDIT/experiment1_backup")
manifest_path = backup / "split_manifest.json"

with open(manifest_path, "r") as f:
    manifest = json.load(f)

print("MANIFEST TYPE:", type(manifest).__name__)

if isinstance(manifest, dict):
    print("\nTOP-LEVEL KEYS:")
    for key, value in manifest.items():
        if isinstance(value, list):
            print(f"{key}: list with {len(value)} entries")
            print("  First 5:", value[:5])
        elif isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} entries")
            print("  Sample:", list(value.items())[:3])
        else:
            print(f"{key}: {value}")
else:
    print("First 10 entries:", manifest[:10])

print("\nBACKUP CONTENTS:")
for p in sorted(backup.iterdir()):
    print(p.name, "(directory)" if p.is_dir() else f"({p.stat().st_size / 1024**2:.2f} MB)")

In [ ]:
import os
from pathlib import Path

roots = [
    Path("/content"),
    Path("/kaggle/input"),
    Path("/content/drive/MyDrive")
]

skip = {".cache", "site-packages", ".git", "__pycache__"}
matches = []

print("SEARCHING FOR LLVIP DATA AND ANNOTATIONS...\n")

for root in roots:
    if not root.exists():
        print(f"Not available: {root}")
        continue

    for current, dirs, files in os.walk(root):
        depth = len(Path(current).relative_to(root).parts)

        dirs[:] = [
            d for d in dirs
            if d not in skip and depth < 5
        ]

        current_path = Path(current)

        if any(word in current_path.name.lower()
               for word in ["llvip", "annotation"]):
            matches.append(str(current_path))

        for filename in files:
            lower = filename.lower()
            if (
                "llvip" in lower
                or "annotation" in lower
                or lower in ["train.txt", "val.txt", "test.txt"]
            ):
                matches.append(str(current_path / filename))

for item in sorted(set(matches)):
    print(item)

print("\nTOTAL MATCHES:", len(set(matches)))

In [ ]:
import requests

url = "https://github.com/bupt-ai-cz/LLVIP"

r = requests.get(url, timeout=30)

print("Repository status:", r.status_code)
print("Repository accessible:", r.status_code == 200)

In [ ]:
!pip -q install py7zr

import requests
import py7zr
import os

url = "https://huggingface.co/datasets/jsonhash/LLVIP/resolve/main/coco_annotations.7z"
archive_path = "/content/llvip_annotations.7z"

response = requests.get(url, timeout=120)
response.raise_for_status()

with open(archive_path, "wb") as f:
    f.write(response.content)

print("Downloaded:", os.path.getsize(archive_path) / 1024, "KB")

with py7zr.SevenZipFile(archive_path, mode="r") as archive:
    names = archive.getnames()

print("\nARCHIVE CONTENTS:")
for name in names[:30]:
    print(name)

print("\nTotal files:", len(names))

In [ ]:
import py7zr
import json
from pathlib import Path

# Extract annotation archive
extract_dir = Path("/content/LLVIP_ANNOTATIONS")
extract_dir.mkdir(exist_ok=True)

with py7zr.SevenZipFile(
    "/content/llvip_annotations.7z", mode="r"
) as archive:
    archive.extractall(path=extract_dir)

# Load original Experiment 1 manifest
backup = Path("/content/EXP1_AUDIT/experiment1_backup")

with open(backup / "split_manifest.json") as f:
    manifest = json.load(f)

# Load test annotations
test_json = extract_dir / "coco_annotations" / "test.json"

with open(test_json) as f:
    coco = json.load(f)

print("=" * 55)
print("LLVIP TEST ANNOTATION AUDIT")
print("=" * 55)

print("\nCOCO STRUCTURE:")
print("Images:", len(coco.get("images", [])))
print("Annotations:", len(coco.get("annotations", [])))
print("Categories:", coco.get("categories", []))

print("\nSAMPLE IMAGE FILENAMES:")
for img in coco.get("images", [])[:10]:
    print(img.get("file_name"))

# Compare image IDs with original manifest
manifest_ids = set(manifest["test"])
annotation_ids = {
    Path(img["file_name"]).stem
    for img in coco.get("images", [])
}

matched = manifest_ids & annotation_ids
missing = manifest_ids - annotation_ids
extra = annotation_ids - manifest_ids

print("\n" + "=" * 55)
print("ORIGINAL TEST SPLIT MATCHING")
print("=" * 55)
print("Manifest test images:", len(manifest_ids))
print("Annotation test images:", len(annotation_ids))
print("Matched:", len(matched))
print("Missing from annotations:", len(missing))
print("Extra annotation images:", len(extra))

print("\nFIRST 15 MISSING:", sorted(missing)[:15])
print("\nFIRST 15 MATCHED:", sorted(matched)[:15])

In [ ]:
import json
import pickle
import numpy as np
from pathlib import Path

backup = Path("/content/EXP1_AUDIT/experiment1_backup")

# Load ground truth
with open("/content/LLVIP_ANNOTATIONS/coco_annotations/test.json") as f:
    coco = json.load(f)

# Load saved predictions
with open(backup / "test_predictions.pkl", "rb") as f:
    predictions = pickle.load(f)

sample_id = "190001"

# Inspect COCO image information
image_info = next(
    img for img in coco["images"]
    if Path(img["file_name"]).stem == sample_id
)

print("=" * 55)
print("IMAGE DIMENSIONS")
print("=" * 55)
print("Image:", image_info["file_name"])
print("COCO width:", image_info["width"])
print("COCO height:", image_info["height"])

# Inspect ground-truth boxes
gt = [
    ann for ann in coco["annotations"]
    if ann["image_id"] == image_info["id"]
]

print("\nGROUND TRUTH")
print("Number of boxes:", len(gt))
print("First 3 COCO boxes (x, y, width, height):")
for ann in gt[:3]:
    print(ann["bbox"])

# Inspect predictions
print("\nSAVED PREDICTIONS")
for model in predictions:
    item = predictions[model][sample_id]

    boxes, scores, dimensions = item

    print(f"\n{model}")
    print("Box shape:", boxes.shape)
    print("Score shape:", scores.shape)
    print("Stored dimensions:", dimensions)
    print("First 3 boxes:")
    print(boxes[:3])
    print("First 3 scores:")
    print(scores[:3])

print("\nINSPECTION COMPLETE")


In [ ]:
from pathlib import Path
import yaml
import pandas as pd

backup = Path("/content/EXP1_AUDIT/experiment1_backup")

print("=" * 65)
print("EXPERIMENT 1 — ORIGINAL EVALUATION SETTINGS")
print("=" * 65)

for model in ["rgb", "thermal", "blend", "vt"]:
    print(f"\n{'='*20} {model.upper()} {'='*20}")

    args_file = backup / "runs" / model / "args.yaml"

    if args_file.exists():
        with open(args_file) as f:
            args = yaml.safe_load(f)

        for key in ["conf", "iou", "imgsz", "batch",
                    "max_det", "split", "mode", "task",
                    "data", "weights"]:
            print(f"{key}: {args.get(key, 'Not recorded')}")
    else:
        print("args.yaml not found")

    results_file = backup / "runs" / model / "results.csv"

    if results_file.exists():
        df = pd.read_csv(results_file)
        print("\nRecorded result columns:")
        print(df.columns.tolist())

print("\n" + "=" * 65)
print("EXPERIMENT 1 RESULTS SUMMARY")
print("=" * 65)

summary = backup / "exp1_results.csv"

if summary.exists():
    print(pd.read_csv(summary).to_string(index=False))

print("\nINSPECTION COMPLETE")

In [ ]:
import pickle
import numpy as np
from pathlib import Path

backup = Path("/content/EXP1_AUDIT/experiment1_backup")

with open(backup / "test_predictions.pkl", "rb") as f:
    predictions = pickle.load(f)

print("=" * 65)
print("SAVED TEST PREDICTION AUDIT")
print("=" * 65)

for model, images in predictions.items():
    all_scores = []
    counts = []

    for image_id, item in images.items():
        boxes, scores, dimensions = item
        all_scores.extend(scores.tolist())
        counts.append(len(scores))

    scores = np.asarray(all_scores)

    print(f"\n{model}")
    print("Images:", len(images))
    print("Total detections:", len(scores))
    print("Minimum confidence:", scores.min() if len(scores) else None)
    print("Maximum confidence:", scores.max() if len(scores) else None)
    print("Mean confidence:", round(float(scores.mean()), 5) if len(scores) else None)
    print("Minimum detections/image:", min(counts))
    print("Maximum detections/image:", max(counts))
    print("Average detections/image:", round(float(np.mean(counts)), 2))

    for threshold in [0.001, 0.01, 0.05, 0.1, 0.25, 0.5]:
        print(
            f"  Detections with confidence >= {threshold}:",
            int(np.sum(scores >= threshold))
        )

print("\nOTHER EVALUATION-RELATED FILES:")
for p in backup.rglob("*"):
    if p.is_file() and any(
        word in p.name.lower()
        for word in ["eval", "predict", "test", "metric", "result"]
    ):
        print(p.relative_to(backup))

print("\nAUDIT COMPLETE")

In [ ]:
!pip -q install pycocotools

import json
import pickle
import numpy as np
from pathlib import Path
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

backup = Path("/content/EXP1_AUDIT/experiment1_backup")
annotation_file = "/content/LLVIP_ANNOTATIONS/coco_annotations/test.json"

# Load ground truth and predictions
coco_gt = COCO(annotation_file)

with open(backup / "test_predictions.pkl", "rb") as f:
    predictions = pickle.load(f)

# Map image filename stems to COCO image IDs and dimensions
image_map = {
    Path(img["file_name"]).stem: img
    for img in coco_gt.dataset["images"]
}

print("=" * 65)
print("INDEPENDENT COCO AP50 EVALUATION")
print("IoU threshold: 0.50 | Maximum detections/image: 300")
print("=" * 65)

results = {}

for model, model_predictions in predictions.items():

    coco_detections = []

    for image_id, (boxes, scores, dimensions) in model_predictions.items():

        image_info = image_map[image_id]
        width = image_info["width"]
        height = image_info["height"]

        boxes = np.asarray(boxes)
        scores = np.asarray(scores)

        for box, score in zip(boxes, scores):

            x1, y1, x2, y2 = box

            # Normalized XYXY -> absolute COCO XYWH
            x1 = np.clip(x1 * width, 0, width)
            y1 = np.clip(y1 * height, 0, height)
            x2 = np.clip(x2 * width, 0, width)
            y2 = np.clip(y2 * height, 0, height)

            bw = x2 - x1
            bh = y2 - y1

            if bw <= 0 or bh <= 0:
                continue

            coco_detections.append({
                "image_id": image_info["id"],
                "category_id": 0,
                "bbox": [
                    float(x1), float(y1),
                    float(bw), float(bh)
                ],
                "score": float(score)
            })

    # Evaluate
    coco_dt = coco_gt.loadRes(coco_detections)

    evaluator = COCOeval(coco_gt, coco_dt, "bbox")
    evaluator.params.imgIds = sorted(image_map[x]["id"] for x in model_predictions)
    evaluator.params.catIds = coco_gt.getCatIds()
    evaluator.params.iouThrs = np.array([0.50])
    evaluator.params.maxDets = [1, 10, 300]

    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    ap50 = float(evaluator.stats[1])

    results[model] = {
        "detections": len(coco_detections),
        "AP50": ap50
    }

    print(f"\n{model}: AP50 = {ap50:.5f}")

print("\n" + "=" * 65)
print("INDEPENDENT AP50 SUMMARY")
print("=" * 65)

for model, values in results.items():
    print(
        f"{model:25s} "
        f"AP50={values['AP50']:.5f} "
        f"Detections={values['detections']}"
    )

In [ ]:
import pandas as pd
import os

audit_dir = "/content/EXP1_AUDIT/experiment1_backup"

results = [
    ["RGB", 0.8698, 0.86981],
    ["Thermal", 0.9497, 0.94972],
    ["Blend", 0.9463, 0.94628],
    ["VT", 0.9504, 0.95042],
    ["Late WBF", 0.9376, 0.93761],
]

df = pd.DataFrame(
    results,
    columns=["Configuration", "Original_AP50", "Reproduced_AP50"]
)

df["Absolute_Difference"] = (
    df["Original_AP50"] - df["Reproduced_AP50"]
).abs()

output_path = os.path.join(audit_dir, "exp1_ap50_independent_audit.csv")
df.to_csv(output_path, index=False)

print("AUDIT FILE SAVED")
print(output_path)
print()
print(df.to_string(index=False))

In [ ]:
import os

root = "/content/EXP1_AUDIT/experiment1_backup"

keywords = [
    "val", "valid", "predict", "threshold",
    "metric", "evaluate", "evaluation",
    "inference", "test", ".py"
]

print("RELEVANT BACKUP FILES")
print("=" * 65)

for current_root, dirs, files in os.walk(root):
    for filename in files:
        if any(k in filename.lower() for k in keywords):
            path = os.path.join(current_root, filename)
            print(path)

In [ ]:
import os
from pathlib import Path

print("DATASET AVAILABILITY")
print("=" * 60)

locations = [
    "/content/work_exp1_corrected",
    "/content/LLVIP",
    "/content/LLVIP_ANNOTATIONS",
    "/content/EXP1_AUDIT/experiment1_backup",
]

for location in locations:
    print(f"\n{location}")
    print("Exists:", os.path.exists(location))

    if os.path.exists(location):
        for item in os.listdir(location)[:12]:
            print("  ", item)

print("\nMODEL WEIGHTS")
print("=" * 60)

backup = Path("/content/EXP1_AUDIT/experiment1_backup")

for model in ["rgb", "thermal", "blend", "vt"]:
    weights = backup / "runs" / model / "weights" / "best.pt"
    print(f"{model}: {weights.exists()}")

print("\nPOSSIBLE VALIDATION IMAGE DIRECTORIES")
print("=" * 60)

for root in ["/content", "/kaggle/input"]:
    if os.path.exists(root):
        matches = []
        for current, dirs, files in os.walk(root):
            dirs[:] = [d for d in dirs if d not in {
                ".cache", "site-packages", "__pycache__"
            }]
            if Path(current).name.lower() in {
                "visible", "infrared", "thermal", "images"
            }:
                matches.append(current)
            if len(matches) >= 30:
                break

        print(root, ":", matches)

In [ ]:
import json
from pathlib import Path

base = Path("/content/EXP1_AUDIT/experiment1_backup")
ann_dir = Path("/content/LLVIP_ANNOTATIONS/coco_annotations")

# Load the original split manifest
with open(base / "split_manifest.json") as f:
    manifest = json.load(f)

# Load recovered training annotations
with open(ann_dir / "train.json") as f:
    train_data = json.load(f)

print("VALIDATION SPLIT")
print("=" * 55)
print("Validation IDs:", len(manifest["val"]))
print("First 10 IDs:", manifest["val"][:10])

print("\nTRAIN ANNOTATION FILE")
print("=" * 55)
print("Images:", len(train_data["images"]))
print("Annotations:", len(train_data["annotations"]))
print("First image entry:")
print(json.dumps(train_data["images"][0], indent=2))

print("\nMODEL DATASET CONFIGURATIONS")
print("=" * 55)

for model in ["rgb", "thermal", "blend", "vt"]:
    path = base / f"{model}.yaml"
    print(f"\n--- {model.upper()} ---")
    print(path.read_text())

In [ ]:
import json

base = "/content/EXP1_AUDIT/experiment1_backup"

with open(f"{base}/split_manifest.json") as f:
    manifest = json.load(f)

with open("/content/LLVIP_ANNOTATIONS/coco_annotations/train.json") as f:
    data = json.load(f)

val_ids = set(map(str, manifest["val"]))
image_ids = {str(x["id"]) for x in data["images"]}

print("Validation IDs:", len(val_ids))
print("Matched:", len(val_ids & image_ids))
print("Missing:", len(val_ids - image_ids))
print("Missing examples:", list(val_ids - image_ids)[:10])

In [ ]:
import requests

url = "https://huggingface.co/datasets/McCheng/LLVIP/resolve/main/LLVIP.zip"

response = requests.head(
    url,
    allow_redirects=True,
    timeout=30
)

print("HTTP status:", response.status_code)
print("Final URL:", response.url[:180])
print("Content length (GB):",
      round(int(response.headers.get("content-length", 0)) / (1024**3), 2))
print("Content type:", response.headers.get("content-type"))


In [ ]:
import requests

url = "https://huggingface.co/datasets/McCheng/LLVIP/resolve/main/LLVIP.zip"

response = requests.get(
    url,
    headers={"Range": "bytes=-65536"},
    allow_redirects=True,
    stream=True,
    timeout=60
)

print("HTTP status:", response.status_code)
print("Content-Range:", response.headers.get("Content-Range"))
print("Accept-Ranges:", response.headers.get("Accept-Ranges"))
print("Content-Length:", response.headers.get("Content-Length"))

response.close()

In [ ]:
!pip -q install remotezip

import json
from remotezip import RemoteZip
from pathlib import PurePosixPath

base = "/content/EXP1_AUDIT/experiment1_backup"
url = "https://huggingface.co/datasets/McCheng/LLVIP/resolve/main/LLVIP.zip"

with open(f"{base}/split_manifest.json") as f:
    manifest = json.load(f)

val_ids = set(map(str, manifest["val"]))

print("Inspecting ZIP directory using HTTP range requests...")

with RemoteZip(url) as z:
    matches = []

    for info in z.infolist():
        path = PurePosixPath(info.filename)

        if path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            if path.stem in val_ids:
                matches.append(info)

    print("\nMATCHING VALIDATION IMAGE FILES:", len(matches))
    print("TOTAL UNCOMPRESSED SIZE (GB):",
          round(sum(x.file_size for x in matches) / (1024**3), 3))

    print("\nSAMPLE MATCHES:")
    for item in matches[:20]:
        print(item.filename, "|", item.file_size, "bytes")

    print("\nTOP-LEVEL ARCHIVE FOLDERS:")
    folders = sorted({
        PurePosixPath(x.filename).parts[0]
        for x in z.infolist()
        if PurePosixPath(x.filename).parts
    })
    print(folders[:30])


In [ ]:
import json
from remotezip import RemoteZip
from pathlib import PurePosixPath

base = "/content/EXP1_AUDIT/experiment1_backup"
url = "https://huggingface.co/datasets/McCheng/LLVIP/resolve/main/LLVIP.zip"

with open(f"{base}/split_manifest.json") as f:
    manifest = json.load(f)

val_ids = set(map(str, manifest["val"]))

with RemoteZip(url) as z:
    image_paths = [
        x.filename for x in z.infolist()
        if PurePosixPath(x.filename).suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    matched_paths = [
        p for p in image_paths
        if PurePosixPath(p).stem in val_ids
    ]

matched_ids = {PurePosixPath(p).stem for p in matched_paths}

print("Validation IDs:", len(val_ids))
print("Matched image files:", len(matched_paths))
print("Unique matched IDs:", len(matched_ids))
print("Missing IDs:", len(val_ids - matched_ids))
print("Unexpected IDs:", len(matched_ids - val_ids))
print("\nFirst 10 validation IDs:", sorted(val_ids)[:10])
print("First 10 matched IDs:", sorted(matched_ids)[:10])
print("\nFirst 10 matched paths:")
for p in matched_paths[:10]:
    print(p)


In [ ]:
from collections import Counter
from pathlib import PurePosixPath
from remotezip import RemoteZip
import json

base = "/content/EXP1_AUDIT/experiment1_backup"
url = "https://huggingface.co/datasets/McCheng/LLVIP/resolve/main/LLVIP.zip"

with open(f"{base}/split_manifest.json") as f:
    val_ids = set(map(str, json.load(f)["val"]))

with RemoteZip(url) as z:
    matches = [
        x.filename for x in z.infolist()
        if PurePosixPath(x.filename).stem in val_ids
        and PurePosixPath(x.filename).suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

counts = Counter(PurePosixPath(p).parts[1] for p in matches)

print("MODALITY AUDIT")
print("=" * 45)
print("Visible:", counts.get("visible", 0))
print("Infrared:", counts.get("infrared", 0))
print("Total:", len(matches))
print("Expected:", 1000)

assert counts.get("visible", 0) == 500
assert counts.get("infrared", 0) == 500
assert len(matches) == 1000

print("\nPASS: Exactly 500 visible and 500 infrared images.")

In [ ]:
import os
import json
from pathlib import PurePosixPath
from remotezip import RemoteZip

base = "/content/EXP1_AUDIT/experiment1_backup"
url = "https://huggingface.co/datasets/McCheng/LLVIP/resolve/main/LLVIP.zip"

output = "/content/EXP1_VALIDATION_SOURCE"

with open(f"{base}/split_manifest.json") as f:
    val_ids = set(map(str, json.load(f)["val"]))

os.makedirs(output, exist_ok=True)

counts = {"visible": 0, "infrared": 0}
total_bytes = 0

with RemoteZip(url) as z:
    for info in z.infolist():
        path = PurePosixPath(info.filename)

        if path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        modality = path.parts[1] if len(path.parts) > 1 else None

        if modality not in counts or path.stem not in val_ids:
            continue

        destination = os.path.join(output, modality, path.name)
        os.makedirs(os.path.dirname(destination), exist_ok=True)

        with z.open(info) as source, open(destination, "wb") as target:
            while True:
                chunk = source.read(1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)
                total_bytes += len(chunk)

        counts[modality] += 1

print("EXTRACTION COMPLETE")
print("=" * 45)
print("Visible:", counts["visible"])
print("Infrared:", counts["infrared"])
print("Total:", sum(counts.values()))
print("Extracted size (MB):", round(total_bytes / (1024**2), 2))

assert counts == {"visible": 500, "infrared": 500}
print("\nPASS: All validation source images extracted.")


In [ ]:
import os
from pathlib import Path

base = Path("/content/EXP1_AUDIT/experiment1_backup")

for model in ["rgb", "thermal", "blend", "vt"]:
    print(f"\n{'='*55}")
    print(model.upper())
    print("="*55)

    args = base / "runs" / model / "args.yaml"
    if args.exists():
        print(args.read_text())

print("\nOTHER BACKUP FILES")
for p in base.rglob("*"):
    if p.is_file() and any(
        word in p.name.lower()
        for word in ["preprocess", "fusion", "blend", "transform", "pipeline"]
    ):
        print(p)

In [ ]:
from pathlib import Path

base = Path("/content/EXP1_AUDIT/experiment1_backup")

for name in ["blend.yaml", "vt.yaml"]:
    path = base / name
    print(f"\n{'='*50}")
    print(name.upper())
    print("="*50)
    print(path.read_text())

print("\nBACKUP ROOT CONTENTS")
for p in sorted(base.iterdir()):
    print(p.name, "<DIR>" if p.is_dir() else f"{p.stat().st_size:,} bytes")

In [ ]:
from pathlib import Path
from PIL import Image

base = Path("/content/EXP1_AUDIT/experiment1_backup/runs")

for model in ["rgb", "thermal", "blend", "vt"]:
    folder = base / model

    print(f"\n--- {model.upper()} ---")

    for p in sorted(folder.glob("train_batch*")):
        try:
            img = Image.open(p)
            print(p.name, img.size, img.mode)
        except Exception as e:
            print(p.name, "Could not inspect:", e)



In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

base = Path("/content/EXP1_AUDIT/experiment1_backup/runs")

fig, axes = plt.subplots(2, 2, figsize=(16, 16))

models = ["rgb", "thermal", "blend", "vt"]

for ax, model in zip(axes.flat, models):
    path = base / model / "train_batch0.jpg"
    img = Image.open(path)

    ax.imshow(img)
    ax.set_title(model.upper(), fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

root = Path("/content")

for folder in ["work_exp1_corrected", "EXP1_AUDIT/experiment1_backup"]:
    path = root / folder

    print(f"\n========== {folder} ==========")

    if not path.exists():
        print("Not found")
        continue

    for item in sorted(path.iterdir()):
        print(item.name, " [DIR]" if item.is_dir() else "")

In [ ]:
from pathlib import Path

root = Path("/content/EXP1_AUDIT/experiment1_backup")

extensions = {".py", ".ipynb", ".yaml", ".yml", ".json", ".txt"}

print("SEARCHING BACKUP FOR IMPLEMENTATION FILES\n")

for p in sorted(root.rglob("*")):
    if p.is_file() and p.suffix.lower() in extensions:
        print(p.relative_to(root))

In [ ]:
from pathlib import Path

backup = Path("/content/EXP1_AUDIT/experiment1_backup")
images = Path("/content/EXP1_VALIDATION_SOURCE")

print("VALIDATION IMAGES")
for modality in ["visible", "infrared"]:
    folder = images / modality
    count = len(list(folder.glob("*.jpg"))) if folder.exists() else 0
    print(f"{modality}: {count} images")

print("\nMODEL WEIGHTS")
for model in ["rgb", "thermal", "blend", "vt"]:
    weights = backup / "runs" / model / "weights" / "best.pt"
    print(f"{model}: {'FOUND' if weights.exists() else 'MISSING'}")

In [ ]:

from pathlib import Path
import pickle
import gc
import torch
from ultralytics import YOLO

backup = Path("/content/EXP1_AUDIT/experiment1_backup")
image_root = Path("/content/EXP1_VALIDATION_SOURCE")
output_dir = Path("/content/EXP1_VALIDATION_RESULTS")
output_dir.mkdir(parents=True, exist_ok=True)

device = 0 if torch.cuda.is_available() else "cpu"

experiments = {
    "rgb": image_root / "visible",
    "thermal": image_root / "infrared"
}

all_predictions = {}

for name, image_dir in experiments.items():
    print(f"\n{'='*55}")
    print(f"RUNNING VALIDATION: {name.upper()}")
    print(f"{'='*55}")

    weights = backup / "runs" / name / "weights" / "best.pt"
    image_paths = sorted(image_dir.glob("*.jpg"))

    assert weights.exists(), f"Missing weights: {weights}"
    assert len(image_paths) == 500, f"Expected 500 images, found {len(image_paths)}"

    model = YOLO(str(weights))
    predictions = {}

    results = model.predict(
        source=[str(p) for p in image_paths],
        imgsz=640,
        conf=0.001,
        iou=0.70,
        max_det=300,
        batch=16,
        device=device,
        workers=0,
        stream=True,
        verbose=False
    )

    for index, result in enumerate(results, start=1):
        image_id = Path(result.path).stem

        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes.xyxy.cpu().numpy()
            scores = result.boxes.conf.cpu().numpy()
        else:
            boxes = []
            scores = []

        predictions[image_id] = {
            "boxes_xyxy": boxes,
            "scores": scores,
            "width": result.orig_shape[1],
            "height": result.orig_shape[0]
        }

        if index % 100 == 0:
            print(f"Processed {index}/500 images")

    all_predictions[name] = predictions

    output_file = output_dir / f"{name}_validation_predictions.pkl"
    with open(output_file, "wb") as f:
        pickle.dump(predictions, f)

    total_detections = sum(
        len(item["scores"]) for item in predictions.values()
    )

    print(f"Completed: {len(predictions)} images")
    print(f"Total detections: {total_detections}")
    print(f"Saved: {output_file}")

    del model, results
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nRGB AND THERMAL VALIDATION INFERENCE COMPLETE.")



In [ ]:
%pip install -q ultralytics

In [ ]:
import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:

import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

annotation_file = Path(
    "/content/LLVIP_ANNOTATIONS/coco_annotations/train.json"
)
manifest_file = Path(
    "/content/EXP1_AUDIT/experiment1_backup/split_manifest.json"
)
prediction_dir = Path("/content/EXP1_VALIDATION_RESULTS")

# Load validation IDs
with open(manifest_file) as f:
    manifest = json.load(f)

val_ids = set(str(x) for x in manifest["val"])

# Load COCO annotations
with open(annotation_file) as f:
    coco = json.load(f)

gt_by_image = {}

for image in coco["images"]:
    image_id = Path(image["file_name"]).stem

    if image_id not in val_ids:
        continue

    gt_by_image[image_id] = []

for ann in coco["annotations"]:
    image_id = str(
        next(
            (Path(img["file_name"]).stem
             for img in coco["images"]
             if img["id"] == ann["image_id"]),
            ""
        )
    )

    if image_id not in gt_by_image:
        continue

    if ann.get("iscrowd", 0):
        continue

    x, y, w, h = ann["bbox"]
    gt_by_image[image_id].append([x, y, x+w, y+h])

assert len(gt_by_image) == 500, (
    f"Expected 500 validation images, got {len(gt_by_image)}"
)

def calculate_iou(box, boxes):
    if len(boxes) == 0:
        return np.array([])

    boxes = np.asarray(boxes)

    ix1 = np.maximum(box[0], boxes[:, 0])
    iy1 = np.maximum(box[1], boxes[:, 1])
    ix2 = np.minimum(box[2], boxes[:, 2])
    iy2 = np.minimum(box[3], boxes[:, 3])

    intersection = np.maximum(0, ix2-ix1) * np.maximum(0, iy2-iy1)

    area1 = max(0, box[2]-box[0]) * max(0, box[3]-box[1])
    area2 = (
        np.maximum(0, boxes[:, 2]-boxes[:, 0]) *
        np.maximum(0, boxes[:, 3]-boxes[:, 1])
    )

    union = area1 + area2 - intersection
    return intersection / np.maximum(union, 1e-9)


def evaluate(predictions, confidence_threshold):
    TP = FP = FN = 0

    for image_id, gt_boxes in gt_by_image.items():
        pred = predictions.get(image_id)

        if pred is None:
            raise ValueError(f"Missing predictions for {image_id}")

        boxes = np.asarray(pred["boxes_xyxy"]).reshape(-1, 4)
        scores = np.asarray(pred["scores"])

        selected = np.where(scores >= confidence_threshold)[0]
        selected = selected[np.argsort(scores[selected])[::-1]]

        matched = set()

        for idx in selected:
            ious = calculate_iou(boxes[idx], gt_boxes)

            if len(ious) == 0:
                FP += 1
                continue

            order = np.argsort(ious)[::-1]

            match = next(
                (j for j in order
                 if ious[j] >= 0.50 and j not in matched),
                None
            )

            if match is not None:
                matched.add(match)
                TP += 1
            else:
                FP += 1

        FN += len(gt_boxes) - len(matched)

    precision = TP / (TP + FP) if TP + FP else 0
    recall = TP / (TP + FN) if TP + FN else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    return TP, FP, FN, precision, recall, f1


thresholds = [round(x, 2) for x in np.arange(0.05, 0.951, 0.05)]

all_results = []

for model_name in ["rgb", "thermal"]:
    file = prediction_dir / f"{model_name}_validation_predictions.pkl"

    with open(file, "rb") as f:
        predictions = pickle.load(f)

    assert len(predictions) == 500

    print(f"\nEvaluating {model_name.upper()}...")

    for threshold in thresholds:
        TP, FP, FN, P, R, F1 = evaluate(predictions, threshold)

        all_results.append({
            "Model": model_name.upper(),
            "Confidence": threshold,
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "Precision": P,
            "Recall": R,
            "F1": F1
        })

results_df = pd.DataFrame(all_results)

results_df.to_csv(
    prediction_dir / "rgb_thermal_validation_thresholds.csv",
    index=False
)

print("\n========== BEST VALIDATION THRESHOLDS ==========")

for model_name in ["RGB", "THERMAL"]:
    subset = results_df[results_df["Model"] == model_name]
    best = subset.loc[subset["F1"].idxmax()]

    print(f"\n{model_name}")
    print(best.to_string())

print("\nFull threshold results saved.")
print(prediction_dir / "rgb_thermal_validation_thresholds.csv")


In [ ]:

import cv2
import numpy as np
from pathlib import Path

source = Path("/content/EXP1_VALIDATION_SOURCE")
output = Path("/content/EXP1_VALIDATION_SOURCE")

visible_dir = source / "visible"
infrared_dir = source / "infrared"

blend_dir = output / "blend"
vt_dir = output / "vt"

blend_dir.mkdir(exist_ok=True)
vt_dir.mkdir(exist_ok=True)

visible_files = sorted(visible_dir.glob("*.jpg"))
infrared_files = sorted(infrared_dir.glob("*.jpg"))

assert len(visible_files) == 500
assert len(infrared_files) == 500

for index, visible_path in enumerate(visible_files, start=1):

    name = visible_path.stem
    infrared_path = infrared_dir / f"{name}.jpg"

    assert infrared_path.exists(), f"Missing IR pair: {name}"

    rgb = cv2.imread(str(visible_path), cv2.IMREAD_COLOR)
    ir = cv2.imread(str(infrared_path), cv2.IMREAD_GRAYSCALE)

    assert rgb is not None and ir is not None

    height, width = rgb.shape[:2]

    if ir.shape[:2] != (height, width):
        ir = cv2.resize(
            ir, (width, height),
            interpolation=cv2.INTER_LINEAR
        )

    ir_bgr = cv2.merge([ir, ir, ir])

    # Exact original Blend preprocessing
    blend = cv2.addWeighted(rgb, 0.5, ir_bgr, 0.5, 0)

    # Exact original VT preprocessing
    vt = cv2.merge([rgb[:, :, 0], rgb[:, :, 1], ir])

    assert cv2.imwrite(str(blend_dir / f"{name}.jpg"), blend)
    assert cv2.imwrite(str(vt_dir / f"{name}.jpg"), vt)

    if index % 100 == 0:
        print(f"Processed {index}/500 pairs")

print("\nREBUILD COMPLETE")
print("Blend images:", len(list(blend_dir.glob("*.jpg"))))
print("VT images:", len(list(vt_dir.glob("*.jpg"))))


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import pickle
import torch

base = Path("/content/EXP1_VALIDATION_SOURCE")
backup = Path("/content/EXP1_AUDIT/experiment1_backup")
out = Path("/content/EXP1_VALIDATION_RESULTS")
out.mkdir(exist_ok=True)

configs = {
    "rgb": base / "visible",
    "thermal": base / "infrared",
    "blend": base / "blend",
    "vt": base / "vt",
}

results = {}

for name, image_dir in configs.items():
    print(f"\n{'='*55}")
    print("PROCESSING:", name.upper())
    print(f"{'='*55}")

    weights = backup / "runs" / name / "weights" / "best.pt"
    assert weights.exists(), f"Missing weights: {weights}"

    model = YOLO(str(weights))

    predictions = {}

    image_paths = sorted(image_dir.glob("*.jpg"))
    assert len(image_paths) == 500, f"{name}: expected 500 images"

    for result in model.predict(
        source=str(image_dir),
        imgsz=640,
        conf=0.001,
        iou=0.6,
        max_det=300,
        batch=16,
        device=0 if torch.cuda.is_available() else "cpu",
        stream=True,
        verbose=False
    ):
        image_id = result.path.split("/")[-1].replace(".jpg", "")

        if result.boxes is None or len(result.boxes) == 0:
            boxes = []
            scores = []
        else:
            boxes = result.boxes.xyxyn.cpu().numpy()
            scores = result.boxes.conf.cpu().numpy()

        predictions[image_id] = (
            boxes,
            scores,
            [result.orig_shape[1], result.orig_shape[0]]
        )

    assert len(predictions) == 500

    output_file = out / f"{name}_validation_predictions.pkl"

    with open(output_file, "wb") as f:
        pickle.dump(predictions, f)

    total_detections = sum(len(v[1]) for v in predictions.values())

    print("Images:", len(predictions))
    print("Detections:", total_detections)
    print("Saved:", output_file)

    results[name] = total_detections

print("\nALL FOUR VALIDATION INFERENCE RUNS COMPLETED")
print(results)


In [ ]:

import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

# Paths
root = Path("/content")
ann_file = root / "LLVIP_ANNOTATIONS/coco_annotations/train.json"
manifest_file = root / "EXP1_AUDIT/experiment1_backup/split_manifest.json"
pred_dir = root / "EXP1_VALIDATION_RESULTS"

# Load validation IDs
with open(manifest_file) as f:
    manifest = json.load(f)

val_ids = set(str(x) for x in manifest["val"])

# Load COCO annotations
with open(ann_file) as f:
    coco = json.load(f)

image_map = {
    str(img["id"]): img
    for img in coco["images"]
}

# Ground truth boxes, indexed by filename stem
gt = {image_id: [] for image_id in val_ids}

for ann in coco["annotations"]:
    if ann.get("iscrowd", 0):
        continue

    image = image_map.get(str(ann["image_id"]))
    if image is None:
        continue

    image_id = Path(image["file_name"]).stem

    if image_id not in val_ids:
        continue

    x, y, w, h = ann["bbox"]
    gt[image_id].append([x, y, x + w, y + h])

assert len(gt) == 500
assert all(image_id in image_map or any(
    Path(im["file_name"]).stem == image_id for im in coco["images"]
) for image_id in val_ids)

total_gt = sum(len(boxes) for boxes in gt.values())
print("Validation images:", len(gt))
print("Ground-truth persons:", total_gt)

def calculate_iou(box, boxes):
    if len(boxes) == 0:
        return np.array([])

    boxes = np.asarray(boxes, dtype=float)

    ix1 = np.maximum(box[0], boxes[:, 0])
    iy1 = np.maximum(box[1], boxes[:, 1])
    ix2 = np.minimum(box[2], boxes[:, 2])
    iy2 = np.minimum(box[3], boxes[:, 3])

    intersection = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)

    area1 = max(0, box[2] - box[0]) * max(0, box[3] - box[1])
    area2 = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * \
            np.maximum(0, boxes[:, 3] - boxes[:, 1])

    union = area1 + area2 - intersection
    return intersection / np.maximum(union, 1e-12)


def evaluate(predictions, threshold, iou_threshold=0.50):
    TP = FP = FN = 0

    for image_id in val_ids:
        gt_boxes = np.asarray(gt[image_id], dtype=float).reshape(-1, 4)

        boxes, scores, dimensions = predictions[image_id]
        boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
        scores = np.asarray(scores, dtype=float)

        width, height = dimensions

        # Convert normalized xyxy predictions to pixel xyxy
        boxes = boxes * np.array([width, height, width, height])

        selected = np.where(scores >= threshold)[0]
        selected = selected[np.argsort(scores[selected])[::-1]]

        matched_gt = set()

        for index in selected:
            if len(gt_boxes) == 0:
                FP += 1
                continue

            ious = calculate_iou(boxes[index], gt_boxes)

            # Highest-IoU unmatched ground-truth box
            order = np.argsort(ious)[::-1]
            match = next(
                (j for j in order
                 if ious[j] >= iou_threshold and j not in matched_gt),
                None
            )

            if match is not None:
                TP += 1
                matched_gt.add(match)
            else:
                FP += 1

        FN += len(gt_boxes) - len(matched_gt)

    precision = TP / (TP + FP) if TP + FP else 0
    recall = TP / (TP + FN) if TP + FN else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    return TP, FP, FN, precision, recall, f1


thresholds = np.round(np.arange(0.05, 1.00, 0.05), 2)
all_results = []

for name in ["rgb", "thermal", "blend", "vt"]:
    file = pred_dir / f"{name}_validation_predictions.pkl"

    with open(file, "rb") as f:
        predictions = pickle.load(f)

    assert set(predictions.keys()) == val_ids, (
        f"{name}: prediction IDs do not match validation IDs"
    )

    for threshold in thresholds:
        TP, FP, FN, P, R, F1 = evaluate(predictions, threshold)

        all_results.append({
            "Configuration": name.upper(),
            "Confidence": threshold,
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "Precision": P,
            "Recall": R,
            "F1": F1
        })

results = pd.DataFrame(all_results)

# Best tested threshold per configuration
best = (
    results.sort_values(
        ["Configuration", "F1", "Confidence"],
        ascending=[True, False, True]
    )
    .groupby("Configuration", as_index=False)
    .first()
)

output_file = pred_dir / "all_four_validation_threshold_results.csv"
best_file = pred_dir / "all_four_validation_best_thresholds.csv"

results.to_csv(output_file, index=False)
best.to_csv(best_file, index=False)

print("\n========== BEST VALIDATION RESULTS ==========")
print(best.to_string(index=False))

print("\nFull threshold sweep:", output_file)
print("Best thresholds:", best_file)


In [ ]:

import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

root = Path("/content")
backup = root / "EXP1_AUDIT/experiment1_backup"
ann_file = root / "LLVIP_ANNOTATIONS/coco_annotations/test.json"

# Frozen thresholds selected using validation data only
thresholds = {
    "rgb": 0.25,
    "thermal": 0.40,
    "blend": 0.35,
    "vt": 0.45
}

with open(backup / "split_manifest.json") as f:
    manifest = json.load(f)

test_ids = set(str(x) for x in manifest["test"])

with open(ann_file) as f:
    coco = json.load(f)

image_map = {
    str(img["id"]): img for img in coco["images"]
}

gt = {image_id: [] for image_id in test_ids}

for ann in coco["annotations"]:
    if ann.get("iscrowd", 0):
        continue

    image = image_map.get(str(ann["image_id"]))
    if image is None:
        continue

    image_id = Path(image["file_name"]).stem

    if image_id in test_ids:
        x, y, w, h = ann["bbox"]
        gt[image_id].append([x, y, x+w, y+h])

assert len(gt) == 3463
total_gt = sum(len(v) for v in gt.values())

print("Test images:", len(gt))
print("Ground-truth persons:", total_gt)

def iou_one_to_many(box, boxes):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)

    if len(boxes) == 0:
        return np.array([])

    ix1 = np.maximum(box[0], boxes[:, 0])
    iy1 = np.maximum(box[1], boxes[:, 1])
    ix2 = np.minimum(box[2], boxes[:, 2])
    iy2 = np.minimum(box[3], boxes[:, 3])

    inter = np.maximum(0, ix2-ix1) * np.maximum(0, iy2-iy1)

    area1 = max(0, box[2]-box[0]) * max(0, box[3]-box[1])
    area2 = np.maximum(0, boxes[:, 2]-boxes[:, 0]) * \
            np.maximum(0, boxes[:, 3]-boxes[:, 1])

    return inter / np.maximum(area1 + area2 - inter, 1e-12)


def evaluate(predictions, threshold):
    TP = FP = FN = 0

    for image_id in test_ids:
        gt_boxes = np.asarray(gt[image_id], dtype=float).reshape(-1, 4)

        boxes, scores, dimensions = predictions[image_id]
        boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
        scores = np.asarray(scores, dtype=float)

        width, height = dimensions
        boxes = boxes * np.array([width, height, width, height])

        selected = np.where(scores >= threshold)[0]
        selected = selected[np.argsort(scores[selected])[::-1]]

        matched = set()

        for index in selected:
            if len(gt_boxes) == 0:
                FP += 1
                continue

            ious = iou_one_to_many(boxes[index], gt_boxes)
            order = np.argsort(ious)[::-1]

            match = next(
                (j for j in order
                 if ious[j] >= 0.50 and j not in matched),
                None
            )

            if match is None:
                FP += 1
            else:
                TP += 1
                matched.add(match)

        FN += len(gt_boxes) - len(matched)

    precision = TP / (TP + FP) if TP + FP else 0
    recall = TP / (TP + FN) if TP + FN else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    return TP, FP, FN, precision, recall, f1


with open(backup / "test_predictions.pkl", "rb") as f:
    test_predictions = pickle.load(f)

rows = []

for name, threshold in thresholds.items():
    predictions = test_predictions[name]

    assert len(predictions) == 3463
    assert set(predictions.keys()) == test_ids

    TP, FP, FN, P, R, F1 = evaluate(predictions, threshold)

    rows.append({
        "Configuration": name.upper(),
        "Frozen_Confidence": threshold,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "Precision": P,
        "Recall": R,
        "F1": F1
    })

results = pd.DataFrame(rows)

output = backup / "exp1_final_test_prf1.csv"
results.to_csv(output, index=False)

print("\n========== FINAL TEST RESULTS ==========")
print(results.to_string(index=False))
print("\nSaved:", output)


In [ ]:
from google.colab import files

files.download(
    "/content/EXP1_AUDIT/experiment1_backup/exp1_final_test_prf1.csv"
)


In [ ]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

folder = Path("/content/EXP1_AUDIT/experiment1_backup")
validation = Path("/content/EXP1_VALIDATION_RESULTS")

csv_files = [
    folder / "exp1_final_test_prf1.csv",
    folder / "exp1_ap50_independent_audit.csv",
    validation / "all_four_validation_threshold_results.csv",
    validation / "all_four_validation_best_thresholds.csv"
]

for path in csv_files:
    assert path.exists(), f"Missing file: {path}"

zip_path = Path("/content/Experiment1_Final_CSV_Evidence.zip")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for path in csv_files:
        archive.write(path, arcname=path.name)

print("Evidence package created:")
print(zip_path)
print("Files included:", len(csv_files))

files.download(str(zip_path))


In [ ]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import os

backup = Path("/content/EXP1_AUDIT/experiment1_backup")
results = Path("/content/EXP1_VALIDATION_RESULTS")

files_to_save = [
    backup / "exp1_final_test_prf1.csv",
    backup / "exp1_ap50_independent_audit.csv",
    results / "all_four_validation_threshold_results.csv",
    results / "all_four_validation_best_thresholds.csv",
    results / "rgb_validation_predictions.pkl",
    results / "thermal_validation_predictions.pkl",
    results / "blend_validation_predictions.pkl",
    results / "vt_validation_predictions.pkl",
]

for path in files_to_save:
    assert path.exists(), f"Missing file: {path}"

archive_path = Path("/content/Experiment1_Final_Evaluation_Evidence.zip")

with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for path in files_to_save:
        archive.write(path, arcname=path.name)

size_mb = archive_path.stat().st_size / (1024**2)

print("ARCHIVE CREATED")
print("Files included:", len(files_to_save))
print(f"Archive size: {size_mb:.2f} MB")
print("Location:", archive_path)


In [ ]:

%pip install -q huggingface_hub

from huggingface_hub import HfApi, login
from getpass import getpass
from pathlib import Path

repo_id = "SakibM19/border-surveillance-research-backups"
archive = Path("/content/Experiment1_Final_Evaluation_Evidence.zip")

assert archive.exists(), "Archive not found!"

token = getpass("Enter your Hugging Face WRITE token: ")
login(token=token)

api = HfApi()

api.upload_file(
    path_or_fileobj=str(archive),
    path_in_repo="Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip",
    repo_id=repo_id,
    repo_type="dataset"
)

print("\nUPLOAD COMPLETED")
print("Repository:", repo_id)
print("Uploaded:", "Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip")


In [ ]:

from huggingface_hub import HfApi

api = HfApi()
repo_id = "SakibM19/border-surveillance-research-backups"

files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

target = "Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip"

if target in files:
    print("VERIFIED: Final evaluation archive exists in Hugging Face.")
    print(target)
else:
    print("NOT FOUND: Please check the repository upload.")


In [ ]:

from pathlib import Path

backup = Path("/content/EXP1_AUDIT/experiment1_backup")
results = Path("/content/EXP1_VALIDATION_RESULTS")

print("=" * 65)
print("EXPERIMENT 1 — COMPLETE EVIDENCE INVENTORY")
print("=" * 65)

# 1. Model checkpoints and training evidence
for name in ["rgb", "thermal", "blend", "vt"]:
    folder = backup / "runs" / name

    print(f"\n--- {name.upper()} ---")

    if not folder.exists():
        print("MODEL FOLDER MISSING")
        continue

    for subfolder in ["weights", ""]:
        target = folder / subfolder if subfolder else folder

        if target.exists():
            files = sorted(
                p for p in target.iterdir()
                if p.is_file()
            )

            if files:
                print(f"\n{target.name or name}:")
                for p in files:
                    print(f"  {p.name} ({p.stat().st_size / 1024:.1f} KB)")

# 2. Existing figures
print("\n" + "=" * 65)
print("EXISTING FIGURES")
print("=" * 65)

for name in ["rgb", "thermal", "blend", "vt"]:
    folder = backup / "runs" / name

    images = sorted(folder.rglob("*.png")) if folder.exists() else []
    images += sorted(folder.rglob("*.jpg")) if folder.exists() else []

    print(f"\n{name.upper()}: {len(images)} image files")

    for p in images:
        print(" ", p.relative_to(backup))

# 3. Evaluation evidence
print("\n" + "=" * 65)
print("EVALUATION FILES")
print("=" * 65)

for folder in [backup, results]:
    print(f"\n{folder}:")
    if folder.exists():
        for p in sorted(folder.iterdir()):
            if p.is_file() and p.suffix.lower() in [".csv", ".pkl", ".json"]:
                print(f"  {p.name} ({p.stat().st_size / 1024:.1f} KB)")

print("\nINVENTORY COMPLETE")


In [ ]:

import pandas as pd
from pathlib import Path

backup = Path("/content/EXP1_AUDIT/experiment1_backup")

for name in ["rgb", "thermal", "blend", "vt"]:
    path = backup / "runs" / name / "results.csv"

    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    print("\n" + "=" * 55)
    print(name.upper())
    print("=" * 55)

    print("Rows (epochs):", len(df))
    print("Columns:")
    for col in df.columns:
        print(" ", col)

    print("\nFirst 2 rows:")
    print(df.head(2).to_string(index=False))

print("\nCSV INSPECTION COMPLETE")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

backup = Path("/content/EXP1_AUDIT/experiment1_backup")
out = Path("/content/EXP1_PAPER_FIGURES")
out.mkdir(parents=True, exist_ok=True)

models = ["RGB", "Thermal", "Blend", "VT"]
data = {}

for name in models:
    df = pd.read_csv(
        backup / "runs" / name.lower() / "results.csv"
    )
    df.columns = df.columns.str.strip()

    assert len(df) == 20, f"{name}: expected 20 epochs"

    data[name] = df

# ---------------------------------------
# FIGURE 1: TRAINING LOSS CURVES
# ---------------------------------------

loss_metrics = [
    ("train/box_loss", "Box loss"),
    ("train/cls_loss", "Classification loss"),
    ("train/dfl_loss", "Distribution focal loss")
]

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))

for ax, (metric, title) in zip(axes, loss_metrics):
    for name in models:
        df = data[name]
        ax.plot(
            df["epoch"],
            df[metric],
            marker="o",
            markersize=2.5,
            linewidth=1.5,
            label=name
        )

    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Epoch", fontsize=9)
    ax.set_ylabel("Loss", fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.set_xticks([1, 5, 10, 15, 20])
    ax.tick_params(labelsize=8)

axes[0].legend(fontsize=8, frameon=False)

fig.tight_layout()

fig.savefig(
    out / "Figure_1_Training_Loss.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
fig.savefig(
    out / "Figure_1_Training_Loss.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

# ---------------------------------------
# FIGURE 2: VALIDATION PERFORMANCE
# ---------------------------------------

metrics = [
    ("metrics/mAP50(B)", "Validation mAP@0.50"),
    ("metrics/mAP50-95(B)", "Validation mAP@0.50:0.95")
]

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3))

for ax, (metric, title) in zip(axes, metrics):
    for name in models:
        df = data[name]
        ax.plot(
            df["epoch"],
            df[metric],
            marker="o",
            markersize=2.5,
            linewidth=1.5,
            label=name
        )

    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Epoch", fontsize=9)
    ax.set_ylabel("mAP", fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.set_xticks([1, 5, 10, 15, 20])
    ax.tick_params(labelsize=8)

axes[0].legend(fontsize=8, frameon=False)

fig.tight_layout()

fig.savefig(
    out / "Figure_2_Validation_Performance.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
fig.savefig(
    out / "Figure_2_Validation_Performance.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

# ---------------------------------------
# VERIFY EXPORTED FILES
# ---------------------------------------

print("\nFIGURE EXPORT COMPLETE")
for path in sorted(out.iterdir()):
    print(
        path.name,
        f"| {path.stat().st_size / 1024:.1f} KB"
    )


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

backup = Path("/content/EXP1_AUDIT/experiment1_backup")
out = Path("/content/EXP1_PAPER_FIGURES")
out.mkdir(exist_ok=True)

# ---------------------------------------
# FIGURE 3: TEST AP50 (FIVE CONFIGURATIONS)
# ---------------------------------------

ap = pd.read_csv(backup / "exp1_ap50_independent_audit.csv")

print("AP50 CSV columns:", ap.columns.tolist())
print(ap.to_string(index=False))

# Detect the relevant columns
config_col = next(c for c in ap.columns if "config" in c.lower())
ap_col = next(
    c for c in ap.columns
    if "reproduced" in c.lower() or "ap50" in c.lower()
)

# Use the independently reproduced AP50
ap = ap[[config_col, ap_col]].copy()
ap.columns = ["Configuration", "AP50"]
ap["Configuration"] = ap["Configuration"].str.upper()

fig, ax = plt.subplots(figsize=(7.0, 3.8))

bars = ax.bar(
    ap["Configuration"],
    ap["AP50"],
    width=0.58
)

ax.set_ylabel("AP@0.50", fontsize=10)
ax.set_xlabel("Configuration", fontsize=10)
ax.set_title("Independent Test AP@0.50 Comparison", fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.35)

for bar, value in zip(bars, ap["AP50"]):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.015,
        f"{value:.4f}",
        ha="center",
        fontsize=8
    )

fig.tight_layout()
fig.savefig(out / "Figure_3_Test_AP50.png", dpi=300,
            bbox_inches="tight", facecolor="white")
fig.savefig(out / "Figure_3_Test_AP50.pdf",
            bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

# ---------------------------------------
# FIGURE 4: TEST PRECISION, RECALL, F1
# ---------------------------------------

prf = pd.read_csv(backup / "exp1_final_test_prf1.csv")

print("\nFinal test P/R/F1:")
print(prf.to_string(index=False))

config_col = next(c for c in prf.columns if "config" in c.lower())

metric_cols = ["Precision", "Recall", "F1"]
assert all(c in prf.columns for c in metric_cols)

prf = prf.set_index(config_col)[metric_cols]
prf.index = prf.index.str.upper()

fig, ax = plt.subplots(figsize=(7.2, 4.0))

x = np.arange(len(prf.index))
width = 0.23

for i, metric in enumerate(metric_cols):
    ax.bar(
        x + (i - 1) * width,
        prf[metric],
        width,
        label=metric
    )

ax.set_xticks(x)
ax.set_xticklabels(prf.index)
ax.set_ylabel("Score", fontsize=10)
ax.set_xlabel("Configuration", fontsize=10)
ax.set_title("Final Test Precision, Recall and F1", fontsize=11)
ax.set_ylim(0, 1.08)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(out / "Figure_4_Test_Precision_Recall_F1.png",
            dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(out / "Figure_4_Test_Precision_Recall_F1.pdf",
            bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)

print("\nFIGURES 3 AND 4 EXPORTED")
for p in sorted(out.glob("Figure_3*")) + sorted(out.glob("Figure_4*")):
    print(p.name, f"{p.stat().st_size/1024:.1f} KB")


In [ ]:
import pandas as pd
from pathlib import Path

file = Path(
    "/content/EXP1_VALIDATION_RESULTS/"
    "all_four_validation_threshold_results.csv"
)

df = pd.read_csv(file)

print("Columns:")
print(df.columns.tolist())

print("\nFirst 12 rows:")
print(df.head(12).to_string(index=False))

print("\nConfigurations:")
print(df.iloc[:, 0].unique())

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

csv_path = Path(
    "/content/EXP1_VALIDATION_RESULTS/"
    "all_four_validation_threshold_results.csv"
)
out = Path("/content/EXP1_PAPER_FIGURES")
out.mkdir(exist_ok=True)

df = pd.read_csv(csv_path)

required = {"Configuration", "Confidence", "F1"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Rows per configuration:")
print(df.groupby("Configuration").size())

fig, ax = plt.subplots(figsize=(7.0, 4.0))

for config in ["RGB", "Thermal", "Blend", "VT"]:
    subset = df[df["Configuration"].str.upper() == config.upper()]
    subset = subset.sort_values("Confidence")

    ax.plot(
        subset["Confidence"],
        subset["F1"],
        marker="o",
        markersize=3,
        linewidth=1.6,
        label=config
    )

    best = subset.loc[subset["F1"].idxmax()]
    ax.scatter(
        best["Confidence"],
        best["F1"],
        marker="*",
        s=100,
        zorder=5
    )

ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("Validation F1-score")
ax.set_title("Validation F1 versus Confidence Threshold")
ax.set_xlim(0.05, 0.95)
ax.set_ylim(0, 1.0)
ax.set_xticks([i / 100 for i in range(5, 100, 10)])
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(frameon=False)
fig.tight_layout()

fig.savefig(
    out / "Figure_5_Validation_Threshold_F1.png",
    dpi=300, bbox_inches="tight", facecolor="white"
)
fig.savefig(
    out / "Figure_5_Validation_Threshold_F1.pdf",
    bbox_inches="tight", facecolor="white"
)

plt.show()
plt.close(fig)

print("\nFIGURE 5 EXPORTED")
for p in sorted(out.glob("Figure_5*")):
    print(p.name, f"{p.stat().st_size/1024:.1f} KB")


In [ ]:

from pathlib import Path
import pickle

base = Path("/content/EXP1_VALIDATION_RESULTS")
source = Path("/content/EXP1_VALIDATION_SOURCE")

files = {
    "RGB": base / "rgb_validation_predictions.pkl",
    "Thermal": base / "thermal_validation_predictions.pkl",
    "Blend": base / "blend_validation_predictions.pkl",
    "VT": base / "vt_validation_predictions.pkl"
}

print("PREDICTION FILES")
for name, path in files.items():
    print(name, "| exists:", path.exists(),
          "| size:", f"{path.stat().st_size/1024:.1f} KB" if path.exists() else "N/A")

print("\nIMAGE FOLDERS")
for name in ["visible", "infrared", "blend", "vt"]:
    folder = source / name
    images = list(folder.glob("*.jpg")) if folder.exists() else []
    print(name, "| exists:", folder.exists(), "| images:", len(images))

print("\nPREDICTION STRUCTURE")
for name, path in files.items():
    if path.exists():
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"\n{name}: type={type(data).__name__}, entries={len(data)}")
        if len(data):
            first_key = next(iter(data))
            first_value = data[first_key]
            print("First key:", first_key)
            print("First entry type:", type(first_value).__name__)
            print("First entry:", str(first_value)[:350])


In [ ]:

from pathlib import Path
import pickle
import numpy as np

base = Path("/content/EXP1_VALIDATION_RESULTS")
source = Path("/content/EXP1_VALIDATION_SOURCE")

configs = {
    "RGB": ("rgb_validation_predictions.pkl", "visible", 0.25),
    "Thermal": ("thermal_validation_predictions.pkl", "infrared", 0.40),
    "Blend": ("blend_validation_predictions.pkl", "blend", 0.35),
    "VT": ("vt_validation_predictions.pkl", "vt", 0.45)
}

predictions = {}

for name, (filename, folder, threshold) in configs.items():
    with open(base / filename, "rb") as f:
        predictions[name] = pickle.load(f)

    data = predictions[name]
    image_id = sorted(data.keys())[0]
    boxes, scores, dimensions = data[image_id]

    image_path = source / folder / f"{image_id}.jpg"

    print(f"\n{name}")
    print("Image ID:", image_id)
    print("Matching image exists:", image_path.exists())
    print("Image path:", image_path)
    print("Boxes shape:", np.asarray(boxes).shape)
    print("Scores shape:", np.asarray(scores).shape)
    print("Image dimensions [width, height]:", dimensions)
    print("Detections above frozen threshold:",
          int(np.sum(np.asarray(scores) >= threshold)))
    print("Confidence range:",
          (float(np.min(scores)), float(np.max(scores)))
          if len(scores) else "No detections")

# Verify that all four models use exactly the same validation IDs
id_sets = {name: set(data.keys()) for name, data in predictions.items()}

print("\nVALIDATION ID MATCH")
print("All four ID sets identical:",
      all(s == next(iter(id_sets.values())) for s in id_sets.values()))
print("Number of shared IDs:", len(set.intersection(*id_sets.values())))


In [ ]:

import json
import pickle
import numpy as np
from pathlib import Path

# Paths
annotation_path = Path(
    "/content/LLVIP_ANNOTATIONS/coco_annotations/train.json"
)
pred_dir = Path("/content/EXP1_VALIDATION_RESULTS")

configs = {
    "RGB": ("rgb_validation_predictions.pkl", 0.25),
    "Thermal": ("thermal_validation_predictions.pkl", 0.40),
    "Blend": ("blend_validation_predictions.pkl", 0.35),
    "VT": ("vt_validation_predictions.pkl", 0.45)
}

# Load annotations
with open(annotation_path, "r") as f:
    coco = json.load(f)

# Map image filename stem to image ID
image_lookup = {
    Path(img["file_name"]).stem: img["id"]
    for img in coco["images"]
}

# Ground truth boxes in xyxy format
gt_by_image = {}

for ann in coco["annotations"]:
    if ann.get("iscrowd", 0):
        continue

    image_id = ann["image_id"]
    x, y, w, h = ann["bbox"]
    gt_by_image.setdefault(image_id, []).append(
        [x, y, x + w, y + h]
    )

# Load predictions
predictions = {}
for name, (filename, threshold) in configs.items():
    with open(pred_dir / filename, "rb") as f:
        predictions[name] = pickle.load(f)

def calculate_iou(box, boxes):
    if len(boxes) == 0:
        return np.array([])

    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])

    intersection = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)

    area_a = max(0, box[2]-box[0]) * max(0, box[3]-box[1])
    area_b = np.maximum(0, boxes[:, 2]-boxes[:, 0]) * \
             np.maximum(0, boxes[:, 3]-boxes[:, 1])

    union = area_a + area_b - intersection
    return intersection / np.maximum(union, 1e-12)

def evaluate_image(image_id, entry, threshold):
    boxes, scores, dimensions = entry
    width, height = dimensions

    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    scores = np.asarray(scores, dtype=float)

    keep = scores >= threshold
    boxes = boxes[keep].copy()
    scores = scores[keep]

    # Convert normalized xyxy to absolute pixel xyxy
    boxes[:, [0, 2]] *= width
    boxes[:, [1, 3]] *= height

    gt = np.asarray(gt_by_image.get(image_id, []), dtype=float)
    if gt.size == 0:
        gt = np.empty((0, 4))

    matched = set()
    tp = 0
    fp = 0

    for idx in np.argsort(-scores):
        if len(gt) == 0:
            fp += 1
            continue

        ious = calculate_iou(boxes[idx], gt)
        order = np.argsort(-ious)

        match = next(
            (j for j in order if ious[j] >= 0.50 and j not in matched),
            None
        )

        if match is None:
            fp += 1
        else:
            tp += 1
            matched.add(match)

    fn = len(gt) - tp
    return tp, fp, fn

# Evaluate every shared validation image
rows = []

for image_stem in sorted(next(iter(predictions.values())).keys()):
    image_id = image_lookup.get(image_stem)
    if image_id is None:
        continue

    row = {"Image": image_stem, "GT": len(gt_by_image.get(image_id, []))}

    for name, (_, threshold) in configs.items():
        tp, fp, fn = evaluate_image(
            image_id, predictions[name][image_stem], threshold
        )
        row[f"{name}_TP"] = tp
        row[f"{name}_FP"] = fp
        row[f"{name}_FN"] = fn

    rows.append(row)

# Rank scenes by useful qualitative characteristics
print("SCENES WHERE ALL FOUR MODELS DETECT EVERY GT PERSON")
for r in rows:
    if r["GT"] > 0 and all(
        r[f"{name}_FN"] == 0 for name in configs
    ) and all(
        r[f"{name}_FP"] == 0 for name in configs
    ):
        print(r)

print("\nSCENES WHERE BLEND HAS FEWER ERRORS THAN RGB")
candidates = []
for r in rows:
    rgb_errors = r["RGB_FP"] + r["RGB_FN"]
    blend_errors = r["Blend_FP"] + r["Blend_FN"]
    if r["GT"] > 0 and blend_errors < rgb_errors:
        candidates.append((rgb_errors - blend_errors, r))

for improvement, r in sorted(candidates, key=lambda x: -x[0])[:10]:
    print("Improvement:", improvement, "|", r)

print("\nCHALLENGING SCENES: MOST TOTAL ERRORS ACROSS FOUR MODELS")
challenging = sorted(
    rows,
    key=lambda r: sum(
        r[f"{name}_FP"] + r[f"{name}_FN"] for name in configs
    ),
    reverse=True
)

for r in challenging[:10]:
    print(r)


In [ ]:

import pickle
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

source = Path("/content/EXP1_VALIDATION_SOURCE")
pred_dir = Path("/content/EXP1_VALIDATION_RESULTS")
out = Path("/content/EXP1_PAPER_FIGURES")
out.mkdir(exist_ok=True)

configs = {
    "RGB": ("visible", "rgb_validation_predictions.pkl", 0.25),
    "Thermal": ("infrared", "thermal_validation_predictions.pkl", 0.40),
    "Blend": ("blend", "blend_validation_predictions.pkl", 0.35),
    "VT": ("vt", "vt_validation_predictions.pkl", 0.45)
}

scene_ids = ["010001", "060436", "080904", "091150"]

predictions = {}
for name, (_, filename, _) in configs.items():
    with open(pred_dir / filename, "rb") as f:
        predictions[name] = pickle.load(f)

fig, axes = plt.subplots(
    len(scene_ids), len(configs),
    figsize=(14, 12)
)

for row, image_id in enumerate(scene_ids):
    for col, (name, (folder, _, threshold)) in enumerate(configs.items()):
        ax = axes[row, col]

        image_path = source / folder / f"{image_id}.jpg"
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

        if image is None:
            raise FileNotFoundError(image_path)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        boxes, scores, dimensions = predictions[name][image_id]
        boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
        scores = np.asarray(scores, dtype=float)

        width, height = dimensions
        keep = scores >= threshold

        for box, score in zip(boxes[keep], scores[keep]):
            x1, y1, x2, y2 = box
            x1, x2 = int(x1 * width), int(x2 * width)
            y1, y2 = int(y1 * height), int(y2 * height)

            cv2.rectangle(
                image, (x1, y1), (x2, y2),
                (0, 255, 0), 2
            )

            cv2.putText(
                image, f"{score:.2f}",
                (x1, max(18, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (0, 255, 0), 1,
                cv2.LINE_AA
            )

        ax.imshow(image)
        ax.axis("off")

        if row == 0:
            ax.set_title(
                f"{name}\n(conf={threshold:.2f})",
                fontsize=10
            )

        if col == 0:
            ax.text(
                -0.04, 0.5, f"Scene {image_id}",
                transform=ax.transAxes,
                rotation=90,
                va="center", ha="right",
                fontsize=9
            )

fig.suptitle(
    "Qualitative Comparison on LLVIP Validation Images",
    fontsize=13,
    y=0.995
)

plt.tight_layout(rect=[0.02, 0, 1, 0.98])

fig.savefig(
    out / "Figure_6_Qualitative_Comparison.png",
    dpi=300, bbox_inches="tight", facecolor="white"
)
fig.savefig(
    out / "Figure_6_Qualitative_Comparison.pdf",
    bbox_inches="tight", facecolor="white"
)

plt.show()
plt.close(fig)

print("\nFIGURE 6 EXPORTED")
for p in sorted(out.glob("Figure_6*")):
    print(p.name, f"{p.stat().st_size/1024:.1f} KB")


In [ ]:
import pandas as pd
from pathlib import Path

out = Path("/content/EXP1_PAPER_TABLES")
out.mkdir(parents=True, exist_ok=True)

# TABLE I: Experimental settings
table1 = pd.DataFrame([
    ["Model", "YOLO11n"],
    ["Training epochs", "20"],
    ["Batch size", "16"],
    ["Input image size", "640 × 640"],
    ["Random seed", "0"],
    ["Deterministic training", "Enabled"],
    ["Inference confidence", "0.001 (prediction generation)"],
    ["Inference IoU", "0.60 (NMS)"],
    ["Maximum detections/image", "300"],
    ["Evaluation IoU", "0.50"],
    ["Dataset split", "4,000 train / 500 validation / 3,463 test"]
], columns=["Parameter", "Setting"])

# TABLE II: Independently reproduced test AP50
table2 = pd.DataFrame([
    ["RGB", 0.86980],
    ["Thermal", 0.94972],
    ["Blend", 0.94628],
    ["VT", 0.95042],
    ["Late WBF (RGB + Thermal)", 0.93761]
], columns=["Configuration", "Independent AP50"])

# TABLE III: Final held-out test performance
table3 = pd.DataFrame([
    ["RGB", 0.25, 7044, 1577, 1258, 0.817075, 0.848470, 0.832477],
    ["Thermal", 0.40, 7151, 172, 1151, 0.976512, 0.861359, 0.915328],
    ["Blend", 0.35, 7260, 233, 1042, 0.968904, 0.874488, 0.919278],
    ["VT", 0.45, 6969, 168, 1333, 0.976461, 0.839436, 0.902779]
], columns=[
    "Configuration", "Frozen confidence",
    "TP", "FP", "FN",
    "Precision", "Recall", "F1"
])

tables = {
    "Table_I_Experimental_Settings": table1,
    "Table_II_Independent_AP50": table2,
    "Table_III_Test_Performance": table3
}

for name, df in tables.items():
    df.to_csv(out / f"{name}.csv", index=False)
    (out / f"{name}.tex").write_text(
        df.to_latex(index=False, float_format="%.4f"),
        encoding="utf-8"
    )
    print(f"\n{name}")
    print(df.to_string(index=False))

print("\nTABLE EXPORT COMPLETE")
print("Saved to:", out)

In [ ]:
from pathlib import Path

fig_dir = Path("/content/EXP1_PAPER_FIGURES")
table_dir = Path("/content/EXP1_PAPER_TABLES")

expected_figures = [
    "Figure_1_Training_Loss",
    "Figure_2_Validation_Performance",
    "Figure_3_Test_AP50",
    "Figure_4_Test_Precision_Recall_F1",
    "Figure_5_Validation_Threshold_F1",
    "Figure_6_Qualitative_Comparison"
]

expected_tables = [
    "Table_I_Experimental_Settings",
    "Table_II_Independent_AP50",
    "Table_III_Test_Performance"
]

missing = []

print("========== FIGURE VERIFICATION ==========")

for name in expected_figures:
    for ext in [".png", ".pdf"]:
        path = fig_dir / f"{name}{ext}"

        if path.exists():
            print(f"[OK] {path.name} — {path.stat().st_size / 1024:.1f} KB")
        else:
            print(f"[MISSING] {path.name}")
            missing.append(str(path))

print("\n========== TABLE VERIFICATION ==========")

for name in expected_tables:
    for ext in [".csv", ".tex"]:
        path = table_dir / f"{name}{ext}"

        if path.exists():
            print(f"[OK] {path.name} — {path.stat().st_size / 1024:.1f} KB")
        else:
            print(f"[MISSING] {path.name}")
            missing.append(str(path))

print("\n========== FINAL STATUS ==========")

if missing:
    print("VERIFICATION FAILED")
    print("Missing files:", missing)
else:
    print("ALL 12 FIGURE FILES AND 6 TABLE FILES VERIFIED.")


In [ ]:
from pathlib import Path
import zipfile

fig_dir = Path("/content/EXP1_PAPER_FIGURES")
table_dir = Path("/content/EXP1_PAPER_TABLES")

zip_path = Path(
    "/content/Experiment1_Paper_Evidence.zip"
)

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as zf:

    # Add all figures
    for file in sorted(fig_dir.iterdir()):
        if file.is_file():
            zf.write(
                file,
                f"Figures/{file.name}"
            )

    # Add all tables
    for file in sorted(table_dir.iterdir()):
        if file.is_file():
            zf.write(
                file,
                f"Tables/{file.name}"
            )

    # Add a README
    readme = """
EXPERIMENT 1 — LLVIP PAPER EVIDENCE

Contents:
- Figures: Six paper figures, each in PNG and PDF format.
- Tables: Three paper tables, each in CSV and LaTeX format.

Evaluation notes:
- AP50 results are independently reproduced from saved test predictions.
- Precision, recall, and F1 use confidence thresholds selected on validation
  and frozen before held-out test evaluation.
- Figure 6 shows qualitative predictions on selected validation scenes.
- All results concern LLVIP person detection.

This archive contains paper presentation assets.
It does not replace the original datasets, model weights, or raw experiment
backups.
"""
    zf.writestr("README.txt", readme.strip())

# Verify archive
with zipfile.ZipFile(zip_path, "r") as zf:
    files = zf.namelist()
    bad_file = zf.testzip()

print("ARCHIVE CREATED")
print("Path:", zip_path)
print("Size:", f"{zip_path.stat().st_size / (1024**2):.2f} MB")
print("Files inside:", len(files))
print("ZIP integrity:", "PASS" if bad_file is None else f"FAILED: {bad_file}")

for name in files:
    print(name)

In [ ]:
from huggingface_hub import HfApi
from getpass import getpass
from pathlib import Path

repo_id = "SakibM19/border-surveillance-research-backups"
repo_type = "dataset"

local_file = Path("/content/Experiment1_Paper_Evidence.zip")
remote_path = "Experiment1/Paper_Evidence/Experiment1_Paper_Evidence.zip"

if not local_file.exists():
    raise FileNotFoundError(local_file)

token = getpass("Enter your Hugging Face WRITE token: ")

api = HfApi(token=token)

print("Uploading paper evidence archive...")

api.upload_file(
    path_or_fileobj=str(local_file),
    path_in_repo=remote_path,
    repo_id=repo_id,
    repo_type=repo_type
)

print("\nUpload command completed.")
print("Checking remote repository...")

remote_files = api.list_repo_files(
    repo_id=repo_id,
    repo_type=repo_type
)

if remote_path in remote_files:
    print("\nSUCCESS: Archive verified in Hugging Face.")
    print("Remote path:", remote_path)
else:
    raise RuntimeError("Archive was not found in the remote repository.")

info = api.repo_info(
    repo_id=repo_id,
    repo_type=repo_type
)

print("Repository private:", info.private)
print("Remote file verified:", remote_path)

In [ ]:
from huggingface_hub import HfApi
from getpass import getpass
from pathlib import Path

repo_id = "SakibM19/border-surveillance-research-backups"
fig_dir = Path("/content/EXP1_PAPER_FIGURES")
remote_dir = "Experiment1/Paper_Evidence/Individual_Figures"

files = sorted(
    p for p in fig_dir.iterdir()
    if p.is_file() and p.suffix.lower() in [".png", ".pdf"]
)

if len(files) != 12:
    raise RuntimeError(f"Expected 12 figure files, found {len(files)}")

api = HfApi(token=getpass("Enter your Hugging Face WRITE token: "))

for file in files:
    remote_path = f"{remote_dir}/{file.name}"
    print(f"Uploading {file.name}...")
    api.upload_file(
        path_or_fileobj=str(file),
        path_in_repo=remote_path,
        repo_id=repo_id,
        repo_type="dataset"
    )

remote_files = api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

verified = [
    f"{remote_dir}/{file.name}"
    for file in files
    if f"{remote_dir}/{file.name}" in remote_files
]

print("\n========== FINAL VERIFICATION ==========")
print(f"Uploaded: {len(files)}")
print(f"Verified remotely: {len(verified)}")

for path in verified:
    print("[OK]", path)

if len(verified) != 12:
    raise RuntimeError("Some figure files could not be verified.")

print("\nSUCCESS: All 12 individual figure files are backed up.")

In [ ]:
from IPython.display import display, Image
from pathlib import Path

figure = Path(
    "/content/EXP1_PAPER_FIGURES/Figure_6_Qualitative_Comparison.png"
)

if not figure.exists():
    raise FileNotFoundError(figure)

display(Image(filename=str(figure), width=1100))

In [ ]:
from IPython.display import display, Image
from pathlib import Path

folder = Path("/content/EXP1_PAPER_FIGURES")

figures = [
    "Figure_1_Training_Loss.png",
    "Figure_2_Validation_Performance.png",
    "Figure_3_Test_AP50.png",
    "Figure_4_Test_Precision_Recall_F1.png",
    "Figure_5_Validation_Threshold_F1.png"
]

for name in figures:
    path = folder / name
    print("\n", name)
    display(Image(filename=str(path), width=1000))

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

# Folder containing all exported figures
figure_dir = Path("/content/EXP1_PAPER_FIGURES")

# ZIP file to download
zip_path = Path("/content/Experiment1_All_Figures.zip")

# Collect all PNG and PDF figures
figure_files = sorted(
    list(figure_dir.glob("*.png")) +
    list(figure_dir.glob("*.pdf"))
)

if len(figure_files) != 12:
    raise RuntimeError(
        f"Expected 12 files, found {len(figure_files)}"
    )

# Create ZIP while preserving original filenames
with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:

    for file in figure_files:
        archive.write(file, arcname=file.name)

# Verify ZIP
with zipfile.ZipFile(zip_path, "r") as archive:
    assert archive.testzip() is None

print("FIGURES READY FOR DOWNLOAD")
print("Total files:", len(figure_files))

for file in figure_files:
    print("✓", file.name)

print("\nDownloading ZIP to your laptop...")

files.download(str(zip_path))

In [ ]:

# STEP 1: Verify Experiment 2 files on Hugging Face

!pip -q install -U huggingface_hub

from huggingface_hub import HfApi
from getpass import getpass

REPO_ID = "SakibM19/border-surveillance-research-backups"

print("Enter your Hugging Face READ or WRITE token.")
token = getpass("Hugging Face token: ")

api = HfApi(token=token)

# Check repository access
try:
    info = api.repo_info(repo_id=REPO_ID, repo_type="model")
    print("\nRepository access: SUCCESS")
    print("Repository:", info.id)
    print("Private:", info.private)

except Exception as e:
    raise RuntimeError(
        "Could not access the repository. Check your token and repository permissions."
    ) from e

# List files without downloading them
files = list(
    api.list_repo_tree(
        repo_id=REPO_ID,
        repo_type="model",
        path_in_repo="Experiment2",
        recursive=True
    )
)

file_entries = [
    f for f in files
    if hasattr(f, "size") and f.size is not None
]

total_bytes = sum(f.size for f in file_entries)

print("\n" + "=" * 55)
print("EXPERIMENT 2 BACKUP VERIFICATION")
print("=" * 55)
print("Files found:", len(file_entries))
print(f"Total size: {total_bytes / (1024**3):.2f} GB")
print("\nUploaded files:")

for f in file_entries:
    print(f"{f.size / (1024**2):8.2f} MB | {f.path}")

print("\n" + "=" * 55)

if file_entries:
    print("RESULT: Experiment 2 files are accessible on Hugging Face.")
else:
    print("RESULT: No files found under Experiment2.")


# Source notebook: `1_Weapon_Detection.ipynb`

Original notebook cells: 16. Non-empty source-code cells included: 16. The original code-cell order is preserved below.

In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/weapon_detection', exist_ok=True)
%cd /content/drive/MyDrive/weapon_detection

In [ ]:
!pip install ultralytics roboflow


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="[REDACTED]")
project = rf.workspace("maheshchhetri").project("weapon-detection-e6otc")
version = project.version(4)
dataset = version.download("yolov8")


In [ ]:
import os, yaml

root = "/content/drive/MyDrive/weapon_detection/weapon-detection-4"  # adjust if your path differs
print(os.listdir(root))

with open(f"{root}/data.yaml") as f:
    cfg = yaml.safe_load(f)
print("Classes:", cfg["names"])

for split in ["train", "valid", "test"]:
    p = f"{root}/{split}/images"
    if os.path.exists(p):
        print(split, len(os.listdir(p)), "images")

In [ ]:
import os, random, shutil, yaml
from collections import Counter

root = "/content/drive/MyDrive/weapon_detection/weapon-detection-4"  # adjust if needed
img_dir = f"{root}/train/images"
lbl_dir = f"{root}/train/labels"

random.seed(42)
files = sorted(os.listdir(img_dir))
random.shuffle(files)

n = len(files)
n_val, n_test = int(n * 0.15), int(n * 0.10)
splits = {
    "valid": files[:n_val],
    "test": files[n_val:n_val + n_test],
}

for split, names in splits.items():
    os.makedirs(f"{root}/{split}/images", exist_ok=True)
    os.makedirs(f"{root}/{split}/labels", exist_ok=True)
    for f in names:
        shutil.move(f"{img_dir}/{f}", f"{root}/{split}/images/{f}")
        lbl = os.path.splitext(f)[0] + ".txt"
        if os.path.exists(f"{lbl_dir}/{lbl}"):
            shutil.move(f"{lbl_dir}/{lbl}", f"{root}/{split}/labels/{lbl}")

# New yaml with absolute paths
cfg = {
    "path": root,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {0: "Handgun", 1: "Knife", 2: "Rifle", 3: "Sword"},
}
with open(f"{root}/my_data.yaml", "w") as f:
    yaml.dump(cfg, f)

# Check class balance and images with no weapons
for split in ["train", "valid", "test"]:
    counts, empty = Counter(), 0
    for lf in os.listdir(f"{root}/{split}/labels"):
        lines = open(f"{root}/{split}/labels/{lf}").read().strip().splitlines()
        if not lines: empty += 1
        for line in lines:
            counts[int(line.split()[0])] += 1
    print(split, len(os.listdir(f"{root}/{split}/images")), "images | boxes per class:", dict(counts), "| empty:", empty)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
model.train(
    data=f"{root}/my_data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="/content/drive/MyDrive/weapon_detection/runs",
    name="weapons_v1",
)

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass("Paste Roboflow API key and press Enter: "))
project = rf.workspace("drone-gpmet").project("drone-xyhff")
dataset = project.version(1).download("yolov8")
print("Saved at:", dataset.location)

In [ ]:
!cp -r /content/Drone-1 /content/drive/MyDrive/weapon_detection/drone-dataset

In [ ]:
import os, yaml
drone_root = "/content/drive/MyDrive/weapon_detection/drone-dataset"

cfg = yaml.safe_load(open(f"{drone_root}/data.yaml"))
print("Classes:", cfg["names"])
for s in ["train", "valid", "test"]:
    p = f"{drone_root}/{s}/images"
    print(s, len(os.listdir(p)) if os.path.exists(p) else "missing")

In [ ]:
import os, shutil, random, yaml
from collections import Counter

weapon_root = "/content/drive/MyDrive/weapon_detection/weapon-detection-4"
drone_root  = "/content/drive/MyDrive/weapon_detection/drone-dataset"
out_root    = "/content/drive/MyDrive/weapon_detection/combined"

DRONE_CLASS_IDS = {0}      # 'drone' is ID 0 in your drone dataset
MAX_DRONE_IMAGES = 2000
random.seed(42)

if os.path.exists(out_root):
    shutil.rmtree(out_root)    # start clean so reruns don't duplicate files

# 1) copy the weapon dataset
for s in ["train", "valid", "test"]:
    shutil.copytree(f"{weapon_root}/{s}", f"{out_root}/{s}")

# 2) collect drone images, shuffle, split 80/10/10
pairs = []
for s in ["train", "valid", "test"]:
    idir, ldir = f"{drone_root}/{s}/images", f"{drone_root}/{s}/labels"
    if not os.path.exists(idir):
        continue
    for f in os.listdir(idir):
        pairs.append((f"{idir}/{f}", f"{ldir}/{os.path.splitext(f)[0]}.txt"))
random.shuffle(pairs)
pairs = pairs[:MAX_DRONE_IMAGES]

n = len(pairs)
bounds = {"train": (0, int(n*0.8)), "valid": (int(n*0.8), int(n*0.9)), "test": (int(n*0.9), n)}

for split, (a, b) in bounds.items():
    os.makedirs(f"{out_root}/{split}/images", exist_ok=True)
    os.makedirs(f"{out_root}/{split}/labels", exist_ok=True)
    for img, lbl in pairs[a:b]:
        base = os.path.splitext(os.path.basename(img))[0]
        shutil.copy(img, f"{out_root}/{split}/images/drone_{os.path.basename(img)}")
        new_lines = []
        if os.path.exists(lbl):
            for line in open(lbl).read().strip().splitlines():
                parts = line.split()
                if parts and int(parts[0]) in DRONE_CLASS_IDS:
                    new_lines.append("4 " + " ".join(parts[1:]))   # drone becomes class 4
        open(f"{out_root}/{split}/labels/drone_{base}.txt", "w").write("\n".join(new_lines))

# 3) yaml file
cfg = {"path": out_root, "train": "train/images", "val": "valid/images", "test": "test/images",
       "names": {0: "Handgun", 1: "Knife", 2: "Rifle", 3: "Sword", 4: "Drone"}}
yaml.dump(cfg, open(f"{out_root}/combined.yaml", "w"))

# 4) check
for split in ["train", "valid", "test"]:
    c = Counter()
    for lf in os.listdir(f"{out_root}/{split}/labels"):
        for line in open(f"{out_root}/{split}/labels/{lf}").read().strip().splitlines():
            if line: c[int(line.split()[0])] += 1
    print(split, len(os.listdir(f"{out_root}/{split}/images")), "images | boxes per class:", dict(sorted(c.items())))

In [ ]:
!pip install ultralytics

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/weapon_detection/runs/weapons_v1/weights/best.pt")
model.train(
    data="/content/drive/MyDrive/weapon_detection/combined/combined.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    project="/content/drive/MyDrive/weapon_detection/runs",
    name="weapons_drone_v1",
)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/weapon_detection/runs/weapons_drone_v1/weights/best.pt")
metrics = model.val(
    data="/content/drive/MyDrive/weapon_detection/combined/combined.yaml",
    split="test",
)

# Source notebook: `framework_exp_2.ipynb`

Original notebook cells: 36. Non-empty source-code cells included: 35. The original code-cell order is preserved below.

In [ ]:

# EXPERIMENT 2 — CELL 1
# Fresh environment and GPU verification

import os
import sys
import platform
import torch

print("=" * 55)
print("EXPERIMENT 2 — ANTI-UAV300")
print("FRESH COLAB ENVIRONMENT")
print("=" * 55)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("WARNING: No GPU detected. Do not start training yet.")

print("\nWorking directory:", os.getcwd())
print("Experiment 2 runtime check completed.")


In [ ]:

# EXPERIMENT 2 — REPOSITORY TYPE DIAGNOSTIC

from huggingface_hub import HfApi
from getpass import getpass

REPO_ID = "SakibM19/border-surveillance-research-backups"

token = getpass("Enter your NEW Hugging Face token: ").strip()

api = HfApi(token=token)

working_type = None

for repo_type in ["model", "dataset"]:
    print(f"\nChecking repository as {repo_type}...")

    try:
        info = api.repo_info(
            repo_id=REPO_ID,
            repo_type=repo_type
        )

        print("SUCCESS!")
        print("Repository:", info.id)
        print("Private:", info.private)
        print("Repository type:", repo_type)

        working_type = repo_type
        break

    except Exception as e:
        print("Not accessible as", repo_type)
        print("Error:", type(e).__name__)

if working_type:
    print("\nCorrect repository type:", working_type)

    entries = list(
        api.list_repo_tree(
            repo_id=REPO_ID,
            repo_type=working_type,
            path_in_repo="Experiment2",
            recursive=True
        )
    )

    file_entries = [
        x for x in entries
        if hasattr(x, "size") and x.size is not None
    ]

    total = sum(x.size for x in file_entries)

    print("\nEXPERIMENT 2 BACKUP")
    print("Files:", len(file_entries))
    print(f"Total size: {total / (1024**3):.3f} GB")

    for x in file_entries:
        print(f"{x.size / (1024**2):8.2f} MB | {x.path}")

else:
    print("\nNeither repository type worked.")
    print("We need to verify the repository URL and token permissions.")


In [ ]:
repo_type="model"

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import os

REPO_ID = "SakibM19/border-surveillance-research-backups"
LOCAL_DIR = Path("/content/EXP2_BACKUP_PARTS")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

files = [
    f"Experiment2/experiment2_part_0{i}.tar.part"
    for i in range(1, 6)
]

for filename in files:
    print(f"\nDownloading: {filename}")

    downloaded = hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset",
        token=token.strip(),
        local_dir=str(LOCAL_DIR)
    )

    size_mb = os.path.getsize(downloaded) / (1024 * 1024)
    print(f"Verified local file: {downloaded}")
    print(f"Size: {size_mb:.2f} MB")

print("\nAll five backup parts downloaded.")
print("Total size:",
      round(sum(p.stat().st_size for p in LOCAL_DIR.rglob("*.part")) /
            (1024**3), 3), "GB")

In [ ]:
%pip uninstall -y huggingface_hub
%pip install --no-cache-dir --force-reinstall "huggingface_hub==1.2.3"

In [ ]:
import huggingface_hub

print("Hugging Face Hub:", huggingface_hub.__version__)

from huggingface_hub import HfApi, hf_hub_download

print("HfApi: OK")
print("hf_hub_download: OK")

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import os

REPO_ID = "SakibM19/border-surveillance-research-backups"
LOCAL_DIR = Path("/content/EXP2_BACKUP_PARTS")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

files = [
    f"Experiment2/experiment2_part_0{i}.tar.part"
    for i in range(1, 6)
]

total_bytes = 0

for filename in files:
    print(f"\nDownloading: {Path(filename).name}")

    downloaded = hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset",
        token=token.strip(),
        local_dir=str(LOCAL_DIR)
    )

    size = os.path.getsize(downloaded)
    total_bytes += size

    print(f"Downloaded: {size / (1024**2):.2f} MB")
    print(f"Location: {downloaded}")

print("\n" + "=" * 55)
print("DOWNLOAD COMPLETE")
print(f"Total downloaded: {total_bytes / (1024**3):.3f} GB")
print(f"Parts found: {len(list(LOCAL_DIR.rglob('*.part')))} / 5")
print("=" * 55)

In [ ]:
from getpass import getpass

token = getpass("Enter your NEW Hugging Face READ token: ").strip()

if not token.startswith("hf_"):
    raise ValueError("Invalid token format. Please check your token.")

print("Token received successfully. It will not be displayed.")

In [ ]:
from pathlib import Path
import shutil
import tarfile
import hashlib

BACKUP_DIR = Path("/content/EXP2_BACKUP_PARTS/Experiment2")
PARTS = [
    BACKUP_DIR / f"experiment2_part_0{i}.tar.part"
    for i in range(1, 6)
]

OUTPUT_TAR = Path("/content/Experiment2_Restored.tar")

print("=" * 60)
print("STEP 1: VERIFY BACKUP PARTS")
print("=" * 60)

for part in PARTS:
    if not part.exists() or part.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty backup part: {part}")

    print(f"{part.name}: {part.stat().st_size / (1024**2):.2f} MB")

total_size = sum(p.stat().st_size for p in PARTS)
free_space = shutil.disk_usage("/content").free

print(f"\nTotal parts: {len(PARTS)}")
print(f"Total size: {total_size / (1024**3):.3f} GB")
print(f"Available storage: {free_space / (1024**3):.2f} GB")

if free_space < total_size + 500 * 1024**2:
    raise RuntimeError("Insufficient free storage to reconstruct the archive.")

print("\n" + "=" * 60)
print("STEP 2: RECONSTRUCT TAR ARCHIVE")
print("=" * 60)

sha256 = hashlib.sha256()

with open(OUTPUT_TAR, "wb") as output:
    for part in PARTS:
        print(f"Joining {part.name}...")
        with open(part, "rb") as source:
            while True:
                chunk = source.read(16 * 1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
                sha256.update(chunk)

restored_size = OUTPUT_TAR.stat().st_size

print("\nReconstructed:", OUTPUT_TAR)
print(f"Archive size: {restored_size / (1024**3):.3f} GB")
print("SHA-256:", sha256.hexdigest())

if restored_size != total_size:
    raise RuntimeError("Size mismatch after reconstruction!")

print("\n" + "=" * 60)
print("STEP 3: VERIFY TAR INTEGRITY")
print("=" * 60)

with tarfile.open(OUTPUT_TAR, "r:*") as archive:
    members = archive.getmembers()
    print("TAR integrity: PASS")
    print("Total archive entries:", len(members))
    print("\nFirst 15 entries:")
    for member in members[:15]:
        print(member.name)

print("\nRECONSTRUCTION AND INTEGRITY CHECK COMPLETE")

In [ ]:
import tarfile
from collections import Counter

ARCHIVE = "/content/Experiment2_Restored.tar"

with tarfile.open(ARCHIVE, "r:") as tar:
    members = tar.getmembers()

    print("TOTAL ENTRIES:", len(members))

    counts = Counter()
    for m in members:
        if m.isfile():
            counts["files"] += 1
        elif m.isdir():
            counts["directories"] += 1

    print("FILES:", counts["files"])
    print("DIRECTORIES:", counts["directories"])

    print("\nTOP-LEVEL CONTENTS:")
    roots = sorted({
        m.name.split("/")[1]
        for m in members
        if m.name.count("/") >= 1
    })

    for name in roots:
        print(" -", name)

    print("\nIMPORTANT FILES:")
    for m in members:
        name = m.name.lower()
        if (
            name.endswith((".yaml", ".yml", ".json", ".py", ".txt"))
            and not m.isdir()
        ):
            print(f"{m.name} ({m.size / 1024:.1f} KB)")

In [ ]:
import tarfile
from collections import Counter
from pathlib import PurePosixPath

ARCHIVE = "/content/Experiment2_Restored.tar"

with tarfile.open(ARCHIVE, "r:") as tar:
    members = tar.getmembers()

    files = [m for m in members if m.isfile()]
    dirs = [m for m in members if m.isdir()]

    print("=" * 55)
    print("EXPERIMENT 2 ARCHIVE SUMMARY")
    print("=" * 55)

    print("Total entries:", len(members))
    print("Files:", len(files))
    print("Directories:", len(dirs))
    print("Archive size (GB):",
          round(sum(m.size for m in files) / (1024**3), 3))

    print("\nTOP-LEVEL FOLDERS:")
    top = Counter(
        PurePosixPath(m.name).parts[1]
        for m in members
        if len(PurePosixPath(m.name).parts) > 1
    )
    for folder, count in sorted(top.items()):
        print(f"  {folder}: {count} entries")

    print("\nFILE TYPES:")
    extensions = Counter(
        PurePosixPath(m.name).suffix.lower() or "[no extension]"
        for m in files
    )
    for ext, count in extensions.most_common(15):
        print(f"  {ext}: {count}")

    print("\nIMPORTANT CONFIGURATION / SCRIPT FILES:")
    important = [
        m.name for m in files
        if (
            PurePosixPath(m.name).suffix.lower()
            in [".yaml", ".yml", ".json", ".py"]
            and any(k in m.name.lower() for k in
                    ["config", "data.yaml", "manifest", "split",
                     "train", "eval", "experiment", "README"])
        )
    ]

    for name in important[:30]:
        print(" ", name)

    if len(important) > 30:
        print(f"  ... and {len(important)-30} more")

print("\nINSPECTION COMPLETE")

In [ ]:
import tarfile
from collections import Counter
from pathlib import PurePosixPath

ARCHIVE = "/content/Experiment2_Restored.tar"

with tarfile.open(ARCHIVE, "r:") as tar:
    members = tar.getmembers()

    print("=== EXPERIMENT RECORDS ===")
    for m in members:
        parts = PurePosixPath(m.name).parts
        if len(parts) >= 2 and parts[1] in [
            "backups", "logs", "manifests", "results", "runs"
        ] and m.isfile():
            print(f"{m.name} ({m.size / 1024:.1f} KB)")

    print("\n=== DATASET IMAGE COUNTS ===")
    counts = Counter()

    for m in members:
        if not m.isfile() or not m.name.lower().endswith(".jpg"):
            continue

        parts = PurePosixPath(m.name).parts

        if len(parts) >= 5 and parts[1] == "datasets":
            modality = parts[2]
            split = parts[4]
            counts[(modality, split)] += 1

    for (modality, split), count in sorted(counts.items()):
        print(f"{modality:12s} | {split:6s} | {count:,}")

    print("\n=== ARCHIVE PATH SAFETY ===")
    unsafe = [
        m.name for m in members
        if PurePosixPath(m.name).is_absolute()
        or ".." in PurePosixPath(m.name).parts
    ]
    print("Unsafe paths:", len(unsafe))
    print("Archive inspection complete.")

In [ ]:
import tarfile
from pathlib import Path
import shutil

ARCHIVE = Path("/content/Experiment2_Restored.tar")
DEST = Path("/content/EXP2_RESTORED")

required_space = ARCHIVE.stat().st_size
free_space = shutil.disk_usage("/content").free

print("Archive size:", round(required_space / (1024**3), 3), "GB")
print("Free storage:", round(free_space / (1024**3), 2), "GB")

if free_space < required_space + 500 * 1024**2:
    raise RuntimeError("Insufficient storage for extraction.")

DEST.mkdir(parents=True, exist_ok=True)

with tarfile.open(ARCHIVE, "r:") as tar:
    tar.extractall(path=DEST, filter="data")

ROOT = DEST / "AntiUAV_Experiment2"

if not ROOT.exists():
    raise RuntimeError("Expected experiment root folder not found.")

print("\nEXTRACTION COMPLETE")
print("Experiment folder:", ROOT)
print("Folder exists:", ROOT.is_dir())

for name in ["datasets", "configs", "manifests",
             "backups", "logs", "results", "runs"]:
    path = ROOT / name
    print(f"{name:12s} | exists={path.exists()} | "
          f"files={sum(1 for p in path.rglob('*') if p.is_file()) if path.exists() else 0}")

In [ ]:
from pathlib import Path
from collections import Counter

ROOT = Path("/content/EXP2_RESTORED/AntiUAV_Experiment2")
DATASETS = ROOT / "datasets"

summary = {}

for modality in ["visible", "infrared"]:
    print(f"\n{'='*50}")
    print(f"VERIFYING {modality.upper()}")
    print("="*50)

    summary[modality] = {}

    for split in ["train", "val", "test"]:
        image_dir = DATASETS / modality / "images" / split
        label_dir = DATASETS / modality / "labels" / split

        images = sorted(image_dir.glob("*.jpg"))
        labels = sorted(label_dir.glob("*.txt"))

        image_stems = {p.stem for p in images}
        label_stems = {p.stem for p in labels}

        missing_labels = image_stems - label_stems
        orphan_labels = label_stems - image_stems

        empty_labels = 0
        malformed_labels = 0
        valid_boxes = 0

        for label in labels:
            content = label.read_text().strip()

            if not content:
                empty_labels += 1
                continue

            for line in content.splitlines():
                values = line.split()

                try:
                    if len(values) != 5:
                        raise ValueError()

                    cls, x, y, w, h = map(float, values)

                    if (
                        cls < 0 or not cls.is_integer()
                        or not all(0 <= v <= 1 for v in [x, y, w, h])
                        or w <= 0 or h <= 0
                    ):
                        raise ValueError()

                    valid_boxes += 1

                except Exception:
                    malformed_labels += 1

        summary[modality][split] = {
            "images": len(images),
            "labels": len(labels),
            "missing": len(missing_labels),
            "orphan": len(orphan_labels),
            "empty": empty_labels,
            "malformed": malformed_labels,
            "boxes": valid_boxes
        }

        print(
            f"{split:5s} | Images: {len(images):6,} | "
            f"Labels: {len(labels):6,} | "
            f"Missing: {len(missing_labels):4,} | "
            f"Orphan: {len(orphan_labels):4,} | "
            f"Empty: {empty_labels:4,} | "
            f"Malformed: {malformed_labels:4,} | "
            f"Boxes: {valid_boxes:,}"
        )

print("\nPAIRING AND ANNOTATION CHECK COMPLETE")

In [ ]:
from pathlib import Path

ROOT = Path("/content/EXP2_RESTORED/AntiUAV_Experiment2")

for filename in ["visible_data.yaml", "infrared_data.yaml"]:
    path = ROOT / "configs" / filename

    print("\n" + "=" * 55)
    print(filename)
    print("=" * 55)

    print(path.read_text())

In [ ]:
from pathlib import Path
import yaml

ROOT = Path("/content/EXP2_RESTORED/AntiUAV_Experiment2")
CONFIG_DIR = ROOT / "configs"

for modality in ["visible", "infrared"]:
    original = CONFIG_DIR / f"{modality}_data.yaml"
    corrected = CONFIG_DIR / f"{modality}_data_restored.yaml"

    config = yaml.safe_load(original.read_text())

    config["path"] = str(ROOT / "datasets" / modality)

    corrected.write_text(
        yaml.safe_dump(config, sort_keys=False)
    )

    print(f"\n{modality.upper()} CONFIGURATION")
    print(corrected.read_text())

    for split in ["train", "val", "test"]:
        image_dir = ROOT / "datasets" / modality / "images" / split
        assert image_dir.exists(), f"Missing: {image_dir}"
        print(f"{split}: {len(list(image_dir.glob('*.jpg'))):,} images")

print("\nBoth corrected configurations created and paths verified.")

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import numpy as np

ROOT = Path("/content/EXP2_RESTORED/AntiUAV_Experiment2")
DATASETS = ROOT / "datasets"

print("=" * 60)
print("IMAGE DIMENSION AUDIT")
print("=" * 60)

for modality in ["visible", "infrared"]:
    image_dir = DATASETS / modality / "images" / "train"
    dimensions = Counter()

    for path in sorted(image_dir.glob("*.jpg"))[:500]:
        with Image.open(path) as img:
            dimensions[img.size] += 1

    print(f"\n{modality.upper()} — first 500 training images")
    for size, count in dimensions.most_common():
        print(f"  {size[0]} x {size[1]}: {count}")

print("\n" + "=" * 60)
print("ANNOTATION DISTRIBUTION AUDIT")
print("=" * 60)

for modality in ["visible", "infrared"]:
    label_dir = DATASETS / modality / "labels" / "train"

    widths, heights, areas = [], [], []
    class_counts = Counter()

    for path in label_dir.glob("*.txt"):
        for line in path.read_text().splitlines():
            values = line.split()
            if len(values) != 5:
                continue

            cls, x, y, w, h = map(float, values)
            class_counts[int(cls)] += 1
            widths.append(w)
            heights.append(h)
            areas.append(w * h)

    print(f"\n{modality.upper()}")
    print("Class counts:", dict(class_counts))

    if areas:
        print("Bounding boxes:", len(areas))
        print("Mean normalized width:", round(float(np.mean(widths)), 5))
        print("Mean normalized height:", round(float(np.mean(heights)), 5))
        print("Mean normalized area:", round(float(np.mean(areas)), 6))
        print("Median normalized area:", round(float(np.median(areas)), 6))
        print("Small boxes (area < 0.01):", sum(a < 0.01 for a in areas))
        print("Medium boxes (0.01–0.10):", sum(0.01 <= a < 0.10 for a in areas))
        print("Large boxes (>= 0.10):", sum(a >= 0.10 for a in areas))

print("\nAUDIT COMPLETE")

In [ ]:
import torch
import ultralytics
import yaml
from pathlib import Path

ROOT = Path("/content/EXP2_RESTORED/AntiUAV_Experiment2")
CONFIGS = ROOT / "configs"

print("=" * 60)
print("EXPERIMENT 2 — TRAINING ENVIRONMENT CHECK")
print("=" * 60)

print("\n1. SOFTWARE")
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)

print("\n2. GPU")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

print("\n3. DATASET CONFIGURATIONS")

for modality in ["visible", "infrared"]:
    config_path = CONFIGS / f"{modality}_data_restored.yaml"

    print(f"\n{modality.upper()}")
    print("Config exists:", config_path.exists())

    if config_path.exists():
        with open(config_path, "r") as f:
            config = yaml.safe_load(f)

        print("Dataset path:", config["path"])
        print("Train:", config["train"])
        print("Validation:", config["val"])
        print("Test:", config["test"])
        print("Number of classes:", config["nc"])
        print("Class names:", config["names"])

        dataset_path = Path(config["path"])

        for split in ["train", "val", "test"]:
            image_dir = dataset_path / "images" / split
            print(
                f"{split} images:",
                len(list(image_dir.glob("*.jpg")))
            )

print("\nENVIRONMENT CHECK COMPLETE")

In [ ]:
!pip install -q ultralytics

In [ ]:
import torch
import ultralytics
import yaml

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Ultralytics:", ultralytics.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import yaml

ROOT = Path("/content/EXP2_RESTORED/AntiUAV_Experiment2")
CONFIGS = ROOT / "configs"

for name in ["visible_data_restored.yaml", "infrared_data_restored.yaml"]:
    path = CONFIGS / name

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    with open(path, "r") as f:
        config = yaml.safe_load(f)

    print("Dataset path:", config["path"])
    print("Train:", config["train"])
    print("Validation:", config["val"])
    print("Test:", config["test"])
    print("Classes:", config["names"])

    for split in ["train", "val", "test"]:
        image_dir = Path(config["path"]) / config[split]
        count = len(list(image_dir.glob("*.jpg")))
        print(f"{split.capitalize()} images:", count)

print("\nCONFIGURATION AUDIT COMPLETE")

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

print("=" * 55)
print("EXPERIMENT 2 TRAINING ENVIRONMENT")
print("=" * 55)

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

print("\nYOLO11n model initialization test")

model = YOLO("yolo11n.pt")

print("Model initialized successfully.")
print("Model task:", model.task)
print("Model type:", model.__class__.__name__)

In [ ]:
from pathlib import Path

ROOT = Path("/content/EXP2_RESTORED/AntiUAV_Experiment2")

print("=" * 65)
print("EXPERIMENT 2 — ORIGINAL SCRIPT AND CONFIGURATION AUDIT")
print("=" * 65)

print("\nPYTHON SCRIPTS:")
scripts = sorted(ROOT.rglob("*.py"))

if scripts:
    for p in scripts:
        print(f"  {p.relative_to(ROOT)}")
else:
    print("  No Python scripts found.")

print("\nCONFIGURATION AND MANIFEST FILES:")
for pattern in ["*.yaml", "*.yml", "*.json", "*.csv", "*.md", "*.txt"]:
    files = sorted(ROOT.rglob(pattern))
    for p in files:
        if "datasets/" not in str(p.relative_to(ROOT)):
            print(f"  {p.relative_to(ROOT)}")

print("\nBACKUP / LOG / RESULT DIRECTORIES:")
for name in ["backups", "logs", "results", "runs"]:
    p = ROOT / name
    if p.exists():
        count = sum(1 for x in p.rglob("*") if x.is_file())
        print(f"  {name}: {count} files")

print("\nAUDIT COMPLETE")

In [ ]:
import shutil

print("COLAB STORAGE AUDIT")
print("=" * 45)

for path in ["/content", "/"]:
    total, used, free = shutil.disk_usage(path)

    print(f"\nPath: {path}")
    print(f"Total: {total / 1024**3:.2f} GB")
    print(f"Used:  {used / 1024**3:.2f} GB")
    print(f"Free:  {free / 1024**3:.2f} GB")

In [ ]:
from pathlib import Path

BASE = Path("/content/EXPERIMENT2_RESULTS")

folders = [
    "visible/weights",
    "visible/logs",
    "infrared/weights",
    "infrared/logs",
    "evaluation",
    "figures",
    "tables",
    "manifests",
    "scripts"
]

for folder in folders:
    (BASE / folder).mkdir(parents=True, exist_ok=True)

print("Experiment 2 result structure created.")
print("Location:", BASE)

for p in sorted(BASE.rglob("*")):
    if p.is_dir():
        print(" ", p.relative_to(BASE))

In [ ]:
from pathlib import Path
import json
import yaml

ROOT = Path("/content/EXP2_RESTORED/AntiUAV_Experiment2")
OUT = Path("/content/EXPERIMENT2_RESULTS")

config = {
    "experiment": "Experiment 2 - AntiUAV300 Visible",
    "model": "yolo11n.pt",
    "pretrained": True,
    "epochs": 50,
    "imgsz": 640,
    "batch": 16,
    "seed": 0,
    "deterministic": True,
    "device": 0,
    "workers": 8,
    "patience": 15,
    "optimizer": "auto",
    "data": str(ROOT / "configs/visible_data_restored.yaml"),
    "project": str(OUT / "visible"),
    "name": "yolo11n_visible",
    "exist_ok": True,
    "save": True,
    "save_period": 5,
    "plots": True,
    "val": True
}

path = OUT / "visible" / "training_config.json"

with open(path, "w") as f:
    json.dump(config, f, indent=4)

print("Visible training configuration saved:")
print(path)
print(json.dumps(config, indent=4))

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import json
import shutil

OUT = Path("/content/EXPERIMENT2_RESULTS")
CONFIG_PATH = OUT / "visible" / "training_config.json"

with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)

print("=" * 60)
print("EXPERIMENT 2A — VISIBLE MODEL TRAINING")
print("=" * 60)
print("Dataset:", cfg["data"])
print("Epochs:", cfg["epochs"])
print("Image size:", cfg["imgsz"])
print("Batch size:", cfg["batch"])
print("GPU: Tesla T4")
print("=" * 60)

model = YOLO(cfg["model"])

results = model.train(
    data=cfg["data"],
    epochs=cfg["epochs"],
    imgsz=cfg["imgsz"],
    batch=cfg["batch"],
    seed=cfg["seed"],
    deterministic=cfg["deterministic"],
    device=cfg["device"],
    workers=cfg["workers"],
    patience=cfg["patience"],
    optimizer=cfg["optimizer"],
    project=cfg["project"],
    name=cfg["name"],
    exist_ok=cfg["exist_ok"],
    save=cfg["save"],
    save_period=cfg["save_period"],
    plots=cfg["plots"],
    val=cfg["val"]
)

RUN_DIR = OUT / "visible" / "yolo11n_visible"
WEIGHTS_DIR = OUT / "visible" / "weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

for filename in ["best.pt", "last.pt"]:
    source = RUN_DIR / "weights" / filename
    if source.exists():
        shutil.copy2(source, WEIGHTS_DIR / filename)
        print(f"Saved: {WEIGHTS_DIR / filename}")

print("\nVISIBLE TRAINING FINISHED")
print("Run directory:", RUN_DIR)
print("Weights directory:", WEIGHTS_DIR)
print("Training results:", RUN_DIR / "results.csv")

In [ ]:
from pathlib import Path

RUN = Path("/content/EXPERIMENT2_RESULTS/visible/yolo11n_visible")

files = [
    RUN / "weights/best.pt",
    RUN / "weights/last.pt",
    RUN / "results.csv",
    RUN / "results.png"
]

print("VISIBLE MODEL RECOVERY CHECK")
print("=" * 50)

for p in files:
    if p.exists():
        print(f"FOUND: {p.name} | {p.stat().st_size / (1024**2):.2f} MB")
    else:
        print(f"MISSING: {p.name}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")

print("SEARCHING GOOGLE DRIVE")
print("=" * 50)

for name in ["best.pt", "last.pt", "results.csv"]:
    matches = list(MYDRIVE.rglob(name))
    print(f"\n{name}: {len(matches)} found")

    for p in matches[:10]:
        print(" ", p)
        print("  Size:", round(p.stat().st_size / 1024**2, 2), "MB")

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

REPO_ID = "SakibM19/border-surveillance-research-backups"

token = userdata.get("HF_TOKEN")
api = HfApi(token=token)

print("=" * 60)
print("SEARCHING HUGGING FACE FOR EXPERIMENT 2 CHECKPOINTS")
print("=" * 60)

files = list(api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
))

matches = [
    f for f in files
    if f.startswith("Experiment2/")
    and (
        f.endswith("best.pt")
        or f.endswith("last.pt")
        or f.endswith("results.csv")
    )
]

if matches:
    for path in matches:
        info = api.get_paths_info(
            repo_id=REPO_ID,
            paths=[path],
            repo_type="dataset"
        )[0]
        print(f"\nFOUND: {path}")
        print("Size:", round(info.size / (1024**2), 2), "MB")
else:
    print("\nNo Experiment 2 model checkpoints found.")

print("\nSEARCH COMPLETE")

In [ ]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

print("HF_TOKEN successfully loaded:", bool(token))

In [ ]:
from google.colab import userdata

try:
    token = userdata.get("HF_TOKEN")
    print("SUCCESS: HF_TOKEN is accessible.")
    print("Token loaded:", bool(token))
except Exception as e:
    print("ERROR:", type(e).__name__, str(e))

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

REPO_ID = "SakibM19/border-surveillance-research-backups"
token = userdata.get("HF_TOKEN")

api = HfApi(token=token)

print("Checking Hugging Face repository...")
files = list(api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
))

exp2_files = [
    f for f in files
    if f.startswith("Experiment2/")
]

checkpoints = [
    f for f in exp2_files
    if f.endswith((".pt", ".pth"))
]

logs = [
    f for f in exp2_files
    if f.endswith(("results.csv", "results.png"))
]

print("\nExperiment 2 files:", len(exp2_files))
print("Model checkpoints:", len(checkpoints))
print("Training logs:", len(logs))

print("\nCHECKPOINTS:")
for f in checkpoints:
    print(f)

print("\nTRAINING LOGS:")
for f in logs:
    print(f)

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

api = HfApi(token=userdata.get("HF_TOKEN"))
repo_id = "SakibM19/border-surveillance-research-backups"

print("FILES UNDER Experiment2:")
for path in api.list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
):
    if path.startswith("Experiment2/"):
        print("-", path)

# Experiment 5 — RGB–Thermal Fusion Robustness

The following cells are preserved from `experiment5-rgb-thermal-fusion-robustness.ipynb` in their original order, including recorded outputs and execution counts. They are included as source/evidence material; they have not been re-executed or modified.

**Environment note:** The original notebook was prepared for Kaggle. Cells that use `/kaggle/working`, Kaggle Secrets, or Kaggle-specific paths may require corresponding environment setup before execution. Review credential and backup cells before running them.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:

import os
import sys
import platform
from pathlib import Path

print("=" * 75)
print("EXPERIMENT 5 — RGB–THERMAL FUSION ROBUSTNESS")
print("ENVIRONMENT AND GPU VERIFICATION")
print("=" * 75)

print("\nPython version:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

try:
    import torch

    print("\nPyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
        print("GPU memory (GB):",
              round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    else:
        print("WARNING: No GPU detected.")

except Exception as e:
    print("PyTorch verification failed:", repr(e))

print("\nWorkspace directories:")

for path in [Path("/kaggle/working"), Path("/kaggle/input")]:
    print(f"\n{path} | Exists: {path.exists()}")

    if path.exists():
        for item in sorted(path.iterdir()):
            print("DIR " if item.is_dir() else "FILE", item.name)

print("\nENVIRONMENT CHECK COMPLETE")
print("No files modified.")


EXPERIMENT 5 — RGB–THERMAL FUSION ROBUSTNESS
ENVIRONMENT AND GPU VERIFICATION

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
Working directory: /kaggle/working

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0
GPU memory (GB): 15.64

Workspace directories:

/kaggle/working | Exists: True

/kaggle/input | Exists: True

ENVIRONMENT CHECK COMPLETE
No files modified.


In [3]:
# EXPERIMENT 5 — HUGGING FACE BACKUP PREFLIGHT
# Upload -> Remote listing -> Download -> SHA-256 verification

import hashlib
import tempfile
from pathlib import Path
from uuid import uuid4

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = "SakibM19/border-surveillance-research-backups"
REPO_TYPE = "dataset"

print("=" * 75)
print("EXPERIMENT 5 — HUGGING FACE BACKUP PREFLIGHT")
print("=" * 75)

# 1. Retrieve token securely
token = UserSecretsClient().get_secret("HF_TOKEN")

if not token:
    raise RuntimeError("HF_TOKEN is missing from Kaggle Secrets.")

api = HfApi()

# 2. Verify account and repository access
identity = api.whoami(token=token)
print("Authenticated account:", identity.get("name", "Unknown"))

repo_info = api.repo_info(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=token
)

print("Repository:", REPO_ID)
print("Repository type:", REPO_TYPE)
print("Private repository:", repo_info.private)

if not repo_info.private:
    raise RuntimeError("Expected a private repository, but it is not private.")

# 3. Create a unique test file
test_id = uuid4().hex
relative_path = f"experiment5_preflight/preflight_{test_id}.txt"

test_content = (
    "Experiment 5 Hugging Face backup preflight\n"
    f"Test ID: {test_id}\n"
    "Purpose: verify upload, listing, download, and integrity.\n"
).encode("utf-8")

with tempfile.TemporaryDirectory() as temp_dir:
    local_file = Path(temp_dir) / "preflight_test.txt"
    local_file.write_bytes(test_content)

    local_sha256 = hashlib.sha256(local_file.read_bytes()).hexdigest()
    print("\nLocal SHA-256:", local_sha256)

    # 4. Upload test file
    api.upload_file(
        path_or_fileobj=str(local_file),
        path_in_repo=relative_path,
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        token=token,
        commit_message="Experiment 5 backup preflight test"
    )
    print("\nUPLOAD: SUCCESS")

    # 5. Verify remote listing
    remote_files = api.list_repo_files(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        token=token
    )

    if relative_path not in remote_files:
        raise RuntimeError("Uploaded test file was not found in remote listing.")

    print("REMOTE LISTING: SUCCESS")
    print("Verified path:", relative_path)

    # 6. Download the uploaded file
    downloaded_file = hf_hub_download(
        repo_id=REPO_ID,
        filename=relative_path,
        repo_type=REPO_TYPE,
        token=token,
        force_download=True
    )

    downloaded_sha256 = hashlib.sha256(
        Path(downloaded_file).read_bytes()
    ).hexdigest()

    print("DOWNLOAD: SUCCESS")
    print("Downloaded SHA-256:", downloaded_sha256)

    # 7. Verify integrity
    if local_sha256 != downloaded_sha256:
        raise RuntimeError("SHA-256 mismatch: backup integrity verification failed.")

    print("\nSHA-256 INTEGRITY: PASS")

print("\n" + "=" * 75)
print("PREFLIGHT COMPLETE — ALL CHECKS PASSED")
print("No experiment data or model checkpoints were modified.")
print("=" * 75)

EXPERIMENT 5 — HUGGING FACE BACKUP PREFLIGHT
Authenticated account: SakibM19
Repository: SakibM19/border-surveillance-research-backups
Repository type: dataset
Private repository: True

Local SHA-256: e160ff62c9e2e01cc37a68c8234efa54b06aa88c30fdc8555b600e0a67d145a4

UPLOAD: SUCCESS
REMOTE LISTING: SUCCESS
Verified path: experiment5_preflight/preflight_71786e1fc413443fa7f4dbbb3420215e.txt


(…)ght_71786e1fc413443fa7f4dbbb3420215e.txt:   0%|          | 0.00/143 [00:00<?, ?B/s]

DOWNLOAD: SUCCESS
Downloaded SHA-256: e160ff62c9e2e01cc37a68c8234efa54b06aa88c30fdc8555b600e0a67d145a4

SHA-256 INTEGRITY: PASS

PREFLIGHT COMPLETE — ALL CHECKS PASSED
No experiment data or model checkpoints were modified.


In [4]:
# EXPERIMENT 5 — INSPECT EXISTING HUGGING FACE BACKUPS
# Read-only: does not upload, delete, or modify repository files.

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
from collections import Counter

REPO_ID = "SakibM19/border-surveillance-research-backups"

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi()

files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=token
)

print("=" * 75)
print("EXPERIMENT 5 — HUGGING FACE BACKUP INVENTORY")
print("=" * 75)

print("Repository:", REPO_ID)
print("Total files:", len(files))

# Count files by top-level directory
top_level = Counter(
    path.split("/")[0] if "/" in path else "[ROOT]"
    for path in files
)

print("\nTOP-LEVEL DIRECTORY SUMMARY:")
for folder, count in sorted(top_level.items()):
    print(f"{folder}: {count} files")

# Identify potentially useful Experiment 1 artifacts
keywords = [
    "experiment1",
    "llvip",
    "best.pt",
    "weights",
    "checkpoint",
    "prediction",
    "test",
    "manifest",
]

matches = [
    path for path in files
    if any(keyword in path.lower() for keyword in keywords)
]

print("\nPOTENTIALLY RELEVANT FILES:")
if matches:
    for path in matches:
        print(path)
else:
    print("No matching files found.")

print("\nINVENTORY COMPLETE")
print("Read-only inspection; repository unchanged.")

EXPERIMENT 5 — HUGGING FACE BACKUP INVENTORY
Repository: SakibM19/border-surveillance-research-backups
Total files: 159

TOP-LEVEL DIRECTORY SUMMARY:
Experiment1: 17 files
Experiment2: 134 files
Experiment3: 5 files
Experiment4: 1 files
[ROOT]: 1 files
experiment5_preflight: 1 files

POTENTIALLY RELEVANT FILES:
Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip
Experiment1/Paper_Evidence/Experiment1_Paper_Evidence.zip
Experiment1/Paper_Evidence/Individual_Figures/Figure_1_Training_Loss.pdf
Experiment1/Paper_Evidence/Individual_Figures/Figure_1_Training_Loss.png
Experiment1/Paper_Evidence/Individual_Figures/Figure_2_Validation_Performance.pdf
Experiment1/Paper_Evidence/Individual_Figures/Figure_2_Validation_Performance.png
Experiment1/Paper_Evidence/Individual_Figures/Figure_3_Test_AP50.pdf
Experiment1/Paper_Evidence/Individual_Figures/Figure_3_Test_AP50.png
Experiment1/Paper_Evidence/Individual_Figures/Figure_4_Test_Precision_Recall_F1.pdf
Experiment1/Paper_Evidence

In [5]:
# EXPERIMENT 5 — INSPECT EXPERIMENT 1 ARCHIVE CONTENTS
# Read-only remote inspection.

from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
from pathlib import Path
import zipfile

REPO_ID = "SakibM19/border-surveillance-research-backups"

token = UserSecretsClient().get_secret("HF_TOKEN")

archives = [
    "Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip",
    "Experiment1/Paper_Evidence/Experiment1_Paper_Evidence.zip",
    "Experiment1/experiment1_backup.zip",
]

for archive in archives:
    print("\n" + "=" * 75)
    print("ARCHIVE:", archive)
    print("=" * 75)

    local_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=archive,
        repo_type="dataset",
        token=token
    )

    path = Path(local_path)

    print("Downloaded size (MB):", round(path.stat().st_size / (1024**2), 2))

    with zipfile.ZipFile(path, "r") as zf:
        members = zf.namelist()

        print("Total entries:", len(members))

        categories = {
            "Model weights": [],
            "Dataset/images": [],
            "Predictions": [],
            "Configuration/scripts": [],
            "Other": []
        }

        for name in members:
            lower = name.lower()

            if lower.endswith((".pt", ".pth", ".onnx")):
                categories["Model weights"].append(name)
            elif any(x in lower for x in ["images/", "datasets/", "llvip"]):
                categories["Dataset/images"].append(name)
            elif any(x in lower for x in ["prediction", "predictions"]):
                categories["Predictions"].append(name)
            elif lower.endswith((".py", ".yaml", ".yml", ".json")):
                categories["Configuration/scripts"].append(name)
            else:
                categories["Other"].append(name)

        for category, items in categories.items():
            print(f"\n{category}: {len(items)} entries")
            for item in items[:35]:
                print(" ", item)

            if len(items) > 35:
                print(f"  ... and {len(items) - 35} more")

print("\nARCHIVE INSPECTION COMPLETE")
print("No repository files were modified.")


ARCHIVE: Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip


Experiment1/Final_Evaluation/Experiment1(…): reconstructing file:   0%|          |  0.00B /  583kB            

Experiment1/Final_Evaluation/Experiment1(…): downloading bytes:           |  0.00B            

Downloaded size (MB): 0.56
Total entries: 8

Model weights: 0 entries

Dataset/images: 0 entries

Predictions: 4 entries
  rgb_validation_predictions.pkl
  thermal_validation_predictions.pkl
  blend_validation_predictions.pkl
  vt_validation_predictions.pkl

Configuration/scripts: 0 entries

Other: 4 entries
  exp1_final_test_prf1.csv
  exp1_ap50_independent_audit.csv
  all_four_validation_threshold_results.csv
  all_four_validation_best_thresholds.csv

ARCHIVE: Experiment1/Paper_Evidence/Experiment1_Paper_Evidence.zip


Experiment1/Paper_Evidence/Experiment1_P(…): reconstructing file:   0%|          |  0.00B / 12.9MB            

Experiment1/Paper_Evidence/Experiment1_P(…): downloading bytes:           |  0.00B            

Downloaded size (MB): 12.3
Total entries: 19

Model weights: 0 entries

Dataset/images: 0 entries

Predictions: 0 entries

Configuration/scripts: 0 entries

Other: 19 entries
  Figures/Figure_1_Training_Loss.pdf
  Figures/Figure_1_Training_Loss.png
  Figures/Figure_2_Validation_Performance.pdf
  Figures/Figure_2_Validation_Performance.png
  Figures/Figure_3_Test_AP50.pdf
  Figures/Figure_3_Test_AP50.png
  Figures/Figure_4_Test_Precision_Recall_F1.pdf
  Figures/Figure_4_Test_Precision_Recall_F1.png
  Figures/Figure_5_Validation_Threshold_F1.pdf
  Figures/Figure_5_Validation_Threshold_F1.png
  Figures/Figure_6_Qualitative_Comparison.pdf
  Figures/Figure_6_Qualitative_Comparison.png
  Tables/Table_III_Test_Performance.csv
  Tables/Table_III_Test_Performance.tex
  Tables/Table_II_Independent_AP50.csv
  Tables/Table_II_Independent_AP50.tex
  Tables/Table_I_Experimental_Settings.csv
  Tables/Table_I_Experimental_Settings.tex
  README.txt

ARCHIVE: Experiment1/experiment1_backup.zip


Experiment1/experiment1_backup.zip: reconstructing file:   0%|          |  0.00B / 72.4MB            

Experiment1/experiment1_backup.zip: downloading bytes:           |  0.00B            

Downloaded size (MB): 69.04
Total entries: 114

Model weights: 8 entries
  experiment1_backup/runs/rgb/weights/last.pt
  experiment1_backup/runs/rgb/weights/best.pt
  experiment1_backup/runs/thermal/weights/last.pt
  experiment1_backup/runs/thermal/weights/best.pt
  experiment1_backup/runs/blend/weights/last.pt
  experiment1_backup/runs/blend/weights/best.pt
  experiment1_backup/runs/vt/weights/last.pt
  experiment1_backup/runs/vt/weights/best.pt

Dataset/images: 0 entries

Predictions: 1 entries
  experiment1_backup/test_predictions.pkl

Configuration/scripts: 9 entries
  experiment1_backup/rgb.yaml
  experiment1_backup/thermal.yaml
  experiment1_backup/vt.yaml
  experiment1_backup/blend.yaml
  experiment1_backup/split_manifest.json
  experiment1_backup/runs/rgb/args.yaml
  experiment1_backup/runs/thermal/args.yaml
  experiment1_backup/runs/blend/args.yaml
  experiment1_backup/runs/vt/args.yaml

Other: 96 entries
  experiment1_backup/
  experiment1_backup/runs/
  experiment1_backup/ex

In [6]:
# EXPERIMENT 5 — AUTOMATED ARCHIVE INVENTORY
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
from pathlib import Path
import zipfile

REPO_ID = "SakibM19/border-surveillance-research-backups"
token = UserSecretsClient().get_secret("HF_TOKEN")

archives = [
    "Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip",
    "Experiment1/Paper_Evidence/Experiment1_Paper_Evidence.zip",
    "Experiment1/experiment1_backup.zip",
]

for archive in archives:
    print("\n" + "=" * 75)
    print(archive)
    print("=" * 75)

    try:
        local_path = hf_hub_download(
            repo_id=REPO_ID,
            filename=archive,
            repo_type="dataset",
            token=token
        )

        path = Path(local_path)

        print("Download: SUCCESS")
        print("Size (MB):", round(path.stat().st_size / (1024**2), 2))

        with zipfile.ZipFile(path, "r") as zf:
            names = zf.namelist()
            lower_names = [n.lower() for n in names]

            weights = [
                n for n in names
                if n.lower().endswith((".pt", ".pth", ".onnx"))
            ]

            images = [
                n for n in names
                if n.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
                and ("image" in n.lower() or "dataset" in n.lower())
            ]

            predictions = [
                n for n in names
                if "prediction" in n.lower()
            ]

            configs = [
                n for n in names
                if n.lower().endswith((".yaml", ".yml", ".json", ".py"))
            ]

            print("Archive entries:", len(names))
            print("Model weights:", len(weights))
            print("Dataset/image candidates:", len(images))
            print("Prediction files:", len(predictions))
            print("Configuration/script files:", len(configs))

            if weights:
                print("\nWeight examples:")
                for n in weights[:12]:
                    print(" ", n)

            if images:
                print("\nImage examples:")
                for n in images[:10]:
                    print(" ", n)

            if predictions:
                print("\nPrediction files:")
                for n in predictions[:10]:
                    print(" ", n)

    except Exception as e:
        print("INSPECTION ERROR:", type(e).__name__, str(e))

print("\nALL THREE ARCHIVES INSPECTED")
print("No remote files modified.")


Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip
Download: SUCCESS
Size (MB): 0.56
Archive entries: 8
Model weights: 0
Dataset/image candidates: 0
Prediction files: 4
Configuration/script files: 0

Prediction files:
  rgb_validation_predictions.pkl
  thermal_validation_predictions.pkl
  blend_validation_predictions.pkl
  vt_validation_predictions.pkl

Experiment1/Paper_Evidence/Experiment1_Paper_Evidence.zip
Download: SUCCESS
Size (MB): 12.3
Archive entries: 19
Model weights: 0
Dataset/image candidates: 0
Prediction files: 0
Configuration/script files: 0

Experiment1/experiment1_backup.zip
Download: SUCCESS
Size (MB): 69.04
Archive entries: 114
Model weights: 8
Dataset/image candidates: 0
Prediction files: 1
Configuration/script files: 9

Weight examples:
  experiment1_backup/runs/rgb/weights/last.pt
  experiment1_backup/runs/rgb/weights/best.pt
  experiment1_backup/runs/thermal/weights/last.pt
  experiment1_backup/runs/thermal/weights/best.pt
  experiment1_backu

In [7]:
# EXPERIMENT 5 — RESTORE AND INSPECT ORIGINAL CONFIGURATIONS
# Extracts only small configuration files into a separate directory.
# Does not modify existing archives or remote backups.

from pathlib import Path
import zipfile
import json
import yaml

archive_path = Path(
    "/kaggle/working/experiment1_backup.zip"
)

# Locate the already downloaded archive if available
if not archive_path.exists():
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import hf_hub_download

    token = UserSecretsClient().get_secret("HF_TOKEN")

    archive_path = Path(hf_hub_download(
        repo_id="SakibM19/border-surveillance-research-backups",
        filename="Experiment1/experiment1_backup.zip",
        repo_type="dataset",
        token=token
    ))

output_dir = Path("/kaggle/working/experiment5_config_inspection")
output_dir.mkdir(parents=True, exist_ok=True)

print("=" * 75)
print("EXPERIMENT 5 — CONFIGURATION INSPECTION")
print("=" * 75)

with zipfile.ZipFile(archive_path, "r") as zf:

    selected = [
        name for name in zf.namelist()
        if name.endswith((
            "rgb.yaml",
            "thermal.yaml",
            "blend.yaml",
            "vt.yaml",
            "split_manifest.json",
            "args.yaml"
        ))
    ]

    for name in selected:
        content = zf.read(name)

        destination = output_dir / Path(name).name
        destination.write_bytes(content)

        print(f"\n--- {name} ---")

        try:
            if name.endswith(".json"):
                data = json.loads(content.decode("utf-8"))
                print(json.dumps(data, indent=2)[:6000])
            else:
                data = yaml.safe_load(content.decode("utf-8"))
                print(yaml.safe_dump(data, sort_keys=False)[:6000])

        except Exception as e:
            print("Could not parse:", str(e))

print("\nCONFIGURATION INSPECTION COMPLETE")
print("Extracted files:", len(selected))
print("Original archive unchanged.")

EXPERIMENT 5 — CONFIGURATION INSPECTION

--- experiment1_backup/rgb.yaml ---
path: /content/work_exp1_corrected/rgb
train: images/train
val: images/val
test: images/test
names:
  0: person


--- experiment1_backup/thermal.yaml ---
path: /content/work_exp1_corrected/thermal
train: images/train
val: images/val
test: images/test
names:
  0: person


--- experiment1_backup/vt.yaml ---
path: /content/work_exp1_corrected/vt
train: images/train
val: images/val
test: images/test
names:
  0: person


--- experiment1_backup/blend.yaml ---
path: /content/work_exp1_corrected/blend
train: images/train
val: images/val
test: images/test
names:
  0: person


--- experiment1_backup/split_manifest.json ---
{
  "seed": 0,
  "n_train": 4000,
  "n_val": 500,
  "n_test": 3463,
  "train": [
    "180191",
    "150163",
    "010047",
    "050073",
    "120297",
    "030455",
    "081344",
    "030098",
    "140479",
    "080783",
    "170251",
    "150461",
    "081313",
    "070190",
    "110239",
    "090099

In [9]:
# EXPERIMENT 5 — SEARCH HUGGING FACE FOR LLVIP DATASET
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

REPO_ID = "SakibM19/border-surveillance-research-backups"

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi()

files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=token
)

keywords = [
    "llvip",
    "dataset",
    "rgb/images",
    "thermal/images",
    "labels/train",
    "images/train",
    "images/test",
    ".tar",
    ".zip"
]

matches = [
    path for path in files
    if any(keyword in path.lower() for keyword in keywords)
]

print("=" * 75)
print("EXPERIMENT 5 — LLVIP DATASET BACKUP SEARCH")
print("=" * 75)

print("Total repository files:", len(files))
print("Matching paths:", len(matches))

for path in matches:
    print(path)

print("\nSEARCH COMPLETE")
print("No repository files modified.")

EXPERIMENT 5 — LLVIP DATASET BACKUP SEARCH
Total repository files: 159
Matching paths: 17
Experiment1/Final_Evaluation/Experiment1_Final_Evaluation_Evidence.zip
Experiment1/Paper_Evidence/Experiment1_Paper_Evidence.zip
Experiment1/exp1_training_logs.zip
Experiment1/experiment1_backup.zip
Experiment2/Dataset_Audits/Experiment2_Dataset_Integrity_Audit.json
Experiment2/Evaluation/Experiment2_Test_Evaluation_Evidence.zip
Experiment2/Paper_Evidence/IEEE_600DPI/Experiment2_IEEE_600DPI_Paper_Evidence.zip
Experiment2/Training/Infrared/final/infrared_complete_results.zip
Experiment2/Training/Visible/final/visible_complete_results.zip
Experiment2/experiment2_part_01.tar.part
Experiment2/experiment2_part_02.tar.part
Experiment2/experiment2_part_03.tar.part
Experiment2/experiment2_part_04.tar.part
Experiment2/experiment2_part_05.tar.part
Experiment3/Paper_Evidence/Experiment3_Range_Distance_Evidence.zip
Experiment3/Paper_Evidence/Experiment3_Range_Distance_Evidence_CORRECTED.zip
Experiment4/Final_

In [10]:
# EXPERIMENT 5 — RESTORE ORIGINAL LLVIP DATASET ARCHIVE

import sys
import subprocess
import shutil
import hashlib
from pathlib import Path

# Install gdown only if unavailable
try:
    import gdown
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "gdown"
    ])
    import gdown

FILE_ID = "1VTlT3Y7e1h-Zsne4zahjx5q0TK2ClMVv"

BASE_DIR = Path("/kaggle/working/Experiment5")
BASE_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_PATH = BASE_DIR / "LLVIP_original_source.zip"

print("=" * 75)
print("EXPERIMENT 5 — ORIGINAL LLVIP DATASET RECOVERY")
print("=" * 75)

disk = shutil.disk_usage("/kaggle/working")

print("Free disk space (GB):", round(disk.free / (1024**3), 2))

if disk.free < 20 * (1024**3):
    raise RuntimeError(
        "Insufficient free disk space. At least 20 GB is required "
        "before attempting dataset recovery."
    )

if ARCHIVE_PATH.exists():
    raise RuntimeError(
        f"Archive already exists at {ARCHIVE_PATH}. "
        "Stopping to avoid overwriting it."
    )

print("\nDownloading from the official LLVIP Google Drive source...")

downloaded = gdown.download(
    id=FILE_ID,
    output=str(ARCHIVE_PATH),
    quiet=False
)

if not downloaded or not ARCHIVE_PATH.exists():
    raise RuntimeError("Dataset download failed.")

if ARCHIVE_PATH.stat().st_size == 0:
    raise RuntimeError("Downloaded archive is empty.")

print("\nDOWNLOAD: SUCCESS")
print("Archive:", ARCHIVE_PATH)
print("Size (GB):", round(ARCHIVE_PATH.stat().st_size / (1024**3), 3))

print("\nCalculating SHA-256...")

sha256 = hashlib.sha256()

with open(ARCHIVE_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)

print("SHA-256:", sha256.hexdigest())

print("\nARCHIVE RECOVERY COMPLETE")
print("No extraction or experiment execution performed.")

EXPERIMENT 5 — ORIGINAL LLVIP DATASET RECOVERY
Free disk space (GB): 19.5


RuntimeError: Insufficient free disk space. At least 20 GB is required before attempting dataset recovery.

In [11]:
# EXPERIMENT 5 — KAGGLE STORAGE DIAGNOSTICS

import shutil
import subprocess
from pathlib import Path

print("=" * 70)
print("STORAGE DIAGNOSTICS")
print("=" * 70)

for path in ["/kaggle/working", "/kaggle/input", "/tmp"]:
    p = Path(path)
    print(f"\n{path}")
    print("Exists:", p.exists())

    if p.exists():
        usage = shutil.disk_usage(path)
        print(f"Total: {usage.total / (1024**3):.2f} GB")
        print(f"Used:  {usage.used / (1024**3):.2f} GB")
        print(f"Free:  {usage.free / (1024**3):.2f} GB")

print("\n" + "=" * 70)
print("FILESYSTEM MOUNTS")
print("=" * 70)

result = subprocess.run(
    ["df", "-h"],
    capture_output=True,
    text=True
)
print(result.stdout)

print("\n" + "=" * 70)
print("EXPERIMENT 5 DIRECTORY")
print("=" * 70)

base = Path("/kaggle/working/Experiment5")
print("Exists:", base.exists())

if base.exists():
    for item in base.iterdir():
        print(item.name, "-", round(
            item.stat().st_size / (1024**2), 2
        ), "MB" if item.is_file() else "(directory)")

STORAGE DIAGNOSTICS

/kaggle/working
Exists: True
Total: 19.52 GB
Used:  0.00 GB
Free:  19.50 GB

/kaggle/input
Exists: True
Total: 19.52 GB
Used:  0.00 GB
Free:  19.50 GB

/tmp
Exists: True
Total: 8156.88 GB
Used:  5643.90 GB
Free:  2512.97 GB

FILESYSTEM MOUNTS
Filesystem        Size  Used Avail Use% Mounted on
overlay           8.0T  5.6T  2.5T  70% /
tmpfs              64M     0   64M   0% /dev
shm                14G     0   14G   0% /dev/shm
/dev/loop2         20G  228K   20G   1% /kaggle/lib
/dev/sda1         122G  5.2G  117G   5% /opt/bin
/dev/mapper/snap  8.0T  5.6T  2.5T  70% /etc/hosts
tmpfs             4.0K     0  4.0K   0% /proc/acpi


EXPERIMENT 5 DIRECTORY
Exists: True


In [12]:
# EXPERIMENT 5 — DOWNLOAD LLVIP TO TEMPORARY STORAGE

import sys
import subprocess
import shutil
import hashlib
from pathlib import Path

try:
    import gdown
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "gdown"
    ])
    import gdown

FILE_ID = "1VTlT3Y7e1h-Zsne4zahjx5q0TK2ClMVv"

DOWNLOAD_DIR = Path("/tmp/experiment5_llvip_recovery")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_PATH = DOWNLOAD_DIR / "LLVIP_original_source.zip"

print("=" * 70)
print("EXPERIMENT 5 — LLVIP TEMPORARY-STORAGE RECOVERY")
print("=" * 70)

disk = shutil.disk_usage("/tmp")
print(f"Temporary storage free: {disk.free / (1024**3):.2f} GB")

if ARCHIVE_PATH.exists():
    raise RuntimeError(
        f"Archive already exists: {ARCHIVE_PATH}. "
        "Stopping to avoid overwriting it."
    )

print("\nDownloading from the official LLVIP source...")

result = gdown.download(
    id=FILE_ID,
    output=str(ARCHIVE_PATH),
    quiet=False
)

if not result or not ARCHIVE_PATH.exists():
    raise RuntimeError("Download failed or archive was not created.")

size = ARCHIVE_PATH.stat().st_size
if size == 0:
    raise RuntimeError("Downloaded archive is empty.")

print("\nDOWNLOAD SUCCESSFUL")
print("Path:", ARCHIVE_PATH)
print(f"Archive size: {size / (1024**3):.3f} GB")

print("\nCalculating SHA-256...")

digest = hashlib.sha256()
with open(ARCHIVE_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        digest.update(chunk)

print("SHA-256:", digest.hexdigest())
print("\nNo extraction or model execution performed.")

EXPERIMENT 5 — LLVIP TEMPORARY-STORAGE RECOVERY
Temporary storage free: 2512.97 GB



FileURLRetrievalError: Failed to retrieve file url:

	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. If the file you are trying to
	access is particularly large or is shared with many people, it may
	take up to 24 hours to be able to view or download the file. If you
	still can't access a file after 24 hours, contact your domain
	administrator.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1VTlT3Y7e1h-Zsne4zahjx5q0TK2ClMVv

but Gdown can't. Please check connections and permissions.

In [13]:
# EXPERIMENT 5 — INSPECT ALTERNATIVE LLVIP REPOSITORY

from huggingface_hub import HfApi

REPO_ID = "McCheng/LLVIP"
api = HfApi()

print("=" * 70)
print("ALTERNATIVE LLVIP REPOSITORY INSPECTION")
print("=" * 70)

info = api.repo_info(
    repo_id=REPO_ID,
    repo_type="dataset",
    files_metadata=True
)

files = api.list_repo_tree(
    repo_id=REPO_ID,
    repo_type="dataset",
    recursive=True,
    expand=True
)

file_items = [x for x in files if hasattr(x, "size")]

total_bytes = sum((x.size or 0) for x in file_items)

print("Repository:", REPO_ID)
print("Revision:", info.sha)
print("Files:", len(file_items))
print("Total size (GB):", round(total_bytes / (1024**3), 3))

print("\nFILE EXTENSION COUNTS:")
from collections import Counter
extensions = Counter(
    x.path.rsplit(".", 1)[-1].lower()
    if "." in x.path else "[no extension]"
    for x in file_items
)
for ext, count in extensions.most_common():
    print(ext, count)

print("\nSAMPLE IMAGE PATHS:")
images = [
    x for x in file_items
    if x.path.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
]
for x in images[:20]:
    print(x.path, x.size)

print("\nSAMPLE ANNOTATION PATHS:")
annotations = [
    x for x in file_items
    if x.path.lower().endswith((".xml", ".txt", ".json"))
]
for x in annotations[:20]:
    print(x.path, x.size)

print("\nREPOSITORY INSPECTION COMPLETE")
print("No files downloaded or modified.")

ALTERNATIVE LLVIP REPOSITORY INSPECTION
Repository: McCheng/LLVIP
Revision: e50976add2fcca00b0db0f3bba59ed486d8c1f6d
Files: 2
Total size (GB): 3.729

FILE EXTENSION COUNTS:
gitattributes 1
zip 1

SAMPLE IMAGE PATHS:

SAMPLE ANNOTATION PATHS:

REPOSITORY INSPECTION COMPLETE
No files downloaded or modified.


In [14]:
# EXPERIMENT 5 — DOWNLOAD AND INSPECT LLVIP CANDIDATE ARCHIVE
# No extraction, model inference, or changes to previous evidence.

import os
import hashlib
import zipfile
import shutil
from pathlib import Path
from collections import Counter
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = "McCheng/LLVIP"
BASE_DIR = Path("/kaggle/working/Experiment5/alternative_dataset")
BASE_DIR.mkdir(parents=True, exist_ok=True)

api = HfApi()
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
zip_files = [p for p in repo_files if p.lower().endswith(".zip")]

if len(zip_files) != 1:
    raise RuntimeError(f"Expected one ZIP archive; found: {zip_files}")

remote_zip = zip_files[0]
print("Remote archive:", remote_zip)

disk = shutil.disk_usage("/kaggle/working")
print("Free disk (GB):", round(disk.free / 1024**3, 2))

if disk.free < 6 * 1024**3:
    raise RuntimeError("Insufficient free space for safe archive download.")

print("\nDownloading candidate archive...")
archive_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename=remote_zip,
    local_dir=str(BASE_DIR)
)

archive_path = Path(archive_path)
print("Downloaded:", archive_path)
print("Size (GB):", round(archive_path.stat().st_size / 1024**3, 3))

print("\nCalculating SHA-256...")
digest = hashlib.sha256()
with open(archive_path, "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        digest.update(chunk)
print("SHA-256:", digest.hexdigest())

print("\nInspecting ZIP directory (without extraction)...")
with zipfile.ZipFile(archive_path, "r") as zf:
    entries = zf.infolist()
    files = [x for x in entries if not x.is_dir()]

    print("Total ZIP entries:", len(entries))
    print("Files:", len(files))
    print("Uncompressed size (GB):",
          round(sum(x.file_size for x in files) / 1024**3, 3))

    extensions = Counter(
        Path(x.filename).suffix.lower() or "[no extension]"
        for x in files
    )

    print("\nFILE EXTENSIONS:")
    for ext, count in extensions.most_common():
        print(ext, count)

    print("\nSAMPLE PATHS:")
    for item in files[:50]:
        print(item.filename)

    print("\nPOSSIBLE SPLIT / MODALITY DIRECTORIES:")
    dirs = sorted({
        "/".join(x.filename.split("/")[:i])
        for x in files
        for i in range(1, min(len(x.filename.split("/")), 5) + 1)
    })
    for d in dirs[:100]:
        print(d)

print("\nARCHIVE INSPECTION COMPLETE")
print("Nothing extracted; previous experiment evidence unchanged.")

Remote archive: LLVIP.zip
Free disk (GB): 19.5



LLVIP.zip: reconstructing file:   0%|          |  0.00B / 4.00GB            

LLVIP.zip: downloading bytes:           |  0.00B            

Downloaded: /kaggle/working/Experiment5/alternative_dataset/LLVIP.zip
Size (GB): 3.729

Calculating SHA-256...
SHA-256: b3b55475c093ac54663c467859eb31a82b840b53a7196ad1887cb547e4518b45

Inspecting ZIP directory (without extraction)...
Total ZIP entries: 46473
Files: 46465
Uncompressed size (GB): 3.776

FILE EXTENSIONS:
.jpg 30976
.xml 15488
.md 1

SAMPLE PATHS:
LLVIP/Annotations/010001.xml
LLVIP/Annotations/010002.xml
LLVIP/Annotations/010003.xml
LLVIP/Annotations/010004.xml
LLVIP/Annotations/010005.xml
LLVIP/Annotations/010006.xml
LLVIP/Annotations/010007.xml
LLVIP/Annotations/010008.xml
LLVIP/Annotations/010009.xml
LLVIP/Annotations/010010.xml
LLVIP/Annotations/010011.xml
LLVIP/Annotations/010012.xml
LLVIP/Annotations/010013.xml
LLVIP/Annotations/010014.xml
LLVIP/Annotations/010015.xml
LLVIP/Annotations/010016.xml
LLVIP/Annotations/010017.xml
LLVIP/Annotations/010018.xml
LLVIP/Annotations/010019.xml
LLVIP/Annotations/010020.xml
LLVIP/Annotations/010021.xml
LLVIP/Annotations/010022.xm

In [15]:
# EXPERIMENT 5 — LLVIP ARCHIVE AND ORIGINAL SPLIT VERIFICATION

import json
import zipfile
from pathlib import Path
from collections import Counter

ARCHIVE = Path(
    "/kaggle/working/Experiment5/alternative_dataset/LLVIP.zip"
)

SEARCH_ROOTS = [
    Path("/kaggle/working/experiment5_config_inspection"),
    Path("/kaggle/working/Experiment5"),
]

print("=" * 75)
print("EXPERIMENT 5 — LLVIP DATASET COMPATIBILITY AUDIT")
print("=" * 75)

# 1. Locate original split manifest
manifest_candidates = []

for root in SEARCH_ROOTS:
    if root.exists():
        manifest_candidates.extend(root.rglob("split_manifest.json"))

if not manifest_candidates:
    raise FileNotFoundError(
        "Original split_manifest.json not found. "
        "Do not create a replacement split."
    )

MANIFEST_PATH = manifest_candidates[0]
print("\nManifest:", MANIFEST_PATH)

with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

print("\nMANIFEST STRUCTURE:")
if isinstance(manifest, dict):
    for key, value in manifest.items():
        if isinstance(value, list):
            print(key, "list length:", len(value),
                  "sample:", value[:5])
        elif isinstance(value, dict):
            print(key, "dict keys:", list(value.keys())[:10])
        else:
            print(key, "=", value)

# 2. Inspect archive layout
print("\n" + "=" * 75)
print("ARCHIVE DIRECTORY STRUCTURE")
print("=" * 75)

with zipfile.ZipFile(ARCHIVE, "r") as zf:
    names = [
        n for n in zf.namelist()
        if not n.endswith("/")
    ]

    top_dirs = Counter(
        "/".join(n.split("/")[:3])
        for n in names
    )

    print("\nTop-level path groups:")
    for path, count in top_dirs.most_common(30):
        print(path, ":", count)

    jpg_paths = [n for n in names if n.lower().endswith(".jpg")]
    xml_paths = [n for n in names if n.lower().endswith(".xml")]

    print("\nJPEG count:", len(jpg_paths))
    print("XML count:", len(xml_paths))

    print("\nSample JPEG paths:")
    for p in jpg_paths[:12]:
        print(p)

    print("\nSample XML paths:")
    for p in xml_paths[:5]:
        print(p)

    # 3. Build filename ID sets
    jpg_ids = {
        Path(p).stem
        for p in jpg_paths
    }
    xml_ids = {
        Path(p).stem
        for p in xml_paths
    }

    print("\nUnique JPEG stems:", len(jpg_ids))
    print("Unique XML stems:", len(xml_ids))
    print("JPEG/XML stem intersection:", len(jpg_ids & xml_ids))

    # 4. Inspect manifest ID fields
    print("\n" + "=" * 75)
    print("SPLIT ID MATCHING")
    print("=" * 75)

    if not isinstance(manifest, dict):
        raise ValueError("Unexpected manifest format.")

    split_keys = [
        k for k, v in manifest.items()
        if isinstance(v, list)
    ]

    for key in split_keys:
        values = manifest[key]
        ids = {Path(str(x)).stem for x in values}

        print(f"\n{key}:")
        print("Manifest entries:", len(values))
        print("Unique IDs:", len(ids))
        print("Found in JPEG archive:", len(ids & jpg_ids))
        print("Found in XML archive:", len(ids & xml_ids))
        print("Missing from JPEG:", len(ids - jpg_ids))
        print("Missing from XML:", len(ids - xml_ids))

    # 5. Inspect one annotation without extracting
    if xml_paths:
        import xml.etree.ElementTree as ET

        sample_xml = ET.fromstring(zf.read(xml_paths[0]))

        print("\n" + "=" * 75)
        print("SAMPLE XML ANNOTATION")
        print("=" * 75)

        for tag in ["filename", "size", "object"]:
            element = sample_xml.find(tag)
            if element is not None:
                print(tag, ET.tostring(
                    element, encoding="unicode"
                )[:1000])

print("\nAUDIT COMPLETE — NO EXTRACTION OR MODIFICATION")

EXPERIMENT 5 — LLVIP DATASET COMPATIBILITY AUDIT

Manifest: /kaggle/working/experiment5_config_inspection/split_manifest.json

MANIFEST STRUCTURE:
seed = 0
n_train = 4000
n_val = 500
n_test = 3463
train list length: 4000 sample: ['180191', '150163', '010047', '050073', '120297']
val list length: 500 sample: ['080028', '081363', '180080', '040223', '100526']
test list length: 3463 sample: ['190001', '190002', '190003', '190004', '190005']

ARCHIVE DIRECTORY STRUCTURE

Top-level path groups:
LLVIP/infrared/train : 12025
LLVIP/visible/train : 12025
LLVIP/infrared/test : 3463
LLVIP/visible/test : 3463
LLVIP/Annotations/010001.xml : 1
LLVIP/Annotations/010002.xml : 1
LLVIP/Annotations/010003.xml : 1
LLVIP/Annotations/010004.xml : 1
LLVIP/Annotations/010005.xml : 1
LLVIP/Annotations/010006.xml : 1
LLVIP/Annotations/010007.xml : 1
LLVIP/Annotations/010008.xml : 1
LLVIP/Annotations/010009.xml : 1
LLVIP/Annotations/010010.xml : 1
LLVIP/Annotations/010011.xml : 1
LLVIP/Annotations/010012.xml : 1

In [16]:
# EXPERIMENT 5 — LLVIP FULL STRUCTURAL AND ANNOTATION AUDIT

import zipfile
import json
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
from io import BytesIO
from PIL import Image

ARCHIVE = Path(
    "/kaggle/working/Experiment5/alternative_dataset/LLVIP.zip"
)
MANIFEST = Path(
    "/kaggle/working/experiment5_config_inspection/split_manifest.json"
)

with open(MANIFEST) as f:
    manifest = json.load(f)

test_ids = set(manifest["test"])
train_ids = set(manifest["train"])
val_ids = set(manifest["val"])

print("=" * 75)
print("LLVIP FULL STRUCTURAL AND ANNOTATION AUDIT")
print("=" * 75)

with zipfile.ZipFile(ARCHIVE) as zf:
    names = zf.namelist()

    def image_ids(modality, split):
        prefix = f"LLVIP/{modality}/{split}/"
        return {
            Path(n).stem for n in names
            if n.startswith(prefix) and n.lower().endswith(".jpg")
        }

    visible_train = image_ids("visible", "train")
    visible_test = image_ids("visible", "test")
    thermal_train = image_ids("infrared", "train")
    thermal_test = image_ids("infrared", "test")

    print("\nIMAGE PAIRING")
    print("Visible train:", len(visible_train))
    print("Infrared train:", len(thermal_train))
    print("Train pairing mismatches:",
          len(visible_train.symmetric_difference(thermal_train)))
    print("Visible test:", len(visible_test))
    print("Infrared test:", len(thermal_test))
    print("Test pairing mismatches:",
          len(visible_test.symmetric_difference(thermal_test)))

    print("\nMANIFEST / ORIGINAL TEST DIRECTORY")
    print("Manifest test IDs:", len(test_ids))
    print("Archive visible test IDs:", len(visible_test))
    print("Manifest IDs missing from visible test:",
          len(test_ids - visible_test))
    print("Extra visible test IDs:",
          len(visible_test - test_ids))
    print("Manifest IDs missing from infrared test:",
          len(test_ids - thermal_test))
    print("Extra infrared test IDs:",
          len(thermal_test - test_ids))

    print("\nMANIFEST TRAIN/VAL LOCATION")
    print("Train IDs found in original train directory:",
          len(train_ids & visible_train), "/", len(train_ids))
    print("Validation IDs found in original train directory:",
          len(val_ids & visible_train), "/", len(val_ids))

    # Check representative image dimensions and readability
    print("\nIMAGE READABILITY / DIMENSIONS")
    sample_ids = sorted(list(test_ids))[:10]
    dimension_counts = Counter()
    image_errors = []

    for image_id in sample_ids:
        for modality in ["visible", "infrared"]:
            path = f"LLVIP/{modality}/test/{image_id}.jpg"
            try:
                with Image.open(BytesIO(zf.read(path))) as im:
                    im.verify()
                with Image.open(BytesIO(zf.read(path))) as im:
                    dimension_counts[(modality, im.size)] += 1
            except Exception as e:
                image_errors.append((path, str(e)))

    for key, count in dimension_counts.items():
        print(key, ":", count)

    print("Sample image errors:", len(image_errors))
    for error in image_errors[:5]:
        print(error)

    # Parse all annotations
    print("\nXML ANNOTATION VALIDATION")
    xml_paths = [
        n for n in names
        if n.startswith("LLVIP/Annotations/")
        and n.lower().endswith(".xml")
    ]

    xml_errors = []
    class_counts = Counter()
    missing_filename = 0
    invalid_boxes = 0
    total_objects = 0

    for xml_path in xml_paths:
        try:
            root = ET.fromstring(zf.read(xml_path))
            filename = root.findtext("filename")
            if not filename:
                missing_filename += 1

            size = root.find("size")
            if size is None:
                raise ValueError("Missing image size")

            width = int(size.findtext("width"))
            height = int(size.findtext("height"))

            for obj in root.findall("object"):
                total_objects += 1
                class_counts[obj.findtext("name", "UNKNOWN")] += 1
                box = obj.find("bndbox")
                if box is None:
                    invalid_boxes += 1
                    continue

                xmin = float(box.findtext("xmin"))
                ymin = float(box.findtext("ymin"))
                xmax = float(box.findtext("xmax"))
                ymax = float(box.findtext("ymax"))

                if not (0 <= xmin < xmax <= width and
                        0 <= ymin < ymax <= height):
                    invalid_boxes += 1

        except Exception as e:
            xml_errors.append((xml_path, str(e)))

    print("XML files:", len(xml_paths))
    print("XML parse errors:", len(xml_errors))
    print("Missing filename fields:", missing_filename)
    print("Object count:", total_objects)
    print("Class counts:", dict(class_counts))
    print("Invalid bounding boxes:", invalid_boxes)

    for error in xml_errors[:5]:
        print(error)

print("\nAUDIT FINISHED — NO EXTRACTION OR MODIFICATION")

LLVIP FULL STRUCTURAL AND ANNOTATION AUDIT

IMAGE PAIRING
Visible train: 12025
Infrared train: 12025
Train pairing mismatches: 0
Visible test: 3463
Infrared test: 3463
Test pairing mismatches: 0

MANIFEST / ORIGINAL TEST DIRECTORY
Manifest test IDs: 3463
Archive visible test IDs: 3463
Manifest IDs missing from visible test: 0
Extra visible test IDs: 0
Manifest IDs missing from infrared test: 0
Extra infrared test IDs: 0

MANIFEST TRAIN/VAL LOCATION
Train IDs found in original train directory: 4000 / 4000
Validation IDs found in original train directory: 500 / 500

IMAGE READABILITY / DIMENSIONS
('visible', (1280, 1024)) : 10
('infrared', (1280, 1024)) : 10
Sample image errors: 0

XML ANNOTATION VALIDATION
XML files: 15488
XML parse errors: 0
Missing filename fields: 0
Object count: 42437
Class counts: {'person': 42437}
Invalid bounding boxes: 5

AUDIT FINISHED — NO EXTRACTION OR MODIFICATION


In [17]:
# EXPERIMENT 5 — INVESTIGATE INVALID LLVIP BOUNDING BOXES

import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

ARCHIVE = Path(
    "/kaggle/working/Experiment5/alternative_dataset/LLVIP.zip"
)

invalid_records = []

with zipfile.ZipFile(ARCHIVE, "r") as zf:
    xml_paths = [
        n for n in zf.namelist()
        if n.startswith("LLVIP/Annotations/")
        and n.lower().endswith(".xml")
    ]

    for xml_path in xml_paths:
        root = ET.fromstring(zf.read(xml_path))
        size = root.find("size")

        width = int(size.findtext("width"))
        height = int(size.findtext("height"))

        for obj_index, obj in enumerate(root.findall("object")):
            box = obj.find("bndbox")

            if box is None:
                invalid_records.append({
                    "file": xml_path,
                    "object": obj_index,
                    "reason": "Missing bounding box"
                })
                continue

            xmin = float(box.findtext("xmin"))
            ymin = float(box.findtext("ymin"))
            xmax = float(box.findtext("xmax"))
            ymax = float(box.findtext("ymax"))

            problems = []

            if not (xmin < xmax):
                problems.append("xmin >= xmax")
            if not (ymin < ymax):
                problems.append("ymin >= ymax")
            if xmin < 0 or ymin < 0:
                problems.append("negative minimum coordinate")
            if xmax > width:
                problems.append("xmax exceeds image width")
            if ymax > height:
                problems.append("ymax exceeds image height")

            if problems:
                invalid_records.append({
                    "file": xml_path,
                    "object": obj_index,
                    "image_size": (width, height),
                    "box": (xmin, ymin, xmax, ymax),
                    "reason": "; ".join(problems)
                })

print("=" * 75)
print("INVALID ANNOTATION DETAILS")
print("=" * 75)

print("Invalid records:", len(invalid_records))

for record in invalid_records:
    for key, value in record.items():
        print(f"{key}: {value}")
    print("-" * 50)

print("\nINVESTIGATION COMPLETE")
print("No annotations or images modified.")

INVALID ANNOTATION DETAILS
Invalid records: 5
file: LLVIP/Annotations/020118.xml
object: 1
image_size: (1280, 1024)
box: (1025.0, 249.0, 1025.0, 250.0)
reason: xmin >= xmax
--------------------------------------------------
file: LLVIP/Annotations/020236.xml
object: 5
image_size: (1280, 1024)
box: (1103.0, 325.0, 1103.0, 326.0)
reason: xmin >= xmax
--------------------------------------------------
file: LLVIP/Annotations/090437.xml
object: 1
image_size: (1280, 1024)
box: (74.0, 863.0, 74.0, 870.0)
reason: xmin >= xmax
--------------------------------------------------
file: LLVIP/Annotations/091180.xml
object: 2
image_size: (1280, 1024)
box: (404.0, 290.0, 404.0, 291.0)
reason: xmin >= xmax
--------------------------------------------------
file: LLVIP/Annotations/170350.xml
object: 0
image_size: (1280, 1024)
box: (695.0, 950.0, 695.0, 957.0)
reason: xmin >= xmax
--------------------------------------------------

INVESTIGATION COMPLETE
No annotations or images modified.


In [18]:
# EXPERIMENT 5 — MAP INVALID ANNOTATIONS TO ORIGINAL SPLITS

import json
from pathlib import Path

MANIFEST = Path(
    "/kaggle/working/experiment5_config_inspection/split_manifest.json"
)

with open(MANIFEST, "r") as f:
    manifest = json.load(f)

invalid_ids = {
    "020118",
    "020236",
    "090437",
    "091180",
    "170350",
}

splits = {
    "train": set(manifest["train"]),
    "validation": set(manifest["val"]),
    "test": set(manifest["test"]),
}

print("=" * 75)
print("INVALID ANNOTATION SPLIT AUDIT")
print("=" * 75)

for image_id in sorted(invalid_ids):
    locations = [
        split for split, ids in splits.items()
        if image_id in ids
    ]

    print(
        image_id,
        "->",
        locations[0] if locations else "NOT FOUND IN MANIFEST"
    )

print("\nSUMMARY")
for split, ids in splits.items():
    affected = sorted(invalid_ids & ids)
    print(f"{split}: {len(affected)} affected IDs", affected)

print("\nNO DATA MODIFIED")

INVALID ANNOTATION SPLIT AUDIT
020118 -> NOT FOUND IN MANIFEST
020236 -> NOT FOUND IN MANIFEST
090437 -> NOT FOUND IN MANIFEST
091180 -> validation
170350 -> NOT FOUND IN MANIFEST

SUMMARY
train: 0 affected IDs []
validation: 1 affected IDs ['091180']
test: 0 affected IDs []

NO DATA MODIFIED


In [19]:
# EXPERIMENT 5 — BACK UP VERIFIED LLVIP ARCHIVE TO PRIVATE HUGGING FACE

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = "SakibM19/border-surveillance-research-backups"
SOURCE_REPO = "McCheng/LLVIP"
SOURCE_REVISION = "e50976add2fcca00b0db0f3bba59ed486d8c1f6d"

ARCHIVE = Path(
    "/kaggle/working/Experiment5/alternative_dataset/LLVIP.zip"
)

REMOTE_ARCHIVE = "Experiment5/Dataset/LLVIP.zip"
REMOTE_AUDIT = "Experiment5/Dataset/LLVIP_dataset_audit.json"

INVALID_ANNOTATIONS = [
    "020118", "020236", "090437", "091180", "170350"
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if not ARCHIVE.exists():
    raise FileNotFoundError(ARCHIVE)

print("=" * 75)
print("EXPERIMENT 5 — HUGGING FACE DATASET BACKUP")
print("=" * 75)

# Authenticate using Kaggle Secret
token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi()

# Verify private repository access
repo_info = api.repo_info(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=token
)

print("Authenticated repository:", repo_info.id)
print("Archive size (GB):", round(ARCHIVE.stat().st_size / 1024**3, 3))

# Recalculate local checksum before upload
local_hash = sha256_file(ARCHIVE)
print("Local SHA-256:", local_hash)

# Create separate audit metadata
audit = {
    "experiment": "Experiment 5 — RGB-Thermal Fusion Robustness",
    "artifact": "Original LLVIP dataset archive",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_repository": SOURCE_REPO,
    "source_revision": SOURCE_REVISION,
    "source_archive": "LLVIP.zip",
    "local_archive": str(ARCHIVE),
    "archive_size_bytes": ARCHIVE.stat().st_size,
    "archive_sha256": local_hash,
    "dataset_counts": {
        "visible_images": 15488,
        "infrared_images": 15488,
        "xml_annotations": 15488,
        "person_objects": 42437
    },
    "original_split_manifest": {
        "seed": 0,
        "train": 4000,
        "validation": 500,
        "test": 3463
    },
    "split_compatibility": {
        "train_ids_found": 4000,
        "validation_ids_found": 500,
        "test_ids_found": 3463,
        "test_visible_pair_mismatches": 0,
        "test_infrared_pair_mismatches": 0
    },
    "annotation_audit": {
        "xml_parse_errors": 0,
        "invalid_degenerate_boxes": 5,
        "invalid_annotation_ids": INVALID_ANNOTATIONS,
        "training_affected": 0,
        "validation_affected": 1,
        "validation_affected_id": "091180",
        "test_affected": 0,
        "source_annotations_modified": False
    },
    "limitations": [
        "The archive is a third-party Hugging Face copy, not the original authors-hosted download.",
        "Split-ID compatibility and structural checks passed.",
        "The five degenerate source annotations are preserved unchanged.",
        "No model inference or experiment execution has been performed."
    ]
}

audit_path = Path(
    "/kaggle/working/Experiment5/LLVIP_dataset_audit.json"
)
audit_path.parent.mkdir(parents=True, exist_ok=True)

with open(audit_path, "w") as f:
    json.dump(audit, f, indent=2)

# Upload archive
print("\nUploading dataset archive...")
api.upload_file(
    path_or_fileobj=str(ARCHIVE),
    path_in_repo=REMOTE_ARCHIVE,
    repo_id=REPO_ID,
    repo_type="dataset",
    token=token,
    commit_message="Experiment 5: back up structurally verified LLVIP archive"
)
print("Archive upload completed.")

# Upload audit
print("\nUploading audit metadata...")
api.upload_file(
    path_or_fileobj=str(audit_path),
    path_in_repo=REMOTE_AUDIT,
    repo_id=REPO_ID,
    repo_type="dataset",
    token=token,
    commit_message="Experiment 5: add LLVIP dataset integrity audit"
)
print("Audit upload completed.")

# Verify remote listing
remote_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=token
)

assert REMOTE_ARCHIVE in remote_files, "Remote archive not listed."
assert REMOTE_AUDIT in remote_files, "Remote audit not listed."

print("\nRemote files confirmed:")
print(REMOTE_ARCHIVE)
print(REMOTE_AUDIT)

# Download remote archive and verify exact bytes
print("\nDownloading remote archive for SHA-256 verification...")

remote_copy = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename=REMOTE_ARCHIVE,
    token=token,
    force_download=True
)

remote_hash = sha256_file(remote_copy)

print("Remote downloaded path:", remote_copy)
print("Remote SHA-256:", remote_hash)

if remote_hash != local_hash:
    raise RuntimeError("SHA-256 mismatch — backup verification FAILED.")

print("\n" + "=" * 75)
print("BACKUP VERIFICATION: PASS")
print("Local and remote SHA-256 hashes match.")
print("Original LLVIP archive remains unchanged.")
print("No extraction or model inference performed.")
print("=" * 75)

EXPERIMENT 5 — HUGGING FACE DATASET BACKUP
Authenticated repository: SakibM19/border-surveillance-research-backups
Archive size (GB): 3.729
Local SHA-256: b3b55475c093ac54663c467859eb31a82b840b53a7196ad1887cb547e4518b45

Uploading dataset archive...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

Archive upload completed.

Uploading audit metadata...
Audit upload completed.

Remote files confirmed:
Experiment5/Dataset/LLVIP.zip
Experiment5/Dataset/LLVIP_dataset_audit.json



Experiment5/Dataset/LLVIP.zip: reconstructing file:   0%|          |  0.00B / 4.00GB            

Experiment5/Dataset/LLVIP.zip: downloading bytes:           |  0.00B            

Remote downloaded path: /root/.cache/huggingface/hub/datasets--SakibM19--border-surveillance-research-backups/snapshots/91d73b4e7c125467f7b3abcb671e1be0943ab530/Experiment5/Dataset/LLVIP.zip
Remote SHA-256: b3b55475c093ac54663c467859eb31a82b840b53a7196ad1887cb547e4518b45

BACKUP VERIFICATION: PASS
Local and remote SHA-256 hashes match.
Original LLVIP archive remains unchanged.
No extraction or model inference performed.


In [20]:
# EXPERIMENT 5 — EXTRACT ORIGINAL LLVIP TEST SET ONLY

import zipfile
import json
import shutil
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET

ARCHIVE = Path(
    "/kaggle/working/Experiment5/alternative_dataset/LLVIP.zip"
)
MANIFEST = Path(
    "/kaggle/working/experiment5_config_inspection/split_manifest.json"
)
OUTPUT = Path(
    "/kaggle/working/Experiment5/LLVIP_test_original"
)

if OUTPUT.exists():
    raise RuntimeError(
        f"Output directory already exists: {OUTPUT}. "
        "Stopping to avoid overwriting existing evidence."
    )

with open(MANIFEST, "r") as f:
    manifest = json.load(f)

test_ids = set(manifest["test"])

if len(test_ids) != 3463:
    raise RuntimeError("Unexpected test split size.")

disk = shutil.disk_usage("/kaggle/working")
print("Free disk before extraction (GB):",
      round(disk.free / 1024**3, 2))

OUTPUT.mkdir(parents=True)

expected = {
    "visible": 3463,
    "infrared": 3463,
    "annotations": 3463
}

extracted = {
    "visible": 0,
    "infrared": 0,
    "annotations": 0
}

print("\nExtracting test images and annotations...")

with zipfile.ZipFile(ARCHIVE, "r") as zf:
    archive_names = set(zf.namelist())

    for image_id in sorted(test_ids):
        paths = {
            "visible": f"LLVIP/visible/test/{image_id}.jpg",
            "infrared": f"LLVIP/infrared/test/{image_id}.jpg",
            "annotations": f"LLVIP/Annotations/{image_id}.xml"
        }

        for category, source_path in paths.items():
            if source_path not in archive_names:
                raise FileNotFoundError(source_path)

            relative_path = Path(source_path).relative_to("LLVIP")
            destination = OUTPUT / relative_path
            destination.parent.mkdir(parents=True, exist_ok=True)

            with zf.open(source_path) as src, open(destination, "wb") as dst:
                shutil.copyfileobj(src, dst)

            extracted[category] += 1

print("\nEXTRACTION COUNTS")
for category in expected:
    print(
        category,
        ":", extracted[category],
        "/", expected[category]
    )
    if extracted[category] != expected[category]:
        raise RuntimeError(f"Incomplete extraction: {category}")

# Verify image readability and XML parsing
print("\nVERIFYING EXTRACTED FILES...")

image_errors = []
xml_errors = []

for modality in ["visible", "infrared"]:
    image_dir = OUTPUT / modality / "test"

    for image_id in test_ids:
        path = image_dir / f"{image_id}.jpg"
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception as e:
            image_errors.append((str(path), str(e)))

annotation_dir = OUTPUT / "Annotations"

for image_id in test_ids:
    path = annotation_dir / f"{image_id}.xml"
    try:
        ET.parse(path)
    except Exception as e:
        xml_errors.append((str(path), str(e)))

print("Image errors:", len(image_errors))
print("XML errors:", len(xml_errors))

if image_errors or xml_errors:
    raise RuntimeError("Extracted dataset integrity check failed.")

# Confirm original test IDs in extracted folders
visible_ids = {
    p.stem for p in (OUTPUT / "visible/test").glob("*.jpg")
}
infrared_ids = {
    p.stem for p in (OUTPUT / "infrared/test").glob("*.jpg")
}
annotation_ids = {
    p.stem for p in annotation_dir.glob("*.xml")
}

assert visible_ids == test_ids
assert infrared_ids == test_ids
assert annotation_ids == test_ids

# Save exact split manifest alongside extracted test set
shutil.copy2(
    MANIFEST,
    OUTPUT / "split_manifest_original.json"
)

print("\n" + "=" * 75)
print("TEST SET EXTRACTION: PASS")
print("Visible IDs:", len(visible_ids))
print("Infrared IDs:", len(infrared_ids))
print("Annotation IDs:", len(annotation_ids))
print("All extracted IDs match the original test manifest.")
print("Training and validation data were not extracted.")
print("Original ZIP archive remains unchanged.")
print("Output:", OUTPUT)
print("=" * 75)

Free disk before extraction (GB): 15.77

Extracting test images and annotations...

EXTRACTION COUNTS
visible : 3463 / 3463
infrared : 3463 / 3463
annotations : 3463 / 3463

VERIFYING EXTRACTED FILES...
Image errors: 0
XML errors: 0

TEST SET EXTRACTION: PASS
Visible IDs: 3463
Infrared IDs: 3463
Annotation IDs: 3463
All extracted IDs match the original test manifest.
Training and validation data were not extracted.
Original ZIP archive remains unchanged.
Output: /kaggle/working/Experiment5/LLVIP_test_original


In [21]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
from zipfile import ZipFile
from pathlib import Path
import hashlib
import os

REPO_ID = "SakibM19/border-surveillance-research-backups"
ARCHIVE_PATH = "Experiment1/experiment1_backup.zip"

token = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token retrieved from Kaggle Secrets.")

archive = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename=ARCHIVE_PATH,
    token=token
)

archive = Path(archive)
print("\nARCHIVE")
print("Path:", archive)
print("Size (MB):", round(archive.stat().st_size / (1024**2), 2))

sha256 = hashlib.sha256()
with open(archive, "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)

print("SHA-256:", sha256.hexdigest())

expected = [
    "experiment1_backup/runs/rgb/weights/best.pt",
    "experiment1_backup/runs/thermal/weights/best.pt",
    "experiment1_backup/runs/blend/weights/best.pt",
    "experiment1_backup/runs/vt/weights/best.pt",
]

print("\nCHECKPOINT INVENTORY")
with ZipFile(archive, "r") as z:
    entries = {item.filename: item for item in z.infolist()}

    for name in expected:
        item = entries.get(name)
        if item:
            print(f"FOUND  | {name} | {item.file_size / (1024**2):.2f} MB")
        else:
            print(f"MISSING | {name}")

    print("\nOTHER RELEVANT FILES")
    for name in sorted(entries):
        lower = name.lower()
        if (
            name.endswith(("args.yaml", ".yaml", ".yml", "split_manifest.json"))
            or "prediction" in lower
        ):
            item = entries[name]
            print(f"{name} | {item.file_size / 1024:.1f} KB")

    print("\nBEST CHECKPOINTS PRESENT:",
          sum(name in entries for name in expected), "/", len(expected))

HF token retrieved from Kaggle Secrets.

ARCHIVE
Path: /root/.cache/huggingface/hub/datasets--SakibM19--border-surveillance-research-backups/snapshots/91d73b4e7c125467f7b3abcb671e1be0943ab530/Experiment1/experiment1_backup.zip
Size (MB): 69.04
SHA-256: 3cb66986951fbd094f397c0b5d87092e38287b72131f498bcd9ea28dd53548fe

CHECKPOINT INVENTORY
FOUND  | experiment1_backup/runs/rgb/weights/best.pt | 5.24 MB
FOUND  | experiment1_backup/runs/thermal/weights/best.pt | 5.24 MB
FOUND  | experiment1_backup/runs/blend/weights/best.pt | 5.24 MB
FOUND  | experiment1_backup/runs/vt/weights/best.pt | 5.24 MB

OTHER RELEVANT FILES
experiment1_backup/blend.yaml | 0.1 KB
experiment1_backup/rgb.yaml | 0.1 KB
experiment1_backup/runs/blend/args.yaml | 1.7 KB
experiment1_backup/runs/rgb/args.yaml | 1.7 KB
experiment1_backup/runs/thermal/args.yaml | 1.7 KB
experiment1_backup/runs/vt/args.yaml | 1.7 KB
experiment1_backup/split_manifest.json | 109.0 KB
experiment1_backup/test_predictions.pkl | 10549.8 KB
experimen

In [22]:
from zipfile import ZipFile
from pathlib import Path
import json
import pickletools
from collections import Counter

archive = Path(
    "/root/.cache/huggingface/hub/datasets--SakibM19--border-surveillance-research-backups/"
    "snapshots/91d73b4e7c125467f7b3abcb671e1be0943ab530/"
    "Experiment1/experiment1_backup.zip"
)

print("=" * 75)
print("EXPERIMENT 1 ARCHIVE INTEGRITY")
print("=" * 75)

with ZipFile(archive, "r") as z:
    bad_file = z.testzip()
    print("ZIP CRC test:", "PASS" if bad_file is None else f"FAIL: {bad_file}")

    print("\nMODEL CONFIGURATIONS")
    for modality in ["rgb", "thermal", "blend", "vt"]:
        path = f"experiment1_backup/{modality}.yaml"
        print(f"\n--- {modality.upper()} ---")
        print(z.read(path).decode("utf-8"))

    print("\nSPLIT MANIFEST SUMMARY")
    manifest = json.loads(
        z.read("experiment1_backup/split_manifest.json").decode("utf-8")
    )
    print("Top-level keys:", list(manifest.keys()))
    for key, value in manifest.items():
        if isinstance(value, list):
            print(f"{key}: {len(value)} entries")
        elif isinstance(value, dict):
            print(f"{key}: dictionary with {len(value)} keys")
        else:
            print(f"{key}: {value}")

    print("\nPREDICTION PICKLE STRUCTURE (SAFE OPCODE INSPECTION)")
    pickle_path = "experiment1_backup/test_predictions.pkl"
    raw = z.read(pickle_path)
    print("File size (MB):", round(len(raw) / (1024 * 1024), 2))

    opcode_counts = Counter()
    global_refs = []
    memo_ops = 0

    for opcode, argument, position in pickletools.genops(raw):
        opcode_counts[opcode.name] += 1
        if opcode.name in ("GLOBAL", "STACK_GLOBAL"):
            global_refs.append((opcode.name, argument, position))
        if opcode.name in ("PUT", "BINPUT", "LONG_BINPUT", "MEMOIZE"):
            memo_ops += 1

    print("Protocol:", raw[1] if raw[:1] == b"\x80" else "legacy/unknown")
    print("Total opcodes:", sum(opcode_counts.values()))
    print("Memo operations:", memo_ops)
    print("Most common opcodes:", opcode_counts.most_common(15))
    print("Global references:")
    for item in global_refs[:30]:
        print(" ", item)
    if len(global_refs) > 30:
        print(" ... additional references:", len(global_refs) - 30)

print("\nINSPECTION COMPLETE")

EXPERIMENT 1 ARCHIVE INTEGRITY
ZIP CRC test: PASS

MODEL CONFIGURATIONS

--- RGB ---
path: /content/work_exp1_corrected/rgb
train: images/train
val: images/val
test: images/test
names: {0: person}


--- THERMAL ---
path: /content/work_exp1_corrected/thermal
train: images/train
val: images/val
test: images/test
names: {0: person}


--- BLEND ---
path: /content/work_exp1_corrected/blend
train: images/train
val: images/val
test: images/test
names: {0: person}


--- VT ---
path: /content/work_exp1_corrected/vt
train: images/train
val: images/val
test: images/test
names: {0: person}


SPLIT MANIFEST SUMMARY
Top-level keys: ['seed', 'n_train', 'n_val', 'n_test', 'train', 'val', 'test']
seed: 0
n_train: 4000
n_val: 500
n_test: 3463
train: 4000 entries
val: 500 entries
test: 3463 entries

PREDICTION PICKLE STRUCTURE (SAFE OPCODE INSPECTION)
File size (MB): 10.3
Protocol: 5
Total opcodes: 644393
Memo operations: 187031
Most common opcodes: [('MEMOIZE', 187031), ('BINGET', 83226), ('BININT1', 65

In [23]:
from zipfile import ZipFile
from pathlib import Path
import pickletools

archive = Path(
    "/root/.cache/huggingface/hub/datasets--SakibM19--border-surveillance-research-backups/"
    "snapshots/91d73b4e7c125467f7b3abcb671e1be0943ab530/"
    "Experiment1/experiment1_backup.zip"
)

with ZipFile(archive, "r") as z:
    data = z.read("experiment1_backup/test_predictions.pkl")

ops = list(pickletools.genops(data))
global_positions = [
    i for i, (op, arg, pos) in enumerate(ops)
    if op.name in ("GLOBAL", "STACK_GLOBAL")
]

print("=" * 75)
print("PREDICTION PICKLE — GLOBAL REFERENCE CONTEXT")
print("=" * 75)
print("Total opcodes:", len(ops))
print("Global reference operations:", len(global_positions))

for number, idx in enumerate(global_positions, 1):
    print(f"\n--- GLOBAL REFERENCE {number} ---")
    start = max(0, idx - 10)
    end = min(len(ops), idx + 4)

    for j in range(start, end):
        op, arg, pos = ops[j]
        marker = " >>> " if j == idx else "     "
        print(f"{marker}{pos:>9} | {op.name:<18} | {repr(arg)[:180]}")

print("\nSTATIC INSPECTION COMPLETE — NO PICKLE DESERIALIZATION PERFORMED")

PREDICTION PICKLE — GLOBAL REFERENCE CONTEXT
Total opcodes: 644393
Global reference operations: 4

--- GLOBAL REFERENCE 1 ---
            19 | MEMOIZE            | None
            20 | EMPTY_DICT         | None
            21 | MEMOIZE            | None
            22 | MARK               | None
            23 | SHORT_BINUNICODE   | '190001'
            31 | MEMOIZE            | None
            32 | SHORT_BINUNICODE   | 'numpy._core.numeric'
            53 | MEMOIZE            | None
            54 | SHORT_BINUNICODE   | '_frombuffer'
            67 | MEMOIZE            | None
 >>>        68 | STACK_GLOBAL       | None
            69 | MEMOIZE            | None
            70 | MARK               | None
            71 | BYTEARRAY8         | bytearray(b'\xf0MM?P\xbc\xa2>\x17Tf?\xbb\x14\x13?\x80@s?\\\xd3\x97>G\xf7\x7f?\xf6s#? RL>\x80\x85\x0b<\xe0:\xa1>\x9a\xa8\x8a>\x1aov?\xc4\xd4\x8e>\x00\x00\x80?r\xb3\x10?X\xfcG>\xc0\x

--- GLOBAL REFERENCE 2 ---
            67 | MEMOIZE            | 

In [24]:
import io
import pickle
import numpy as np
from pathlib import Path
from zipfile import ZipFile

ARCHIVE = Path(
    "/root/.cache/huggingface/hub/datasets--SakibM19--border-surveillance-research-backups/"
    "snapshots/91d73b4e7c125467f7b3abcb671e1be0943ab530/"
    "Experiment1/experiment1_backup.zip"
)

ALLOWED_GLOBALS = {
    ("numpy._core.numeric", "_frombuffer"),
    ("numpy", "dtype"),
    ("numpy._core.multiarray", "_reconstruct"),
    ("numpy", "ndarray"),
}

class RestrictedNumpyUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if (module, name) in ALLOWED_GLOBALS:
            imported = __import__(module, fromlist=[name])
            return getattr(imported, name)
        raise pickle.UnpicklingError(
            f"Blocked global: {module}.{name}"
        )

with ZipFile(ARCHIVE, "r") as z:
    raw = z.read("experiment1_backup/test_predictions.pkl")

print("Loading with restricted NumPy-only allowlist...")
obj = RestrictedNumpyUnpickler(io.BytesIO(raw)).load()
print("Restricted loading: PASS")
print("Root type:", type(obj).__name__)

def inspect(value, path="root", depth=0):
    indent = "  " * depth

    if isinstance(value, dict):
        print(f"{indent}{path}: dict, {len(value)} entries")
        for i, (key, item) in enumerate(value.items()):
            if i >= 15:
                print(f"{indent}  ... remaining keys omitted")
                break
            print(f"{indent}  key: {repr(key)[:120]}")
            inspect(item, f"{path}[{repr(key)[:50]}]", depth + 2)

    elif isinstance(value, (list, tuple)):
        print(f"{indent}{path}: {type(value).__name__}, length={len(value)}")
        for i, item in enumerate(value[:3]):
            inspect(item, f"{path}[{i}]", depth + 1)
        if len(value) > 3:
            print(f"{indent}  ... remaining items omitted")

    elif isinstance(value, np.ndarray):
        print(
            f"{indent}{path}: ndarray, shape={value.shape}, "
            f"dtype={value.dtype}"
        )

    elif isinstance(value, np.generic):
        print(f"{indent}{path}: NumPy scalar, dtype={value.dtype}")

    else:
        print(f"{indent}{path}: {type(value).__name__} = {repr(value)[:120]}")

print("\nPREDICTION OBJECT STRUCTURE")
inspect(obj)

print("\nINSPECTION COMPLETE")

Loading with restricted NumPy-only allowlist...
Restricted loading: PASS
Root type: dict

PREDICTION OBJECT STRUCTURE
root: dict, 5 entries
  key: 'rgb'
    root['rgb']: dict, 3463 entries
      key: '190001'
        root['rgb']['190001']: tuple, length=3
          root['rgb']['190001'][0]: ndarray, shape=(27, 4), dtype=float32
          root['rgb']['190001'][1]: ndarray, shape=(27,), dtype=float32
          root['rgb']['190001'][2]: tuple, length=2
            root['rgb']['190001'][2][0]: int = 1280
            root['rgb']['190001'][2][1]: int = 1024
      key: '190002'
        root['rgb']['190002']: tuple, length=3
          root['rgb']['190002'][0]: ndarray, shape=(19, 4), dtype=float32
          root['rgb']['190002'][1]: ndarray, shape=(19,), dtype=float32
          root['rgb']['190002'][2]: tuple, length=2
            root['rgb']['190002'][2][0]: int = 1280
            root['rgb']['190002'][2][1]: int = 1024
      key: '190003'
        root['rgb']['190003']: tuple, length=3
      

In [25]:
import json
import numpy as np
from pathlib import Path
from zipfile import ZipFile

ARCHIVE = Path(
    "/root/.cache/huggingface/hub/datasets--SakibM19--border-surveillance-research-backups/"
    "snapshots/91d73b4e7c125467f7b3abcb671e1be0943ab530/"
    "Experiment1/experiment1_backup.zip"
)

# Reuse the safely loaded prediction object: obj
with ZipFile(ARCHIVE, "r") as z:
    manifest = json.loads(
        z.read("experiment1_backup/split_manifest.json")
    )

expected_ids = set(manifest["test"])
print("=" * 75)
print("EXPERIMENT 1 — BASELINE PREDICTION AUDIT")
print("=" * 75)
print("Manifest test IDs:", len(expected_ids))

all_pass = True

for name, predictions in obj.items():
    ids = set(predictions.keys())
    missing = expected_ids - ids
    extra = ids - expected_ids

    total_boxes = 0
    empty_images = 0
    invalid_arrays = 0
    invalid_scores = 0
    invalid_boxes = 0
    score_min = float("inf")
    score_max = float("-inf")
    box_min = float("inf")
    box_max = float("-inf")
    dimension_mismatches = 0

    for image_id, record in predictions.items():
        boxes, scores, dimensions = record
        boxes = np.asarray(boxes)
        scores = np.asarray(scores)

        total_boxes += len(scores)
        empty_images += int(len(scores) == 0)

        if boxes.shape != (len(scores), 4):
            invalid_arrays += 1
            continue

        if not np.isfinite(boxes).all():
            invalid_arrays += 1
        if not np.isfinite(scores).all():
            invalid_arrays += 1

        if len(scores):
            score_min = min(score_min, float(scores.min()))
            score_max = max(score_max, float(scores.max()))
            box_min = min(box_min, float(boxes.min()))
            box_max = max(box_max, float(boxes.max()))

            invalid_scores += int(
                ((scores < 0) | (scores > 1)).sum()
            )

            invalid_boxes += int(
                (
                    (boxes[:, 2] < boxes[:, 0]) |
                    (boxes[:, 3] < boxes[:, 1])
                ).sum()
            )

        if tuple(dimensions) != (1280, 1024):
            dimension_mismatches += 1

    if score_min == float("inf"):
        score_min = None
        score_max = None
        box_min = None
        box_max = None

    passed = (
        not missing and not extra and
        invalid_arrays == 0 and
        invalid_scores == 0 and
        invalid_boxes == 0 and
        dimension_mismatches == 0
    )
    all_pass &= passed

    print(f"\n[{name}]")
    print("Images:", len(predictions))
    print("Missing IDs:", len(missing))
    print("Extra IDs:", len(extra))
    print("Total predicted boxes:", total_boxes)
    print("Images with zero predictions:", empty_images)
    print("Malformed/non-finite arrays:", invalid_arrays)
    print("Scores outside [0,1]:", invalid_scores)
    print("Invalid box coordinate ordering:", invalid_boxes)
    print("Unexpected image dimensions:", dimension_mismatches)
    print("Confidence range:", score_min, "to", score_max)
    print("Coordinate range:", box_min, "to", box_max)
    print("AUDIT:", "PASS" if passed else "REVIEW REQUIRED")

# Search archive for any original source files
with ZipFile(ARCHIVE, "r") as z:
    names = z.namelist()

    source_files = [
        n for n in names
        if n.lower().endswith((".py", ".ipynb"))
    ]

    fusion_related = [
        n for n in names
        if any(term in n.lower() for term in (
            "fusion", "blend", "thermal", "visible",
            "preprocess", "inference", "predict"
        ))
    ]

print("\n" + "=" * 75)
print("ARCHIVE SOURCE-CODE DISCOVERY")
print("=" * 75)
print("Python/notebook files:", len(source_files))
for n in source_files[:50]:
    print(n)

print("\nFusion/modality-related archive paths:", len(fusion_related))
for n in fusion_related[:60]:
    print(n)

print("\nOVERALL BASELINE AUDIT:", "PASS" if all_pass else "REVIEW REQUIRED")

EXPERIMENT 1 — BASELINE PREDICTION AUDIT
Manifest test IDs: 3463

[rgb]
Images: 3463
Missing IDs: 0
Extra IDs: 0
Total predicted boxes: 101060
Images with zero predictions: 0
Malformed/non-finite arrays: 0
Scores outside [0,1]: 0
Invalid box coordinate ordering: 0
Unexpected image dimensions: 0
Confidence range: 0.0010000041220337152 to 0.8510403633117676
Coordinate range: 0.0 to 1.0
AUDIT: PASS

[thermal]
Images: 3463
Missing IDs: 0
Extra IDs: 0
Total predicted boxes: 50303
Images with zero predictions: 0
Malformed/non-finite arrays: 0
Scores outside [0,1]: 0
Invalid box coordinate ordering: 0
Unexpected image dimensions: 0
Confidence range: 0.0010001151822507381 to 0.8955950736999512
Coordinate range: 0.0 to 1.0
AUDIT: PASS

[blend]
Images: 3463
Missing IDs: 0
Extra IDs: 0
Total predicted boxes: 58616
Images with zero predictions: 0
Malformed/non-finite arrays: 0
Scores outside [0,1]: 0
Invalid box coordinate ordering: 0
Unexpected image dimensions: 0
Confidence range: 0.001000001793

In [26]:
import cv2
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path
import json
import random

ROOT = Path("/kaggle/working/Experiment5/LLVIP_test_original")

VISIBLE_DIR = ROOT / "visible" / "test"
INFRARED_DIR = ROOT / "infrared" / "test"
ANNOTATION_DIR = ROOT / "Annotations"

# Load the original test manifest
manifest_path = ROOT / "split_manifest.json"
with open(manifest_path, "r") as f:
    manifest = json.load(f)

test_ids = sorted(manifest["test"])

# Deterministic sample: first, middle, last, and seven seeded samples
rng = random.Random(0)
sample_ids = sorted(set([
    test_ids[0],
    test_ids[len(test_ids) // 2],
    test_ids[-1],
    *rng.sample(test_ids, 7)
]))

print("=" * 75)
print("EXPERIMENT 5 — ORIGINAL FUSION CONSTRUCTION CHECK")
print("=" * 75)
print("Sample count:", len(sample_ids))

all_pass = True

for image_id in sample_ids:
    visible_path = VISIBLE_DIR / f"{image_id}.jpg"
    infrared_path = INFRARED_DIR / f"{image_id}.jpg"
    annotation_path = ANNOTATION_DIR / f"{image_id}.xml"

    visible = cv2.imread(str(visible_path), cv2.IMREAD_COLOR)
    infrared = cv2.imread(str(infrared_path), cv2.IMREAD_GRAYSCALE)

    if visible is None or infrared is None or not annotation_path.exists():
        print(f"FAIL {image_id}: missing/unreadable source")
        all_pass = False
        continue

    # Match the recovered Experiment 1 implementation
    if infrared.shape != visible.shape[:2]:
        infrared = cv2.resize(
            infrared,
            (visible.shape[1], visible.shape[0]),
            interpolation=cv2.INTER_LINEAR
        )

    infrared_3ch = cv2.merge([infrared, infrared, infrared])

    blend = cv2.addWeighted(
        visible, 0.5,
        infrared_3ch, 0.5,
        0
    )

    vt = cv2.merge([
        visible[:, :, 0],
        visible[:, :, 1],
        infrared
    ])

    # Verify dimensions and channel count
    checks = {
        "source_dimensions_match": visible.shape[:2] == infrared.shape,
        "blend_shape_valid": blend.shape == visible.shape,
        "vt_shape_valid": vt.shape == visible.shape,
        "blend_is_3_channel": blend.ndim == 3 and blend.shape[2] == 3,
        "vt_is_3_channel": vt.ndim == 3 and vt.shape[2] == 3,
    }

    # Verify annotation dimensions
    root = ET.parse(annotation_path).getroot()
    size = root.find("size")
    xml_width = int(size.findtext("width"))
    xml_height = int(size.findtext("height"))

    checks["annotation_dimensions_match"] = (
        xml_width == visible.shape[1] and
        xml_height == visible.shape[0]
    )

    passed = all(checks.values())
    all_pass &= passed

    print(
        f"{image_id}: {'PASS' if passed else 'FAIL'} | "
        f"visible={visible.shape} | "
        f"infrared={infrared.shape} | "
        f"blend={blend.shape} | "
        f"VT={vt.shape} | "
        f"annotation={xml_width}x{xml_height}"
    )

    if not passed:
        print("  Failed checks:", [k for k, v in checks.items() if not v])

print("\n" + "=" * 75)
print("FUSION CONSTRUCTION CHECK:", "PASS" if all_pass else "REVIEW REQUIRED")
print("Original source images and annotations were not modified.")
print("=" * 75)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/Experiment5/LLVIP_test_original/split_manifest.json'

In [27]:
import cv2
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path
import json
import random

ROOT = Path("/kaggle/working/Experiment5/LLVIP_test_original")

VISIBLE_DIR = ROOT / "visible" / "test"
INFRARED_DIR = ROOT / "infrared" / "test"
ANNOTATION_DIR = ROOT / "Annotations"

# Use the already extracted original manifest
manifest_path = Path(
    "/kaggle/working/experiment5_config_inspection/split_manifest.json"
)

assert manifest_path.exists(), f"Manifest not found: {manifest_path}"

with open(manifest_path, "r") as f:
    manifest = json.load(f)

test_ids = sorted(manifest["test"])

rng = random.Random(0)
sample_ids = sorted(set([
    test_ids[0],
    test_ids[len(test_ids) // 2],
    test_ids[-1],
    *rng.sample(test_ids, 7)
]))

print("=" * 75)
print("EXPERIMENT 5 — ORIGINAL FUSION CONSTRUCTION CHECK")
print("=" * 75)
print("Manifest test IDs:", len(test_ids))
print("Sample count:", len(sample_ids))

all_pass = True

for image_id in sample_ids:
    visible_path = VISIBLE_DIR / f"{image_id}.jpg"
    infrared_path = INFRARED_DIR / f"{image_id}.jpg"
    annotation_path = ANNOTATION_DIR / f"{image_id}.xml"

    visible = cv2.imread(str(visible_path), cv2.IMREAD_COLOR)
    infrared = cv2.imread(str(infrared_path), cv2.IMREAD_GRAYSCALE)

    if visible is None or infrared is None or not annotation_path.exists():
        print(f"FAIL {image_id}: missing/unreadable source")
        all_pass = False
        continue

    if infrared.shape != visible.shape[:2]:
        infrared = cv2.resize(
            infrared,
            (visible.shape[1], visible.shape[0]),
            interpolation=cv2.INTER_LINEAR
        )

    infrared_3ch = cv2.merge([infrared, infrared, infrared])

    blend = cv2.addWeighted(
        visible, 0.5,
        infrared_3ch, 0.5,
        0
    )

    vt = cv2.merge([
        visible[:, :, 0],
        visible[:, :, 1],
        infrared
    ])

    root = ET.parse(annotation_path).getroot()
    size = root.find("size")
    xml_width = int(size.findtext("width"))
    xml_height = int(size.findtext("height"))

    checks = {
        "source_dimensions_match": visible.shape[:2] == infrared.shape,
        "blend_shape_valid": blend.shape == visible.shape,
        "vt_shape_valid": vt.shape == visible.shape,
        "blend_is_3_channel": blend.ndim == 3 and blend.shape[2] == 3,
        "vt_is_3_channel": vt.ndim == 3 and vt.shape[2] == 3,
        "annotation_dimensions_match": (
            xml_width == visible.shape[1] and
            xml_height == visible.shape[0]
        )
    }

    passed = all(checks.values())
    all_pass &= passed

    print(
        f"{image_id}: {'PASS' if passed else 'FAIL'} | "
        f"visible={visible.shape} | infrared={infrared.shape} | "
        f"blend={blend.shape} | VT={vt.shape}"
    )

    if not passed:
        print("Failed checks:", [k for k, v in checks.items() if not v])

print("\nFUSION CONSTRUCTION CHECK:",
      "PASS" if all_pass else "REVIEW REQUIRED")
print("Original source images and annotations were not modified.")

EXPERIMENT 5 — ORIGINAL FUSION CONSTRUCTION CHECK
Manifest test IDs: 3463
Sample count: 10
190001: PASS | visible=(1024, 1280, 3) | infrared=(1024, 1280) | blend=(1024, 1280, 3) | VT=(1024, 1280, 3)
190167: PASS | visible=(1024, 1280, 3) | infrared=(1024, 1280) | blend=(1024, 1280, 3) | VT=(1024, 1280, 3)
200234: PASS | visible=(1024, 1280, 3) | infrared=(1024, 1280) | blend=(1024, 1280, 3) | VT=(1024, 1280, 3)
210432: PASS | visible=(1024, 1280, 3) | infrared=(1024, 1280) | blend=(1024, 1280, 3) | VT=(1024, 1280, 3)
220141: PASS | visible=(1024, 1280, 3) | infrared=(1024, 1280) | blend=(1024, 1280, 3) | VT=(1024, 1280, 3)
220150: PASS | visible=(1024, 1280, 3) | infrared=(1024, 1280) | blend=(1024, 1280, 3) | VT=(1024, 1280, 3)
230132: PASS | visible=(1024, 1280, 3) | infrared=(1024, 1280) | blend=(1024, 1280, 3) | VT=(1024, 1280, 3)
260175: PASS | visible=(1024, 1280, 3) | infrared=(1024, 1280) | blend=(1024, 1280, 3) | VT=(1024, 1280, 3)
260532: PASS | visible=(1024, 1280, 3) | infr

In [28]:
from pathlib import Path
import zipfile
import yaml

ARCHIVE = Path(
    "/root/.cache/huggingface/hub/datasets--SakibM19--border-surveillance-research-backups/"
    "snapshots/91d73b4e7c125467f7b3abcb671e1be0943ab530/"
    "Experiment1/experiment1_backup.zip"
)

assert ARCHIVE.exists(), "Original Experiment 1 archive not found."

with zipfile.ZipFile(ARCHIVE, "r") as z:
    files = z.namelist()

    print("=" * 75)
    print("ORIGINAL EXPERIMENT 1 — INFERENCE CONFIGURATION INSPECTION")
    print("=" * 75)

    for model in ["rgb", "thermal", "blend", "vt"]:
        matches = [
            f for f in files
            if f.endswith(f"runs/{model}/args.yaml")
        ]

        if not matches:
            print(f"\n[{model.upper()}] args.yaml NOT FOUND")
            continue

        with z.open(matches[0]) as f:
            args = yaml.safe_load(f)

        print(f"\n[{model.upper()}]")
        for key in [
            "imgsz", "conf", "iou", "max_det",
            "batch", "device", "half", "agnostic_nms"
        ]:
            print(f"{key}: {args.get(key, 'NOT RECORDED')}")

print("\nCONFIGURATION INSPECTION COMPLETE")

ORIGINAL EXPERIMENT 1 — INFERENCE CONFIGURATION INSPECTION

[RGB]
imgsz: 640
conf: None
iou: None
max_det: 300
batch: 16
device: 
half: NOT RECORDED
agnostic_nms: False

[THERMAL]
imgsz: 640
conf: None
iou: None
max_det: 300
batch: 16
device: 
half: NOT RECORDED
agnostic_nms: False

[BLEND]
imgsz: 640
conf: None
iou: None
max_det: 300
batch: 16
device: 
half: NOT RECORDED
agnostic_nms: False

[VT]
imgsz: 640
conf: None
iou: None
max_det: 300
batch: 16
device: 
half: NOT RECORDED
agnostic_nms: False

CONFIGURATION INSPECTION COMPLETE


In [29]:
from pathlib import Path
import zipfile
import re

ARCHIVE = Path(
    "/root/.cache/huggingface/hub/datasets--SakibM19--border-surveillance-research-backups/"
    "snapshots/91d73b4e7c125467f7b3abcb671e1be0943ab530/"
    "Experiment1/experiment1_backup.zip"
)

patterns = re.compile(
    r"\b(conf|confidence|iou|iou_thres|nms|predict|prediction|"
    r"max_det|agnostic_nms|weighted.box|wbf)\b",
    re.IGNORECASE
)

text_extensions = {
    ".txt", ".log", ".yaml", ".yml", ".json",
    ".csv", ".md", ".py"
}

print("=" * 75)
print("EXPERIMENT 1 — INFERENCE METADATA SEARCH")
print("=" * 75)

with zipfile.ZipFile(ARCHIVE, "r") as z:
    candidates = [
        name for name in z.namelist()
        if Path(name).suffix.lower() in text_extensions
        and not name.endswith("/")
    ]

    print("Text/log files inspected:", len(candidates))
    matches_found = 0

    for name in candidates:
        try:
            raw = z.read(name)
            if len(raw) > 5_000_000:
                continue
            content = raw.decode("utf-8", errors="ignore")
            matching_lines = [
                (i, line.strip())
                for i, line in enumerate(content.splitlines(), 1)
                if patterns.search(line)
            ]

            if matching_lines:
                matches_found += 1
                print(f"\nFILE: {name}")
                for line_no, line in matching_lines[:15]:
                    print(f"  L{line_no}: {line[:250]}")
        except Exception:
            continue

    print("\nFiles containing relevant terms:", matches_found)

print("\nMETADATA SEARCH COMPLETE")

EXPERIMENT 1 — INFERENCE METADATA SEARCH
Text/log files inspected: 15

FILE: experiment1_backup/runs/rgb/args.yaml
  L42: conf: null
  L43: iou: null
  L44: max_det: 300
  L48: nms: null
  L55: agnostic_nms: false

FILE: experiment1_backup/runs/thermal/args.yaml
  L42: conf: null
  L43: iou: null
  L44: max_det: 300
  L48: nms: null
  L55: agnostic_nms: false

FILE: experiment1_backup/runs/blend/args.yaml
  L42: conf: null
  L43: iou: null
  L44: max_det: 300
  L48: nms: null
  L55: agnostic_nms: false

FILE: experiment1_backup/runs/vt/args.yaml
  L42: conf: null
  L43: iou: null
  L44: max_det: 300
  L48: nms: null
  L55: agnostic_nms: false

Files containing relevant terms: 4

METADATA SEARCH COMPLETE


In [30]:
import inspect
import ultralytics
from ultralytics.cfg import DEFAULT_CFG

print("=" * 75)
print("EXPERIMENT 5 — INFERENCE IMPLEMENTATION CHECK")
print("=" * 75)

print("Ultralytics version:", ultralytics.__version__)

for key in [
    "imgsz",
    "conf",
    "iou",
    "max_det",
    "agnostic_nms",
    "classes",
    "half",
    "rect"
]:
    print(f"{key}: {getattr(DEFAULT_CFG, key, 'NOT FOUND')}")

from ultralytics.engine.predictor import BasePredictor
print("\nPredictor postprocess implementation:")
print(inspect.getsource(BasePredictor.postprocess))

print("\nIMPLEMENTATION CHECK COMPLETE")

ModuleNotFoundError: No module named 'ultralytics'

In [31]:
import sys
import importlib.util
import subprocess

print("=" * 75)
print("EXPERIMENT 5 — ULTRALYTICS ENVIRONMENT DIAGNOSTIC")
print("=" * 75)

print("Python executable:", sys.executable)
print("Python version:", sys.version)

spec = importlib.util.find_spec("ultralytics")
print("Ultralytics importable:", spec is not None)

result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "ultralytics"],
    capture_output=True,
    text=True
)

print("\nPIP SHOW OUTPUT:")
print(result.stdout if result.stdout else "Ultralytics package not installed.")

print("Diagnostic complete.")

EXPERIMENT 5 — ULTRALYTICS ENVIRONMENT DIAGNOSTIC
Python executable: /usr/bin/python3
Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Ultralytics importable: False

PIP SHOW OUTPUT:
Ultralytics package not installed.
Diagnostic complete.


In [32]:
import sys
import subprocess

print("Installing Ultralytics 8.3.228...")

result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "ultralytics==8.3.228"
    ],
    text=True
)

if result.returncode != 0:
    raise RuntimeError("Ultralytics installation failed.")

print("\nInstallation command completed.")

Installing Ultralytics 8.3.228...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 7.1 MB/s  0:00:00

Installation command completed.


In [33]:
import sys
import torch
import ultralytics
from pathlib import Path
from ultralytics import YOLO

BASE = Path(
    "/root/.cache/huggingface/hub/datasets--SakibM19--border-surveillance-research-backups/"
    "snapshots/91d73b4e7c125467f7b3abcb671e1be0943ab530/"
)

ARCHIVE = BASE / "Experiment1/experiment1_backup.zip"

import zipfile

CHECKPOINTS = {
    "RGB": "experiment1_backup/runs/rgb/weights/best.pt",
    "Thermal": "experiment1_backup/runs/thermal/weights/best.pt",
    "Blend": "experiment1_backup/runs/blend/weights/best.pt",
    "VT": "experiment1_backup/runs/vt/weights/best.pt",
}

print("=" * 75)
print("EXPERIMENT 5 — ENVIRONMENT AND CHECKPOINT COMPATIBILITY")
print("=" * 75)

print("Python:", sys.version.split()[0])
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

with zipfile.ZipFile(ARCHIVE, "r") as z:
    for name, member in CHECKPOINTS.items():
        destination = Path("/kaggle/working/Experiment5/checkpoints") / f"{name.lower()}_best.pt"
        destination.parent.mkdir(parents=True, exist_ok=True)

        if not destination.exists():
            with z.open(member) as src, open(destination, "wb") as dst:
                dst.write(src.read())

        print(f"\nLoading {name}: {destination.name}")
        model = YOLO(str(destination))
        print("  Task:", model.task)
        print("  Model:", model.model.__class__.__name__)
        print("  Classes:", model.names)

        del model

print("\nALL FOUR CHECKPOINT LOAD TESTS COMPLETED")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
EXPERIMENT 5 — ENVIRONMENT AND CHECKPOINT COMPATIBILITY
Python: 3.13.16
Ultralytics: 8.3.228
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

Loading RGB: rgb_best.pt
  Task: detect
  Model: DetectionModel
  Classes: {0: 'person'}

Loading Thermal: thermal_best.pt
  Task: detect
  Model: DetectionModel
  Classes: {0: 'person'}

Loading Blend: blend_best.pt
  Task: detect
  Model: DetectionModel
  Classes: {0: 'person'}

Loading VT: vt_best.pt
  Task: detect
  Model: DetectionModel
  Classes: {0: 'person'}

ALL FOUR CHECKPOINT LOAD TESTS COMPLETED


In [35]:
from pathlib import Path
from ultralytics import YOLO
import torch

ROOT = Path("/kaggle/working/Experiment5/checkpoints")

print("=" * 75)
print("EXPERIMENT 5 — CHECKPOINT ARCHITECTURE AUDIT")
print("=" * 75)

for name in ["rgb", "thermal", "blend", "vt"]:
    path = ROOT / f"{name}_best.pt"
    model = YOLO(str(path))
    net = model.model

    first_conv = None
    for module_name, module in net.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            first_conv = (module_name, module)
            break

    print(f"\n[{name.upper()}]")
    print("Task:", model.task)
    print("Class names:", model.names)
    print("Stride:", net.stride.tolist())

    if first_conv:
        module_name, conv = first_conv
        print("First convolution:", module_name)
        print("Input channels:", conv.in_channels)
        print("Output channels:", conv.out_channels)
        print("Kernel:", conv.kernel_size)

    print("Parameters:", sum(p.numel() for p in net.parameters()))

print("\nARCHITECTURE AUDIT COMPLETE")

EXPERIMENT 5 — CHECKPOINT ARCHITECTURE AUDIT

[RGB]
Task: detect
Class names: {0: 'person'}
Stride: [8.0, 16.0, 32.0]
First convolution: model.0.conv
Input channels: 3
Output channels: 16
Kernel: (3, 3)
Parameters: 2604883

[THERMAL]
Task: detect
Class names: {0: 'person'}
Stride: [8.0, 16.0, 32.0]
First convolution: model.0.conv
Input channels: 3
Output channels: 16
Kernel: (3, 3)
Parameters: 2604883

[BLEND]
Task: detect
Class names: {0: 'person'}
Stride: [8.0, 16.0, 32.0]
First convolution: model.0.conv
Input channels: 3
Output channels: 16
Kernel: (3, 3)
Parameters: 2604883

[VT]
Task: detect
Class names: {0: 'person'}
Stride: [8.0, 16.0, 32.0]
First convolution: model.0.conv
Input channels: 3
Output channels: 16
Kernel: (3, 3)
Parameters: 2604883

ARCHITECTURE AUDIT COMPLETE


In [36]:
from pathlib import Path
import cv2
import numpy as np
import random

ROOT = Path("/kaggle/working/Experiment5/LLVIP_test_original")

VISIBLE = ROOT / "visible/test"
INFRARED = ROOT / "infrared/test"

MANIFEST = Path(
    "/kaggle/working/experiment5_config_inspection/split_manifest.json"
)

import json

with open(MANIFEST, "r") as f:
    manifest = json.load(f)

ids = manifest["test"][:20]

print("=" * 75)
print("EXPERIMENT 5 — ORIGINAL INPUT PIXEL AUDIT")
print("=" * 75)

for image_id in ids:
    visible_path = VISIBLE / f"{image_id}.jpg"
    infrared_path = INFRARED / f"{image_id}.jpg"

    visible = cv2.imread(str(visible_path), cv2.IMREAD_COLOR)
    infrared = cv2.imread(str(infrared_path), cv2.IMREAD_GRAYSCALE)

    if visible is None or infrared is None:
        raise RuntimeError(f"Unreadable image pair: {image_id}")

    infrared_3ch = cv2.cvtColor(infrared, cv2.COLOR_GRAY2BGR)

    blend = cv2.addWeighted(visible, 0.5, infrared_3ch, 0.5, 0)
    vt = cv2.merge([
        visible[:, :, 0],
        visible[:, :, 1],
        infrared
    ])

    print(f"\nID: {image_id}")
    for label, image in [
        ("RGB", visible),
        ("Thermal", infrared),
        ("Blend", blend),
        ("VT", vt)
    ]:
        print(
            f"{label}: shape={image.shape}, dtype={image.dtype}, "
            f"min={image.min()}, max={image.max()}, "
            f"mean={image.mean():.3f}, std={image.std():.3f}"
        )

print("\nPIXEL AUDIT COMPLETE")
print("Only source copies were read; no images or annotations were modified.")

EXPERIMENT 5 — ORIGINAL INPUT PIXEL AUDIT

ID: 190001
RGB: shape=(1024, 1280, 3), dtype=uint8, min=0, max=255, mean=51.570, std=54.719
Thermal: shape=(1024, 1280), dtype=uint8, min=0, max=255, mean=101.144, std=51.800
Blend: shape=(1024, 1280, 3), dtype=uint8, min=0, max=247, mean=76.357, std=43.576
VT: shape=(1024, 1280, 3), dtype=uint8, min=0, max=255, mean=63.199, std=58.419

ID: 190002
RGB: shape=(1024, 1280, 3), dtype=uint8, min=0, max=255, mean=49.267, std=57.609
Thermal: shape=(1024, 1280), dtype=uint8, min=0, max=255, mean=97.724, std=48.381
Blend: shape=(1024, 1280, 3), dtype=uint8, min=0, max=254, mean=73.496, std=43.512
VT: shape=(1024, 1280, 3), dtype=uint8, min=0, max=255, mean=62.160, std=59.127

ID: 190003
RGB: shape=(1024, 1280, 3), dtype=uint8, min=0, max=255, mean=58.413, std=59.504
Thermal: shape=(1024, 1280), dtype=uint8, min=0, max=255, mean=98.987, std=52.124
Blend: shape=(1024, 1280, 3), dtype=uint8, min=0, max=246, mean=78.700, std=46.459
VT: shape=(1024, 1280, 

In [38]:
import json
from pathlib import Path

MANIFEST = Path(
    "/kaggle/working/experiment5_config_inspection/split_manifest.json"
)

with open(MANIFEST, "r") as f:
    manifest = json.load(f)

print("=" * 70)
print("SPLIT MANIFEST STRUCTURE DIAGNOSTIC")
print("=" * 70)

print("Root type:", type(manifest).__name__)

if isinstance(manifest, dict):
    print("Root keys:", list(manifest.keys()))

    for key, value in manifest.items():
        print(f"\nKey: {key}")
        print("Type:", type(value).__name__)

        if isinstance(value, (list, tuple)):
            print("Length:", len(value))
            print("First 5:", value[:5])

        elif isinstance(value, dict):
            print("Nested keys:", list(value.keys())[:15])

        else:
            print("Value:", value)

print("\nDIAGNOSTIC COMPLETE")

SPLIT MANIFEST STRUCTURE DIAGNOSTIC
Root type: dict
Root keys: ['seed', 'n_train', 'n_val', 'n_test', 'train', 'val', 'test']

Key: seed
Type: int
Value: 0

Key: n_train
Type: int
Value: 4000

Key: n_val
Type: int
Value: 500

Key: n_test
Type: int
Value: 3463

Key: train
Type: list
Length: 4000
First 5: ['180191', '150163', '010047', '050073', '120297']

Key: val
Type: list
Length: 500
First 5: ['080028', '081363', '180080', '040223', '100526']

Key: test
Type: list
Length: 3463
First 5: ['190001', '190002', '190003', '190004', '190005']

DIAGNOSTIC COMPLETE


In [39]:
import json
import cv2
import hashlib
import numpy as np
from pathlib import Path

BASE = Path("/kaggle/working/Experiment5")
SOURCE = BASE / "LLVIP_test_original"
OUTPUT = BASE / "pilot_degraded"

MANIFEST = Path(
    "/kaggle/working/experiment5_config_inspection/split_manifest.json"
)

with open(MANIFEST, "r") as f:
    manifest = json.load(f)

test_ids = manifest["test"][:100]

VISIBLE = SOURCE / "visible/test"
INFRARED = SOURCE / "infrared/test"

SEED = 20261125

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def brightness_reduction(image, factor):
    return np.clip(
        image.astype(np.float32) * factor,
        0, 255
    ).astype(np.uint8)

def gaussian_noise(image, sigma, rng):
    noise = rng.normal(0, sigma, image.shape).astype(np.float32)
    return np.clip(
        image.astype(np.float32) + noise,
        0, 255
    ).astype(np.uint8)

def gaussian_blur(image, kernel):
    return cv2.GaussianBlur(image, (kernel, kernel), 0)

def make_variants(image, rng):
    variants = {"clean": image.copy()}

    for factor in [0.75, 0.50, 0.25]:
        variants[f"brightness_{factor}"] = brightness_reduction(
            image, factor
        )

    for sigma in [5, 15, 25]:
        variants[f"noise_{sigma}"] = gaussian_noise(
            image, sigma, rng
        )

    for kernel in [3, 5, 7]:
        variants[f"blur_{kernel}"] = gaussian_blur(
            image, kernel
        )

    return variants

print("=" * 75)
print("EXPERIMENT 5 — 100-IMAGE DEGRADATION PILOT")
print("=" * 75)

assert manifest["n_test"] == 3463
assert len(manifest["test"]) == 3463
assert len(test_ids) == 100
assert len(set(test_ids)) == 100
assert not OUTPUT.exists(), (
    "Pilot output directory already exists. Stopping to protect existing evidence."
)

for image_id in test_ids:
    assert (VISIBLE / f"{image_id}.jpg").is_file()
    assert (INFRARED / f"{image_id}.jpg").is_file()

counts = {}
hash_records = []

conditions = [
    "clean",
    "brightness_0.75", "brightness_0.5", "brightness_0.25",
    "noise_5", "noise_15", "noise_25",
    "blur_3", "blur_5", "blur_7"
]

for image_id in test_ids:
    visible = cv2.imread(
        str(VISIBLE / f"{image_id}.jpg"), cv2.IMREAD_COLOR
    )
    infrared = cv2.imread(
        str(INFRARED / f"{image_id}.jpg"), cv2.IMREAD_GRAYSCALE
    )

    if visible is None or infrared is None:
        raise RuntimeError(f"Image read failure: {image_id}")

    rgb_rng = np.random.default_rng(SEED + int(image_id) * 2)
    ir_rng = np.random.default_rng(SEED + int(image_id) * 2 + 1)

    rgb_variants = make_variants(visible, rgb_rng)
    ir_variants = make_variants(infrared, ir_rng)

    for modality, variants in [
        ("RGB", rgb_variants),
        ("Thermal", ir_variants)
    ]:
        for condition, image in variants.items():
            out_dir = OUTPUT / modality / condition
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = out_dir / f"{image_id}.png"

            if not cv2.imwrite(str(out_path), image):
                raise RuntimeError(f"Write failure: {out_path}")

            counts[f"{modality}/{condition}"] = (
                counts.get(f"{modality}/{condition}", 0) + 1
            )

            hash_records.append({
                "image_id": image_id,
                "modality": modality,
                "condition": condition,
                "sha256": sha256_file(out_path)
            })

    # Reconstruct fusion from the corresponding modality variants.
    for condition in conditions:
        rgb = rgb_variants[condition]
        ir = ir_variants[condition]
        ir_3ch = cv2.cvtColor(ir, cv2.COLOR_GRAY2BGR)

        blend = cv2.addWeighted(rgb, 0.5, ir_3ch, 0.5, 0)
        vt = cv2.merge([rgb[:, :, 0], rgb[:, :, 1], ir])

        for modality, image in [("Blend", blend), ("VT", vt)]:
            out_dir = OUTPUT / modality / condition
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = out_dir / f"{image_id}.png"

            if not cv2.imwrite(str(out_path), image):
                raise RuntimeError(f"Write failure: {out_path}")

            counts[f"{modality}/{condition}"] = (
                counts.get(f"{modality}/{condition}", 0) + 1
            )

            hash_records.append({
                "image_id": image_id,
                "modality": modality,
                "condition": condition,
                "sha256": sha256_file(out_path)
            })

summary = {
    "experiment": "Experiment 5",
    "pilot_images": 100,
    "full_test_images": 3463,
    "seed": SEED,
    "conditions": conditions,
    "modalities": ["RGB", "Thermal", "Blend", "VT"],
    "counts": counts,
    "hash_record_count": len(hash_records),
    "source_images_modified": False
}

with open(OUTPUT / "pilot_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

with open(OUTPUT / "pilot_hashes.json", "w") as f:
    json.dump(hash_records, f, indent=2)

print("\nPILOT GENERATION SUMMARY")
print("Full test IDs:", len(manifest["test"]))
print("Pilot IDs:", len(test_ids))
print("Conditions:", len(conditions))
print("Modalities:", 4)
print("Generated images:", sum(counts.values()))
print("Hash records:", len(hash_records))

for key in sorted(counts):
    print(f"{key}: {counts[key]}")

print("\nPILOT GENERATION COMPLETE")

EXPERIMENT 5 — 100-IMAGE DEGRADATION PILOT

PILOT GENERATION SUMMARY
Full test IDs: 3463
Pilot IDs: 100
Conditions: 10
Modalities: 4
Generated images: 4000
Hash records: 4000
Blend/blur_3: 100
Blend/blur_5: 100
Blend/blur_7: 100
Blend/brightness_0.25: 100
Blend/brightness_0.5: 100
Blend/brightness_0.75: 100
Blend/clean: 100
Blend/noise_15: 100
Blend/noise_25: 100
Blend/noise_5: 100
RGB/blur_3: 100
RGB/blur_5: 100
RGB/blur_7: 100
RGB/brightness_0.25: 100
RGB/brightness_0.5: 100
RGB/brightness_0.75: 100
RGB/clean: 100
RGB/noise_15: 100
RGB/noise_25: 100
RGB/noise_5: 100
Thermal/blur_3: 100
Thermal/blur_5: 100
Thermal/blur_7: 100
Thermal/brightness_0.25: 100
Thermal/brightness_0.5: 100
Thermal/brightness_0.75: 100
Thermal/clean: 100
Thermal/noise_15: 100
Thermal/noise_25: 100
Thermal/noise_5: 100
VT/blur_3: 100
VT/blur_5: 100
VT/blur_7: 100
VT/brightness_0.25: 100
VT/brightness_0.5: 100
VT/brightness_0.75: 100
VT/clean: 100
VT/noise_15: 100
VT/noise_25: 100
VT/noise_5: 100

PILOT GENERATI

In [40]:
from pathlib import Path
from PIL import Image
import hashlib
import json

ROOT = Path("/kaggle/working/Experiment5")
PILOT = ROOT / "pilot_degraded"
ORIGINAL = ROOT / "LLVIP_test_original"

with open(PILOT / "pilot_hashes.json", "r") as f:
    hash_data = json.load(f)

print("Hash manifest type:", type(hash_data).__name__)
print("Pilot directory exists:", PILOT.exists())
print("Original directory exists:", ORIGINAL.exists())
print("")

# Inspect manifest structure without assuming its schema.
if isinstance(hash_data, dict):
    print("Manifest keys:", list(hash_data.keys())[:20])
elif isinstance(hash_data, list):
    print("Manifest records:", len(hash_data))
    if hash_data:
        print("First record:", hash_data[0])

print("\nDirectory image counts:")
for modality in ["RGB", "Thermal", "Blend", "VT"]:
    for condition in sorted(p.name for p in (PILOT / modality).iterdir()
                            if p.is_dir()):
        files = list((PILOT / modality / condition).glob("*.png"))
        print(f"{modality}/{condition}: {len(files)}")

print("\nClean-image checks:")
for modality, source_folder in [
    ("RGB", "visible"),
    ("Thermal", "infrared"),
]:
    clean_dir = PILOT / modality / "clean"
    source_dir = ORIGINAL / source_folder / "test"

    checked = 0
    mismatches = []

    for generated in sorted(clean_dir.glob("*.png")):
        source = source_dir / generated.name
        if not source.exists():
            mismatches.append((generated.name, "source missing"))
            continue

        generated_hash = hashlib.sha256(generated.read_bytes()).hexdigest()
        source_hash = hashlib.sha256(source.read_bytes()).hexdigest()

        if generated_hash != source_hash:
            mismatches.append((generated.name, "hash mismatch"))

        checked += 1

    print(f"{modality}: checked={checked}, mismatches={len(mismatches)}")
    if mismatches:
        print("Examples:", mismatches[:5])

print("\nImage readability and dimensions:")
for modality in ["RGB", "Thermal", "Blend", "VT"]:
    total = 0
    errors = []
    dimensions = set()

    for condition_dir in (PILOT / modality).iterdir():
        if not condition_dir.is_dir():
            continue
        for path in condition_dir.glob("*.png"):
            try:
                with Image.open(path) as im:
                    im.load()
                    dimensions.add(im.size)
                    total += 1
            except Exception as e:
                errors.append((str(path), str(e)))

    print(
        f"{modality}: readable={total}, "
        f"errors={len(errors)}, dimensions={sorted(dimensions)}"
    )
    if errors:
        print("Examples:", errors[:3])

print("\nPILOT FILE AUDIT FINISHED")

Hash manifest type: list
Pilot directory exists: True
Original directory exists: True

Manifest records: 4000
First record: {'image_id': '190001', 'modality': 'RGB', 'condition': 'clean', 'sha256': '16dbcc19320d194642f6ae69d2948a6fa2f256d72025d6221895ac3c72b5f029'}

Directory image counts:
RGB/blur_3: 100
RGB/blur_5: 100
RGB/blur_7: 100
RGB/brightness_0.25: 100
RGB/brightness_0.5: 100
RGB/brightness_0.75: 100
RGB/clean: 100
RGB/noise_15: 100
RGB/noise_25: 100
RGB/noise_5: 100
Thermal/blur_3: 100
Thermal/blur_5: 100
Thermal/blur_7: 100
Thermal/brightness_0.25: 100
Thermal/brightness_0.5: 100
Thermal/brightness_0.75: 100
Thermal/clean: 100
Thermal/noise_15: 100
Thermal/noise_25: 100
Thermal/noise_5: 100
Blend/blur_3: 100
Blend/blur_5: 100
Blend/blur_7: 100
Blend/brightness_0.25: 100
Blend/brightness_0.5: 100
Blend/brightness_0.75: 100
Blend/clean: 100
Blend/noise_15: 100
Blend/noise_25: 100
Blend/noise_5: 100
VT/blur_3: 100
VT/blur_5: 100
VT/blur_7: 100
VT/brightness_0.25: 100
VT/brightn

In [41]:
from pathlib import Path

ROOT = Path("/kaggle/working/Experiment5/LLVIP_test_original")

print("Original extraction directory:")
print(ROOT)

for path in sorted(ROOT.rglob("*")):
    if path.is_dir():
        png = len(list(path.glob("*.png")))
        jpg = len(list(path.glob("*.jpg"))) + len(list(path.glob("*.jpeg")))
        if png or jpg:
            print(f"{path} | PNG={png} | JPEG={jpg}")

Original extraction directory:
/kaggle/working/Experiment5/LLVIP_test_original
/kaggle/working/Experiment5/LLVIP_test_original/infrared/test | PNG=0 | JPEG=3463
/kaggle/working/Experiment5/LLVIP_test_original/visible/test | PNG=0 | JPEG=3463


In [42]:
from pathlib import Path
from PIL import Image
import numpy as np
import json

ROOT = Path("/kaggle/working/Experiment5")
PILOT = ROOT / "pilot_degraded"
ORIGINAL = ROOT / "LLVIP_test_original"

with open(PILOT / "pilot_hashes.json", "r") as f:
    records = json.load(f)

manifest_lookup = {
    (r["image_id"], r["modality"], r["condition"]): r["sha256"]
    for r in records
}

results = {}

for modality, source_folder in [
    ("RGB", "visible"),
    ("Thermal", "infrared")
]:
    clean_dir = PILOT / modality / "clean"
    source_dir = ORIGINAL / source_folder / "test"

    checked = 0
    exact_matches = 0
    pixel_mismatches = 0
    missing = []

    for generated in sorted(clean_dir.glob("*.png")):
        source = source_dir / f"{generated.stem}.jpg"

        if not source.exists():
            missing.append(generated.stem)
            continue

        with Image.open(generated) as a, Image.open(source) as b:
            a.load()
            b.load()

            arr_a = np.asarray(a)
            arr_b = np.asarray(b)

        checked += 1

        if arr_a.shape == arr_b.shape and np.array_equal(arr_a, arr_b):
            exact_matches += 1
        else:
            pixel_mismatches += 1

        expected_hash = manifest_lookup.get(
            (generated.stem, modality, "clean")
        )

        if expected_hash is None:
            raise AssertionError(
                f"Missing hash record: {generated.stem}/{modality}/clean"
            )

    results[modality] = {
        "checked": checked,
        "exact_pixel_matches": exact_matches,
        "pixel_mismatches": pixel_mismatches,
        "missing_sources": len(missing)
    }

    print(f"\n{modality}")
    for key, value in results[modality].items():
        print(f"{key}: {value}")

print("\nCLEAN IMAGE COMPARISON COMPLETE")


RGB
checked: 100
exact_pixel_matches: 100
pixel_mismatches: 0
missing_sources: 0

Thermal
checked: 100
exact_pixel_matches: 0
pixel_mismatches: 100
missing_sources: 0

CLEAN IMAGE COMPARISON COMPLETE


In [43]:
from pathlib import Path
from PIL import Image
import numpy as np

ROOT = Path("/kaggle/working/Experiment5")

original_dir = ROOT / "LLVIP_test_original/infrared/test"
clean_dir = ROOT / "pilot_degraded/Thermal/clean"

for image_id in ["190001", "190002", "190003"]:
    original_path = original_dir / f"{image_id}.jpg"
    clean_path = clean_dir / f"{image_id}.png"

    with Image.open(original_path) as im:
        original = np.asarray(im)

    with Image.open(clean_path) as im:
        generated = np.asarray(im)

    print(f"\nImage: {image_id}")
    print("Original shape:", original.shape)
    print("Generated shape:", generated.shape)
    print("Original dtype:", original.dtype)
    print("Generated dtype:", generated.dtype)
    print("Original range:", original.min(), original.max())
    print("Generated range:", generated.min(), generated.max())

    if original.ndim == 2 and generated.ndim == 3:
        print("Generated channels identical:",
              np.array_equal(generated[:,:,0], generated[:,:,1]) and
              np.array_equal(generated[:,:,1], generated[:,:,2]))
        print("Original equals generated channel 0:",
              np.array_equal(original, generated[:,:,0]))

    elif original.shape == generated.shape:
        print("Exact match:", np.array_equal(original, generated))

    print("Original first pixels:", original.reshape(-1)[:10])
    print("Generated first pixels:", generated.reshape(-1)[:10])


Image: 190001
Original shape: (1024, 1280, 3)
Generated shape: (1024, 1280)
Original dtype: uint8
Generated dtype: uint8
Original range: 0 255
Generated range: 0 255
Original first pixels: [70 70 70 71 71 71 71 71 71 72]
Generated first pixels: [70 71 71 72 72 71 71 70 70 70]

Image: 190002
Original shape: (1024, 1280, 3)
Generated shape: (1024, 1280)
Original dtype: uint8
Generated dtype: uint8
Original range: 0 255
Generated range: 0 255
Original first pixels: [65 65 65 65 65 65 65 65 65 65]
Generated first pixels: [65 65 65 65 65 65 65 65 65 65]

Image: 190003
Original shape: (1024, 1280, 3)
Generated shape: (1024, 1280)
Original dtype: uint8
Generated dtype: uint8
Original range: 0 255
Generated range: 0 255
Original first pixels: [66 66 66 67 67 67 68 68 68 69]
Generated first pixels: [66 67 68 69 69 69 69 68 67 67]


In [44]:
from pathlib import Path
from PIL import Image
import numpy as np
import json
import hashlib

ROOT = Path("/kaggle/working/Experiment5")
ORIGINAL = ROOT / "LLVIP_test_original/infrared/test"
CLEAN = ROOT / "pilot_degraded/Thermal/clean"

with open(ROOT / "pilot_degraded/pilot_hashes.json") as f:
    records = json.load(f)

hash_lookup = {
    (r["image_id"], r["modality"], r["condition"]): r["sha256"]
    for r in records
}

checked = 0
channel_identical = 0
intensity_matches = 0
hash_matches = 0
errors = []

for generated in sorted(CLEAN.glob("*.png")):
    image_id = generated.stem
    source = ORIGINAL / f"{image_id}.jpg"

    with Image.open(source) as im:
        original = np.asarray(im)

    with Image.open(generated) as im:
        result = np.asarray(im)

    checked += 1

    channels_ok = (
        original.ndim == 3
        and original.shape[2] == 3
        and np.array_equal(original[:, :, 0], original[:, :, 1])
        and np.array_equal(original[:, :, 1], original[:, :, 2])
    )

    if channels_ok:
        channel_identical += 1

    if channels_ok and np.array_equal(original[:, :, 0], result):
        intensity_matches += 1
    else:
        errors.append(image_id)

    actual_hash = hashlib.sha256(generated.read_bytes()).hexdigest()
    expected_hash = hash_lookup.get((image_id, "Thermal", "clean"))

    if actual_hash == expected_hash:
        hash_matches += 1

print("THERMAL CLEAN REPRESENTATION AUDIT")
print("Images checked:", checked)
print("Original RGB channels identical:", channel_identical)
print("Exact grayscale intensity matches:", intensity_matches)
print("Generated PNG hash matches manifest:", hash_matches)
print("Intensity mismatch count:", len(errors))
print("Mismatch examples:", errors[:10])

assert checked == 100
assert channel_identical == 100
assert intensity_matches == 100
assert hash_matches == 100

print("\nTHERMAL CLEAN BASELINE: PASS")

THERMAL CLEAN REPRESENTATION AUDIT
Images checked: 100
Original RGB channels identical: 100
Exact grayscale intensity matches: 100
Generated PNG hash matches manifest: 100
Intensity mismatch count: 0
Mismatch examples: []

THERMAL CLEAN BASELINE: PASS


In [45]:
from pathlib import Path
import hashlib
import json
import torch
import ultralytics
from ultralytics import YOLO

ROOT = Path("/kaggle/working/Experiment5")
CHECKPOINT_DIR = ROOT / "checkpoints"

EXPECTED = {
    "rgb": "rgb_best.pt",
    "thermal": "thermal_best.pt",
    "blend": "blend_best.pt",
    "vt": "vt_best.pt",
}

# Reconstructed evaluation protocol from the saved Experiment 1 evaluation record.
CONFIG = {
    "imgsz": 640,
    "conf": 0.001,
    "iou": 0.60,
    "max_det": 300,
    "agnostic_nms": False,
    "evaluation_iou": 0.50,
    "batch": 16,
    "half": False,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "pilot_images": 100,
    "seed": 20261125,
}

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Inference configuration:")
print(json.dumps(CONFIG, indent=2))

models = {}
checkpoint_audit = {}

for name, filename in EXPECTED.items():
    path = CHECKPOINT_DIR / filename
    assert path.is_file(), f"Missing checkpoint: {path}"

    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    model = YOLO(str(path))

    assert model.task == "detect", f"{name}: unexpected task"
    assert model.names == {0: "person"}, f"{name}: unexpected class mapping"

    first_conv = model.model.model[0].conv
    input_channels = first_conv.in_channels

    assert input_channels == 3, (
        f"{name}: expected 3-channel input, found {input_channels}"
    )

    models[name] = model
    checkpoint_audit[name] = {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": digest,
        "task": model.task,
        "classes": model.names,
        "input_channels": input_channels,
    }

    print(f"\n{name.upper()}: PASS")
    print("SHA-256:", digest)
    print("Size:", path.stat().st_size)
    print("Input channels:", input_channels)

audit = {
    "experiment": "Experiment 5 — RGB-Thermal Fusion Robustness",
    "protocol_note": (
        "Inference settings are explicitly reconstructed from the "
        "saved Experiment 1 evaluation record; they are not claimed "
        "to be recovered from the original training args.yaml files."
    ),
    "configuration": CONFIG,
    "checkpoints": checkpoint_audit,
}

output = ROOT / "inference_configuration_audit.json"
output.write_text(json.dumps(audit, indent=2))

print("\nAudit saved:", output)
print("ALL FOUR CHECKPOINTS AND CONFIGURATION: PASS")

Ultralytics: 8.3.228
PyTorch: 2.11.0+cu130
CUDA available: True
Inference configuration:
{
  "imgsz": 640,
  "conf": 0.001,
  "iou": 0.6,
  "max_det": 300,
  "agnostic_nms": false,
  "evaluation_iou": 0.5,
  "batch": 16,
  "half": false,
  "device": 0,
  "pilot_images": 100,
  "seed": 20261125
}

RGB: PASS
SHA-256: 3c115ea3572521f91180d7fada2d2d80394fe1e7e2b51434f1db9e8b7167091b
Size: 5491226
Input channels: 3

THERMAL: PASS
SHA-256: 726748ab5d96f7d9c5bd058384449c9f11db65d2e10062ef645b3433e5b5645b
Size: 5491226
Input channels: 3

BLEND: PASS
SHA-256: ca29376157c33d5ebc505bf9aba8488a734a74f77097ca26cbc5a996e70d445a
Size: 5491226
Input channels: 3

VT: PASS
SHA-256: 9655871592607a08481e05a020322be553d84f87a84556dd2a97de91a52ca42a
Size: 5491226
Input channels: 3

Audit saved: /kaggle/working/Experiment5/inference_configuration_audit.json
ALL FOUR CHECKPOINTS AND CONFIGURATION: PASS


In [46]:
from pathlib import Path
import json
import hashlib
import time
from ultralytics import YOLO

ROOT = Path("/kaggle/working/Experiment5")
PILOT = ROOT / "pilot_degraded"
OUTPUT = ROOT / "pilot_predictions"

assert not OUTPUT.exists(), (
    f"Output already exists: {OUTPUT}. Do not overwrite existing evidence."
)

# Reuse the four models loaded and verified in the previous cell.
assert "models" in globals(), "Verified models not found. Run the checkpoint audit cell again."

model_map = {
    "RGB": "rgb",
    "Thermal": "thermal",
    "Blend": "blend",
    "VT": "vt",
}

conditions = sorted(
    p.name for p in (PILOT / "RGB").iterdir() if p.is_dir()
)

reference_ids = {
    p.stem for p in (PILOT / "RGB" / "clean").glob("*.png")
}

assert len(reference_ids) == 100
assert len(conditions) == 10

CONFIG = {
    "imgsz": 640,
    "conf": 0.001,
    "iou": 0.60,
    "max_det": 300,
    "agnostic_nms": False,
    "batch": 16,
    "half": False,
    "device": 0,
    "augment": False,
}

OUTPUT.mkdir(parents=True)

run_summary = {
    "experiment": "Experiment 5 — 100-image degradation pilot",
    "configuration": CONFIG,
    "conditions": conditions,
    "modalities": {},
}

total_images = 0
start_all = time.time()

for modality, model_key in model_map.items():
    model = models[model_key]
    modality_dir = PILOT / modality
    modality_output = OUTPUT / modality
    modality_output.mkdir()

    modality_count = 0
    modality_start = time.time()

    print(f"\n{'='*65}")
    print(f"INFERENCE: {modality}")
    print(f"{'='*65}")

    for condition in conditions:
        image_dir = modality_dir / condition
        image_paths = sorted(image_dir.glob("*.png"))

        assert len(image_paths) == 100, (
            f"{modality}/{condition}: expected 100 images, "
            f"found {len(image_paths)}"
        )

        current_ids = {p.stem for p in image_paths}
        assert current_ids == reference_ids, (
            f"Image ID mismatch in {modality}/{condition}"
        )

        output_file = modality_output / f"{condition}.jsonl"
        seen_ids = set()
        count = 0

        with output_file.open("w") as f:
            results = model.predict(
                source=[str(p) for p in image_paths],
                stream=True,
                verbose=False,
                save=False,
                **CONFIG
            )

            for result in results:
                image_id = Path(result.path).stem
                assert image_id in reference_ids
                assert image_id not in seen_ids

                seen_ids.add(image_id)

                boxes = result.boxes

                if boxes is None or len(boxes) == 0:
                    xyxyn = []
                    confidences = []
                    classes = []
                else:
                    xyxyn = boxes.xyxyn.cpu().tolist()
                    confidences = boxes.conf.cpu().tolist()
                    classes = boxes.cls.cpu().int().tolist()

                record = {
                    "image_id": image_id,
                    "modality": modality,
                    "condition": condition,
                    "image_width": int(result.orig_shape[1]),
                    "image_height": int(result.orig_shape[0]),
                    "boxes_xyxy_normalized": xyxyn,
                    "confidence": confidences,
                    "class_ids": classes,
                }

                f.write(json.dumps(record) + "\n")
                count += 1

        assert count == 100
        assert seen_ids == reference_ids

        modality_count += count
        total_images += count

        print(f"{condition}: {count}/100 images processed")

    elapsed = time.time() - modality_start

    run_summary["modalities"][modality] = {
        "model": model_key,
        "images_processed": modality_count,
        "conditions": len(conditions),
        "elapsed_seconds": round(elapsed, 2),
    }

    print(f"{modality} complete: {modality_count} images")
    print(f"Elapsed: {elapsed/60:.2f} minutes")

run_summary["total_images_processed"] = total_images
run_summary["total_elapsed_seconds"] = round(
    time.time() - start_all, 2
)

summary_path = OUTPUT / "inference_run_summary.json"
summary_path.write_text(json.dumps(run_summary, indent=2))

print("\n" + "="*65)
print("PILOT INFERENCE SUMMARY")
print("="*65)
print(json.dumps(run_summary, indent=2))

assert total_images == 4000
print("\nPILOT INFERENCE COMPLETE — 4000/4000")


INFERENCE: RGB
blur_3: 100/100 images processed
blur_5: 100/100 images processed
blur_7: 100/100 images processed
brightness_0.25: 100/100 images processed
brightness_0.5: 100/100 images processed
brightness_0.75: 100/100 images processed
clean: 100/100 images processed
noise_15: 100/100 images processed
noise_25: 100/100 images processed
noise_5: 100/100 images processed
RGB complete: 1000 images
Elapsed: 1.17 minutes

INFERENCE: Thermal
blur_3: 100/100 images processed
blur_5: 100/100 images processed
blur_7: 100/100 images processed
brightness_0.25: 100/100 images processed
brightness_0.5: 100/100 images processed
brightness_0.75: 100/100 images processed
clean: 100/100 images processed
noise_15: 100/100 images processed
noise_25: 100/100 images processed
noise_5: 100/100 images processed
Thermal complete: 1000 images
Elapsed: 0.58 minutes

INFERENCE: Blend
blur_3: 100/100 images processed
blur_5: 100/100 images processed
blur_7: 100/100 images processed
brightness_0.25: 100/100 im

In [47]:
from pathlib import Path
import json
import math

OUTPUT = Path("/kaggle/working/Experiment5/pilot_predictions")

modalities = ["RGB", "Thermal", "Blend", "VT"]

conditions = [
    "blur_3", "blur_5", "blur_7",
    "brightness_0.25", "brightness_0.5", "brightness_0.75",
    "clean", "noise_15", "noise_25", "noise_5"
]

reference_ids = {
    p.stem for p in Path(
        "/kaggle/working/Experiment5/pilot_degraded/RGB/clean"
    ).glob("*.png")
}

assert len(reference_ids) == 100

total_records = 0
total_boxes = 0
total_errors = 0
file_results = []

for modality in modalities:
    for condition in conditions:
        path = OUTPUT / modality / f"{condition}.jsonl"

        assert path.is_file(), f"Missing file: {path}"

        seen = set()
        records = 0
        boxes_count = 0
        errors = []

        with path.open("r") as f:
            for line_number, line in enumerate(f, 1):
                try:
                    record = json.loads(line)

                    image_id = record["image_id"]
                    boxes = record["boxes_xyxy_normalized"]
                    confs = record["confidence"]
                    classes = record["class_ids"]

                    if image_id in seen:
                        errors.append(f"Duplicate ID: {image_id}")
                    seen.add(image_id)

                    if record["modality"] != modality:
                        errors.append(f"Wrong modality: {image_id}")

                    if record["condition"] != condition:
                        errors.append(f"Wrong condition: {image_id}")

                    if record["image_width"] != 1280:
                        errors.append(f"Wrong width: {image_id}")

                    if record["image_height"] != 1024:
                        errors.append(f"Wrong height: {image_id}")

                    if not (len(boxes) == len(confs) == len(classes)):
                        errors.append(f"Prediction length mismatch: {image_id}")

                    for box in boxes:
                        if len(box) != 4:
                            errors.append(f"Invalid box shape: {image_id}")
                            continue

                        if not all(math.isfinite(float(v)) for v in box):
                            errors.append(f"Non-finite box: {image_id}")
                            continue

                        x1, y1, x2, y2 = box

                        if not (
                            -1e-5 <= x1 <= 1.00001
                            and -1e-5 <= y1 <= 1.00001
                            and -1e-5 <= x2 <= 1.00001
                            and -1e-5 <= y2 <= 1.00001
                            and x2 >= x1
                            and y2 >= y1
                        ):
                            errors.append(f"Invalid box coordinates: {image_id}")

                    for confidence in confs:
                        if not math.isfinite(float(confidence)):
                            errors.append(f"Non-finite confidence: {image_id}")
                        elif not 0 <= confidence <= 1:
                            errors.append(f"Confidence out of range: {image_id}")

                    if any(class_id != 0 for class_id in classes):
                        errors.append(f"Unexpected class ID: {image_id}")

                    records += 1
                    boxes_count += len(boxes)

                except Exception as e:
                    errors.append(f"Line {line_number}: {str(e)}")

        missing_ids = reference_ids - seen
        extra_ids = seen - reference_ids

        if missing_ids:
            errors.append(f"Missing IDs: {len(missing_ids)}")

        if extra_ids:
            errors.append(f"Unexpected IDs: {len(extra_ids)}")

        if records != 100:
            errors.append(f"Expected 100 records, found {records}")

        total_records += records
        total_boxes += boxes_count
        total_errors += len(errors)

        result = {
            "modality": modality,
            "condition": condition,
            "records": records,
            "predicted_boxes": boxes_count,
            "errors": len(errors),
            "examples": errors[:5]
        }

        file_results.append(result)

        print(
            f"{modality}/{condition}: "
            f"records={records}, boxes={boxes_count}, "
            f"errors={len(errors)}"
        )

print("\n" + "=" * 65)
print("PREDICTION INTEGRITY AUDIT")
print("=" * 65)
print("Prediction files:", len(file_results))
print("Total records:", total_records)
print("Total predicted boxes:", total_boxes)
print("Total errors:", total_errors)

audit = {
    "prediction_files": len(file_results),
    "total_records": total_records,
    "total_predicted_boxes": total_boxes,
    "total_errors": total_errors,
    "results": file_results
}

audit_path = OUTPUT / "prediction_integrity_audit.json"
audit_path.write_text(json.dumps(audit, indent=2))

print("Audit saved:", audit_path)

assert len(file_results) == 40
assert total_records == 4000
assert total_errors == 0

print("\nPREDICTION INTEGRITY AUDIT: PASS")


RGB/blur_3: records=100, boxes=4457, errors=0
RGB/blur_5: records=100, boxes=4538, errors=0
RGB/blur_7: records=100, boxes=4716, errors=0
RGB/brightness_0.25: records=100, boxes=3592, errors=0
RGB/brightness_0.5: records=100, boxes=3670, errors=0
RGB/brightness_0.75: records=100, boxes=4057, errors=0
RGB/clean: records=100, boxes=4353, errors=0
RGB/noise_15: records=100, boxes=3467, errors=0
RGB/noise_25: records=100, boxes=3556, errors=0
RGB/noise_5: records=100, boxes=4569, errors=0
Thermal/blur_3: records=100, boxes=1025, errors=0
Thermal/blur_5: records=100, boxes=1043, errors=0
Thermal/blur_7: records=100, boxes=1059, errors=0
Thermal/brightness_0.25: records=100, boxes=1027, errors=0
Thermal/brightness_0.5: records=100, boxes=1299, errors=0
Thermal/brightness_0.75: records=100, boxes=1006, errors=0
Thermal/clean: records=100, boxes=1012, errors=0
Thermal/noise_15: records=100, boxes=831, errors=0
Thermal/noise_25: records=100, boxes=1041, errors=0
Thermal/noise_5: records=100, bo

In [48]:
from pathlib import Path
import xml.etree.ElementTree as ET
import json

ROOT = Path("/kaggle/working/Experiment5")
ORIGINAL = ROOT / "LLVIP_test_original"
PILOT = ROOT / "pilot_degraded"

pilot_ids = {
    p.stem
    for p in (PILOT / "RGB" / "clean").glob("*.png")
}

xml_files = list(ORIGINAL.rglob("*.xml"))
xml_lookup = {p.stem: p for p in xml_files}

print("Pilot image IDs:", len(pilot_ids))
print("XML files discovered:", len(xml_files))
print("Annotation directories:")
for directory in sorted({str(p.parent) for p in xml_files}):
    print(directory)

missing = sorted(pilot_ids - set(xml_lookup))
duplicates = len(xml_files) - len(xml_lookup)

total_objects = 0
valid_objects = 0
invalid_objects = []
classes = {}
dimension_errors = []
annotation_records = []

for image_id in sorted(pilot_ids):
    if image_id not in xml_lookup:
        continue

    root = ET.parse(xml_lookup[image_id]).getroot()

    size = root.find("size")
    width = int(size.findtext("width"))
    height = int(size.findtext("height"))

    objects = root.findall("object")
    image_objects = 0

    for index, obj in enumerate(objects):
        name = obj.findtext("name", default="").strip()
        classes[name] = classes.get(name, 0) + 1

        box = obj.find("bndbox")
        xmin = float(box.findtext("xmin"))
        ymin = float(box.findtext("ymin"))
        xmax = float(box.findtext("xmax"))
        ymax = float(box.findtext("ymax"))

        total_objects += 1
        image_objects += 1

        valid = (
            xmin >= 0 and ymin >= 0
            and xmax <= width and ymax <= height
            and xmax > xmin and ymax > ymin
        )

        if valid:
            valid_objects += 1
        else:
            invalid_objects.append({
                "image_id": image_id,
                "object_index": index,
                "class": name,
                "bbox": [xmin, ymin, xmax, ymax],
                "image_size": [width, height]
            })

    if width != 1280 or height != 1024:
        dimension_errors.append({
            "image_id": image_id,
            "width": width,
            "height": height
        })

    annotation_records.append({
        "image_id": image_id,
        "width": width,
        "height": height,
        "objects": image_objects
    })

print("\n" + "=" * 65)
print("PILOT GROUND-TRUTH AUDIT")
print("=" * 65)
print("Annotations matched:", len(pilot_ids) - len(missing))
print("Missing annotations:", len(missing))
print("Duplicate XML stems:", duplicates)
print("Total ground-truth objects:", total_objects)
print("Valid bounding boxes:", valid_objects)
print("Invalid bounding boxes:", len(invalid_objects))
print("Classes:", classes)
print("Image dimension errors:", len(dimension_errors))

if missing:
    print("Missing ID examples:", missing[:10])

if invalid_objects:
    print("Invalid box examples:", invalid_objects[:10])

if dimension_errors:
    print("Dimension error examples:", dimension_errors[:10])

audit = {
    "pilot_images": len(pilot_ids),
    "matched_annotations": len(pilot_ids) - len(missing),
    "missing_annotations": missing,
    "duplicate_xml_stems": duplicates,
    "total_ground_truth_objects": total_objects,
    "valid_boxes": valid_objects,
    "invalid_boxes": invalid_objects,
    "classes": classes,
    "dimension_errors": dimension_errors,
    "records": annotation_records
}

out = ROOT / "pilot_ground_truth_audit.json"
out.write_text(json.dumps(audit, indent=2))

print("\nAudit saved:", out)
print("GROUND-TRUTH AUDIT COMPLETE")

Pilot image IDs: 100
XML files discovered: 3463
Annotation directories:
/kaggle/working/Experiment5/LLVIP_test_original/Annotations

PILOT GROUND-TRUTH AUDIT
Annotations matched: 100
Missing annotations: 0
Duplicate XML stems: 0
Total ground-truth objects: 303
Valid bounding boxes: 303
Invalid bounding boxes: 0
Classes: {'person': 303}
Image dimension errors: 0

Audit saved: /kaggle/working/Experiment5/pilot_ground_truth_audit.json
GROUND-TRUTH AUDIT COMPLETE


In [49]:
from pathlib import Path
import json
import csv
import numpy as np
import xml.etree.ElementTree as ET

ROOT = Path("/kaggle/working/Experiment5")
PRED_DIR = ROOT / "pilot_predictions"
ANNOTATIONS = ROOT / "LLVIP_test_original/Annotations"
OUTPUT = ROOT / "pilot_metrics"

assert not OUTPUT.exists(), "Metrics output already exists; refusing to overwrite."
OUTPUT.mkdir(parents=True)

THRESHOLDS = {
    "RGB": 0.25,
    "Thermal": 0.40,
    "Blend": 0.35,
    "VT": 0.45
}

CONDITIONS = [
    "clean",
    "brightness_0.75", "brightness_0.5", "brightness_0.25",
    "noise_5", "noise_15", "noise_25",
    "blur_3", "blur_5", "blur_7"
]

IOU_THRESHOLD = 0.50
IMAGE_WIDTH = 1280
IMAGE_HEIGHT = 1024

# ---------------------------------------------------------
# Load and validate pilot ground truth
# ---------------------------------------------------------
gt = {}

for xml_path in ANNOTATIONS.glob("*.xml"):
    image_id = xml_path.stem

    if image_id not in {
        p.stem for p in
        (ROOT / "pilot_degraded/RGB/clean").glob("*.png")
    }:
        continue

    root = ET.parse(xml_path).getroot()
    size = root.find("size")

    width = int(size.findtext("width"))
    height = int(size.findtext("height"))

    assert width == IMAGE_WIDTH and height == IMAGE_HEIGHT

    boxes = []

    for obj in root.findall("object"):
        assert obj.findtext("name").strip() == "person"

        b = obj.find("bndbox")
        box = [
            float(b.findtext("xmin")),
            float(b.findtext("ymin")),
            float(b.findtext("xmax")),
            float(b.findtext("ymax"))
        ]

        assert box[2] > box[0] and box[3] > box[1]
        boxes.append(box)

    gt[image_id] = boxes

assert len(gt) == 100
total_gt = sum(len(v) for v in gt.values())
assert total_gt == 303

# ---------------------------------------------------------
# IoU and one-to-one matching
# ---------------------------------------------------------
def calculate_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)

    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])

    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


def match_detections(detections, confidence_floor=None):
    """
    detections: list of (image_id, confidence, pixel_box)
    Greedy one-to-one matching, globally sorted by confidence.
    """
    selected = [
        d for d in detections
        if confidence_floor is None or d[1] >= confidence_floor
    ]

    selected.sort(key=lambda d: d[1], reverse=True)

    matched = {image_id: set() for image_id in gt}
    tp = []
    fp = []

    for image_id, confidence, pred_box in selected:
        gt_boxes = gt[image_id]
        best_iou = 0.0
        best_index = None

        for index, gt_box in enumerate(gt_boxes):
            if index in matched[image_id]:
                continue

            overlap = calculate_iou(pred_box, gt_box)

            if overlap > best_iou:
                best_iou = overlap
                best_index = index

        if best_index is not None and best_iou >= IOU_THRESHOLD:
            matched[image_id].add(best_index)
            tp.append(1)
            fp.append(0)
        else:
            tp.append(0)
            fp.append(1)

    return np.asarray(tp), np.asarray(fp)


def calculate_ap101(tp, fp, number_gt):
    if len(tp) == 0 or number_gt == 0:
        return 0.0

    cumulative_tp = np.cumsum(tp)
    cumulative_fp = np.cumsum(fp)

    recall = cumulative_tp / number_gt
    precision = cumulative_tp / np.maximum(
        cumulative_tp + cumulative_fp, 1
    )

    recall_levels = np.linspace(0.0, 1.0, 101)
    interpolated = []

    for level in recall_levels:
        valid_precision = precision[recall >= level]
        interpolated.append(
            float(np.max(valid_precision))
            if len(valid_precision) else 0.0
        )

    return float(np.mean(interpolated))


# ---------------------------------------------------------
# Evaluate all modalities and conditions
# ---------------------------------------------------------
rows = []

for modality, threshold in THRESHOLDS.items():
    for condition in CONDITIONS:

        path = PRED_DIR / modality / f"{condition}.jsonl"
        assert path.is_file(), f"Missing prediction file: {path}"

        detections = []
        seen = set()

        with path.open() as f:
            for line in f:
                record = json.loads(line)
                image_id = record["image_id"]

                assert image_id in gt
                assert image_id not in seen
                seen.add(image_id)

                for box, confidence, class_id in zip(
                    record["boxes_xyxy_normalized"],
                    record["confidence"],
                    record["class_ids"]
                ):
                    assert class_id == 0

                    pixel_box = [
                        box[0] * IMAGE_WIDTH,
                        box[1] * IMAGE_HEIGHT,
                        box[2] * IMAGE_WIDTH,
                        box[3] * IMAGE_HEIGHT
                    ]

                    detections.append((
                        image_id,
                        float(confidence),
                        pixel_box
                    ))

        assert len(seen) == 100

        # AP50 using all detections above the inference floor.
        tp_all, fp_all = match_detections(detections)
        ap50 = calculate_ap101(tp_all, fp_all, total_gt)

        # Fixed validation-selected confidence threshold.
        tp_fixed, fp_fixed = match_detections(
            detections,
            confidence_floor=threshold
        )

        TP = int(tp_fixed.sum())
        FP = int(fp_fixed.sum())
        FN = total_gt - TP

        precision = TP / (TP + FP) if TP + FP else 0.0
        recall = TP / total_gt if total_gt else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        rows.append({
            "modality": modality,
            "condition": condition,
            "confidence_threshold": threshold,
            "images": len(seen),
            "ground_truth_objects": total_gt,
            "predicted_boxes": len(detections),
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "AP50_101point": ap50
        })

        print(
            f"{modality:8s} | {condition:16s} | "
            f"AP50={ap50:.4f} | F1={f1:.4f} | "
            f"P={precision:.4f} | R={recall:.4f}"
        )

# ---------------------------------------------------------
# Save CSV and JSON evidence
# ---------------------------------------------------------
csv_path = OUTPUT / "pilot_metrics.csv"

with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

json_path = OUTPUT / "pilot_metrics.json"
json_path.write_text(json.dumps({
    "experiment": "Experiment 5 — 100-image degradation pilot",
    "iou_threshold": IOU_THRESHOLD,
    "ap_method": "101-point interpolated AP50",
    "matching": "confidence-sorted, class-aware, one-to-one",
    "confidence_thresholds": THRESHOLDS,
    "pilot_images": 100,
    "ground_truth_objects": total_gt,
    "results": rows
}, indent=2))

assert len(rows) == 40

print("\n" + "=" * 65)
print("PILOT METRIC CALCULATION COMPLETE")
print("=" * 65)
print("Evaluation rows:", len(rows))
print("Ground-truth objects per condition:", total_gt)
print("CSV:", csv_path)
print("JSON:", json_path)

RGB      | clean            | AP50=0.8735 | F1=0.8298 | P=0.8311 | R=0.8284
RGB      | brightness_0.75  | AP50=0.8860 | F1=0.8542 | P=0.8780 | R=0.8317
RGB      | brightness_0.5   | AP50=0.8772 | F1=0.8475 | P=0.8711 | R=0.8251
RGB      | brightness_0.25  | AP50=0.8180 | F1=0.7149 | P=0.6471 | R=0.7987
RGB      | noise_5          | AP50=0.8848 | F1=0.8552 | P=0.8838 | R=0.8284
RGB      | noise_15         | AP50=0.8814 | F1=0.8632 | P=0.9213 | R=0.8119
RGB      | noise_25         | AP50=0.8354 | F1=0.8075 | P=0.9310 | R=0.7129
RGB      | blur_3           | AP50=0.8787 | F1=0.8276 | P=0.8235 | R=0.8317
RGB      | blur_5           | AP50=0.8796 | F1=0.8272 | P=0.8328 | R=0.8218
RGB      | blur_7           | AP50=0.8811 | F1=0.8267 | P=0.8350 | R=0.8185
Thermal  | clean            | AP50=0.9781 | F1=0.9398 | P=0.9820 | R=0.9010
Thermal  | brightness_0.75  | AP50=0.9753 | F1=0.9414 | P=0.9856 | R=0.9010
Thermal  | brightness_0.5   | AP50=0.9697 | F1=0.9434 | P=0.9821 | R=0.9076
Thermal  | b

In [50]:
from pathlib import Path
import csv
import json
import math

ROOT = Path("/kaggle/working/Experiment5/pilot_metrics")

csv_path = ROOT / "pilot_metrics.csv"
json_path = ROOT / "pilot_metrics.json"

with csv_path.open() as f:
    csv_rows = list(csv.DictReader(f))

with json_path.open() as f:
    json_data = json.load(f)

json_rows = json_data["results"]

assert len(csv_rows) == 40
assert len(json_rows) == 40

errors = []

for i, (csv_row, json_row) in enumerate(zip(csv_rows, json_rows)):
    for key in [
        "modality", "condition", "TP", "FP", "FN",
        "precision", "recall", "F1", "AP50_101point"
    ]:
        if key in ["modality", "condition"]:
            if csv_row[key] != json_row[key]:
                errors.append(f"Row {i}: {key} mismatch")
        elif key in ["TP", "FP", "FN"]:
            if int(csv_row[key]) != int(json_row[key]):
                errors.append(f"Row {i}: {key} mismatch")
        else:
            if not math.isclose(
                float(csv_row[key]),
                float(json_row[key]),
                rel_tol=1e-10,
                abs_tol=1e-10
            ):
                errors.append(f"Row {i}: {key} mismatch")

    tp = int(csv_row["TP"])
    fp = int(csv_row["FP"])
    fn = int(csv_row["FN"])

    p = tp / (tp + fp) if tp + fp else 0
    r = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * p * r / (p + r) if p + r else 0

    if tp + fn != 303:
        errors.append(f"Row {i}: GT count is not 303")

    if not math.isclose(p, float(csv_row["precision"]), abs_tol=1e-9):
        errors.append(f"Row {i}: precision arithmetic mismatch")

    if not math.isclose(r, float(csv_row["recall"]), abs_tol=1e-9):
        errors.append(f"Row {i}: recall arithmetic mismatch")

    if not math.isclose(f1, float(csv_row["F1"]), abs_tol=1e-9):
        errors.append(f"Row {i}: F1 arithmetic mismatch")

    ap = float(csv_row["AP50_101point"])
    if not 0 <= ap <= 1:
        errors.append(f"Row {i}: AP outside [0,1]")

print("CSV rows:", len(csv_rows))
print("JSON rows:", len(json_rows))
print("Ground-truth consistency: 303 objects per condition")
print("Metric arithmetic and file consistency errors:", len(errors))

if errors:
    print("\nERROR EXAMPLES:")
    for error in errors[:20]:
        print(error)
else:
    print("\nMETRIC CONSISTENCY AUDIT: PASS")

CSV rows: 40
JSON rows: 40
Ground-truth consistency: 303 objects per condition
Metric arithmetic and file consistency errors: 0

METRIC CONSISTENCY AUDIT: PASS


In [51]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download
import hashlib
import json

ROOT = Path("/kaggle/working/Experiment5")
REPO_ID = "SakibM19/border-surveillance-research-backups"

# Retrieve token securely from Kaggle Secrets.
token = UserSecretsClient().get_secret("HF_TOKEN")
assert token, "HF_TOKEN secret is empty."

api = HfApi(token=token)

# Confirm local evidence before uploading.
required = [
    ROOT / "pilot_degraded",
    ROOT / "pilot_predictions",
    ROOT / "pilot_metrics",
    ROOT / "inference_configuration_audit.json",
    ROOT / "pilot_ground_truth_audit.json",
]

for path in required:
    assert path.exists(), f"Missing evidence: {path}"

image_count = sum(
    1 for p in (ROOT / "pilot_degraded").rglob("*.png")
)
prediction_count = sum(
    1 for p in (ROOT / "pilot_predictions").glob("*/*.jsonl")
)

assert image_count == 4000, image_count
assert prediction_count == 40, prediction_count

print("Local evidence preflight: PASS")
print("Generated images:", image_count)
print("Prediction files:", prediction_count)

# Upload only Experiment 5 pilot evidence.
allow_patterns = [
    "pilot_degraded/**",
    "pilot_predictions/**",
    "pilot_metrics/**",
    "inference_configuration_audit.json",
    "pilot_ground_truth_audit.json",
]

result = api.upload_folder(
    repo_id=REPO_ID,
    repo_type="dataset",
    folder_path=str(ROOT),
    path_in_repo="Experiment5/Pilot100",
    allow_patterns=allow_patterns,
    commit_message="Back up Experiment 5 100-image degradation pilot evidence"
)

print("\nUpload result:", result)

# Verify remote file listing.
remote_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

prefix = "Experiment5/Pilot100/"
pilot_remote = [p for p in remote_files if p.startswith(prefix)]

print("\nRemote pilot files:", len(pilot_remote))

required_remote = [
    "Experiment5/Pilot100/inference_configuration_audit.json",
    "Experiment5/Pilot100/pilot_ground_truth_audit.json",
    "Experiment5/Pilot100/pilot_metrics/pilot_metrics.csv",
    "Experiment5/Pilot100/pilot_metrics/pilot_metrics.json",
    "Experiment5/Pilot100/pilot_predictions/inference_run_summary.json",
    "Experiment5/Pilot100/pilot_predictions/prediction_integrity_audit.json",
    "Experiment5/Pilot100/pilot_degraded/pilot_hashes.json",
    "Experiment5/Pilot100/pilot_degraded/pilot_summary.json",
]

missing_remote = [p for p in required_remote if p not in pilot_remote]

assert not missing_remote, f"Missing remote files: {missing_remote}"

# Download key artifacts and verify SHA-256.
verification_files = [
    "pilot_metrics/pilot_metrics.csv",
    "pilot_metrics/pilot_metrics.json",
    "pilot_predictions/inference_run_summary.json",
    "pilot_degraded/pilot_hashes.json",
]

for relative in verification_files:
    local_path = ROOT / relative
    remote_path = f"Experiment5/Pilot100/{relative}"

    downloaded = Path(hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=remote_path,
        token=token
    ))

    local_hash = hashlib.sha256(local_path.read_bytes()).hexdigest()
    remote_hash = hashlib.sha256(downloaded.read_bytes()).hexdigest()

    assert local_hash == remote_hash, f"SHA-256 mismatch: {relative}"
    print(f"SHA-256 PASS: {relative}")

print("\n" + "=" * 65)
print("EXPERIMENT 5 PILOT BACKUP VERIFICATION: PASS")
print("Repository:", REPO_ID)
print("Remote files:", len(pilot_remote))
print("Images backed up:", image_count)
print("Prediction files backed up:", prediction_count)
print("=" * 65)

Local evidence preflight: PASS
Generated images: 4000
Prediction files: 40

Upload result: https://huggingface.co/datasets/SakibM19/border-surveillance-research-backups/commit/682f867defd4b29c925ba5fb726c8a86f30d3d3d

Remote pilot files: 4048


pilot_metrics.csv:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

SHA-256 PASS: pilot_metrics/pilot_metrics.csv


pilot_metrics.json:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

SHA-256 PASS: pilot_metrics/pilot_metrics.json


inference_run_summary.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

SHA-256 PASS: pilot_predictions/inference_run_summary.json


pilot_hashes.json:   0%|          | 0.00/681k [00:00<?, ?B/s]

SHA-256 PASS: pilot_degraded/pilot_hashes.json

EXPERIMENT 5 PILOT BACKUP VERIFICATION: PASS
Repository: SakibM19/border-surveillance-research-backups
Remote files: 4048
Images backed up: 4000
Prediction files backed up: 40


In [52]:
from pathlib import Path
import json

ROOT = Path("/kaggle/working/Experiment5")

print("=" * 70)
print("EXPERIMENT 5 — BOOTSTRAP INPUT STRUCTURE INSPECTION")
print("=" * 70)

# Inspect one prediction file and one record.
prediction_dir = ROOT / "pilot_predictions"
prediction_files = sorted(prediction_dir.glob("*/*.jsonl"))

print("\nPrediction files:", len(prediction_files))

for path in prediction_files[:2]:
    print("\nFile:", path.relative_to(ROOT))
    with path.open("r") as f:
        record = json.loads(next(f))

    print("Record keys:", list(record.keys()))
    for key, value in record.items():
        if isinstance(value, list):
            print(f"  {key}: list, length={len(value)}")
            if value:
                print("    first item:", value[0])
        else:
            print(f"  {key}: {value}")

# Inspect metrics JSON.
metrics_path = ROOT / "pilot_metrics" / "pilot_metrics.json"
metrics = json.loads(metrics_path.read_text())

print("\nMetrics JSON type:", type(metrics).__name__)

if isinstance(metrics, dict):
    print("Top-level keys:", list(metrics.keys()))
    for key, value in metrics.items():
        if isinstance(value, list):
            print(f"  {key}: list, length={len(value)}")
            if value:
                print("    first item:", value[0])
        elif isinstance(value, dict):
            print(f"  {key}: dict, keys={list(value.keys())[:15]}")
        else:
            print(f"  {key}: {value}")

elif isinstance(metrics, list):
    print("Rows:", len(metrics))
    if metrics:
        print("First row:", metrics[0])

# Confirm ground-truth location.
gt_dir = ROOT / "LLVIP_test_original" / "Annotations"
print("\nGround-truth directory:", gt_dir)
print("Exists:", gt_dir.exists())
if gt_dir.exists():
    print("XML files:", len(list(gt_dir.glob("*.xml"))))

print("\nINSPECTION COMPLETE")

EXPERIMENT 5 — BOOTSTRAP INPUT STRUCTURE INSPECTION

Prediction files: 40

File: pilot_predictions/Blend/blur_3.jsonl
Record keys: ['image_id', 'modality', 'condition', 'image_width', 'image_height', 'boxes_xyxy_normalized', 'confidence', 'class_ids']
  image_id: 190001
  modality: Blend
  condition: blur_3
  image_width: 1280
  image_height: 1024
  boxes_xyxy_normalized: list, length=23
    first item: [0.7952168583869934, 0.3190682530403137, 0.8963736891746521, 0.5660580992698669]
  confidence: list, length=23
    first item: 0.7878214120864868
  class_ids: list, length=23
    first item: 0

File: pilot_predictions/Blend/blur_5.jsonl
Record keys: ['image_id', 'modality', 'condition', 'image_width', 'image_height', 'boxes_xyxy_normalized', 'confidence', 'class_ids']
  image_id: 190001
  modality: Blend
  condition: blur_5
  image_width: 1280
  image_height: 1024
  boxes_xyxy_normalized: list, length=26
    first item: [0.794801652431488, 0.31871163845062256, 0.8968411684036255, 0.5677

In [53]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET
from collections import Counter

ROOT = Path("/kaggle/working/Experiment5")
PRED_ROOT = ROOT / "pilot_predictions"
GT_ROOT = ROOT / "LLVIP_test_original" / "Annotations"
OUT = ROOT / "pilot_metrics" / "bootstrap_inputs.jsonl"

assert GT_ROOT.exists(), f"Missing annotations: {GT_ROOT}"

# Index ground-truth XML files by image ID.
gt_files = {p.stem: p for p in GT_ROOT.glob("*.xml")}
assert len(gt_files) == 3463, f"Unexpected XML count: {len(gt_files)}"

def load_ground_truth(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")
    width = int(size.findtext("width"))
    height = int(size.findtext("height"))

    boxes = []
    for obj in root.findall("object"):
        name = obj.findtext("name")
        assert name == "person", f"Unexpected class: {name}"

        box = obj.find("bndbox")
        xmin = float(box.findtext("xmin"))
        ymin = float(box.findtext("ymin"))
        xmax = float(box.findtext("xmax"))
        ymax = float(box.findtext("ymax"))

        assert 0 <= xmin < xmax <= width
        assert 0 <= ymin < ymax <= height

        boxes.append([
            xmin / width,
            ymin / height,
            xmax / width,
            ymax / height
        ])

    return width, height, boxes

# Cache ground truth for the pilot IDs.
gt_cache = {}
for image_id in sorted(gt_files):
    if image_id in {
        str(i) for i in []
    }:
        pass

# Read the 40 prediction files and join each image to its XML.
prediction_files = sorted(PRED_ROOT.glob("*/*.jsonl"))
assert len(prediction_files) == 40, len(prediction_files)

records = []
seen = set()
combination_counts = Counter()
image_ids_by_condition = {}

for pred_file in prediction_files:
    with pred_file.open("r", encoding="utf-8") as f:
        for line in f:
            pred = json.loads(line)

            image_id = str(pred["image_id"])
            modality = pred["modality"]
            condition = pred["condition"]
            key = (modality, condition, image_id)

            assert key not in seen, f"Duplicate prediction record: {key}"
            seen.add(key)

            assert image_id in gt_files, f"Missing XML: {image_id}"

            width, height, gt_boxes = load_ground_truth(gt_files[image_id])

            assert width == pred["image_width"] == 1280
            assert height == pred["image_height"] == 1024

            boxes = pred["boxes_xyxy_normalized"]
            confidences = pred["confidence"]
            class_ids = pred["class_ids"]

            assert len(boxes) == len(confidences) == len(class_ids)
            assert all(c == 0 for c in class_ids)

            for box in boxes:
                assert len(box) == 4
                assert all(0 <= float(v) <= 1 for v in box)
                assert box[0] <= box[2] and box[1] <= box[3]

            assert all(
                0 <= float(c) <= 1 for c in confidences
            )

            record = {
                "image_id": image_id,
                "modality": modality,
                "condition": condition,
                "image_width": width,
                "image_height": height,
                "gt_boxes_xyxy_normalized": gt_boxes,
                "pred_boxes_xyxy_normalized": boxes,
                "confidence": confidences,
                "class_ids": class_ids
            }

            records.append(record)
            combination_counts[(modality, condition)] += 1
            image_ids_by_condition.setdefault(
                (modality, condition), set()
            ).add(image_id)

# Validate all 40 modality-condition combinations.
assert len(records) == 4000, len(records)
assert len(combination_counts) == 40, len(combination_counts)
assert all(n == 100 for n in combination_counts.values())

reference_ids = None
for key, ids in image_ids_by_condition.items():
    assert len(ids) == 100
    if reference_ids is None:
        reference_ids = ids
    else:
        assert ids == reference_ids, (
            f"Image pairing mismatch in {key}"
        )

# Write deterministic image-level bootstrap inputs.
OUT.parent.mkdir(parents=True, exist_ok=True)

records.sort(key=lambda r: (
    r["modality"], r["condition"], r["image_id"]
))

with OUT.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, separators=(",", ":")) + "\n")

# Re-read and verify the generated file.
with OUT.open("r", encoding="utf-8") as f:
    verified = [json.loads(line) for line in f]

assert len(verified) == 4000
assert sum(
    len(r["gt_boxes_xyxy_normalized"]) for r in verified
    if r["modality"] == "RGB" and r["condition"] == "clean"
) == 303

print("=" * 70)
print("EXPERIMENT 5 — BOOTSTRAP INPUT BUILD AUDIT")
print("=" * 70)
print("Prediction records:", len(records))
print("Unique image IDs:", len(reference_ids))
print("Modality-condition combinations:", len(combination_counts))
print("Records per combination:", sorted(set(combination_counts.values())))
print("Paired image IDs across all combinations: PASS")
print("Ground-truth and prediction structure: PASS")
print("Ground-truth objects in RGB clean:", 303)
print("Output:", OUT)
print("Output size (MB):", round(OUT.stat().st_size / (1024**2), 2))
print("\nBOOTSTRAP INPUT BUILD: PASS")

EXPERIMENT 5 — BOOTSTRAP INPUT BUILD AUDIT
Prediction records: 4000
Unique image IDs: 100
Modality-condition combinations: 40
Records per combination: [100]
Paired image IDs across all combinations: PASS
Ground-truth and prediction structure: PASS
Ground-truth objects in RGB clean: 303
Output: /kaggle/working/Experiment5/pilot_metrics/bootstrap_inputs.jsonl
Output size (MB): 9.52

BOOTSTRAP INPUT BUILD: PASS


In [54]:
from pathlib import Path
import json
import csv
import numpy as np
from collections import defaultdict

ROOT = Path("/kaggle/working/Experiment5")
INPUT = ROOT / "pilot_metrics" / "bootstrap_inputs.jsonl"
METRICS_JSON = ROOT / "pilot_metrics" / "pilot_metrics.json"
OUT_DIR = ROOT / "pilot_metrics" / "bootstrap"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_BOOT = 2000
SEED = 20261125
IOU_THRESHOLD = 0.50

# Load fixed validation-selected thresholds.
with METRICS_JSON.open("r", encoding="utf-8") as f:
    metric_config = json.load(f)

thresholds = metric_config["confidence_thresholds"]

# Load paired image-level records.
records = []
with INPUT.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

image_ids = sorted({r["image_id"] for r in records})
modalities = ["RGB", "Thermal", "Blend", "VT"]

conditions = sorted({r["condition"] for r in records})
assert "clean" in conditions
assert len(image_ids) == 100

record_index = {
    (r["modality"], r["condition"], r["image_id"]): r
    for r in records
}

assert len(record_index) == 4000

def calculate_iou(box, boxes):
    if len(boxes) == 0:
        return np.zeros(0, dtype=np.float64)

    boxes = np.asarray(boxes, dtype=np.float64)
    box = np.asarray(box, dtype=np.float64)

    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])

    intersection = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)

    area_a = max(0, box[2]-box[0]) * max(0, box[3]-box[1])
    area_b = (
        np.maximum(0, boxes[:, 2]-boxes[:, 0]) *
        np.maximum(0, boxes[:, 3]-boxes[:, 1])
    )

    union = area_a + area_b - intersection
    return np.divide(
        intersection,
        union,
        out=np.zeros_like(intersection),
        where=union > 0
    )

def image_level_counts(record, confidence_threshold):
    gt = record["gt_boxes_xyxy_normalized"]
    pred_boxes = record["pred_boxes_xyxy_normalized"]
    pred_conf = record["confidence"]
    pred_classes = record["class_ids"]

    # Apply the frozen modality-specific confidence threshold.
    selected = [
        i for i, conf in enumerate(pred_conf)
        if float(conf) >= confidence_threshold
    ]

    selected.sort(key=lambda i: float(pred_conf[i]), reverse=True)

    matched_gt = set()
    tp = 0
    fp = 0

    for pred_i in selected:
        if int(pred_classes[pred_i]) != 0:
            fp += 1
            continue

        ious = calculate_iou(pred_boxes[pred_i], gt)

        if len(ious) == 0:
            fp += 1
            continue

        # Greedy one-to-one matching: highest-IoU unmatched GT.
        candidate_order = np.argsort(-ious)
        match = None

        for gt_i in candidate_order:
            if ious[gt_i] >= IOU_THRESHOLD and int(gt_i) not in matched_gt:
                match = int(gt_i)
                break

        if match is not None:
            matched_gt.add(match)
            tp += 1
        else:
            fp += 1

    fn = len(gt) - tp
    return tp, fp, fn

# Calculate per-image TP/FP/FN once for every combination.
counts = {}

for modality in modalities:
    threshold = float(thresholds[modality])

    for condition in conditions:
        rows = []

        for image_id in image_ids:
            record = record_index[(modality, condition, image_id)]
            rows.append(
                image_level_counts(record, threshold)
            )

        counts[(modality, condition)] = np.asarray(
            rows, dtype=np.int64
        )

# Verify aggregate counts against the existing pilot metrics.
existing = {
    (r["modality"], r["condition"]): r
    for r in metric_config["results"]
}

for key, values in counts.items():
    tp, fp, fn = values.sum(axis=0)
    old = existing[key]

    assert int(tp) == int(old["TP"]), (key, tp, old["TP"])
    assert int(fp) == int(old["FP"]), (key, fp, old["FP"])
    assert int(fn) == int(old["FN"]), (key, fn, old["FN"])

print("Per-image matching and aggregate reconciliation: PASS")

# Generate identical bootstrap image samples for all conditions.
rng = np.random.default_rng(SEED)
bootstrap_indices = rng.integers(
    0, len(image_ids), size=(N_BOOT, len(image_ids))
)

def metrics_from_counts(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    return precision, recall, f1

def bootstrap_metrics(image_counts):
    samples = image_counts[bootstrap_indices].sum(axis=1)

    output = np.zeros((N_BOOT, 3), dtype=np.float64)

    for i, (tp, fp, fn) in enumerate(samples):
        output[i] = metrics_from_counts(
            int(tp), int(fp), int(fn)
        )

    return output

def percentile_ci(values):
    return (
        float(np.percentile(values, 2.5)),
        float(np.percentile(values, 97.5))
    )

bootstrap_results = {}
summary_rows = []

for modality in modalities:
    for condition in conditions:
        key = (modality, condition)
        samples = bootstrap_metrics(counts[key])
        bootstrap_results[key] = samples

        tp, fp, fn = counts[key].sum(axis=0)
        point = metrics_from_counts(
            int(tp), int(fp), int(fn)
        )

        for metric_i, metric_name in enumerate(
            ["precision", "recall", "F1"]
        ):
            low, high = percentile_ci(samples[:, metric_i])

            summary_rows.append({
                "modality": modality,
                "condition": condition,
                "metric": metric_name,
                "point_estimate": point[metric_i],
                "CI95_lower": low,
                "CI95_upper": high,
                "bootstrap_replicates": N_BOOT,
                "seed": SEED
            })

# Paired bootstrap differences versus each modality's clean baseline.
delta_rows = []

for modality in modalities:
    clean_samples = bootstrap_results[(modality, "clean")]

    for condition in conditions:
        if condition == "clean":
            continue

        condition_samples = bootstrap_results[(modality, condition)]
        paired_delta = condition_samples - clean_samples

        for metric_i, metric_name in enumerate(
            ["precision", "recall", "F1"]
        ):
            low, high = percentile_ci(paired_delta[:, metric_i])

            delta_rows.append({
                "modality": modality,
                "condition": condition,
                "comparison": "condition_minus_clean",
                "metric": metric_name,
                "mean_bootstrap_difference": float(
                    np.mean(paired_delta[:, metric_i])
                ),
                "CI95_lower": low,
                "CI95_upper": high,
                "bootstrap_replicates": N_BOOT,
                "seed": SEED
            })

# Save outputs.
summary_csv = OUT_DIR / "bootstrap_metric_confidence_intervals.csv"
delta_csv = OUT_DIR / "bootstrap_paired_differences.csv"
summary_json = OUT_DIR / "bootstrap_metric_confidence_intervals.json"
delta_json = OUT_DIR / "bootstrap_paired_differences.json"

for path, rows in [
    (summary_csv, summary_rows),
    (delta_csv, delta_rows)
]:
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

for path, rows in [
    (summary_json, summary_rows),
    (delta_json, delta_rows)
]:
    with path.open("w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)

# Final output checks.
assert len(summary_rows) == 40 * 3
assert len(delta_rows) == 36 * 3

for row in summary_rows + delta_rows:
    assert row["CI95_lower"] <= row["CI95_upper"]

print("\n" + "=" * 70)
print("EXPERIMENT 5 — PAIRED BOOTSTRAP RESULTS")
print("=" * 70)
print("Images:", len(image_ids))
print("Conditions:", len(conditions))
print("Modalities:", len(modalities))
print("Bootstrap resamples:", N_BOOT)
print("Seed:", SEED)
print("Metric confidence intervals:", len(summary_rows))
print("Paired condition-clean comparisons:", len(delta_rows))

print("\nF1 confidence intervals:")
for row in summary_rows:
    if row["metric"] == "F1":
        print(
            f'{row["modality"]:8s} {row["condition"]:16s} '
            f'F1={row["point_estimate"]:.4f} '
            f'95% CI [{row["CI95_lower"]:.4f}, '
            f'{row["CI95_upper"]:.4f}]'
        )

print("\nSaved:")
print(summary_csv)
print(delta_csv)
print(summary_json)
print(delta_json)

print("\nPAIRED BOOTSTRAP ANALYSIS: PASS")

Per-image matching and aggregate reconciliation: PASS

EXPERIMENT 5 — PAIRED BOOTSTRAP RESULTS
Images: 100
Conditions: 10
Modalities: 4
Bootstrap resamples: 2000
Seed: 20261125
Metric confidence intervals: 120
Paired condition-clean comparisons: 108

F1 confidence intervals:
RGB      blur_3           F1=0.8276 95% CI [0.7941, 0.8605]
RGB      blur_5           F1=0.8272 95% CI [0.7943, 0.8599]
RGB      blur_7           F1=0.8267 95% CI [0.7928, 0.8586]
RGB      brightness_0.25  F1=0.7149 95% CI [0.6803, 0.7479]
RGB      brightness_0.5   F1=0.8475 95% CI [0.8131, 0.8793]
RGB      brightness_0.75  F1=0.8542 95% CI [0.8215, 0.8868]
RGB      clean            F1=0.8298 95% CI [0.7955, 0.8642]
RGB      noise_15         F1=0.8632 95% CI [0.8297, 0.8951]
RGB      noise_25         F1=0.8075 95% CI [0.7681, 0.8465]
RGB      noise_5          F1=0.8552 95% CI [0.8201, 0.8889]
Thermal  blur_3           F1=0.9381 95% CI [0.9128, 0.9602]
Thermal  blur_5           F1=0.9398 95% CI [0.9144, 0.9623]
Ther

In [55]:
from pathlib import Path
import csv
import json

ROOT = Path("/kaggle/working/Experiment5/pilot_metrics")

delta_path = ROOT / "bootstrap" / "bootstrap_paired_differences.csv"
metrics_path = ROOT / "pilot_metrics.json"

with metrics_path.open("r", encoding="utf-8") as f:
    metrics = json.load(f)

with delta_path.open("r", encoding="utf-8") as f:
    delta_rows = list(csv.DictReader(f))

baseline = {
    (r["modality"], r["condition"]): float(r["F1"])
    for r in metrics["results"]
}

f1_rows = [
    r for r in delta_rows
    if r["metric"] == "F1"
]

assert len(f1_rows) == 36, len(f1_rows)

print("=" * 72)
print("EXPERIMENT 5 — PAIRED F1 DIFFERENCE ANALYSIS")
print("=" * 72)

print(
    f'{"Modality":8s} {"Condition":16s} '
    f'{"Point Δ":>9s} {"95% CI":>25s} {"Conclusion"}'
)

summary = {}

for r in sorted(f1_rows, key=lambda x: (x["modality"], x["condition"])):
    modality = r["modality"]
    condition = r["condition"]

    point_delta = (
        baseline[(modality, condition)]
        - baseline[(modality, "clean")]
    )

    low = float(r["CI95_lower"])
    high = float(r["CI95_upper"])

    if low > 0:
        conclusion = "Above clean"
    elif high < 0:
        conclusion = "Below clean"
    else:
        conclusion = "Includes zero"

    summary.setdefault(modality, Counter() if False else {
        "above": 0, "below": 0, "includes_zero": 0
    })

    if low > 0:
        summary[modality]["above"] += 1
    elif high < 0:
        summary[modality]["below"] += 1
    else:
        summary[modality]["includes_zero"] += 1

    print(
        f"{modality:8s} {condition:16s} "
        f"{point_delta:+.4f} "
        f"[{low:+.4f}, {high:+.4f}] "
        f"{conclusion}"
    )

print("\n" + "-" * 72)
print("EXPLORATORY COMPARISON COUNTS")
for modality, counts in summary.items():
    print(
        f'{modality:8s} '
        f'Below clean: {counts["below"]:2d} | '
        f'Above clean: {counts["above"]:2d} | '
        f'Includes zero: {counts["includes_zero"]:2d}'
    )

print("\nTotal F1 paired comparisons:", len(f1_rows))
print("Confidence intervals excluding zero:",
      sum(
          float(r["CI95_lower"]) > 0 or float(r["CI95_upper"]) < 0
          for r in f1_rows
      ))

print("\nPAIRED DIFFERENCE INSPECTION: COMPLETE")


EXPERIMENT 5 — PAIRED F1 DIFFERENCE ANALYSIS
Modality Condition          Point Δ                    95% CI Conclusion
Blend    blur_3           +0.0000 [-0.0056, +0.0052] Includes zero
Blend    blur_5           -0.0021 [-0.0129, +0.0088] Includes zero
Blend    blur_7           -0.0056 [-0.0183, +0.0071] Includes zero
Blend    brightness_0.25  -0.0435 [-0.0672, -0.0200] Below clean
Blend    brightness_0.5   -0.0118 [-0.0298, +0.0071] Includes zero
Blend    brightness_0.75  -0.0021 [-0.0137, +0.0086] Includes zero
Blend    noise_15         +0.0048 [-0.0094, +0.0211] Includes zero
Blend    noise_25         -0.0210 [-0.0424, +0.0005] Includes zero
Blend    noise_5          -0.0016 [-0.0148, +0.0101] Includes zero
RGB      blur_3           -0.0022 [-0.0156, +0.0107] Includes zero
RGB      blur_5           -0.0025 [-0.0179, +0.0125] Includes zero
RGB      blur_7           -0.0031 [-0.0243, +0.0161] Includes zero
RGB      brightness_0.25  -0.1148 [-0.1486, -0.0817] Below clean
RGB      bright

In [56]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download
import hashlib
import json

ROOT = Path("/kaggle/working/Experiment5")
METRICS = ROOT / "pilot_metrics"
BOOT = METRICS / "bootstrap"

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)

files = [
    METRICS / "bootstrap_inputs.jsonl",
    BOOT / "bootstrap_metric_confidence_intervals.csv",
    BOOT / "bootstrap_metric_confidence_intervals.json",
    BOOT / "bootstrap_paired_differences.csv",
    BOOT / "bootstrap_paired_differences.json",
]

for path in files:
    assert path.exists(), f"Missing: {path}"

metadata = {
    "experiment": "Experiment 5 — 100-image RGB-Thermal degradation pilot",
    "analysis": "Paired image-level bootstrap",
    "images": 100,
    "modalities": ["RGB", "Thermal", "Blend", "VT"],
    "conditions": 10,
    "bootstrap_resamples": 2000,
    "random_seed": 20261125,
    "iou_threshold": 0.50,
    "confidence_thresholds": {
        "source": "Validation-selected thresholds from pilot metrics JSON",
        "frozen": True
    },
    "sampling_unit": "Image ID",
    "paired_sampling": True,
    "interval": "Percentile 95% CI (2.5th and 97.5th percentiles)",
    "metrics": ["Precision", "Recall", "F1"],
    "comparisons": "Each degraded condition minus its modality's clean condition",
    "multiple_comparison_correction": "Not applied; comparisons are exploratory",
    "scope_limitations": [
        "100-image pilot only",
        "Degradation applied jointly to RGB and thermal source modalities",
        "Not a single-sensor failure evaluation",
        "Not a full-test-set evaluation"
    ]
}

metadata_path = BOOT / "bootstrap_methodology.json"
metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)
files.append(metadata_path)

# Upload the new bootstrap evidence.
for path in files:
    relative = path.relative_to(ROOT).as_posix()
    remote = f"Experiment5/Pilot100/{relative}"

    api.upload_file(
        path_or_fileobj=str(path),
        path_in_repo=remote,
        repo_id="SakibM19/border-surveillance-research-backups",
        repo_type="dataset",
        commit_message="Back up Experiment 5 paired bootstrap evidence"
    )
    print("Uploaded:", remote)

# Verify remote files and hashes.
remote_files = set(api.list_repo_files(
    repo_id="SakibM19/border-surveillance-research-backups",
    repo_type="dataset"
))

for path in files:
    relative = path.relative_to(ROOT).as_posix()
    remote = f"Experiment5/Pilot100/{relative}"

    assert remote in remote_files, f"Missing remote file: {remote}"

    downloaded = Path(hf_hub_download(
        repo_id="SakibM19/border-surveillance-research-backups",
        repo_type="dataset",
        filename=remote,
        token=token
    ))

    local_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    remote_hash = hashlib.sha256(downloaded.read_bytes()).hexdigest()

    assert local_hash == remote_hash, f"Hash mismatch: {relative}"
    print("SHA-256 PASS:", relative)

print("\nBOOTSTRAP EVIDENCE BACKUP: PASS")


Uploaded: Experiment5/Pilot100/pilot_metrics/bootstrap_inputs.jsonl
Uploaded: Experiment5/Pilot100/pilot_metrics/bootstrap/bootstrap_metric_confidence_intervals.csv
Uploaded: Experiment5/Pilot100/pilot_metrics/bootstrap/bootstrap_metric_confidence_intervals.json
Uploaded: Experiment5/Pilot100/pilot_metrics/bootstrap/bootstrap_paired_differences.csv
Uploaded: Experiment5/Pilot100/pilot_metrics/bootstrap/bootstrap_paired_differences.json
Uploaded: Experiment5/Pilot100/pilot_metrics/bootstrap/bootstrap_methodology.json


bootstrap_inputs.jsonl:   0%|          | 0.00/9.99M [00:00<?, ?B/s]

SHA-256 PASS: pilot_metrics/bootstrap_inputs.jsonl


(…)ootstrap_metric_confidence_intervals.csv:   0%|          | 0.00/11.1k [00:00<?, ?B/s]

SHA-256 PASS: pilot_metrics/bootstrap/bootstrap_metric_confidence_intervals.csv


(…)otstrap_metric_confidence_intervals.json:   0%|          | 0.00/31.0k [00:00<?, ?B/s]

... (3 outputs left out)

bootstrap_paired_differences.json:   0%|          | 0.00/33.9k [00:00<?, ?B/s]

SHA-256 PASS: pilot_metrics/bootstrap/bootstrap_paired_differences.json


bootstrap_methodology.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

SHA-256 PASS: pilot_metrics/bootstrap/bootstrap_methodology.json

BOOTSTRAP EVIDENCE BACKUP: PASS


In [57]:
from pathlib import Path

ROOT = Path("/kaggle/working/Experiment5")
DATA = ROOT / "LLVIP_test_original"
CHECKPOINTS = ROOT / "checkpoints"

paths = {
    "Visible images": DATA / "visible",
    "Thermal images": DATA / "infrared",
    "Annotations": DATA / "Annotations",
    "RGB checkpoint": CHECKPOINTS / "rgb_best.pt",
    "Thermal checkpoint": CHECKPOINTS / "thermal_best.pt",
    "Blend checkpoint": CHECKPOINTS / "blend_best.pt",
    "VT checkpoint": CHECKPOINTS / "vt_best.pt",
}

print("=" * 68)
print("EXPERIMENT 5 — FULL-TEST CLEAN BASELINE PREFLIGHT")
print("=" * 68)

for name, path in paths.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")
    print(f"  {path}")

assert all(p.exists() for p in paths.values()), "Missing required path."

visible = {p.stem for p in paths["Visible images"].glob("*.jpg")}
thermal = {p.stem for p in paths["Thermal images"].glob("*.jpg")}
annotations = {p.stem for p in paths["Annotations"].glob("*.xml")}

print("\nDataset counts:")
print("Visible:", len(visible))
print("Thermal:", len(thermal))
print("Annotations:", len(annotations))

print("\nPairing audit:")
print("Visible-only IDs:", len(visible - thermal))
print("Thermal-only IDs:", len(thermal - visible))
print("Missing annotations:", len((visible | thermal) - annotations))
print("Extra annotations:", len(annotations - (visible | thermal)))

assert len(visible) == 3463
assert len(thermal) == 3463
assert len(annotations) == 3463
assert visible == thermal == annotations

for name in ["RGB", "Thermal", "Blend", "VT"]:
    checkpoint = CHECKPOINTS / f"{name.lower()}_best.pt"
    assert checkpoint.stat().st_size > 1_000_000
    print(f"{name} checkpoint size: {checkpoint.stat().st_size:,} bytes")

print("\nFULL-TEST CLEAN BASELINE PREFLIGHT: PASS")
print("No inference performed. No source files modified.")


EXPERIMENT 5 — FULL-TEST CLEAN BASELINE PREFLIGHT
Visible images: FOUND
  /kaggle/working/Experiment5/LLVIP_test_original/visible
Thermal images: FOUND
  /kaggle/working/Experiment5/LLVIP_test_original/infrared
Annotations: FOUND
  /kaggle/working/Experiment5/LLVIP_test_original/Annotations
RGB checkpoint: FOUND
  /kaggle/working/Experiment5/checkpoints/rgb_best.pt
Thermal checkpoint: FOUND
  /kaggle/working/Experiment5/checkpoints/thermal_best.pt
Blend checkpoint: FOUND
  /kaggle/working/Experiment5/checkpoints/blend_best.pt
VT checkpoint: FOUND
  /kaggle/working/Experiment5/checkpoints/vt_best.pt

Dataset counts:
Visible: 0
Thermal: 0
Annotations: 3463

Pairing audit:
Visible-only IDs: 0
Thermal-only IDs: 0
Missing annotations: 0
Extra annotations: 3463


AssertionError: 

In [58]:
from pathlib import Path

ROOT = Path("/kaggle/working/Experiment5/LLVIP_test_original")

print("=" * 65)
print("LLVIP TEST DIRECTORY DIAGNOSTIC")
print("=" * 65)

for folder in [ROOT, ROOT / "visible", ROOT / "infrared", ROOT / "Annotations"]:
    print(f"\n{folder}")
    print("Exists:", folder.exists())

    if folder.exists():
        print("Subdirectories:", [p.name for p in folder.iterdir() if p.is_dir()])
        print("JPG files:", len(list(folder.rglob("*.jpg"))))
        print("XML files:", len(list(folder.rglob("*.xml"))))

print("\nDiagnostic complete. No files modified.")

LLVIP TEST DIRECTORY DIAGNOSTIC

/kaggle/working/Experiment5/LLVIP_test_original
Exists: True
Subdirectories: ['infrared', 'visible', 'Annotations']
JPG files: 6926
XML files: 3463

/kaggle/working/Experiment5/LLVIP_test_original/visible
Exists: True
Subdirectories: ['test']
JPG files: 3463
XML files: 0

/kaggle/working/Experiment5/LLVIP_test_original/infrared
Exists: True
Subdirectories: ['test']
JPG files: 3463
XML files: 0

/kaggle/working/Experiment5/LLVIP_test_original/Annotations
Exists: True
Subdirectories: []
JPG files: 0
XML files: 3463

Diagnostic complete. No files modified.


In [59]:
from pathlib import Path

ROOT = Path("/kaggle/working/Experiment5")
DATA = ROOT / "LLVIP_test_original"
CHECKPOINTS = ROOT / "checkpoints"

visible_dir = DATA / "visible" / "test"
thermal_dir = DATA / "infrared" / "test"
annotation_dir = DATA / "Annotations"

visible = {p.stem for p in visible_dir.glob("*.jpg")}
thermal = {p.stem for p in thermal_dir.glob("*.jpg")}
annotations = {p.stem for p in annotation_dir.glob("*.xml")}

print("=" * 68)
print("EXPERIMENT 5 — CORRECTED FULL-TEST PREFLIGHT")
print("=" * 68)

print("Visible images:", len(visible))
print("Thermal images:", len(thermal))
print("Annotations:", len(annotations))

print("\nPairing audit:")
print("Visible-only:", len(visible - thermal))
print("Thermal-only:", len(thermal - visible))
print("Missing annotations:", len((visible | thermal) - annotations))
print("Extra annotations:", len(annotations - (visible | thermal)))

assert len(visible) == 3463
assert len(thermal) == 3463
assert len(annotations) == 3463
assert visible == thermal == annotations

for modality in ["rgb", "thermal", "blend", "vt"]:
    path = CHECKPOINTS / f"{modality}_best.pt"
    assert path.exists() and path.stat().st_size > 1_000_000
    print(f"{modality.upper()} checkpoint: PASS ({path.stat().st_size:,} bytes)")

print("\nFULL-TEST PREFLIGHT: PASS")
print("No inference performed.")
print("No source files modified.")

EXPERIMENT 5 — CORRECTED FULL-TEST PREFLIGHT
Visible images: 3463
Thermal images: 3463
Annotations: 3463

Pairing audit:
Visible-only: 0
Thermal-only: 0
Missing annotations: 0
Extra annotations: 0
RGB checkpoint: PASS (5,491,226 bytes)
THERMAL checkpoint: PASS (5,491,226 bytes)
BLEND checkpoint: PASS (5,491,226 bytes)
VT checkpoint: PASS (5,491,226 bytes)

FULL-TEST PREFLIGHT: PASS
No inference performed.
No source files modified.


In [60]:
from pathlib import Path
from ultralytics import YOLO
import torch
import json
import hashlib
import time
import gc

ROOT = Path("/kaggle/working/Experiment5")
DATA = ROOT / "LLVIP_test_original"
OUT = ROOT / "full_test_clean_predictions"
OUT.mkdir(parents=True, exist_ok=True)

MODALITIES = {
    "RGB": {
        "images": DATA / "visible" / "test",
        "checkpoint": ROOT / "checkpoints" / "rgb_best.pt"
    },
    "Thermal": {
        "images": DATA / "infrared" / "test",
        "checkpoint": ROOT / "checkpoints" / "thermal_best.pt"
    },
    "Blend": {
        "images": DATA / "visible" / "test",
        "checkpoint": ROOT / "checkpoints" / "blend_best.pt"
    },
    "VT": {
        "images": DATA / "visible" / "test",
        "checkpoint": ROOT / "checkpoints" / "vt_best.pt"
    }
}

assert torch.cuda.is_available(), "CUDA is unavailable."
print("GPU:", torch.cuda.get_device_name(0))

config = {
    "experiment": "Experiment 5 — Full-test clean baseline",
    "images_per_modality": 3463,
    "model": "YOLO11n",
    "imgsz": 640,
    "confidence": 0.001,
    "nms_iou": 0.60,
    "max_det": 300,
    "agnostic_nms": False,
    "batch": 16,
    "device": 0,
    "half": False,
    "augment": False,
    "training": False
}

summary = {
    "configuration": config,
    "results": {}
}

expected_ids = {
    p.stem for p in (DATA / "visible" / "test").glob("*.jpg")
}
assert len(expected_ids) == 3463

for modality, settings in MODALITIES.items():
    print("\n" + "=" * 65)
    print("RUNNING:", modality)
    print("=" * 65)

    image_paths = sorted(settings["images"].glob("*.jpg"))
    checkpoint = settings["checkpoint"]

    assert len(image_paths) == 3463
    assert checkpoint.exists()

    checkpoint_hash = hashlib.sha256(
        checkpoint.read_bytes()
    ).hexdigest()

    model = YOLO(str(checkpoint))
    output_file = OUT / f"{modality.lower()}_clean.jsonl"

    start = time.perf_counter()
    seen = set()
    total_boxes = 0

    with output_file.open("w", encoding="utf-8") as f:
        results = model.predict(
            source=[str(p) for p in image_paths],
            stream=True,
            imgsz=640,
            conf=0.001,
            iou=0.60,
            max_det=300,
            agnostic_nms=False,
            batch=16,
            device=0,
            half=False,
            augment=False,
            verbose=False
        )

        for result in results:
            image_id = Path(result.path).stem
            assert image_id in expected_ids
            assert image_id not in seen, f"Duplicate ID: {image_id}"
            seen.add(image_id)

            boxes = []
            confidences = []
            class_ids = []

            if result.boxes is not None and len(result.boxes) > 0:
                boxes = result.boxes.xyxyn.cpu().numpy().tolist()
                confidences = result.boxes.conf.cpu().numpy().tolist()
                class_ids = result.boxes.cls.cpu().numpy().astype(int).tolist()

            assert len(boxes) == len(confidences) == len(class_ids)

            for box in boxes:
                assert len(box) == 4
                assert all(0 <= float(v) <= 1 for v in box)

            record = {
                "image_id": image_id,
                "modality": modality,
                "condition": "clean",
                "image_width": int(result.orig_shape[1]),
                "image_height": int(result.orig_shape[0]),
                "boxes_xyxy_normalized": boxes,
                "confidence": confidences,
                "class_ids": class_ids
            }

            f.write(json.dumps(record, separators=(",", ":")) + "\n")
            total_boxes += len(boxes)

    elapsed = time.perf_counter() - start

    assert len(seen) == 3463
    assert seen == expected_ids

    file_hash = hashlib.sha256(output_file.read_bytes()).hexdigest()

    summary["results"][modality] = {
        "images": len(seen),
        "predicted_boxes": total_boxes,
        "runtime_seconds": round(elapsed, 3),
        "checkpoint_sha256": checkpoint_hash,
        "prediction_file": str(output_file),
        "prediction_sha256": file_hash
    }

    print("Images processed:", len(seen))
    print("Predicted boxes:", total_boxes)
    print("Runtime (seconds):", round(elapsed, 2))
    print("Prediction SHA-256:", file_hash)
    print("Status: PASS")

    del model
    gc.collect()
    torch.cuda.empty_cache()

summary_path = OUT / "full_test_clean_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 65)
print("FULL-TEST CLEAN INFERENCE COMPLETE")
print("=" * 65)

for modality, result in summary["results"].items():
    print(
        f'{modality:8s}: {result["images"]} images, '
        f'{result["predicted_boxes"]} boxes, '
        f'{result["runtime_seconds"]:.1f}s'
    )

print("Total image inferences:",
      sum(r["images"] for r in summary["results"].values()))
print("Summary:", summary_path)
print("\nINFERENCE COMPLETE — INTEGRITY CHECKS PASSED")

GPU: Tesla T4

RUNNING: RGB


KeyboardInterrupt: 

In [61]:
from pathlib import Path
import shutil
import os

ROOT = Path("/kaggle/working/Experiment5")
PILOT = ROOT / "pilot_degraded"
DATA = ROOT / "LLVIP_test_original"

print("=" * 65)
print("OVERNIGHT RUN — RESOURCE PREFLIGHT")
print("=" * 65)

usage = shutil.disk_usage("/kaggle/working")

print(f"Disk total: {usage.total / 1024**3:.2f} GB")
print(f"Disk used: {usage.used / 1024**3:.2f} GB")
print(f"Disk free: {usage.free / 1024**3:.2f} GB")

for folder in [
    ROOT / "checkpoints",
    DATA / "visible" / "test",
    DATA / "infrared" / "test",
    PILOT
]:
    print(f"\n{folder}")
    print("Exists:", folder.exists())

    if folder.exists():
        files = list(folder.rglob("*"))
        files = [p for p in files if p.is_file()]
        total = sum(p.stat().st_size for p in files)
        print("Files:", len(files))
        print(f"Size: {total / 1024**3:.3f} GB")

# Estimate fusion PNG size from existing pilot outputs
for name in ["blend", "vt"]:
    folder = PILOT / name / "clean"
    if not folder.exists():
        # Check alternate pilot directory layouts
        matches = list(PILOT.rglob("*.png"))
        matches = [p for p in matches if name.lower() in str(p).lower()
                   and "clean" in str(p).lower()]
    else:
        matches = list(folder.glob("*.png"))

    sample = matches[:20]
    if sample:
        avg = sum(p.stat().st_size for p in sample) / len(sample)
        estimate = avg * 3463 / 1024**3
        print(f"\n{name.upper()} estimated full-test PNG size: {estimate:.2f} GB")
    else:
        print(f"\n{name.upper()}: pilot sample path not identified")

print("\nRESOURCE PREFLIGHT COMPLETE — NO FILES MODIFIED")

OVERNIGHT RUN — RESOURCE PREFLIGHT
Disk total: 19.52 GB
Disk used: 10.38 GB
Disk free: 9.12 GB

/kaggle/working/Experiment5/checkpoints
Exists: True
Files: 4
Size: 0.020 GB

/kaggle/working/Experiment5/LLVIP_test_original/visible/test
Exists: True
Files: 3463
Size: 0.456 GB

/kaggle/working/Experiment5/LLVIP_test_original/infrared/test
Exists: True
Files: 3463
Size: 0.268 GB

/kaggle/working/Experiment5/pilot_degraded
Exists: True
Files: 4002
Size: 5.853 GB

BLEND estimated full-test PNG size: 5.19 GB

VT estimated full-test PNG size: 5.69 GB

RESOURCE PREFLIGHT COMPLETE — NO FILES MODIFIED


In [62]:
from pathlib import Path

ROOT = Path("/kaggle/working/Experiment5")
PILOT = ROOT / "pilot_degraded"

print("PILOT DIRECTORY STRUCTURE")

for p in sorted(PILOT.iterdir()):
    if p.is_dir():
        files = list(p.rglob("*.png"))
        print(f"\n{p.name}: {len(files)} PNG files")
        for f in files[:3]:
            print(" ", f.relative_to(PILOT))

print("\nPILOT STRUCTURE CHECK COMPLETE")

PILOT DIRECTORY STRUCTURE

Blend: 1000 PNG files
  Blend/blur_5/190027.png
  Blend/blur_5/190020.png
  Blend/blur_5/190028.png

RGB: 1000 PNG files
  RGB/blur_5/190027.png
  RGB/blur_5/190020.png
  RGB/blur_5/190028.png

Thermal: 1000 PNG files
  Thermal/blur_5/190027.png
  Thermal/blur_5/190020.png
  Thermal/blur_5/190028.png

VT: 1000 PNG files
  VT/blur_5/190027.png
  VT/blur_5/190020.png
  VT/blur_5/190028.png

PILOT STRUCTURE CHECK COMPLETE


In [63]:
import os
import zipfile
from getpass import getpass
from huggingface_hub import snapshot_download

REPO_ID = "SakibM19/border-surveillance-research-backups"
REPO_TYPE = "dataset"

REMOTE_FOLDER = "Experiment5/Pilot100"
LOCAL_FOLDER = "/kaggle/working/Experiment5_Pilot100_Results"
ZIP_PATH = "/kaggle/working/Experiment5_Pilot100_Results.zip"

# Get Hugging Face access token securely
token = os.environ.get("HF_TOKEN")

if not token:
    token = getpass("Enter your Hugging Face READ token: ")

# Download confirmed pilot result artifacts
patterns = [
    f"{REMOTE_FOLDER}/pilot_metrics/bootstrap_inputs.jsonl",
    f"{REMOTE_FOLDER}/pilot_metrics/bootstrap/*.csv",
    f"{REMOTE_FOLDER}/pilot_metrics/bootstrap/*.json",
]

print("Downloading Experiment 5 pilot results...")

snapshot_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    token=token,
    allow_patterns=patterns,
    local_dir=LOCAL_FOLDER
)

# Create a ZIP archive
print("\nCreating ZIP archive...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for root, dirs, files in os.walk(LOCAL_FOLDER):
        for file in files:
            full_path = os.path.join(root, file)
            relative_path = os.path.relpath(full_path, LOCAL_FOLDER)
            archive.write(full_path, relative_path)

print("\nDOWNLOAD COMPLETE")
print("Local folder:", LOCAL_FOLDER)
print("ZIP archive:", ZIP_PATH)
print("ZIP size:", round(os.path.getsize(ZIP_PATH) / (1024 * 1024), 2), "MB")

print("\nDownloaded files:")
for root, dirs, files in os.walk(LOCAL_FOLDER):
    for file in files:
        print(os.path.relpath(os.path.join(root, file), LOCAL_FOLDER))

Enter your Hugging Face READ token:  ········


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]


Creating ZIP archive...

DOWNLOAD COMPLETE
Local folder: /kaggle/working/Experiment5_Pilot100_Results
ZIP archive: /kaggle/working/Experiment5_Pilot100_Results.zip
ZIP size: 4.37 MB

Downloaded files:
.cache/huggingface/.gitignore
.cache/huggingface/CACHEDIR.TAG
.cache/huggingface/trees/6c1f37ace68f8221c28f9dbb46e71b9be2d49769.json
.cache/huggingface/download/Experiment5/Pilot100/pilot_metrics/bootstrap_inputs.jsonl.metadata
.cache/huggingface/download/Experiment5/Pilot100/pilot_metrics/bootstrap_inputs.jsonl.lock
.cache/huggingface/download/Experiment5/Pilot100/pilot_metrics/bootstrap/bootstrap_metric_confidence_intervals.csv.metadata
.cache/huggingface/download/Experiment5/Pilot100/pilot_metrics/bootstrap/bootstrap_metric_confidence_intervals.csv.lock
.cache/huggingface/download/Experiment5/Pilot100/pilot_metrics/bootstrap/bootstrap_methodology.json.metadata
.cache/huggingface/download/Experiment5/Pilot100/pilot_metrics/bootstrap/bootstrap_paired_differences.json.metadata
.cache/hug

In [64]:
import os
import zipfile

BASE = "/kaggle/working/Experiment5_Pilot100_Results"
SOURCE = os.path.join(BASE, "Experiment5", "Pilot100", "pilot_metrics")

ZIP_PATH = "/kaggle/working/Experiment5_Pilot100_Results_CLEAN.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(SOURCE):
        for file in files:
            if file.endswith((".csv", ".json", ".jsonl")):
                full_path = os.path.join(root, file)
                relative_path = os.path.relpath(full_path, SOURCE)
                z.write(full_path, relative_path)

print("CLEAN ZIP CREATED")
print("Path:", ZIP_PATH)
print("Size:", round(os.path.getsize(ZIP_PATH) / (1024 * 1024), 2), "MB")

print("\nZIP CONTENTS:")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for name in z.namelist():
        print(name)

CLEAN ZIP CREATED
Path: /kaggle/working/Experiment5_Pilot100_Results_CLEAN.zip
Size: 3.88 MB

ZIP CONTENTS:
bootstrap_inputs.jsonl
bootstrap/bootstrap_metric_confidence_intervals.csv
bootstrap/bootstrap_metric_confidence_intervals.json
bootstrap/bootstrap_methodology.json
bootstrap/bootstrap_paired_differences.json
bootstrap/bootstrap_paired_differences.csv


# Consolidation manifest

| Original notebook | Original cells | Source-code cells included |
|---|---:|---:|
| `Framework_SeniorProject.ipynb` | 98 | 98 |
| `exp-2(1).ipynb` | 40 | 40 |
| `framework_2(1).ipynb` | 96 | 94 |
| `1_Weapon_Detection.ipynb` | 16 | 16 |
| `framework_exp_2.ipynb` | 36 | 35 |
| `experiment5-rgb-thermal-fusion-robustness.ipynb` | 61 | 61 |

**Total source-code cells included:** 344

**Credential-pattern substitutions applied in the original consolidated sources:** 2

Experiment 5 cells were appended without changing their source, recorded outputs, or execution counts. The Experiment 5 source notebook was prepared for Kaggle; environment-specific paths and secret access may need adjustment before execution.

This is a source-code collection, not a claim that every cell is necessary, safe to execute, or independently reproducible. Review cells before execution and preserve the original experiment outputs as evidence.